In [1]:
# =====================================================================
# MC AI BRAIN 5 - CELL 0: DATA - PINNED, CHECKED, GIVEN ROLES
# Runs first. Each file is kept only with the SHA-256 written here: used
# if already in mc_ai_brain5/data, copied if attached (any folder under
# /kaggle/input), else downloaded at a fixed commit (internet on).
# Writes data/sources.json (address, commit, SHA-256, bytes of each file)
# and data/roles.json. Copies the attached Brain 4 and Brain 3 ledgers,
# each only if its hash chain verifies, into mc_ai_brain5/earlier/, so
# Brain 5's ledger keeps their looks after the inputs are removed. Makes
# /kaggle/working/mcai for the library cells, on the import path.
# SOURCES  Brain 4's seventeen files, unchanged (see Brain 4 Cell 0), and
#   from Hugging Face at fixed commits (fetch check, 2026-11-16):
#   arc      ARC Easy and Challenge (Clark et al. 2018), allenai/ai2_arc
#            210d026f: train, validation, test. CC BY-SA 4.0
#   obqa     OpenBookQA (Mihaylov et al. 2018), allenai/openbookqa
#            388097ea: main, additional (with each question's fact).
#            Licence not stated: used here, never re-hosted
#   mmlu     MMLU (Hendrycks et al. 2021), cais/mmlu c30699e8, all 57
#            subjects: dev, validation, test. MIT
#   mmlupro  MMLU-Pro (Wang et al. 2024), TIGER-Lab/MMLU-Pro b189ec76:
#            validation, test. MIT
# ROLES
#   fit        counted into the model: SemCor and the train splits
#   practice   scored as often as needed: SemEval-2007, EWT dev, MMLU
#              dev, the validation splits
#   test       each model scored on it is a look, counted: TEST4 (SE2,
#              SE3, SE13, SE15), the test splits, the word-pair tests
#   knowledge  read, never scored: WordNet, gloss corpus, thesauri,
#              Simple Wikipedia
#   sealed     42D (Maru et al. 2022): pinned in roles.json, not fetched;
#              one final run, after the freeze
# PREDICTIONS (written before running)
#   P1 every file has the SHA-256 written here
#   P2 under 5 minutes
# =====================================================================
import hashlib, json, os, shutil, sys, tarfile, time, urllib.error, urllib.request, zipfile
from pathlib import Path

print("=" * 64); print("MC AI BRAIN 5 - CELL 0: DATA - PINNED, CHECKED, GIVEN ROLES"); print("=" * 64)
T0 = time.time()
HOME = Path(os.environ.get("MC_AI_HOME", "/kaggle/working"))
WORK = HOME / "mc_ai_brain5"
DATA, EARLIER = WORK / "data", WORK / "earlier"
BASE = os.environ.get("MC_AI_FETCH_BASE", "")
INPUT = Path(os.environ.get("MC_AI_INPUT", "/kaggle/input"))
AGENT = "MC-AI-Brain5/1.0 (Kaggle notebook; pinned SHA-256 downloads)"
GITHUB = "https://raw.githubusercontent.com/{repo}/{commit}/{file}"
HF = "https://huggingface.co/datasets/{repo}/resolve/{commit}/{file}"
(HOME / "mcai").mkdir(parents=True, exist_ok=True)
(HOME / "mcai" / "__init__.py").touch()
if str(HOME) not in sys.path:
    sys.path.insert(0, str(HOME))
P = "-00000-of-00001.parquet"
SOURCES = {  # source: (repo, commit, {file: SHA-256 or (SHA-256, bytes)}, address template(s))
    "ewt": ("UniversalDependencies/UD_English-EWT", "4a4d77f599ea53cc405f85d0cec4b2f14f81d42b", {
        "en_ewt-ud-train.conllu": "d68e06122a702464c613076523d56740f047e5bbe89dd90ec32737e04d952143",
        "en_ewt-ud-dev.conllu": "39239e0a60db3ae68f4b7036189f11b6692741d10ff8240dd91f74f2760d90f8",
        "en_ewt-ud-test.conllu": "fa024f43dc5da3c5ac02563bc9bd0e974f46cbb1560823976a8f342a37dc494a"}),
    "wsd": ("akranlu/wsd-data", "697cb66cf7ba874c276c863f2e9e2ea32b6c52d9", {
        "data/SemCor/semcor.data.xml": "45003c51336fcb15b9d8a6556be65eaadf5bf3285ddbae763675000e28b00753",
        "data/SemCor/semcor.gold.key.txt": "d5bb7966417fc1861c43a3851e03aa7b762b9004d9ffe7636637420d779259af",
        "data/ALL/ALL.data.xml": "27a9da01b6d4a1e4dc7c76d15148a38eaa56cc0df371cae4416d04e541b6fe36",
        "data/ALL/ALL.gold.key.txt": "0bf2ffc38f572587c72b59ea928da676423b99b5a54bd1a83eb4848b517d3170"}),
    "wordnet": ("nltk/nltk_data", "550b6625bcef1f2abff2ff770a5a0d272c9c6b2a", {
        "packages/corpora/wordnet.zip": "cbda5ea6eef7f36a97a43d4a75f85e07fccbb4f23657d27b4ccbc93e2646ab59"}),
    "gsm": ("openai/grade-school-math", "3101c7d5072418e28b9008a6636bde82a006892c", {
        "grade_school_math/data/train.jsonl": "17f347dc51477c50d4efb83959dbb7c56297aba886e5544ee2aaed3024813465",
        "grade_school_math/data/test.jsonl": "3730d312f6e3440559ace48831e51066acaca737f6eabec99bccb9e4b3c39d14"}),
    "encyclopedia": ("wikimedia/wikipedia", "b04c8d1ceb2f5cd4588862100d08de323dccfbaa", {
        "20231101.simple/train" + P: "31bded16768a47c286becd292079122f5d7d4397a17b87d4250a00ccd581e6f0"}, HF),
    "wordsim": ("mfaruqui/eval-word-vectors", "94b3dd9fdbac7ed08dbee7179cd5ee464acf0594", {
        "data/word-sim/EN-MEN-TR-3k.txt": "888b13580faa362c7b11da0a670258cf1b1eb4bf0ecbcd418918729a3bd3f43a",
        "data/word-sim/EN-SimVerb-3500.txt": "9d84274e7ad029fd3cf69dd0516e9ade624bbef9bd4e2050f6cc1fa01471e0f5",
        "data/word-sim/EN-RW-STANFORD.txt": "5108d54dfc746f44f9c3d7e31d2857c37cf71430bc1594175a61a3b89bb0ae1f"}),
    "simlex": ("fh295.github.io", "-", {
        "SimLex-999.zip": "30f40eb18f41b08b65dfbc328f473e8b69ba444b1c9d053c1adde2324fc76315"},
        "https://fh295.github.io/{file}"),
    "thesaurus": ("www.gutenberg.org", "-", {
        "3/2/0/3202/files.zip": "3a54e61dd3b4153313e00c3113bda1e939af505d1336a350a0fd93a80538a6d8"},
        ["https://gutenberg.pglaf.org/{file}", "https://mirror.cs.odu.edu/gutenberg/{file}",
         "https://www.gutenberg.org/dirs/{file}"]),
    "glosstag": ("wordnetcode.princeton.edu", "-", {
        "glosstag-files/WordNet-3.0-glosstag.tar.bz2":
            "c7b8318ac873ecbea14b225f33465210ec20ce19d9c6f7fc82bbbbe1e6b3080a"},
        "https://wordnetcode.princeton.edu/{file}"),
    "arc": ("allenai/ai2_arc", "210d026faf9955653af8916fad021475a3f00453", {
    "ARC-Challenge/train" + P: ("e488c1587ffdcfc8443f916c53488a95cd471c5790e0746c6bfe4cecf20962cb", 189909),
    "ARC-Challenge/validation" + P: ("395a5c88d1580d69855fbaee9450270578df1ad5af6259771cd0a42c20e99f05", 55743),
    "ARC-Challenge/test" + P: ("62f03257e737aed263f55c6abf87c7bb0028a44a6bdd2a26eb1279eb42c1d1e9", 203808),
    "ARC-Easy/train" + P: ("b315db8a4be597dc7daa50a4e70d48dd7c990c32085629e6ccd8c926beaa80b5", 330598),
    "ARC-Easy/validation" + P: ("ed890ff1e4cef7a7140d3a30dcea3ed2c9d467c6458f447ad9ef0176d8dcbb74", 86080),
    "ARC-Easy/test" + P: ("4160597d618ae851c7eb04e281574f3f654776216ac6b6641588d64527b47177", 346257)}, HF),
    "obqa": ("allenai/openbookqa", "388097ea7776314e93a529163e0fea805b8a6454", {
    "main/train" + P: ("98148f8a54e62eb862346a75192d5fb824d6cbb68f2f59aecd793d39ecb5cd8b", 495845),
    "main/validation" + P: ("35370b9cfee8c1ff325ccc74adc434d12c47ca0ac3244aa87f3fa77069285206", 58233),
    "main/test" + P: ("cd5483e366daa230c1c87bbdc512d8b7229f14f6dd04d19fc8b1a3855aaaa8a3", 55535),
    "additional/train" + P: ("d16d719e87efb86ed0a2ac4c8cdf380f7bfb94b602088393674c0a64ce9ed3d3", 635446),
    "additional/validation" + P: ("92e5e68e4da7bec7d130d925385abf377c2d82b89a16de502b4e1b9cf3f50a26", 75882),
    "additional/test" + P: ("33b318ea8e2354484868bc601c1b30a58149e9deb93162ff422bb8de980c7105", 72461)}, HF),
    "mmlu": ("cais/mmlu", "c30699e8356da336a370243923dbaf21066bb9fe", {
    "all/dev" + P: ("2b19bde1ed8ca6b482fb283abc90e8e0d9d228947029c0b91795d64b28b3bc3f", 76504),
    "all/validation" + P: ("66cdf0b090ccb657d18d13cd81e31fdc55c3467da9642ffb178653268a97c8ef", 408449),
    "all/test" + P: ("74a41822ce7d3def56e1682f958469c04642a5336a5ce912fa375fdb90fb25d7", 3504718)}, HF),
    "mmlupro": ("TIGER-Lab/MMLU-Pro", "b189ec765aa7ed75c8acfea42df31fdae71f97be", {
    "data/validation" + P: ("139423c23722e480c807ac4a191409a710cfce4eba744c1d641cf88e730e2078", 42857),
    "data/test" + P: ("0e24a191921c2f453518a537a8b2117bd137e7714d4ef1565e9ba06c1ecb9ad8", 4144185)}, HF),
}
KEEP = {"arc", "obqa", "mmlu", "mmlupro"}  # their files keep their folder: one name is in several folders
SEALED = {"source": "https://raw.githubusercontent.com/SapienzaNLP/wsd-hard-benchmark/"
                    "74ff055519b3c426330df9974a6603a0e638a3db/wsd_hard_benchmark/42D/",
          "42D.data.xml": ["45ae38a0f3758c03ca4a568e5d031ac0baa09784b70cb257bbd64e3a1be5cf1a", 334360],
          "42D.gold.key.txt": ["d6b1a81dec6f6418238a880727f033cca8a3317142a25012183483f731d53c78", 14741]}
ARCHIVES = {"wordnet.zip": ("wordnet/", "wordnet/"), "SimLex-999.zip": ("SimLex-999/", "simlex/"),
            "files.zip": ("files/", "thesaurus/"),
            "WordNet-3.0-glosstag.tar.bz2": ("WordNet-3.0/glosstag/", "glosstag/", ("standoff/", "dtd/"))}
# archive: (folder inside it, folder in DATA, subfolders left out)
SETS = {"semeval2007": "practice", "senseval2": "test", "senseval3": "test", "semeval2013": "test",
        "semeval2015": "test"}  # the five sets in ALL, by the start of their sentence ids


def role_of(source: str, rel: str):
    """A file's role, from its source and name; ALL holds five sets, so it gets a role for each."""
    name = rel.rsplit("/", 1)[-1]
    if source in ("wordnet", "encyclopedia", "thesaurus", "glosstag"):
        return "knowledge"
    if source in ("wordsim", "simlex"):
        return "test"
    if source == "wsd":
        return "fit" if "/SemCor/" in rel else dict(SETS)
    for split, role in (("train", "fit"), ("dev", "practice"), ("validation", "practice"), ("test", "test")):
        if split in name:
            return role
    raise ValueError(f"no role written for {source} {rel}")


def sha256(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()


def size(n: float) -> str:
    for unit in ("B", "KB", "MB"):
        if n < 1024:
            return f"{n:.0f} {unit}"
        n /= 1024
    return f"{n:.1f} GB"


def origin(p: Path) -> str:
    """The input a path is in (Kaggle mounts inputs at <kind>/<owner>/<name>)."""
    parts = p.relative_to(INPUT).parts
    return parts[2] if parts[0] in ("notebooks", "datasets", "models") and len(parts) > 3 else parts[0]


def address(repo: str, commit: str, rel: str, template: str) -> str:
    """The file's address at that commit (MC_AI_FETCH_BASE, if set, replaces the host: for tests)."""
    return f"{BASE}{repo}/{commit}/{rel}" if BASE else template.format(repo=repo, commit=commit, file=rel)


def fetch(source: str, repo: str, commit: str, rel: str, want: str, nbytes, templates: list):
    """(path, how, address) of the file at repo/commit/rel, kept only with the SHA-256 wanted: already here,
    or copied from an input, or downloaded from the first address that gives those bytes."""
    name = rel.rsplit("/", 1)[-1]
    sub = rel.rsplit("/", 2)[-2] if source in KEEP else ""
    folder = DATA / ("_archives" if name in ARCHIVES else source) / sub
    folder.mkdir(parents=True, exist_ok=True)
    target = folder / name
    first = address(repo, commit, rel, templates[0])
    if target.exists() and sha256(target) == want:
        return target, "already here", first
    for p in sorted(INPUT.rglob(name)) if INPUT.exists() else []:
        if p.is_file() and (nbytes is None or p.stat().st_size == nbytes) and sha256(p) == want:
            shutil.copyfile(p, target)
            return target, f"copied from {origin(p)}", str(p)
    tried, wrong = [], []
    for template in templates:
        url = address(repo, commit, rel, template)
        part = target.with_name(name + ".part")
        try:
            req = urllib.request.Request(url, headers={"User-Agent": AGENT})
            with urllib.request.urlopen(req, timeout=120) as r, open(part, "wb") as f:
                shutil.copyfileobj(r, f, 1 << 20)
        except (urllib.error.URLError, OSError) as err:
            part.unlink(missing_ok=True)
            tried.append(f"{url}: {err}")
            continue
        got = sha256(part)
        if got != want:
            part.unlink()
            wrong.append(f"{url} gave SHA-256 {got[:16]}...")
            continue
        part.replace(target)
        return target, "downloaded" + (f" from {url.split('/')[2]}" if len(templates) > 1 else ""), url
    if wrong:
        return target, f"REFUSED: {'; '.join(wrong)}, not {want[:16]}...", first
    raise RuntimeError(f"could not download {name}: {'; '.join(tried)}. Is the internet on (Settings -> "
                       f"Internet)? Or attach the file (Add Input): it is found by its SHA-256")


def wanted(n: str, inside: str, skip: tuple) -> bool:
    if not n.startswith(inside) or n.endswith("/") or any(n[len(inside):].startswith(s) for s in skip):
        return False
    if n.startswith("/") or ".." in Path(n).parts:
        raise ValueError(f"archive entry outside its folder: {n}")
    return True


def unpack(archive: Path, inside: str, dest: str, skip: tuple = ()) -> int:
    """Unpack the entries under `inside` (not under `skip`) into DATA/dest; returns how many files. An entry
    with an absolute path or '..' is refused."""
    count = 0
    if zipfile.is_zipfile(archive):
        with zipfile.ZipFile(archive) as z:
            for info in z.infolist():
                if wanted(info.filename, inside, skip):
                    out, count = DATA / dest / info.filename[len(inside):], count + 1
                    if not (out.exists() and out.stat().st_size == info.file_size):
                        out.parent.mkdir(parents=True, exist_ok=True)
                        with z.open(info) as src, open(out, "wb") as dst:
                            shutil.copyfileobj(src, dst, 1 << 20)
        return count
    with tarfile.open(archive, "r:*") as t:
        for m in t:
            if m.isfile() and wanted(m.name, inside, skip):
                out, count = DATA / dest / m.name[len(inside):], count + 1
                if not (out.exists() and out.stat().st_size == m.size):
                    out.parent.mkdir(parents=True, exist_ok=True)
                    with t.extractfile(m) as src, open(out, "wb") as dst:
                        shutil.copyfileobj(src, dst, 1 << 20)
    return count


def canonical(obj) -> str:
    return json.dumps(obj, sort_keys=True, separators=(",", ":"), default=str)


def chain(path: Path):
    """The ledger's entries if every entry's number, link and SHA-256 are right (Brain 4's test), else None."""
    try:
        entries = [json.loads(line) for line in Path(path).read_text().splitlines() if line.strip()]
    except (OSError, ValueError):
        return None
    prev = ""
    for i, e in enumerate(entries):
        body = {k: v for k, v in e.items() if k != "sha"}
        if e.get("n") != i or e.get("prev") != prev or e.get("sha") != hashlib.sha256(
                canonical(body).encode()).hexdigest():
            return None
        prev = e["sha"]
    return entries


# ---------------- files: kept, copied or downloaded, and checked ----------------
RECORD, REFUSED, ROLES, FROM = [], [], {}, {}
print(f"\n  into {DATA}")
for source, (repo, commit, files, *rest) in SOURCES.items():
    templates = [rest[0]] if rest and isinstance(rest[0], str) else list(rest[0]) if rest else [GITHUB]
    print(f"\n  {source}: {repo} " + (f"at {commit[:8]}" if commit != "-" else "(no commit: pinned by SHA-256)"))
    for rel, pin in files.items():
        want, nbytes = (pin, None) if isinstance(pin, str) else pin
        t1 = time.time()
        path, how, url = fetch(source, repo, commit, rel, want, nbytes, templates)
        ok = not how.startswith("REFUSED")
        if not ok:
            REFUSED.append(rel)
        shown = ("/".join(rel.split("/")[-2:]) if source in KEEP else rel.rsplit("/", 1)[-1]).replace(P, ".parquet")
        print(f"    {shown:<36} {size(path.stat().st_size) if ok else '':>7}  {how} ({time.time() - t1:.1f}s)")
        if ok:
            kept = str(path.relative_to(DATA))
            RECORD.append(dict(source=source, url=url, repo=repo, commit=commit, file=kept, sha256=want,
                               bytes=path.stat().st_size))
            ROLES[kept], FROM[kept] = role_of(source, rel), source
            if path.name in ARCHIVES:
                n = unpack(path, *ARCHIVES[path.name])
                print(f"    {'':<36} {'':>7}  unpacked: {n} files in {DATA / ARCHIVES[path.name][1].rstrip('/')}")
(DATA / "sources.json").write_text(json.dumps(RECORD, indent=1))
(DATA / "roles.json").write_text(json.dumps(dict(roles=ROLES, sealed=SEALED, TEST4=[k for k, v in SETS.items()
                                                                                    if v == "test"]), indent=1))

# ---------------- roles ----------------
print("\n  ROLES (data/roles.json)")
for role in ("fit", "practice", "test", "knowledge"):
    names = [k for k, v in ROLES.items() if v == role or (isinstance(v, dict) and role in v.values())]
    print(f"    {role:<9} {len(names):>2} files: " + ", ".join(sorted({FROM[n] for n in names})))
print("    sealed     2 files: 42D.data.xml, 42D.gold.key.txt (pinned, not fetched)")
print("    ALL (wsd) by set: " + ", ".join(f"{k} {v}" for k, v in SETS.items()))

# ---------------- earlier ledgers ----------------
EARLIER.mkdir(parents=True, exist_ok=True)
print(f"\n  EARLIER LEDGERS (into {EARLIER})")
MISSING = []
for brain in ("mc_ai_brain4", "mc_ai_brain3"):
    target = EARLIER / f"{brain}_ledger.jsonl"
    here = chain(target) if target.exists() else None
    attached = [(p, chain(p)) for p in sorted(INPUT.rglob(f"{brain}/ledger.jsonl"))] if INPUT.exists() else []
    good = [(p, e) for p, e in attached if e]
    how = "kept from an earlier run" if here else ""
    if good:
        p, e = max(good, key=lambda pe: len(pe[1]))
        if here is None or len(e) > len(here):
            shutil.copyfile(p, target)
            here, how = e, f"copied from {origin(p)}"
    bad = len(attached) - len(good)
    if here is None:
        MISSING.append(brain)
        print(f"    {brain}: none here and none attached that verifies" + (f" ({bad} did not)" if bad else ""))
        continue
    looks = {}
    for e in here:
        if e.get("kind") == "look":
            looks.setdefault(e["split"], set()).add(e["model"])
    print(f"    {brain}: {len(here)} entries, chain verifies; {how}" + (f"; {bad} attached did not verify" if bad
                                                                       else ""))
    print(f"      head {here[-1]['sha']}")
    print("      looks: " + ", ".join(f"{k} {len(v)}" for k, v in sorted(looks.items())))

total = time.time() - T0
print(f"\n  {len(RECORD)} files kept, {size(sum(r['bytes'] for r in RECORD))}; refused: "
      f"{', '.join(REFUSED) if REFUSED else 'none'}")
print("\n  PREDICTIONS vs RESULTS")
print(f"    P1 every file has its SHA-256: {len(RECORD)} of {len(RECORD) + len(REFUSED)} -> "
      f"{'HELD' if not REFUSED else 'DID NOT HOLD'}")
print(f"    P2 under 5 minutes: {total:.0f}s -> {'HELD' if total <= 300 else 'DID NOT HOLD'}")
if REFUSED:
    raise RuntimeError(f"files with other bytes than pinned were not kept: {', '.join(REFUSED)}")
if "mc_ai_brain4" in MISSING:
    raise RuntimeError("Brain 4's ledger is not here: attach the MC AI Brain 4 notebook (Add Input) and run "
                       "this cell again; without it TEST4's looks would be counted from zero")
print(f"\n  ({total:.0f}s)")
print("  LAST LINE OF MC AI BRAIN 5 CELL 0")


MC AI BRAIN 5 - CELL 0: DATA - PINNED, CHECKED, GIVEN ROLES

  into /kaggle/working/mc_ai_brain5/data

  ewt: UniversalDependencies/UD_English-EWT at 4a4d77f5
    en_ewt-ud-train.conllu                 14 MB  copied from mc-ai-brain-4 (1.3s)
    en_ewt-ud-dev.conllu                    2 MB  copied from mc-ai-brain-4 (0.5s)
    en_ewt-ud-test.conllu                   2 MB  copied from mc-ai-brain-4 (0.5s)

  wsd: akranlu/wsd-data at 697cb66c
    semcor.data.xml                        37 MB  copied from mc-ai-brain-4 (0.6s)
    semcor.gold.key.txt                     7 MB  copied from mc-ai-brain-4 (0.5s)
    ALL.data.xml                            1 MB  copied from mc-ai-brain-4 (0.4s)
    ALL.gold.key.txt                      317 KB  copied from mc-ai-brain-4 (0.4s)

  wordnet: nltk/nltk_data at 550b6625
    wordnet.zip                            10 MB  copied from mc-ai-brain-4 (0.5s)
                                                  unpacked: 18 files in /kaggle/working/mc_ai_brain5/

In [2]:
# =====================================================================
# MC AI BRAIN 5 - CELL 1: LIBRARY AND LEDGER - BRAIN 4'S CODE, CHECKED;
# THE LEDGER, CONTINUED
# Needs Cell 0. No internet, nothing trained.
# LIBRARY  Copies Brain 4's seven modules (mcai core, stats, read, book,
#   words, count, sense) from the attached MC AI Brain 4 notebook into
#   /kaggle/working/mcai, each only if its text is the code written for
#   Brain 4: the SHA-256 of the text with line ends trimmed, written here.
#   A file already there with that text is used as it is. Then imports
#   all seven and prints the versions of Python, NumPy and SciPy.
# LEDGER   Opens Brain 5's ledger, /kaggle/working/mc_ai_brain5/
#   ledger.jsonl. A new one starts as Brain 4's 348 entries (copied by
#   Cell 0 into earlier/). Each look in Brain 3's ledger (also in
#   earlier/) is added as a look by the model brain3:<its model>, on the
#   split Brain 4 names for the same data, so TEST4's count covers both
#   brains. Brain 5's keys start with B5 (Brain 4 used the plain ones).
# PREDICTIONS (written before running)
#   B5-K1 all seven modules match the code written for Brain 4
#   B5-L1 the ledger's first 348 entries are Brain 4's, unchanged
#   B5-L2 TEST4 (WSD) has 51 looks: Brain 4's 50 and Brain 3's 1
#   B5-T1 under 30 seconds
# =====================================================================
import hashlib, importlib, platform, shutil, sys, time
from pathlib import Path

import numpy, scipy

print("=" * 64); print("MC AI BRAIN 5 - CELL 1: LIBRARY AND LEDGER"); print("=" * 64)
T0 = time.time()
if "EARLIER" not in globals() or "chain" not in globals():
    raise RuntimeError("run Cell 0 first")
LIB = HOME / "mcai"
MODULES = {  # module: SHA-256 of its text with line ends trimmed, as written for Brain 4 (Cells 2-10)
    "core.py": "29b3fae477711f8f85eba914dd802f4353a1274b75f1c54430d6f7df60fc7b57",
    "stats.py": "fcefe440002348e2f24978eefedb9e2e5f7d693d8e91211291488afa241f378b",
    "read.py": "cf2c00f21cd4f7e1d5e4ca990bf04eb5a6ff79af98c9cf365271517a1d43c990",
    "book.py": "d5aa08db951fbe70495e4df52fa2191ebc3e810a07e4cbdb399cc6e80a2cacd4",
    "words.py": "5690e7300e8de75de778d7d84083e38963f94a741ea46175a24b6ce1f0404cfc",
    "count.py": "02ab5624f1a30dc760b24e0c5f3de88d1e6d3701ebeae3210fa17d386e20219c",
    "sense.py": "d0cbf80548d650eff5ef4b3313d2611efc3a9f6c0dac965ad7d196b58f406257",
}
SPLITS = {"WSD TEST4 (SE2, SE3, SE13, SE15)": "TEST4 (WSD)", "EWT test (XPOS tags)": "EWT test (XPOS)"}
# Brain 3's name for a split -> Brain 4's name for the same data; any other name is kept
B4, B3 = EARLIER / "mc_ai_brain4_ledger.jsonl", EARLIER / "mc_ai_brain3_ledger.jsonl"


def text_sha(path) -> str:
    """SHA-256 of a file's text with each line's trailing spaces and the file's final newlines trimmed: the same
    for the code as written and as pasted into a cell."""
    t = Path(path).read_text(encoding="utf-8", errors="replace").replace("\r\n", "\n")
    return hashlib.sha256(("\n".join(line.rstrip() for line in t.split("\n")).rstrip() + "\n").encode()).hexdigest()


# ---------------- library ----------------
print(f"\n  LIBRARY into {LIB}")
LIB.mkdir(parents=True, exist_ok=True)
(LIB / "__init__.py").touch()
MATCH, PROBLEMS = 0, []
for name, want in MODULES.items():
    target, how = LIB / name, ""
    if target.exists() and text_sha(target) == want:
        how = "already here"
    else:
        seen = sorted(INPUT.rglob(f"mcai/{name}")) if INPUT.exists() else []
        for p in seen:
            if text_sha(p) == want:
                shutil.copyfile(p, target)
                how = f"copied from {origin(p)}"
                break
        else:
            how = (f"{len(seen)} attached, none with the text written for Brain 4" if seen else "not attached")
            PROBLEMS.append(name)
    MATCH += name not in PROBLEMS
    print(f"    {name:<10} {target.stat().st_size if target.exists() else 0:>7,} bytes  {how}")
if PROBLEMS:
    raise RuntimeError(f"not the code written for Brain 4: {', '.join(PROBLEMS)}. Send this output: those modules "
                       f"will be given as cells to paste")
if str(HOME) not in sys.path:
    sys.path.insert(0, str(HOME))
importlib.invalidate_caches()
for name in MODULES:
    importlib.reload(importlib.import_module(f"mcai.{name[:-3]}"))
from mcai.core import Ledger, at_most, between

print(f"    imported all {len(MODULES)}; Python {platform.python_version()}, NumPy {numpy.__version__}, "
      f"SciPy {scipy.__version__}")

# ---------------- ledger ----------------
B4_ENTRIES = chain(B4) if B4.exists() else None
if not B4_ENTRIES:
    raise RuntimeError("Brain 4's ledger is not in earlier/: attach MC AI Brain 4 (Add Input) and run Cell 0")
LEDGER = Ledger(WORK / "ledger.jsonl", earlier=[B4])
LEDGER.predict("B5-K1", "Brain 4 modules whose text is the code written for Brain 4", between(7, 7))
LEDGER.predict("B5-L1", "Brain 4's entries at the start of this ledger, unchanged", between(348, 348))
LEDGER.predict("B5-L2", "models scored on TEST4 (WSD), Brain 3's included", between(51, 51))
LEDGER.predict("B5-T1", "Cell 1 seconds", at_most(30))
B3_ENTRIES = chain(B3) if B3.exists() else None
added = 0
for e in B3_ENTRIES or []:
    if e.get("kind") == "look":
        split = SPLITS.get(e["split"], e["split"])
        before = LEDGER.looks(split)
        added += LEDGER.look(split, f"brain3:{e['model']}", note=f"carried from Brain 3's ledger, entry {e['n']} "
                             f"(head {B3_ENTRIES[-1]['sha'][:16]}); Brain 3 named the split {e['split']}") > before
same = 0
for a, b in zip(LEDGER.entries, B4_ENTRIES):
    if a != b:
        break
    same += 1
print("\n  PREDICTIONS vs RESULTS")
LEDGER.result("B5-K1", MATCH)
LEDGER.result("B5-L1", same)
LEDGER.result("B5-L2", LEDGER.looks("TEST4 (WSD)"))
LEDGER.result("B5-T1", time.time() - T0)
print(f"\n  LEDGER {LEDGER.path}")
print(f"    {len(LEDGER.entries)} entries; starts with {same} of Brain 4's {len(B4_ENTRIES)} unchanged; "
      f"Brain 3's ledger: {'not in earlier/' if B3_ENTRIES is None else f'{len(B3_ENTRIES)} entries'}; "
      f"looks carried from it this run: {added}")
splits = sorted({e["split"] for e in LEDGER.entries if e["kind"] == "look"})
print("    looks: " + ", ".join(f"{s} {LEDGER.looks(s)}" for s in splits))
print(f"    chain verifies: {LEDGER.verify()}; head {LEDGER.head}")
print(f"\n  ({time.time() - T0:.0f}s)")
print("  LAST LINE OF MC AI BRAIN 5 CELL 1")


MC AI BRAIN 5 - CELL 1: LIBRARY AND LEDGER

  LIBRARY into /kaggle/working/mcai
    core.py     16,234 bytes  copied from mc-ai-brain-4
    stats.py     6,788 bytes  copied from mc-ai-brain-4
    read.py     18,116 bytes  copied from mc-ai-brain-4
    book.py      4,337 bytes  copied from mc-ai-brain-4
    words.py     7,630 bytes  copied from mc-ai-brain-4
    count.py     7,763 bytes  copied from mc-ai-brain-4
    sense.py     7,302 bytes  copied from mc-ai-brain-4
    imported all 7; Python 3.13.18, NumPy 2.1.3, SciPy 1.16.3

  PREDICTIONS vs RESULTS
    B5-K1 Brain 4 modules whose text is the code written for Brain 4: 7 (predicted between 7 and 7) -> HELD
    B5-L1 Brain 4's entries at the start of this ledger, unchanged: 348 (predicted between 348 and 348) -> HELD
    B5-L2 models scored on TEST4 (WSD), Brain 3's included: 50 (predicted between 51 and 51) -> DID NOT HOLD
    B5-T1 Cell 1 seconds: 2.042 (predicted <= 30) -> HELD

  LEDGER /kaggle/working/mc_ai_brain5/ledger.jsonl
 

In [3]:
# =====================================================================
# MC AI BRAIN 5 - CELL 1B: FEVER DATA - PINNED, CHECKED, GIVEN ROLES
# Needs Cells 0 and 1 (put it right after Cell 1).
# FEVER 1.0 (Thorne et al. 2018), from fever.ai, CC BY-SA 3.0. A file is
# used where an attached input holds it (found by name, size and
# SHA-256; nothing is copied), else downloaded into data/fever (fever.ai,
# then the old Amazon S3 address; a broken download resumes).
# ROLES  train: fit; paper_dev: practice; paper_test: test;
#   shared_task_dev: test (it is paper_dev and paper_test together:
#   18,999 + 18,567 = 37,566 evidence rows); shared_task_dev_public and
#   shared_task_test: unlabelled (no gold); wiki-pages.zip: knowledge
#   (the June 2017 Wikipedia pages the labels were made against)
# The zip also holds a macOS "._" copy of each page file; those are left
# out. Pages are counted once per zip (kept in fever_pages.json).
# PREDICTIONS (written before running)
#   B5-D1 FEVER files kept with their SHA-256: 7 of 7
#   B5-D2 page files in wiki-pages.zip (macOS copies left out): 109
#   B5-D3 pages in them: 5,416,537
# =====================================================================
import json, time, urllib.request, zipfile
from pathlib import Path

print("=" * 64); print("MC AI BRAIN 5 - CELL 1B: FEVER DATA - PINNED, CHECKED, GIVEN ROLES"); print("=" * 64)
T0 = time.time()
for need in ("DATA", "WORK", "INPUT", "RECORD", "ROLES", "FROM", "SEALED", "SETS", "LEDGER", "sha256", "size"):
    if need not in globals():
        raise RuntimeError(f"run Cells 0 and 1 first ({need} is missing)")
from mcai.core import between

FEVER_BASES = ["https://fever.ai/download/fever/", "https://s3-eu-west-1.amazonaws.com/fever.public/"]
FEVER_PINS = {
    "train.jsonl": ("eba7e8f87076753f8494718b9a857827af7bf73e76c9e4b75420207d26e588b6", 33_024_303, "fit"),
    "paper_dev.jsonl": ("41158707810008747946bf23471e82df53e77a513524b9e3ec1c2e674ef5ef8c", 2_168_767, "practice"),
    "paper_test.jsonl": ("fb7b0280a0adc2302bbb29bfb7af37274fa585de3171bcf908f180642d11d88e", 2_181_168, "test"),
    "shared_task_dev.jsonl": ("e89865bfe1b4dd054e03dd57d7241a6fde24862905f31117cf0cd719f7c78df7", 4_349_935,
                              "test"),
    "shared_task_dev_public.jsonl": ("acda01ae5ee7e75c73909a665f465cec20704ea26e9d676cd7423ff2c8ab0e8b",
                                     1_530_640, "unlabelled"),
    "shared_task_test.jsonl": ("76dd0872d8fa1f49efe1194fe8a88b7dd4c715c77d87a142b615d4be583e1e51", 1_599_159,
                               "unlabelled"),
    "wiki-pages.zip": ("4b06d95da6adf7fe02d2796176c670dacccb21348da89cba4c50676ab99665f2", 1_713_485_474,
                       "knowledge")}
for key, claim, rule in (("B5-D1", "FEVER files kept with their SHA-256 (of 7)", between(7, 7)),
                         ("B5-D2", "page files in wiki-pages.zip, macOS copies left out", between(109, 109)),
                         ("B5-D3", "pages in them", between(5_416_537, 5_416_537))):
    LEDGER.predict(key, claim, rule)
CHECKED = WORK / "fever_checked.json"            # SHA-256 already computed, by path, size and time
MEMO = json.loads(CHECKED.read_text()) if CHECKED.exists() else {}


def sha_of(p: Path) -> str:
    st = p.stat()
    key = f"{p}|{st.st_size}|{st.st_mtime_ns}"
    if key not in MEMO:
        MEMO[key] = sha256(p)
    return MEMO[key]


def fever_get(name: str, want: str, nbytes: int):
    """(path, how, address): kept here, used in an attached input, or downloaded and checked"""
    here = DATA / "fever" / name
    if here.exists() and here.stat().st_size == nbytes and sha_of(here) == want:
        return here, "already here", FEVER_BASES[0] + name
    for p in sorted(INPUT.rglob(name)) if INPUT.exists() else []:
        if p.is_file() and p.stat().st_size == nbytes and sha_of(p) == want:
            return p, f"used where it is ({p.parent})", FEVER_BASES[0] + name
    here.parent.mkdir(parents=True, exist_ok=True)
    tried = []
    for base in FEVER_BASES:
        url = base + name
        for attempt in range(3):
            have = here.stat().st_size if here.exists() else 0
            if have > nbytes:
                here.unlink()
                have = 0
            head = {"User-Agent": "MC_AI_Brain5/1.0", **({"Range": f"bytes={have}-"} if have else {})}
            try:
                with urllib.request.urlopen(urllib.request.Request(url, headers=head), timeout=60) as r:
                    mode = "ab" if have and r.status == 206 else "wb"
                    with open(here, mode) as f:
                        for b in iter(lambda: r.read(1 << 20), b""):
                            f.write(b)
                if here.stat().st_size == nbytes:
                    if sha_of(here) == want:
                        return here, "downloaded", url
                    here.unlink()
                    tried.append(f"{url}: other bytes than pinned")
                    break
            except Exception as err:
                tried.append(f"{url} try {attempt + 1}: {type(err).__name__}: {err}")
    return None, "REFUSED: " + " | ".join(tried[-3:]), FEVER_BASES[0] + name


def fever_parts(z: zipfile.ZipFile) -> list:
    """The page files in wiki-pages.zip, without the macOS '._' copies"""
    return sorted(n for n in z.namelist() if n.endswith(".jsonl") and not n.startswith("__MACOSX/")
                  and not n.rsplit("/", 1)[-1].startswith("._"))


FEVER, FEVER_REFUSED = {}, []
print("\n  fever: fever.ai (no commit: pinned by SHA-256)")
for name, (want, nbytes, role) in FEVER_PINS.items():
    t1 = time.time()
    path, how, url = fever_get(name, want, nbytes)
    print(f"    {name:<30} {size(nbytes) if path else '':>7}  {role:<10} {how} ({time.time() - t1:.1f}s)")
    if path is None:
        FEVER_REFUSED.append(name)
        continue
    FEVER[name] = path
    kept = str(path.relative_to(DATA)) if path.is_relative_to(DATA) else str(path)
    RECORD[:] = [r for r in RECORD if not (r["source"] == "fever" and r["file"].endswith(name))]
    RECORD.append(dict(source="fever", url=url, repo="fever.ai", commit="-", file=kept, sha256=want, bytes=nbytes))
    ROLES[kept], FROM[kept] = role, "fever"
CHECKED.write_text(json.dumps(MEMO, indent=1))
(DATA / "sources.json").write_text(json.dumps(RECORD, indent=1))
(DATA / "roles.json").write_text(json.dumps(dict(roles=ROLES, sealed=SEALED, TEST4=[k for k, v in SETS.items()
                                                                                    if v == "test"]), indent=1))
FEVER_COUNT = {}
if "wiki-pages.zip" in FEVER:
    counted = WORK / "fever_pages.json"
    FEVER_COUNT = json.loads(counted.read_text()) if counted.exists() else {}
    if FEVER_COUNT.get("sha256") != FEVER_PINS["wiki-pages.zip"][0]:
        t1 = time.time()
        with zipfile.ZipFile(FEVER["wiki-pages.zip"]) as z:
            parts = fever_parts(z)
            pages = 0
            for n in parts:
                with z.open(n) as f:
                    pages += sum(1 for line in f if line.strip())
            other = len([n for n in z.namelist() if n.endswith(".jsonl")]) - len(parts)
        FEVER_COUNT = dict(sha256=FEVER_PINS["wiki-pages.zip"][0], files=len(parts), pages=pages, macos=other)
        counted.write_text(json.dumps(FEVER_COUNT, indent=1))
        print(f"\n  wiki-pages.zip counted ({time.time() - t1:.0f}s)")
    print(f"  wiki-pages.zip: {FEVER_COUNT['files']} page files ({FEVER_COUNT['macos']} macOS copies left out), "
          f"{FEVER_COUNT['pages']:,} pages")
print("\n  ROLES (data/roles.json): " + ", ".join(f"{n} {r}" for n, (_, _, r) in FEVER_PINS.items()))

print("\n  PREDICTIONS vs RESULTS")
LEDGER.result("B5-D1", len(FEVER))
LEDGER.result("B5-D2", FEVER_COUNT.get("files", 0))
LEDGER.result("B5-D3", FEVER_COUNT.get("pages", 0))
if FEVER_REFUSED:
    raise RuntimeError(f"FEVER files not kept: {', '.join(FEVER_REFUSED)} (attach the MC AI FEVER notebook, "
                       f"Add Input, or turn the internet on)")
if "sync" in globals():
    sync("Cell 1B")
print(f"\n  ({time.time() - T0:.0f}s)")
print("  LAST LINE OF MC AI BRAIN 5 CELL 1B")


MC AI BRAIN 5 - CELL 1B: FEVER DATA - PINNED, CHECKED, GIVEN ROLES

  fever: fever.ai (no commit: pinned by SHA-256)
    train.jsonl                      31 MB  fit        used where it is (/kaggle/input/notebooks/sasquilly/mc-ai-fever/fever_data) (0.3s)
    paper_dev.jsonl                   2 MB  practice   used where it is (/kaggle/input/notebooks/sasquilly/mc-ai-fever/fever_data) (0.4s)
    paper_test.jsonl                  2 MB  test       used where it is (/kaggle/input/notebooks/sasquilly/mc-ai-fever/fever_data) (0.4s)
    shared_task_dev.jsonl             4 MB  test       used where it is (/kaggle/input/notebooks/sasquilly/mc-ai-fever/fever_data) (0.4s)
    shared_task_dev_public.jsonl      1 MB  unlabelled used where it is (/kaggle/input/notebooks/sasquilly/mc-ai-fever/fever_data) (0.4s)
    shared_task_test.jsonl            2 MB  unlabelled used where it is (/kaggle/input/notebooks/sasquilly/mc-ai-fever/fever_data) (0.4s)
    wiki-pages.zip                  1.6 GB  knowledge  

In [4]:
# =====================================================================
# MC AI BRAIN 5 - SYNC: THE LEDGER TO GITHUB
# Run after Cell 1 in each session. Needs the notebook's internet and
# the Kaggle secret GITHUB_TOKEN (Add-ons -> Secrets): a fine-grained
# token for Coffey333/mc-ai-brain5 with Contents: Read and write. The
# token is read from the secret, handed to git through environment
# variables for one command at a time, and never printed or written to
# a file.
# ACCESS CHECK, one line per step, stopping at the first that fails: the
#   secret can be read (its length and kind are printed, nothing of it
#   else); GitHub can be reached; the token can see the repository.
# SYNC  Clones the repository's main branch (one commit deep) into a
#   temporary folder outside /kaggle/working and puts into its kaggle/:
#   ledger.jsonl          Brain 5's ledger, only if its chain verifies
#   earlier/              the Brain 4 and Brain 3 ledgers it continues
#   data/                 roles.json and sources.json (no data file)
#   reports/              any report a cell wrote to mc_ai_brain5/reports
#   runs/<time>.json      this sync: time, machine, versions, the ledger's
#                         length and head, the SHA-256 of each file synced
#   Commits only if something changed, pushes, and reads back from
#   GitHub that main is the commit just pushed. If GitHub moved on
#   meanwhile, it clones again and tries once more.
# Defines sync(note) for later cells: the same steps; a failure is
# printed and returns False instead of stopping the cell. Any failure
# here stops this cell with the reason in the error itself.
# =====================================================================
import base64, json, os, platform, shutil, subprocess, tempfile, time, urllib.error, urllib.request
from pathlib import Path

import numpy

print("=" * 64); print("MC AI BRAIN 5 - SYNC: THE LEDGER TO GITHUB"); print("=" * 64)
if "LEDGER" not in globals() or "sha256" not in globals():
    raise RuntimeError("run Cells 0 and 1 first")
OWNER_REPO = "Coffey333/mc-ai-brain5"
REMOTE = os.environ.get("MC_AI_SYNC_REMOTE", f"https://github.com/{OWNER_REPO}.git")
BRANCH = "main"
SYNC_ERROR = ""


def _token() -> str:
    t = os.environ.get("MC_AI_GITHUB_TOKEN")  # for tests outside Kaggle
    if t:
        return t
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("GITHUB_TOKEN")
    except Exception as err:
        raise RuntimeError(f"no Kaggle secret named GITHUB_TOKEN could be read ({type(err).__name__}): Add-ons -> "
                           f"Secrets; its label must be exactly GITHUB_TOKEN and its box checked for this "
                           f"notebook") from None


def _api(path: str, token: str = "") -> int | None:
    """HTTP status GitHub's API gives for path (None if GitHub cannot be reached); the token, if given, goes
    in the Authorization header."""
    head = {"Accept": "application/vnd.github+json", "User-Agent": "MC-AI-Brain5"}
    if token:
        head["Authorization"] = f"Bearer {token}"
    try:
        with urllib.request.urlopen(urllib.request.Request(f"https://api.github.com{path}", headers=head),
                                    timeout=30) as r:
            return r.status
    except urllib.error.HTTPError as err:
        return err.code
    except OSError:
        return None


def _git(args: list, cwd, token: str) -> str:
    """Run one git command; the token goes in as an HTTP header set through environment variables, and is cut
    out of anything git prints."""
    header = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    env = dict(os.environ, GIT_TERMINAL_PROMPT="0", GIT_CONFIG_COUNT="1",
               GIT_CONFIG_KEY_0="http.https://github.com/.extraheader",
               GIT_CONFIG_VALUE_0=f"AUTHORIZATION: basic {header}")
    r = subprocess.run(["git", *args], cwd=str(cwd), env=env, capture_output=True, text=True, timeout=300)
    out = (r.stdout + r.stderr).replace(token, "***").replace(header, "***")
    if r.returncode != 0:
        hint = {"clone": " (can the token read mc-ai-brain5? is the internet on?)",
                "push": " (the token can read but not write: GitHub -> the token -> Contents: Read and write)"}
        raise RuntimeError(f"git {args[0]} failed{hint.get(args[0], '')}: {out.strip()[-400:]}")
    return out


def check_access() -> str:
    """One line per step; raises at the first step that fails. Returns the token."""
    token = _token()
    kind = ("fine-grained" if token.startswith("github_pat_") else "classic" if token.startswith("ghp_")
            else "not a GitHub token shape")
    print(f"  1 secret GITHUB_TOKEN read: {len(token)} characters, {kind}")
    if "MC_AI_SYNC_REMOTE" in os.environ:  # tests: no GitHub
        return token
    status = _api(f"/repos/{OWNER_REPO}")
    if status is None:
        raise RuntimeError("GitHub cannot be reached: is the notebook's internet on? (Settings -> Internet)")
    print(f"  2 GitHub reached (without the token it answers {status}, as for any private repository)")
    status = _api(f"/repos/{OWNER_REPO}", token)
    if status == 401:
        raise RuntimeError("GitHub refuses the token (401): it is mistyped, expired or deleted; make a new one")
    if status != 200:
        raise RuntimeError(f"the token cannot see {OWNER_REPO} ({status}): on GitHub, edit the token -> "
                           f"Repository access -> Only select repositories -> add mc-ai-brain5")
    print(f"  3 the token can see {OWNER_REPO}")
    return token


def _attempt(note: str, token: str, tmp: Path):
    repo = tmp / "repo"
    shutil.rmtree(repo, ignore_errors=True)
    _git(["clone", "--quiet", "--depth", "1", "--branch", BRANCH, REMOTE, str(repo)], tmp, token)
    dest = repo / "kaggle"
    files = {"ledger.jsonl": WORK / "ledger.jsonl"}
    files.update({f"earlier/{p.name}": p for p in sorted(EARLIER.glob("*.jsonl"))})
    files.update({f"data/{n}": DATA / n for n in ("roles.json", "sources.json") if (DATA / n).exists()})
    if (WORK / "reports").exists():
        files.update({f"reports/{p.name}": p for p in sorted((WORK / "reports").glob("*.txt"))})
    for rel, src in files.items():
        (dest / rel).parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(src, dest / rel)
    if not _git(["status", "--porcelain", "--", "kaggle"], repo, token).strip():
        return None, files
    stamp = time.strftime("%Y-%m-%dT%H-%M-%SZ", time.gmtime())
    run = dict(time=stamp, note=note, machine=platform.platform(), cpus=os.cpu_count(),
               python=platform.python_version(), numpy=numpy.__version__, ledger_entries=len(LEDGER.entries),
               ledger_head=LEDGER.head, files={rel: sha256(dest / rel) for rel in files})
    (dest / "runs").mkdir(parents=True, exist_ok=True)
    (dest / "runs" / f"{stamp}.json").write_text(json.dumps(run, indent=1))
    _git(["add", "kaggle"], repo, token)
    message = f"Kaggle {stamp}: ledger {len(LEDGER.entries)} entries, head {LEDGER.head[:16]}"
    _git(["-c", "user.name=MC AI Brain 5 (Kaggle run)", "-c", "user.email=kaggle-run@localhost", "commit", "-q",
          "-m", message + (f"; {note}" if note else "")], repo, token)
    _git(["push", "--quiet", "origin", f"HEAD:{BRANCH}"], repo, token)
    mine = _git(["rev-parse", "HEAD"], repo, token).strip()
    there = _git(["ls-remote", "origin", f"refs/heads/{BRANCH}"], repo, token).split()[0]
    if there != mine:
        raise RuntimeError(f"GitHub's {BRANCH} is {there[:12]}, not the commit pushed ({mine[:12]})")
    return mine, files


def sync(note: str = "", token: str = "") -> bool:
    """Push the ledger and reports to kaggle/ in mc-ai-brain5. True if GitHub now holds them; on failure the
    reason is printed and kept in SYNC_ERROR."""
    global SYNC_ERROR
    t0 = time.time()
    try:
        if not LEDGER.verify():
            raise RuntimeError("the ledger's chain does not verify: nothing synced")
        token = token or _token()
        with tempfile.TemporaryDirectory() as tmp:
            for attempt in (1, 2):
                try:
                    commit, files = _attempt(note, token, Path(tmp))
                    break
                except RuntimeError as err:
                    if attempt == 2 or "git push failed" not in str(err):
                        raise
        if commit is None:
            print(f"  sync: GitHub already has these {len(files)} files (ledger head {LEDGER.head[:16]}) "
                  f"({time.time() - t0:.0f}s)")
        else:
            print(f"  sync: pushed {len(files)} files to kaggle/ on {BRANCH} as commit {commit[:12]}; GitHub "
                  f"confirms it ({len(LEDGER.entries)} entries, head {LEDGER.head[:16]}) ({time.time() - t0:.0f}s)")
        SYNC_ERROR = ""
        return True
    except Exception as err:
        SYNC_ERROR = str(err)
        print(f"  sync FAILED: {err}")
        return False


print(f"  {_git(['--version'], WORK, 'none').strip()}; repository {REMOTE}, branch {BRANCH}")
_TOKEN = None
try:
    _TOKEN = check_access()
    SYNCED = sync("sync cell", _TOKEN)
except RuntimeError as err:
    raise RuntimeError(f"access check: {err}") from None
finally:
    _TOKEN = None
if not SYNCED:
    raise RuntimeError(f"the sync failed: {SYNC_ERROR}")
print("  LAST LINE OF MC AI BRAIN 5 SYNC")


MC AI BRAIN 5 - SYNC: THE LEDGER TO GITHUB
  git version 2.43.0; repository https://github.com/Coffey333/mc-ai-brain5.git, branch main
  1 secret GITHUB_TOKEN read: 93 characters, fine-grained
  2 GitHub reached (without the token it answers 404, as for any private repository)
  3 the token can see Coffey333/mc-ai-brain5
  sync: pushed 4 files to kaggle/ on main as commit fe0650b35a5d; GitHub confirms it (362 entries, head a334b36a5c636385) (4s)
  LAST LINE OF MC AI BRAIN 5 SYNC


In [5]:
# =====================================================================
# MC AI BRAIN 5 - CELL 2: WORD VECTORS AND DEFINITION VECTORS - BUILT
# ONCE, SAVED
# Needs Cells 0 and 1. No internet, nothing trained.
# Builds the vectors of Brain 4's Cells 14 and 16 with the same code,
# settings and seeds, and saves them with mcai.core's Artifacts (.npz and
# .json, never pickle) in mc_ai_brain5/artifacts, in a folder named by
# what they depend on: the data's SHA-256s and the settings below. A later
# session with the same dependencies loads them instead of building them.
# WORD VECTORS (Cell 14)  Simple Wikipedia, lower case, words of letters
#   only; the 50,000 commonest seen >= 5 times; the words 1 and 2 places
#   either side counted with weight 1 and 1/2, never across two articles;
#   PPMI with context counts to the power 0.75 (Levy, Goldberg & Dagan
#   2015); the 300 strongest directions by randomized SVD (Halko et al.
#   2011; seed 14, 5 power iterations), each scaled by the square root of
#   its singular value; rows at length 1
# DEFINITION VECTORS (Cell 16)  each WordNet synset: the SVD-PPMI vectors
#   of the words in its definition and examples (not its own lemmas), each
#   weighted a / (a + p), a = 0.001, p = the word's share of the counted
#   text; the synsets' first principal component removed; length 1: SIF
#   (Arora, Liang & Ma 2017)
# CHECK  Brain 4's synonym test, rebuilt with Cell 14's seed: 2,000 held-
#   out WordNet synonym questions, four choices each; answer = the most
#   similar word vector. Brain 4 scored 0.651 (its V1). Run every time, on
#   built or loaded vectors
# PREDICTIONS (written before running)
#   B5-V1 synonym test minus Brain 4's 0.651: exactly 0 (same data, code,
#         seeds, Python and NumPy as Brain 4's run)
#   B5-V2 the size of that difference: at most 0.002 (4 of 2,000)
#   B5-T2 the cell, when it builds: under 300 seconds
#   B5-T3 the cell, when it loads: under 60 seconds
# =====================================================================
import importlib, re, time

import numpy as np
import scipy
import scipy.sparse as sp
import sklearn
from sklearn.utils.extmath import randomized_svd

print("=" * 64); print("MC AI BRAIN 5 - CELL 2: WORD VECTORS AND DEFINITION VECTORS"); print("=" * 64)
T0 = time.time()
if "LEDGER" not in globals() or "RECORD" not in globals():
    raise RuntimeError("run Cells 0 and 1 first")
import mcai.book, mcai.read
importlib.reload(mcai.book); importlib.reload(mcai.read)
from mcai.book import read_encyclopedia
from mcai.core import Artifacts, at_most, between
from mcai.read import read_wordnet

LEDGER.predict("B5-V1", "synonym test minus Brain 4's V1 (0.651)", between(0, 0))
LEDGER.predict("B5-V2", "size of the synonym test's difference from Brain 4's", at_most(0.002))
LEDGER.predict("B5-T2", "Cell 2 seconds, building", at_most(300))
LEDGER.predict("B5-T3", "Cell 2 seconds, loading", at_most(60))
ARTS = Artifacts(WORK / "artifacts")
SETTINGS = dict(vocabulary=50000, min_count=5, window={"1": 1.0, "2": 0.5}, context_power=0.75, dims=300,
                svd_seed=14, svd_iterations=5, sif_a=0.001)
DEPENDS = dict(what="Brain 4 Cells 14 and 16 vectors, rebuilt in Brain 5 Cell 2", settings=SETTINGS,
               data={r["file"]: r["sha256"] for r in RECORD if r["source"] in ("encyclopedia", "wordnet")})
WN = read_wordnet(DATA / "wordnet")
SYNLIST = list(WN.synsets.values())


def unit(X: np.ndarray) -> np.ndarray:
    n = np.linalg.norm(X, axis=1, keepdims=True)
    return (X / np.where(n == 0, 1, n)).astype(np.float32)


def norm(w: str) -> str:
    return " ".join(w.lower().replace("_", " ").split())


def build() -> dict:
    """Brain 4 Cell 14's counting, PPMI and SVD, then Cell 16's definition vectors."""
    t1 = time.time()
    enc = read_encyclopedia(sorted((DATA / "encyclopedia").rglob("*.parquet"))[0])
    type_id, chunks = {}, []
    letters = str.maketrans({c: " " for c in map(chr, range(128)) if not c.isalpha()})
    for a in enc.articles():
        toks = [t for t in a.text.lower().translate(letters).split() if t.isascii()]
        chunks.append(np.fromiter((type_id.setdefault(t, len(type_id)) for t in toks), np.int32, len(toks)))
        chunks.append(np.array([-1], np.int32))        # article boundary
    ids = np.concatenate(chunks)
    del chunks
    types = sorted(type_id, key=type_id.get)
    freq = np.bincount(ids[ids >= 0], minlength=len(types))
    top = [int(i) for i in np.argsort(-freq, kind="stable") if freq[i] >= SETTINGS["min_count"]]
    top = top[:SETTINGS["vocabulary"]]
    v = len(top)
    rank = np.full(len(types) + 1, -2, np.int32)       # -2: outside the vocabulary
    rank[-1] = -1                                       # ids -1 (boundary) read rank[-1]
    rank[np.array(top)] = np.arange(v)
    vocab, vfreq = [types[i] for i in top], freq[top]
    r_ids, tokens = rank[ids], int((ids >= 0).sum())
    del ids
    print(f"\n  COUNT: {tokens:,} words in {enc.groups[-1]:,} articles; {len(types):,} different words; vocabulary "
          f"{v:,} (seen >= {vfreq[-1]:,} times) covers {vfreq.sum() / tokens:.1%} of the text "
          f"({time.time() - t1:.0f}s)")
    t1 = time.time()
    m = sp.csr_matrix((v, v), dtype=np.float32)
    step = 20_000_000
    for s0 in range(0, len(r_ids), step):
        part = r_ids[s0:s0 + step + 2]
        rows, cols, wts = [], [], []
        for d, w in ((1, 1.0), (2, 0.5)):
            n = min(step, len(part) - d)
            a, b = part[:n], part[d:d + n]
            ok = (a >= 0) & (b >= 0)
            if d == 2:
                ok &= part[1:1 + n] != -1              # no pair across an article boundary
            a, b = a[ok], b[ok]
            rows += [a, b]; cols += [b, a]; wts.append(np.full(2 * len(a), w, np.float32))
        m = m + sp.coo_matrix((np.concatenate(wts), (np.concatenate(rows), np.concatenate(cols))),
                              shape=(v, v)).tocsr()
    del r_ids, rows, cols, wts
    row = np.asarray(m.sum(1)).ravel()
    col = np.asarray(m.sum(0)).ravel() ** SETTINGS["context_power"]
    coo = m.tocoo()
    del m
    pmi = np.log(coo.data) - np.log(row[coo.row]) - np.log(col[coo.col] / col.sum())
    keep = pmi > 0
    ppmi = sp.csr_matrix((pmi[keep].astype(np.float32), (coo.row[keep], coo.col[keep])), shape=(v, v))
    del coo, pmi, keep
    u, s, _ = randomized_svd(ppmi, n_components=min(SETTINGS["dims"], v - 1), n_iter=SETTINGS["svd_iterations"],
                             random_state=SETTINGS["svd_seed"])
    svd = unit(u * np.sqrt(s))
    print(f"    PPMI: {ppmi.nnz:,} positive pairs; SVD-PPMI {svd.shape[1]} numbers per word ({time.time() - t1:.0f}s)")
    del u, ppmi
    t1 = time.time()
    wrow = {w: i for i, w in enumerate(vocab)}
    gl = []
    for si, syn in enumerate(SYNLIST):
        own = {x for w in syn.words for x in norm(w).split()}
        for x in re.findall(r"[a-z]+", (syn.definition + " " + " ".join(syn.examples)).lower()):
            r = wrow.get(x, -1)
            if r >= 0 and x not in own:
                gl.append((si, r))
    gl = np.array(gl).T
    sif = (SETTINGS["sif_a"] / (SETTINGS["sif_a"] + vfreq / vfreq.sum())).astype(np.float32)
    g = sp.csr_matrix((sif[gl[1]], (gl[0], gl[1])), shape=(len(SYNLIST), v))
    synv = g @ svd
    has_def = np.abs(synv).sum(1) > 0
    pc = np.linalg.svd(synv[has_def], full_matrices=False)[2][0]
    synv = unit(np.where(has_def[:, None], synv - np.outer(synv @ pc, pc), 0))
    print(f"  DEFINITIONS: {int(has_def.sum()):,} of {len(SYNLIST):,} synsets have a definition vector "
          f"({time.time() - t1:.0f}s)")
    return dict(vocab=np.array(vocab), vfreq=vfreq.astype(np.int64), svd=svd,
                synsets=np.array([x.id for x in SYNLIST]), synv=synv, pc=pc.astype(np.float32), has_def=has_def,
                tokens=np.array(tokens), articles=np.array(enc.groups[-1]))


VEC, BUILT = ARTS.get("vectors", DEPENDS, build)
if list(VEC["synsets"]) != [x.id for x in SYNLIST]:
    raise RuntimeError("the saved definition vectors are for another order of synsets: delete mc_ai_brain5/"
                       "artifacts and run this cell again")
VOCAB, VFREQ, SVD, SYNV = [str(w) for w in VEC["vocab"]], VEC["vfreq"], VEC["svd"], VEC["synv"]
V, WROW = len(VOCAB), {w: i for i, w in enumerate(VEC["vocab"].tolist())}
SID = {s: i for i, s in enumerate(VEC["synsets"].tolist())}
print(f"\n  VECTORS {'built and saved' if BUILT else 'loaded'}: {V:,} words x {SVD.shape[1]}, {SYNV.shape[0]:,} "
      f"synsets x {SYNV.shape[1]}; {int(VEC['tokens']):,} words counted in {int(VEC['articles']):,} articles "
      f"({time.time() - T0:.0f}s)")

# ---------------- check: Brain 4's synonym test (Cell 14), same seed ----------------
t1 = time.time()
RNG = np.random.default_rng(14)
HASPOS = {p: np.array([bool(WN.senses.get(f"{w}.{p}")) for w in VOCAB]) for p in "nvar"}
PAIRPOS = {}
for s in WN.synsets.values():
    ws = sorted({x.lower() for x in s.words if x.isalpha() and x.lower() in WROW})
    for i, a in enumerate(ws):
        for b in ws[i + 1:]:
            PAIRPOS.setdefault((WROW[a], WROW[b]), s.id[0])
PAIRS = np.array(sorted(PAIRPOS))
HELD = RNG.random(len(PAIRS)) < 0.2
TEST = PAIRS[HELD]
LINKED = {int(i): set() for i in range(V)}
for s in WN.synsets.values():
    near = {s.id} | {p.target for p in s.pointers if p.symbol in ("@", "@i", "~", "~i")}
    parents = [p.target for p in s.pointers if p.symbol in ("@", "@i")]
    near |= {p.target for q in parents for p in WN.synsets[q].pointers if p.symbol in ("~", "~i")}
    words = {WROW[x.lower()] for n in near for x in WN.synsets[n].words if x.lower() in WROW}
    for x in s.words:
        if x.lower() in WROW:
            LINKED[WROW[x.lower()]] |= words
QUESTIONS = []
for k in RNG.permutation(len(TEST)):
    w, s = (int(x) for x in TEST[k][RNG.permutation(2)])
    pos = PAIRPOS[tuple(TEST[k])]
    if VFREQ[w] < 50 or VFREQ[s] < 50:
        continue
    lo, hi = max(0, s - 200), min(V, s + 200)
    pool = [x for x in RNG.permutation(np.arange(lo, hi)).tolist()
            if x not in (w, s) and x not in LINKED[w] and VFREQ[x] >= 50 and HASPOS[pos][x]][:3]
    if len(pool) == 3:
        QUESTIONS.append((w, s, pool))
    if len(QUESTIONS) == 2000:
        break
RIGHT = np.array([([s] + pool)[int(np.argmax(SVD[[s] + pool] @ SVD[w]))] == s for w, s, pool in QUESTIONS])
SCORE = float(RIGHT.mean())
print(f"\n  SYNONYM TEST: {len(PAIRS):,} synonym pairs in the vocabulary, {len(TEST):,} held out; {len(QUESTIONS):,} "
      f"questions; {int(RIGHT.sum()):,} right = {SCORE:.4f} (Brain 4: 0.651) ({time.time() - t1:.0f}s)")
for w, s, pool in QUESTIONS[:3]:
    pick = ([s] + pool)[int(np.argmax(SVD[[s] + pool] @ SVD[w]))]
    print(f"    {VOCAB[w]}: {', '.join(VOCAB[x] for x in [s] + pool)} -> {VOCAB[pick]} "
          f"({'right' if pick == s else 'wrong'})")
print(f"  NumPy {np.__version__}, SciPy {scipy.__version__}, scikit-learn {sklearn.__version__}")

print("\n  PREDICTIONS vs RESULTS")
LEDGER.result("B5-V1", round(SCORE - 0.651, 6))
LEDGER.result("B5-V2", round(abs(SCORE - 0.651), 6))
LEDGER.result("B5-T2" if BUILT else "B5-T3", round(time.time() - T0))
if "sync" in globals():
    sync("Cell 2")
print(f"\n  ({time.time() - T0:.0f}s)")
print("  LAST LINE OF MC AI BRAIN 5 CELL 2")


MC AI BRAIN 5 - CELL 2: WORD VECTORS AND DEFINITION VECTORS

  COUNT: 39,324,314 words in 241,787 articles; 522,412 different words; vocabulary 50,000 (seen >= 22 times) covers 96.4% of the text (31s)
    PPMI: 14,032,154 positive pairs; SVD-PPMI 300 numbers per word (39s)
  DEFINITIONS: 117,228 of 117,659 synsets have a definition vector (4s)

  VECTORS built and saved: 50,000 words x 300, 117,659 synsets x 300; 39,324,314 words counted in 241,787 articles (77s)

  SYNONYM TEST: 18,132 synonym pairs in the vocabulary, 3,669 held out; 2,000 questions; 1,302 right = 0.6510 (Brain 4: 0.651) (4s)
    folk: sept, vasari, bending, lapland -> bending (wrong)
    restore: repair, slang, imagine, vacuum -> repair (right)
    civilized: cultivated, seasonal, thirteenth, digestive -> cultivated (right)
  NumPy 2.1.3, SciPy 1.16.3, scikit-learn 1.6.1

  PREDICTIONS vs RESULTS
    B5-V1 synonym test minus Brain 4's V1 (0.651): 0 (predicted between 0 and 0) -> HELD
    B5-V2 size of the synonym tes

In [6]:
# =====================================================================
# MC AI BRAIN 5 - CELL 3: WHICH MEANING - BRAIN 4'S POOLED (hand) MODEL,
# BUILT ONCE, SAVED, CHECKED ON SEMEVAL-2007
# Needs Cells 0-2. No internet, nothing trained. TEST4 is not scored here.
# Rebuilds Brain 4's word-sense model of Cells 18, 19 and 21 with their
# code, settings and seeds, and saves it with Artifacts (mc_ai_brain5/
# artifacts; .npz and .json, never pickle). A later session loads it.
# TASK     each tagged word of SemCor's text and of ALL (Raganato et al.
#          2017) gets one WordNet synset; SemEval-2007 (SE07, 455 words)
#          is the practice set that chooses the weights
# CONTEXT  the sentence's content lemmas: their SVD-PPMI vectors (Cell 2),
#          weighted a / (a + p), the definition vectors' common direction
#          removed, length 1
# SCORE of each candidate synset, all from counting and the dictionary:
#   (g + e) / 2 + beta x prior + g1 x ow + g2 x om
#   g      cosine of the context with the definition vector (Cell 2)
#   e      cosine with the synset's stored examples: the sum of the context
#          vectors of every word tagged with it in SemCor and, by hand, in
#          the Princeton WordNet Gloss Corpus (POOLED (hand)); a synset with
#          no example uses its definition vector
#   prior  log((times SemCor tagged it for the lemma + 1) / (times any
#          candidate was tagged + candidates))
#   ow     WORDS AT PLACES: FHRR, a random phasor per lemma and per place
#          -3 ... +3 (D = 1,024, seeds 19 and 21); context = sum of place x
#          lemma; cosine with the synset's stored sum
#   om     MEANINGS AT PLACES: the SVD-PPMI vectors of the words 2 either
#          side, one block per place; cosine with the synset's stored sum
# WEIGHTS  chosen on SE07 only, as Brain 4 chose them: beta from 0, 0.01,
#          0.02, 0.05, 0.1, 0.2, 0.5 (SemCor examples, no order); then g1
#          and g2 from 0, 0.25, 0.5, 1, 2 (ties to the smaller sum)
# BRAIN 4 ON SE07 (its printed output): BOTH + PRIOR 62.2, ALL 64.6,
#          POOLED (hand) 63.5; beta 0.1, g1 0.5, g2 0.25
# PREDICTIONS (written before running)
#   B5-S1 beta chosen on SE07: 0.1
#   B5-S2 g1 chosen on SE07: 0.5
#   B5-S3 g2 chosen on SE07: 0.25
#   B5-S4 SE07 words right, BOTH + PRIOR: 283 of 455 (Brain 4's 62.2%)
#   B5-S5 SE07 words right, ALL: 294 of 455 (64.6%)
#   B5-S6 SE07 words right, POOLED (hand): 289 of 455 (63.5%)
#   B5-S7 synsets with stored examples, POOLED (hand): 36,615
#   B5-T4 the cell, when it builds: under 600 seconds
#   B5-T5 the cell, when it loads: under 90 seconds
# =====================================================================
import importlib, time

import numpy as np
import scipy.sparse as sp

print("=" * 64); print("MC AI BRAIN 5 - CELL 3: WHICH MEANING - POOLED (hand)"); print("=" * 64)
T0 = time.time()
for need in ("SVD", "SYNV", "SID", "WROW", "ARTS", "WN"):
    if need not in globals():
        raise RuntimeError(f"run Cells 0-2 first ({need} is missing)")
import mcai.sense, mcai.words
importlib.reload(mcai.sense); importlib.reload(mcai.words)
from mcai.core import at_most, between
from mcai.read import read_wsd
from mcai.sense import SenseCounts, candidates, items_of
from mcai.words import read_glosstag

for key, claim, rule in (("B5-S1", "beta chosen on SE07", between(0.1, 0.1)),
                         ("B5-S2", "g1 chosen on SE07", between(0.5, 0.5)),
                         ("B5-S3", "g2 chosen on SE07", between(0.25, 0.25)),
                         ("B5-S4", "SE07 words right of 455, BOTH + PRIOR", between(283, 283)),
                         ("B5-S5", "SE07 words right of 455, ALL", between(294, 294)),
                         ("B5-S6", "SE07 words right of 455, POOLED (hand)", between(289, 289)),
                         ("B5-S7", "synsets with stored examples, POOLED (hand)", between(36615, 36615)),
                         ("B5-T4", "Cell 3 seconds, building", at_most(600)),
                         ("B5-T5", "Cell 3 seconds, loading", at_most(90))):
    LEDGER.predict(key, claim, rule)
SIF = (0.001 / (0.001 + VFREQ / VFREQ.sum())).astype(np.float32)
PC = VEC["pc"]
ODIM, PLACES, MPLACES = 1024, (-3, -2, -1, 1, 2, 3), (-2, -1, 1, 2)
BETAS, GRID = (0, 0.01, 0.02, 0.05, 0.1, 0.2, 0.5), (0, 0.25, 0.5, 1, 2)

t1 = time.time()
TEXTS = {n: read_wsd(DATA / "wsd" / f"{n}.data.xml", DATA / "wsd" / f"{n}.gold.key.txt") for n in ("semcor", "ALL")}
TRAIN, ITEMS = items_of(TEXTS["semcor"], WN), items_of(TEXTS["ALL"], WN)
SET_OF = np.array([it.id.split(".")[0] for it in ITEMS])
DEV = SET_OF == "semeval2007"
WHERE = {t.instance: (toks, k) for name in ("semcor", "ALL") for toks in TEXTS[name].sentences.values()
         for k, t in enumerate(toks) if t.instance}
print(f"\n  TEXT ({time.time() - t1:.0f}s): {len(TRAIN):,} SemCor words; {len(ITEMS):,} words in ALL, {int(DEV.sum())} "
      f"of them SE07 (practice); the other {int((~DEV).sum()):,} are TEST4 and are not scored here")


def cunit(Z: np.ndarray) -> np.ndarray:
    n = np.sqrt((np.abs(Z) ** 2).sum(1, keepdims=True))
    return Z / np.where(n == 0, 1, n)


def contexts(items) -> np.ndarray:
    """One context vector per item (rows), from its content lemmas, as Cell 2 made definition vectors."""
    rows, cols = [], []
    for i, it in enumerate(items):
        for lem in it.context:
            r = WROW.get(lem, -1)
            if r >= 0:
                rows.append(i); cols.append(r)
    A = sp.csr_matrix((SIF[np.array(cols, int)], (np.array(rows, int), np.array(cols, int))), shape=(len(items), V))
    C = A @ SVD
    C -= np.outer(C @ PC, PC)
    return unit(C)


def around(toks, k) -> list:
    """The lemma at each place -3 ... +3 (None outside the sentence or not a word)."""
    out = []
    for d in PLACES:
        j = k + d
        lem = toks[j].lemma.lower() if 0 <= j < len(toks) else ""
        out.append(lem if lem.replace("'", "").isalpha() else None)
    return out


def ids_of(arounds) -> np.ndarray:
    return np.array([[LID.get(x, -1) if x else -1 for x in a] for a in arounds])


def words_at_places(ids: np.ndarray) -> np.ndarray:
    """Sum over places of place x lemma (FHRR binding), one row per word."""
    return sum(PLACEV[p][None, :] * LEMV[ids[:, p]] for p in range(len(PLACES)))


def meanings_at_places(arounds) -> np.ndarray:
    X = np.zeros((len(arounds), len(MPLACES) * SVD.shape[1]), np.float32)
    for i, a in enumerate(arounds):
        for b, d in enumerate(MPLACES):
            lem = a[PLACES.index(d)]
            r = WROW.get(lem, -1) if lem else -1
            if r >= 0:
                X[i, b * SVD.shape[1]:(b + 1) * SVD.shape[1]] = SVD[r]
    return X


def stored(items, ctx) -> dict:
    """The synsets these items are examples of, and per synset the sums of their context, order-word and
    order-meaning vectors (each example at length 1)."""
    pairs = np.array([(SID[s], i) for i, it in enumerate(items) for s in it.gold if s in SID])
    rows = np.unique(pairs[:, 0])
    pos = np.full(len(SID), -1)
    pos[rows] = np.arange(len(rows))
    T = sp.csr_matrix((np.ones(len(pairs), np.float32), (pos[pairs[:, 0]], pairs[:, 1])), shape=(len(rows), len(items)))
    E = T @ ctx
    W = np.zeros((len(rows), ODIM), np.complex64)
    M = np.zeros((len(rows), len(MPLACES) * SVD.shape[1]), np.float32)
    arounds = [around(*WHERE[it.id]) for it in items]
    for c0 in range(0, len(items), 20000):
        c1 = min(c0 + 20000, len(items))
        Z = cunit(words_at_places(ids_of(arounds[c0:c1])))
        W += (T[:, c0:c1] @ Z.real) + 1j * (T[:, c0:c1] @ Z.imag)
        M += T[:, c0:c1] @ unit(meanings_at_places(arounds[c0:c1]))
    return dict(rows=rows, E=E, W=W, M=M)


def combine(srcs: list) -> dict:
    """Stored examples of several sources summed per synset; a synset with none keeps its definition vector."""
    rows = np.unique(np.concatenate([s["rows"] for s in srcs]))
    pos = np.full(len(SID), -1)
    pos[rows] = np.arange(len(rows))
    out = {k: np.zeros((len(rows),) + srcs[0][k].shape[1:], srcs[0][k].dtype) for k in ("E", "W", "M")}
    for s in srcs:
        for k in ("E", "W", "M"):
            out[k][pos[s["rows"]]] += s[k]
    exv = SYNV.copy()
    exv[rows] = unit(out["E"])
    return dict(pos=pos, n=len(rows), EXV=exv, W=cunit(out["W"]).astype(np.complex64), M=unit(out["M"]))


def parts(counts_of) -> list:
    """Per word of ALL: candidate synsets, their rows, definition cosines and log priors."""
    out = []
    for k, it in enumerate(ITEMS):
        cands = candidates(WN, it)
        ids = np.array([SID[s] for s in cands])
        n = np.array([counts_of(it.lemma, s) for s in cands], float)
        out.append((cands, ids, SYNV[ids] @ C_ALL[k], np.log((n + 1) / (n.sum() + len(cands)))))
    return out


def right(v: dict, beta: float, g1: float, g2: float, mask: np.ndarray) -> np.ndarray:
    """1 where the best-scoring candidate is a gold synset, for the words in mask."""
    out = np.zeros(len(ITEMS), bool)
    for k in np.flatnonzero(mask):
        cands, ids, g, prior = PARTS[k]
        e = v["EXV"][ids] @ C_ALL[k]
        j = v["pos"][ids]
        ow = np.where(j >= 0, (v["W"][np.maximum(j, 0)] @ np.conj(O_W[k])).real, 0.0) if g1 else 0.0
        om = np.where(j >= 0, v["M"][np.maximum(j, 0)] @ O_M[k], 0.0) if g2 else 0.0
        out[k] = cands[int(np.argmax((g + e) / 2 + beta * prior + g1 * ow + g2 * om))] in ITEMS[k].gold
    return out


def build() -> dict:
    """Brain 4 Cells 18, 19 and 21 in one pass; the weights chosen on SE07; the POOLED (hand) arrays."""
    global LID, LEMV, PLACEV, C_ALL, O_W, O_M, PARTS
    t1 = time.time()
    counts = SenseCounts.count(TRAIN)
    orng = np.random.default_rng(19)
    at_train, at_all = [around(*WHERE[it.id]) for it in TRAIN], [around(*WHERE[it.id]) for it in ITEMS]
    lemmas = sorted({x for a in at_train + at_all for x in a if x})
    LID = {x: i for i, x in enumerate(lemmas)}
    LEMV = np.exp(1j * orng.uniform(0, 2 * np.pi, (len(lemmas) + 1, ODIM))).astype(np.complex64)
    LEMV[-1] = 0                                      # no word at that place
    PLACEV = np.exp(1j * orng.uniform(0, 2 * np.pi, (len(PLACES), ODIM))).astype(np.complex64)
    gfiles = sorted(p for p in (DATA / "glosstag").rglob("*.xml") if p.name in ("noun.xml", "verb.xml", "adj.xml",
                                                                                 "adv.xml"))
    gc = read_glosstag(gfiles)
    g_items = [it for it in items_of(gc.text, WN) if any(s in SID for s in it.gold)]
    g_hand = [it for it in g_items if it.id in gc.manual]
    WHERE.update({t.instance: (toks, k) for toks in gc.text.sentences.values() for k, t in enumerate(toks)
                  if t.instance})
    newlem = sorted({x for it in g_items for x in around(*WHERE[it.id]) if x} - set(LID))
    LID.update({x: len(lemmas) + i for i, x in enumerate(newlem)})
    lemmas = lemmas + newlem
    LEMV = np.vstack([LEMV[:-1], np.exp(1j * np.random.default_rng(21).uniform(0, 2 * np.pi, (len(newlem), ODIM)))
                      .astype(np.complex64), np.zeros((1, ODIM), np.complex64)])
    print(f"  GLOSS CORPUS ({time.time() - t1:.0f}s): {len(gc.text.sentences):,} definitions and examples; "
          f"{len(g_hand):,} words tagged by hand; {len(lemmas):,} lemmas at places")
    t1 = time.time()
    semcor = stored(TRAIN, contexts(TRAIN))
    hand = stored(g_hand, contexts(g_hand))
    only, pooled = combine([semcor]), combine([semcor, hand])
    del semcor, hand
    C_ALL = contexts(ITEMS)
    at = [around(*WHERE[it.id]) for it in ITEMS]
    O_W, O_M = cunit(words_at_places(ids_of(at))), unit(meanings_at_places(at))
    PARTS = parts(lambda lem, s: counts.tagged[(lem, s)])
    print(f"  STORED EXAMPLES ({time.time() - t1:.0f}s): synsets with examples, SemCor {only['n']:,}; SemCor and "
          f"gloss by hand {pooled['n']:,}")
    dev_b = {b: int(right(only, b, 0, 0, DEV).sum()) for b in BETAS}
    beta = max(BETAS, key=lambda b: (round(100 * dev_b[b] / DEV.sum(), 6), -b))
    dev_g = {(a, b): int(right(only, beta, a, b, DEV).sum()) for a in GRID for b in GRID}
    g1, g2 = max(dev_g, key=lambda ab: (round(100 * dev_g[ab] / DEV.sum(), 6), -(ab[0] + ab[1])))
    print("  BETA on SE07 (words right of 455): " + ", ".join(f"{b:g} -> {n}" for b, n in dev_b.items()) +
          f"; chosen {beta:g}\n  g1, g2 on SE07: best {g1:g}, {g2:g} -> {dev_g[g1, g2]} (no order: {dev_g[0, 0]})")
    keys = sorted(counts.tagged)
    return dict(exv=pooled["EXV"], w=pooled["W"], m=pooled["M"], pos=pooled["pos"].astype(np.int32),
                n=np.array(pooled["n"]), lemmas=np.array(lemmas), lemv=LEMV, placev=PLACEV,
                weights=np.array([beta, g1, g2]), dev_both=np.array(dev_b[beta]), dev_all=np.array(dev_g[g1, g2]),
                tagged_lemma=np.array([k[0] for k in keys]), tagged_synset=np.array([k[1] for k in keys]),
                tagged_n=np.array([counts.tagged[k] for k in keys], np.int64))


DEPENDS3 = dict(what="Brain 4 Cells 18, 19 and 21: POOLED (hand), rebuilt in Brain 5 Cell 3",
                vectors=ARTS.key("vectors", DEPENDS), places=[ODIM, list(PLACES), list(MPLACES)], seeds=[19, 21],
                betas=list(BETAS), grid=list(GRID),
                data={r["file"]: r["sha256"] for r in RECORD if r["source"] in ("wsd", "wordnet", "glosstag")})
MODEL, BUILT3 = ARTS.get("pooled_hand", DEPENDS3, build)
BETA, G1, G2 = (float(x) for x in MODEL["weights"])
POOLED = dict(EXV=MODEL["exv"], W=MODEL["w"], M=MODEL["m"], pos=MODEL["pos"], n=int(MODEL["n"]))
if not BUILT3:
    LID = {str(x): i for i, x in enumerate(MODEL["lemmas"].tolist())}
    LEMV, PLACEV = MODEL["lemv"], MODEL["placev"]
    C_ALL = contexts(ITEMS)
    at = [around(*WHERE[it.id]) for it in ITEMS]
    O_W, O_M = cunit(words_at_places(ids_of(at))), unit(meanings_at_places(at))
    TAGGED = {(str(a), str(b)): int(n) for a, b, n in zip(MODEL["tagged_lemma"].tolist(),
                                                          MODEL["tagged_synset"].tolist(), MODEL["tagged_n"])}
    PARTS = parts(lambda lem, s: TAGGED.get((lem, s), 0))
DEV_POOLED = int(right(POOLED, BETA, G1, G2, DEV).sum())
print(f"\n  MODEL {'built and saved' if BUILT3 else 'loaded'}: POOLED (hand), {POOLED['n']:,} synsets with stored "
      f"examples; beta {BETA:g}, g1 {G1:g}, g2 {G2:g} ({time.time() - T0:.0f}s)")
print(f"  SE07, words right of {int(DEV.sum())}: BOTH + PRIOR {int(MODEL['dev_both'])} "
      f"({100 * int(MODEL['dev_both']) / DEV.sum():.1f}%), ALL {int(MODEL['dev_all'])} "
      f"({100 * int(MODEL['dev_all']) / DEV.sum():.1f}%), POOLED (hand) {DEV_POOLED} "
      f"({100 * DEV_POOLED / DEV.sum():.1f}%); Brain 4: 62.2, 64.6, 63.5")

print("\n  PREDICTIONS vs RESULTS")
for key, value in (("B5-S1", BETA), ("B5-S2", G1), ("B5-S3", G2), ("B5-S4", int(MODEL["dev_both"])),
                   ("B5-S5", int(MODEL["dev_all"])), ("B5-S6", DEV_POOLED), ("B5-S7", POOLED["n"])):
    LEDGER.result(key, value)
LEDGER.result("B5-T4" if BUILT3 else "B5-T5", round(time.time() - T0))
if "sync" in globals():
    sync("Cell 3")
print(f"\n  ({time.time() - T0:.0f}s)")
print("  LAST LINE OF MC AI BRAIN 5 CELL 3")


MC AI BRAIN 5 - CELL 3: WHICH MEANING - POOLED (hand)

  TEXT (7s): 226,036 SemCor words; 7,253 words in ALL, 455 of them SE07 (practice); the other 6,798 are TEST4 and are not scored here
  GLOSS CORPUS (32s): 169,336 definitions and examples; 290,582 words tagged by hand; 37,927 lemmas at places
  STORED EXAMPLES (36s): synsets with examples, SemCor 25,942; SemCor and gloss by hand 36,615
  BETA on SE07 (words right of 455): 0 -> 210, 0.01 -> 236, 0.02 -> 251, 0.05 -> 279, 0.1 -> 283, 0.2 -> 277, 0.5 -> 268; chosen 0.1
  g1, g2 on SE07: best 0.5, 0.25 -> 294 (no order: 283)

  MODEL built and saved: POOLED (hand), 36,615 synsets with stored examples; beta 0.1, g1 0.5, g2 0.25 (78s)
  SE07, words right of 455: BOTH + PRIOR 283 (62.2%), ALL 294 (64.6%), POOLED (hand) 289 (63.5%); Brain 4: 62.2, 64.6, 63.5

  PREDICTIONS vs RESULTS
    B5-S1 beta chosen on SE07: 0.1 (predicted between 0.1 and 0.1) -> HELD
    B5-S2 g1 chosen on SE07: 0.5 (predicted between 0.5 and 0.5) -> HELD
    B5-S3

In [7]:
# =====================================================================
# MC AI BRAIN 5 - CELL 4: THE THESAURUS CHECK - DOES A WORD THAT COULD
# STAND IN FOR EACH MEANING FIT THE SENTENCE?
# Needs Cells 0-3 in the same session. No internet, nothing trained.
# Mark's process: look the word up; if a meaning's definition does not
# fit, take a thesaurus word for it and check that word against the
# sentence. The published form is the substitution test with WordNet
# relatives (Leacock, Chodorow & Miller 1998).
# RELATIVES of a candidate synset: the one-word lemmas of the synset and
#   of the synsets one hypernym, hyponym or similar-to link away, the
#   target word itself left out, each with a word vector (Cell 2).
#   ALL: every relative; MONO: only relatives with one WordNet meaning
# FIT of a relative, from counted text only:
#   CONTEXT  cosine of its word vector with the sentence's context vector
#   PLACES   mean cosine with the words 1 and 2 places either side
#   a meaning's SUB = its best-fitting relative's fit; a meaning with no
#   relative gets the mean of the other candidates' SUB
# SCORE    POOLED (hand)'s score (Cell 3) + g3 x SUB
# PRACTICE SET  Brain 4 Cell 23's: every 5th SemCor document (sorted ids)
#   held out (45,727 words); the model rebuilt from the other documents
#   and the hand-tagged gloss examples, with Cell 3's weights. Brain 4:
#   73.2% right. The variant (CONTEXT or PLACES, ALL or MONO) and g3 (0.25,
#   0.5, 1, 2, 4) are chosen here, most words right, ties to smaller g3
# TEST4    on Kaggle only (never in the lab): POOLED (hand) scored again
#   under Brain 4's own name for it (the same model: not a new look); the
#   chosen variant once (one new look) only if B5-U1 and B5-U2 both held
# NOT BLIND  the lab ran this cell on GitHub (practice data only) before
#   Kaggle; the predictions below were committed to the repository first
#   (commit 7027a01). The gain did not hold there (+0.04 points), so the
#   rule above, added after that run, keeps the variant off TEST4
# PREDICTIONS (written before running)
#   B5-H1 held-out SemCor, POOLED (hand) from the other documents: between
#         73.15% and 73.25% right (Brain 4: 73.2)
#   B5-U1 chosen variant minus POOLED (hand), held-out SemCor: >= 0.2 points
#   B5-U2 low end of that gain's 95% interval (documents resampled): >= 0
#   B5-U3 chosen variant minus POOLED (hand), SE07: >= 0 points
#   B5-U4 TEST4 words right, POOLED (hand) scored again: 4,853 (Brain 4's
#         71.4%)
#   B5-U5 chosen variant minus POOLED (hand), TEST4: >= 0 points
#   B5-T6 the cell, when it builds: under 600 seconds
#   B5-T7 the cell, when it loads: under 240 seconds
# =====================================================================
import os, time
from collections import Counter

import numpy as np

print("=" * 64); print("MC AI BRAIN 5 - CELL 4: THE THESAURUS CHECK"); print("=" * 64)
T0 = time.time()
for need in ("POOLED", "PARTS", "stored", "combine", "contexts", "LID", "BETA"):
    if need not in globals():
        raise RuntimeError(f"run Cells 0-3 first ({need} is missing)")
from mcai.core import at_least, at_most, between
from mcai.sense import SenseCounts, candidates, items_of
from mcai.stats import paired_bootstrap
from mcai.words import read_glosstag

LAB = os.environ.get("MC_AI_LAB") == "1"
for key, claim, rule in (("B5-H1", "held-out SemCor % right, POOLED (hand) from the other documents",
                          between(73.15, 73.25)),
                         ("B5-U1", "chosen substitution variant minus POOLED (hand), held-out SemCor, points",
                          at_least(0.2)),
                         ("B5-U2", "low end of that gain's 95% interval, documents resampled, points", at_least(0.0)),
                         ("B5-U3", "chosen substitution variant minus POOLED (hand), SE07, points", at_least(0.0)),
                         ("B5-U4", "TEST4 words right of 6,798, POOLED (hand) scored again", between(4853, 4853)),
                         ("B5-U5", "chosen substitution variant minus POOLED (hand), TEST4, points", at_least(0.0)),
                         ("B5-T6", "Cell 4 seconds, building", at_most(600)),
                         ("B5-T7", "Cell 4 seconds, loading", at_most(240))):
    LEDGER.predict(key, claim, rule)
VARIANTS = ("CONTEXT, ALL", "CONTEXT, MONO", "PLACES, ALL", "PLACES, MONO")
G3S = (0.25, 0.5, 1, 2, 4)

# ---------------- the practice set: held-out SemCor documents ----------------
t1 = time.time()
DOCS = sorted({it.doc for it in TRAIN})
HELD = set(DOCS[::5])
HOLD_I = np.array([i for i, it in enumerate(TRAIN) if it.doc in HELD])
STORE_I = np.array([i for i, it in enumerate(TRAIN) if it.doc not in HELD])
C_TRAIN = contexts(TRAIN)


def build_store() -> dict:
    """POOLED (hand) from the store documents and the hand-tagged gloss examples (Brain 4 Cell 23)."""
    gfiles = sorted(p for p in (DATA / "glosstag").rglob("*.xml") if p.name in ("noun.xml", "verb.xml", "adj.xml",
                                                                                 "adv.xml"))
    gc = read_glosstag(gfiles)
    g_hand = [it for it in items_of(gc.text, WN) if any(s in SID for s in it.gold) and it.id in gc.manual]
    WHERE.update({t.instance: (toks, k) for toks in gc.text.sentences.values() for k, t in enumerate(toks)
                  if t.instance})
    store_items = [TRAIN[i] for i in STORE_I]
    model = combine([stored(store_items, C_TRAIN[STORE_I]), stored(g_hand, contexts(g_hand))])
    counts = SenseCounts.count(store_items).tagged
    keys = sorted(counts)
    return dict(exv=model["EXV"], w=model["W"], m=model["M"], pos=model["pos"].astype(np.int32),
                n=np.array(model["n"]), tagged_lemma=np.array([k[0] for k in keys]),
                tagged_synset=np.array([k[1] for k in keys]), tagged_n=np.array([counts[k] for k in keys], np.int64))


DEPENDS4 = dict(what="POOLED (hand) without every 5th SemCor document (Brain 4 Cell 23), Brain 5 Cell 4",
                model=ARTS.key("pooled_hand", DEPENDS3), held="sorted documents [::5]")
SM, BUILT4 = ARTS.get("pooled_hand_store", DEPENDS4, build_store)
STORE = dict(EXV=SM["exv"], W=SM["w"], M=SM["m"], pos=SM["pos"])
SCOUNT = Counter({(str(a), str(b)): int(n) for a, b, n in zip(SM["tagged_lemma"].tolist(),
                                                               SM["tagged_synset"].tolist(), SM["tagged_n"])})
print(f"\n  PRACTICE SET ({time.time() - t1:.0f}s; store model {'built and saved' if BUILT4 else 'loaded'}): "
      f"{len(DOCS)} SemCor documents, {len(HELD)} held out ({len(HOLD_I):,} words); the model from the other "
      f"{len(DOCS) - len(HELD)} and the gloss examples: {int(SM['n']):,} synsets with stored examples")

# ---------------- one record per word: what the model and the thesaurus check need ----------------
NEAR = (1, 2, 3, 4)                                       # places -2, -1, +1, +2 in around()'s order


def records(items, ctxs, arounds, model, counts_of, cands_ids=None) -> list:
    out = []
    ow_all, om_all = cunit(words_at_places(ids_of(arounds))), unit(meanings_at_places(arounds))
    for k, it in enumerate(items):
        if cands_ids is None:
            cands = candidates(WN, it)
            ids = np.array([SID[s] for s in cands])
            n = np.array([counts_of(it.lemma, s) for s in cands], float)
            g, prior = SYNV[ids] @ ctxs[k], np.log((n + 1) / (n.sum() + len(cands)))
        else:
            cands, ids, g, prior = cands_ids[k]
        e = model["EXV"][ids] @ ctxs[k]
        j = model["pos"][ids]
        ow = np.where(j >= 0, (model["W"][np.maximum(j, 0)] @ np.conj(ow_all[k])).real, 0.0)
        om = np.where(j >= 0, model["M"][np.maximum(j, 0)] @ om_all[k], 0.0)
        near = [WROW[arounds[k][p]] for p in NEAR if arounds[k][p] in WROW]
        out.append((cands, (g + e) / 2 + BETA * prior + G1 * ow + G2 * om, ctxs[k], near, it))
    return out


t1 = time.time()
HOLD_ITEMS = [TRAIN[i] for i in HOLD_I]
HOLD = records(HOLD_ITEMS, C_TRAIN[HOLD_I], [around(*WHERE[it.id]) for it in HOLD_ITEMS], STORE,
               lambda lem, s: SCOUNT[(lem, s)])
ALLR = records(ITEMS, C_ALL, [around(*WHERE[it.id]) for it in ITEMS], POOLED, None, PARTS)
print(f"  WORDS ({time.time() - t1:.0f}s): {len(HOLD):,} held-out SemCor, {len(ALLR):,} in ALL")

# ---------------- relatives and their fit ----------------
NSENSES = Counter()
for key, ss in WN.senses.items():
    NSENSES[key.rsplit(".", 1)[0]] += len(ss)
LINKS, REL = ("@", "@i", "~", "~i", "&"), {}


def relatives(sid: str) -> dict:
    """{word: vocabulary row} for the one-word lemmas of the synset and of the synsets one link away."""
    if sid not in REL:
        s = WN.synsets[sid]
        near = [s] + [WN.synsets[p.target] for p in s.pointers if p.symbol in LINKS]
        REL[sid] = {w: WROW[w] for w in {x.lower() for n in near for x in n.words if x.isalpha()} if w in WROW}
    return REL[sid]


def sub(rec) -> np.ndarray:
    """4 x candidates: SUB for CONTEXT ALL, CONTEXT MONO, PLACES ALL, PLACES MONO."""
    cands, _, ctx, near, it = rec
    out = np.full((4, len(cands)), np.nan)
    nv = SVD[near] if near else None
    for i, s in enumerate(cands):
        rel = relatives(s)
        for m, mono in enumerate((False, True)):
            rows = [r for w, r in rel.items() if w != it.lemma.lower() and (not mono or NSENSES[w] == 1)]
            if rows:
                R = SVD[rows]
                out[m, i] = float((R @ ctx).max())
                if nv is not None:
                    out[2 + m, i] = float((R @ nv.T).mean(1).max())
    for f in range(4):
        ok = ~np.isnan(out[f])
        out[f, ~ok] = out[f, ok].mean() if ok.any() else 0.0
    return out


t1 = time.time()
SUB_HOLD = [sub(r) for r in HOLD]
SUB_ALL = [sub(r) for r in ALLR]
print(f"  RELATIVES ({time.time() - t1:.0f}s): {len(REL):,} synsets looked at; candidates with a relative in the "
      f"vocabulary, held-out SemCor: {np.mean([np.isfinite(s).all() for s in SUB_HOLD]):.1%} of words fully covered")


def right(recs, subs, f: int, g3: float, mask=None) -> np.ndarray:
    idx = range(len(recs)) if mask is None else np.flatnonzero(mask)
    out = np.zeros(len(recs), bool)
    for k in idx:
        cands, base, _, _, it = recs[k]
        out[k] = cands[int(np.argmax(base + g3 * subs[k][f] if g3 else base))] in it.gold
    return out


BASE_H = right(HOLD, SUB_HOLD, 0, 0)
GRID4 = {(f, g3): int(right(HOLD, SUB_HOLD, f, g3).sum()) for f in range(4) for g3 in G3S}
F, G3 = max(GRID4, key=lambda fg: (GRID4[fg], -fg[1], -fg[0]))
BEST_H = right(HOLD, SUB_HOLD, F, G3)
DOC_H = np.array([it.doc for it in HOLD_ITEMS])
D_H = paired_bootstrap(BEST_H, BASE_H, groups=DOC_H)
BASE_D, BEST_D = right(ALLR, SUB_ALL, 0, 0, DEV), right(ALLR, SUB_ALL, F, G3, DEV)
print(f"\n  HELD-OUT SEMCOR, % right of {len(HOLD):,} (POOLED (hand): {100 * BASE_H.mean():.2f}; Brain 4: 73.2)")
print("    variant          " + "".join(f"{'g3 ' + format(g, 'g'):>9}" for g in G3S))
for f, name in enumerate(VARIANTS):
    print(f"    {name:<17}" + "".join(f"{100 * GRID4[f, g] / len(HOLD):>9.2f}" for g in G3S))
print(f"  CHOSEN: {VARIANTS[F]}, g3 {G3:g}: {100 * BEST_H.mean():.2f} on held-out SemCor, "
      f"{100 * (BEST_H.mean() - BASE_H.mean()):+.2f} points (95% interval {100 * D_H[1]:+.2f} to {100 * D_H[2]:+.2f}); "
      f"SE07 {int(BEST_D.sum())} of {int(DEV.sum())} (POOLED (hand) {int(BASE_D.sum())})")
CHANGED = np.flatnonzero(BEST_H != BASE_H)
print(f"    words whose answer changed: {len(CHANGED):,}; now right {int(BEST_H[CHANGED].sum()):,}, now wrong "
      f"{int(BASE_H[CHANGED].sum()):,}")
for k in CHANGED[:4]:
    cands, base, _, _, it = HOLD[k]
    a, b = cands[int(np.argmax(base))], cands[int(np.argmax(base + G3 * SUB_HOLD[k][F]))]
    print(f"    {it.lemma}: {a} ({WN.synsets[a].definition[:40]}) -> {b} ({WN.synsets[b].definition[:40]}); "
          f"{'right' if b in it.gold else 'wrong'} now")

# ---------------- TEST4: Kaggle only ----------------
TEST4 = ~DEV
results = [("B5-H1", round(100 * BASE_H.mean(), 4)), ("B5-U1", round(100 * (BEST_H.mean() - BASE_H.mean()), 4)),
           ("B5-U2", round(100 * D_H[1], 4)), ("B5-U3", round(100 * (BEST_D.sum() - BASE_D.sum()) / DEV.sum(), 4))]
if LAB:
    print("\n  TEST4: not scored (lab run)")
else:
    LEDGER.look("TEST4 (WSD)", "ALL with stored examples POOLED (hand): SemCor + Princeton gloss corpus (c7b8318a)")
    BASE_T = right(ALLR, SUB_ALL, 0, 0, TEST4)
    results.append(("B5-U4", int(BASE_T.sum())))
    print(f"\n  TEST4 ({int(TEST4.sum()):,} words): POOLED (hand) scored again: {int(BASE_T.sum()):,} right "
          f"({100 * BASE_T[TEST4].mean():.2f}; Brain 4: 71.39); looks {LEDGER.looks('TEST4 (WSD)')}")
    if 100 * (BEST_H.mean() - BASE_H.mean()) >= 0.2 and 100 * D_H[1] >= 0:
        n_looks = LEDGER.look("TEST4 (WSD)", f"POOLED (hand) + SUBSTITUTION {VARIANTS[F]}, g3 {G3:g}, chosen on "
                                             f"held-out SemCor (Brain 5 Cell 4)")
        BEST_T = right(ALLR, SUB_ALL, F, G3, TEST4)
        D_T = paired_bootstrap(BEST_T[TEST4], BASE_T[TEST4], groups=np.array([it.doc for it in ITEMS])[TEST4])
        print(f"    {VARIANTS[F]} g3 {G3:g}: {int(BEST_T.sum()):,} right ({100 * BEST_T[TEST4].mean():.2f}); "
              f"difference {100 * D_T[0]:+.2f} points (95% interval {100 * D_T[1]:+.2f} to {100 * D_T[2]:+.2f}); "
              f"looks now {n_looks}")
        results.append(("B5-U5", round(100 * D_T[0], 4)))
    else:
        print("    the substitution variant is not scored on TEST4: its gain did not hold on the practice set")
print("\n  PREDICTIONS vs RESULTS")
for key, value in results:
    LEDGER.result(key, value)
LEDGER.result("B5-T6" if BUILT4 else "B5-T7", round(time.time() - T0))
if "sync" in globals():
    sync("Cell 4")
print(f"\n  ({time.time() - T0:.0f}s)")
print("  LAST LINE OF MC AI BRAIN 5 CELL 4")


MC AI BRAIN 5 - CELL 4: THE THESAURUS CHECK

  PRACTICE SET (66s; store model built and saved): 352 SemCor documents, 71 held out (45,727 words); the model from the other 281 and the gloss examples: 34,780 synsets with stored examples
  WORDS (8s): 45,727 held-out SemCor, 7,253 in ALL
  RELATIVES (16s): 24,232 synsets looked at; candidates with a relative in the vocabulary, held-out SemCor: 100.0% of words fully covered

  HELD-OUT SEMCOR, % right of 45,727 (POOLED (hand): 73.23; Brain 4: 73.2)
    variant            g3 0.25   g3 0.5     g3 1     g3 2     g3 4
    CONTEXT, ALL         73.21    72.86    71.80    68.94    64.60
    CONTEXT, MONO        73.27    73.23    73.11    72.63    71.66
    PLACES, ALL          73.27    73.21    72.46    70.60    66.73
    PLACES, MONO         73.27    73.26    73.21    72.95    72.29
  CHOSEN: CONTEXT, MONO, g3 0.25: 73.27 on held-out SemCor, +0.04 points (95% interval -0.01 to +0.10); SE07 289 of 455 (POOLED (hand) 289)
    words whose answer ch

In [8]:
# =====================================================================
# MC AI BRAIN 5 - CELL 5: QUESTIONS - FIRST BASELINES ON ARC, OPENBOOKQA
# AND MMLU, FROM COUNTING ONLY
# Needs Cells 0-2 (Cell 3 not needed). No internet, nothing trained.
# Practice splits only: ARC-Easy and ARC-Challenge validation, OpenBookQA
# validation, MMLU validation (and ARC train, to choose two weights). No
# test split is read.
# KNOWLEDGE  Simple Wikipedia (Cell 0's copy) cut into sentences, and
#   WordNet's definitions with their examples: an index of each one's
#   words (lower case, letters only, common words left out), weighted by
#   BM25 (Robertson et al. 1994; k1 1.2, b 0.75); saved with Artifacts
# SOLVERS  each scores every answer choice; the highest is the answer
#   (a tie of k choices holding the right one earns 1/k, as ARC scores):
#   SEARCH  Clark et al. (2016)'s IR solver: the best BM25 score, for the
#           question's and the choice's words, of a sentence holding a word
#           of the question and a word of the choice not in the question
#   ASSOC   their PMI solver's idea with Cell 2's word vectors: the mean
#           cosine of the choice's words with the question's words (SVD-
#           PPMI cosines standing in for pointwise mutual information)
#   VECTOR  cosine of the question's and the choice's context vectors
#           (SIF over Cell 2's word vectors, as Cell 3's contexts)
#   ALL     the three z-scored over each question's choices and added;
#           SEARCH weight 1, ASSOC and VECTOR 0, 0.5 or 1 each, chosen on
#           the ARC train splits (most right, ties to the smaller sum)
# SURE     margin = ALL's best minus second best; accuracy on the 30% of
#   questions with the largest margins
# PUBLISHED  ARC test sets (Clark et al. 2018): IR with the 14-million-
#   sentence ARC Corpus 62.6 Easy, 20.3 Challenge. Chance: 25 on ARC
#   (some questions have 3 or 5 choices), OpenBookQA and MMLU
# PREDICTIONS (written before running)
#   B5-Q1 SEARCH, ARC-Easy validation: between 40% and 62%
#   B5-Q2 SEARCH, ARC-Challenge validation: at most 30%
#   B5-Q3 best solver, MMLU validation: between 24% and 33%
#   B5-Q4 best solver, OpenBookQA validation: between 25% and 40%
#   B5-Q5 ALL, ARC-Easy validation, 30% most sure minus all: >= 10 points
#   B5-T8 the cell, when it builds the index: under 900 seconds
#   B5-T9 the cell, when it loads the index: under 300 seconds
# =====================================================================
import re, time
from array import array

import numpy as np
import pyarrow.parquet as pq
import scipy.sparse as sp

print("=" * 64); print("MC AI BRAIN 5 - CELL 5: QUESTIONS - FIRST BASELINES"); print("=" * 64)
T0 = time.time()
for need in ("SVD", "VEC", "VFREQ", "WROW", "ARTS", "WN", "SYNLIST", "LEDGER", "RECORD"):
    if need not in globals():
        raise RuntimeError(f"run Cells 0-2 first ({need} is missing)")
from mcai.book import read_encyclopedia
from mcai.core import at_least, at_most, between

for key, claim, rule in (("B5-Q1", "SEARCH, ARC-Easy validation, % right", between(40, 62)),
                         ("B5-Q2", "SEARCH, ARC-Challenge validation, % right", at_most(30)),
                         ("B5-Q3", "best solver, MMLU validation, % right", between(24, 33)),
                         ("B5-Q4", "best solver, OpenBookQA validation, % right", between(25, 40)),
                         ("B5-Q5", "ALL, ARC-Easy validation: 30% most sure minus all, points", at_least(10)),
                         ("B5-T8", "Cell 5 seconds, building the index", at_most(900)),
                         ("B5-T9", "Cell 5 seconds, loading the index", at_most(300))):
    LEDGER.predict(key, claim, rule)
STOPW = set("""a an the of to in on at for by with from as is are was were be been being and or but if then than
that this these those it its they them their there here what which who whom whose when where why how not no do does
did done have has had having can could would should will shall may might must also into about over under after before
between during each other some any all most more many much such only own same so too very just both either neither
nor per via up down out off again once i me my we our you your he him his she her us""".split())
LETTERS = str.maketrans({c: " " for c in map(chr, range(128)) if not c.isalpha()})
SPLIT = re.compile(r"(?<=[.!?])\s+|\n+")
K1, B, MAX_LEN, MIN_WORDS, MIN_DF = 1.2, 0.75, 25, 3, 2


def content_words(text: str) -> list:
    return [t for t in text.lower().translate(LETTERS).split()
            if t.isascii() and len(t) <= MAX_LEN and t not in STOPW]


# ---------------- the index ----------------
def build_index() -> dict:
    """BM25-weighted words of every Simple Wikipedia sentence and every WordNet definition, column by column."""
    t1 = time.time()
    enc = read_encyclopedia(sorted((DATA / "encyclopedia").rglob("*.parquet"))[0])
    term, cols, lens, art, sen = {}, [], array("i"), array("i"), array("i")

    def add(ws: list, a: int, s: int, out: list) -> None:
        out.extend(term.setdefault(x, len(term)) for x in ws)
        lens.append(len(ws)); art.append(a); sen.append(s)

    for a_i, a in enumerate(enc.articles()):
        ids = []
        for s_i, s in enumerate(SPLIT.split(a.text)):
            w = content_words(s)
            if len(w) >= MIN_WORDS:
                add(w, a_i, s_i, ids)
        cols.append(np.array(ids, np.int32))
    n_wiki = len(lens)
    ids = []
    for si, syn in enumerate(SYNLIST):
        w = content_words(" ".join(x.replace("_", " ") for x in syn.words) + " " + syn.definition + " "
                          + " ".join(syn.examples))
        if w:
            add(w, -1, si, ids)
    cols.append(np.array(ids, np.int32))
    n, c = len(lens), np.concatenate(cols)
    del cols, ids
    r = np.repeat(np.arange(n, dtype=np.int32), np.frombuffer(lens, np.int32))
    tf = sp.csr_matrix((np.ones(len(r), np.float32), (r, c)), shape=(n, len(term)))
    del r, c
    tf.sum_duplicates()
    length = np.frombuffer(lens, np.int32).astype(np.float32)
    df = np.bincount(tf.indices, minlength=len(term))
    idf = np.log(1 + (n - df + 0.5) / (df + 0.5)).astype(np.float32)
    norm_ = K1 * (1 - B + B * length / length.mean())
    tf.data = (idf[tf.indices] * tf.data * (K1 + 1) / (tf.data + np.repeat(norm_, np.diff(tf.indptr))))
    tf.data = tf.data.astype(np.float32)
    keep = np.flatnonzero(df >= MIN_DF)
    X = tf.tocsc()[:, keep]
    del tf
    terms = sorted(term, key=term.get)
    print(f"  INDEX ({time.time() - t1:.0f}s): {n_wiki:,} Simple Wikipedia sentences and {n - n_wiki:,} WordNet "
          f"definitions; {len(term):,} different words, {len(keep):,} of them in >= {MIN_DF} entries and kept; "
          f"{X.nnz:,} entries")
    return dict(data=X.data, indices=X.indices, indptr=X.indptr.astype(np.int64), n=np.array(n),
                n_wiki=np.array(n_wiki), terms=np.array([terms[k] for k in keep]),
                src=np.stack([np.frombuffer(art, np.int32), np.frombuffer(sen, np.int32)], 1))


DEPENDS5 = dict(what="sentence index (BM25) of Simple Wikipedia and WordNet definitions, Brain 5 Cell 5",
                version=1, k1=K1, b=B, stopwords=sorted(STOPW), max_len=MAX_LEN, min_words=MIN_WORDS,
                min_df=MIN_DF, split=SPLIT.pattern,
                data={r["file"]: r["sha256"] for r in RECORD if r["source"] in ("encyclopedia", "wordnet")})
IDX, BUILT5 = ARTS.get("sentence_index", DEPENDS5, build_index)
N_DOCS = int(IDX["n"])
XI, XP, XD = IDX["indices"], IDX["indptr"], IDX["data"]
TERM = {str(w): i for i, w in enumerate(IDX["terms"].tolist())}
print(f"\n  INDEX {'built and saved' if BUILT5 else 'loaded'}: {N_DOCS:,} entries to search, {len(TERM):,} words "
      f"({time.time() - T0:.0f}s)")

# ---------------- the questions ----------------
def table_rows(path) -> list:
    return pq.read_table(path).to_pylist()


def arc(split: str, part: str) -> list:
    out = []
    for r in table_rows(DATA / "arc" / part / f"{split}-00000-of-00001.parquet"):
        lab = list(r["choices"]["label"])
        out.append((r["question"], list(r["choices"]["text"]), lab.index(r["answerKey"])))
    return out


QSETS = {"ARC-Easy validation": arc("validation", "ARC-Easy"),
         "ARC-Challenge validation": arc("validation", "ARC-Challenge"),
         "OpenBookQA validation": [(r["question_stem"], list(r["choices"]["text"]),
                                    list(r["choices"]["label"]).index(r["answerKey"]))
                                   for r in table_rows(DATA / "obqa" / "main" / "validation-00000-of-00001.parquet")],
         "MMLU validation": [(r["question"], list(r["choices"]), int(r["answer"]))
                             for r in table_rows(DATA / "mmlu" / "all" / "validation-00000-of-00001.parquet")]}
TRAIN5 = arc("train", "ARC-Easy") + arc("train", "ARC-Challenge")
print("  QUESTIONS: " + ", ".join(f"{k} {len(v):,}" for k, v in QSETS.items()) + f"; ARC train {len(TRAIN5):,}")

# ---------------- the solvers ----------------
SIF5 = (0.001 / (0.001 + VFREQ / VFREQ.sum())).astype(np.float32)
PC5 = VEC["pc"]


def postings(ids: list):
    if not ids:
        return np.zeros(0, np.int64), np.zeros(0, np.float32)
    return (np.concatenate([XI[XP[t]:XP[t + 1]] for t in ids]), np.concatenate([XD[XP[t]:XP[t + 1]] for t in ids]))


def vec(ws: list):
    rows = [WROW[w] for w in ws if w in WROW]
    if not rows:
        return None
    v = (SIF5[rows][:, None] * SVD[rows]).sum(0)
    v -= (v @ PC5) * PC5
    n = np.linalg.norm(v)
    return v / n if n > 0 else None


def solve(question: str, choices: list) -> tuple:
    """(3 x choices: SEARCH, ASSOC, VECTOR scores; the index row of SEARCH's best sentence per choice)"""
    qw = list(dict.fromkeys(content_words(question)))
    q_ids = [TERM[w] for w in qw if w in TERM]
    qr, qv = postings(q_ids)
    sq = np.bincount(qr, weights=qv, minlength=N_DOCS) if len(qr) else None
    q_rows = [WROW[w] for w in qw if w in WROW]
    q_vec = vec(qw)
    out, best = np.zeros((3, len(choices))), []
    for j, ch in enumerate(choices):
        cw = [w for w in dict.fromkeys(content_words(ch)) if w not in qw]
        r, v = postings([TERM[w] for w in cw if w in TERM])
        b = -1
        if len(r) and sq is not None:
            u, inv = np.unique(r, return_inverse=True)
            so = np.bincount(inv, weights=v)
            ok = sq[u] > 0
            if ok.any():
                k = int(np.argmax(np.where(ok, so + sq[u], -1)))
                out[0, j], b = so[k] + sq[u[k]], int(u[k])
        best.append(b)
        c_rows = [WROW[w] for w in cw if w in WROW]
        if c_rows and q_rows:
            out[1, j] = float((SVD[c_rows] @ SVD[q_rows].T).mean())
        c_vec = vec(cw or content_words(ch))
        if c_vec is not None and q_vec is not None:
            out[2, j] = float(c_vec @ q_vec)
    return out, best


def z(s: np.ndarray) -> np.ndarray:
    sd = s.std()
    return (s - s.mean()) / sd if sd > 0 else np.zeros_like(s)


def credit(scores: np.ndarray, gold: int) -> float:
    top = np.flatnonzero(np.isclose(scores, scores.max(), rtol=0, atol=1e-9))
    return 1 / len(top) if gold in top else 0.0


t1 = time.time()
SOLVED = {name: [solve(q, ch) for q, ch, _ in qs] for name, qs in QSETS.items()}
SOLVED_TRAIN = [solve(q, ch) for q, ch, _ in TRAIN5]
print(f"  SOLVED ({time.time() - t1:.0f}s): {sum(len(v) for v in SOLVED.values()) + len(SOLVED_TRAIN):,} questions")


def combined(s: np.ndarray, wa: float, wv: float) -> np.ndarray:
    return z(s[0]) + wa * z(s[1]) + wv * z(s[2])


GRID5 = {(wa, wv): sum(credit(combined(s, wa, wv), g) for (s, _), (_, _, g) in zip(SOLVED_TRAIN, TRAIN5))
         for wa in (0, 0.5, 1) for wv in (0, 0.5, 1)}
WA, WV = max(GRID5, key=lambda k: (round(GRID5[k], 6), -(k[0] + k[1])))
print(f"  ALL's weights on ARC train ({len(TRAIN5):,} questions): ASSOC {WA:g}, VECTOR {WV:g} -> "
      f"{100 * GRID5[WA, WV] / len(TRAIN5):.1f}% (SEARCH alone {100 * GRID5[0, 0] / len(TRAIN5):.1f}%)")

# ---------------- results ----------------
ACC, SURE = {}, {}
print("\n  % right (ties share credit)        SEARCH   ASSOC  VECTOR     ALL  chance   ALL, 30% most sure")
for name, qs in QSETS.items():
    sv = SOLVED[name]
    row = [100 * np.mean([credit(s[f], g) for (s, _), (_, _, g) in zip(sv, qs)]) for f in range(3)]
    allc = np.array([credit(combined(s, WA, WV), g) for (s, _), (_, _, g) in zip(sv, qs)])
    margin = np.array([np.diff(np.sort(combined(s, WA, WV))[-2:])[0] if len(s[0]) > 1 else 0 for s, _ in sv])
    top = np.argsort(-margin, kind="stable")[:max(1, int(round(0.3 * len(qs))))]
    ACC[name], SURE[name] = row + [100 * allc.mean()], 100 * allc[top].mean()
    chance = 100 * np.mean([1 / len(ch) for _, ch, _ in qs])
    print(f"    {name:<30}" + "".join(f"{x:>8.1f}" for x in ACC[name]) + f"{chance:>8.1f}   {SURE[name]:>8.1f}")


ENC5 = read_encyclopedia(sorted((DATA / "encyclopedia").rglob("*.parquet"))[0])


def sentence(row: int) -> str:
    a_i, s_i = (int(x) for x in IDX["src"][row])
    if a_i < 0:
        syn = SYNLIST[s_i]
        return f"[WordNet] {syn.words[0]}: {syn.definition}"
    return f"[Wikipedia: {ENC5.titles[a_i]}] " + SPLIT.split(ENC5.texts([a_i])[a_i])[s_i][:150]


print("\n  TWO ARC-EASY QUESTIONS, with SEARCH's best sentence for its answer")
for (q, ch, g), (s, best) in list(zip(QSETS["ARC-Easy validation"], SOLVED["ARC-Easy validation"]))[:2]:
    pick = int(np.argmax(combined(s, WA, WV)))
    said = "right" if pick == g else "wrong; right: " + ch[g][:40]
    print(f"    Q: {q[:110]}\n       answer: {ch[pick][:60]} ({said})")
    if best[pick] >= 0:
        print(f"       {sentence(best[pick])}")

print("\n  PREDICTIONS vs RESULTS")
LEDGER.result("B5-Q1", round(ACC["ARC-Easy validation"][0], 2))
LEDGER.result("B5-Q2", round(ACC["ARC-Challenge validation"][0], 2))
LEDGER.result("B5-Q3", round(max(ACC["MMLU validation"]), 2))
LEDGER.result("B5-Q4", round(max(ACC["OpenBookQA validation"]), 2))
LEDGER.result("B5-Q5", round(SURE["ARC-Easy validation"] - ACC["ARC-Easy validation"][3], 2))
LEDGER.result("B5-T8" if BUILT5 else "B5-T9", round(time.time() - T0))
if "sync" in globals():
    sync("Cell 5")
print(f"\n  ({time.time() - T0:.0f}s)")
print("  LAST LINE OF MC AI BRAIN 5 CELL 5")


MC AI BRAIN 5 - CELL 5: QUESTIONS - FIRST BASELINES
  INDEX (43s): 3,353,949 Simple Wikipedia sentences and 117,659 WordNet definitions; 503,403 different words, 260,674 of them in >= 2 entries and kept; 23,113,969 entries

  INDEX built and saved: 3,471,608 entries to search, 260,674 words (43s)
  QUESTIONS: ARC-Easy validation 570, ARC-Challenge validation 299, OpenBookQA validation 500, MMLU validation 1,531; ARC train 3,370
  SOLVED (40s): 6,270 questions
  ALL's weights on ARC train (3,370 questions): ASSOC 0, VECTOR 0.5 -> 42.2% (SEARCH alone 41.8%)

  % right (ties share credit)        SEARCH   ASSOC  VECTOR     ALL  chance   ALL, 30% most sure
    ARC-Easy validation               46.7    43.9    42.0    47.6    25.0       57.9
    ARC-Challenge validation          26.5    18.4    16.4    20.9    25.1       24.4
    OpenBookQA validation             27.6    20.8    23.5    27.5    25.0       24.0
    MMLU validation                   29.2    26.1    26.3    29.7    25.0       3

In [9]:
# =====================================================================
# MC AI BRAIN 5 - CELL 6: QUESTIONS - WORD FORMS, CHAPTERS, OPPOSITES,
# PINS AND NOT-QUESTIONS
# Needs Cells 0-2 and 5. No internet, nothing trained. Settings are
# chosen on ARC train (Easy + Challenge, 3,370 questions); the practice
# validation splits are scored once, with the chosen settings.
# WORD FORMS  each word becomes its WordNet base form (morphy: exception
#   lists, then suffix rules; noun, verb, adjective, adverb in that order)
# INDEXES  BM25 (k1 1.2, b 0.75) over base forms:
#   SENTENCES  Simple Wikipedia sentences and WordNet entries (as Cell 5)
#   CHAPTERS   whole Simple Wikipedia articles
# PER CHOICE (the question's words taken out of the choice):
#   S  sentence support: the best BM25 score of a sentence holding a word
#      of the question and a word of the choice (Cell 5's SEARCH)
#   C  chapter support: the same over whole articles
#   X  contradiction: S for the choice words' WordNet antonyms ("!"
#      links); 0 for a choice without any
#   PINNED  S = C = 0: no evidence either way
# SCORE  z(S) + wc z(C) - wx z(X), z over the question's choices
#   OPEN WORLD  a pinned choice gets the mean score of the other choices
#     (closed world, as Cell 5: no evidence scores as the lowest)
#   FLIP  if the question's last sentence asks NOT, EXCEPT or LEAST ("at
#     least" left out), the lowest score is the answer
#   wc 0/0.25/0.5/1, wx 0/0.5/1, open world and flip off/on: 48 settings
#   scored on ARC train; the most right is chosen (ties: fewest changes)
# PIN LOG  right answers that were pinned (ARC train, MMLU validation)
#   and their words in no index entry: reports/cell06_pins.txt
# PREDICTIONS (written before running; points = percentage points)
#   B5-R1 base forms alone, ARC train, points vs Cell 5's SEARCH: -1..+2
#   B5-R2 flip alone, the ARC train questions it flips, points: >= 5
#   B5-R3 chapters alone (best wc), ARC-Challenge train, points: 0..+4
#   B5-R4 open world alone, ARC train, points: -1..+1
#   B5-R5 contradiction alone (best wx), ARC train, points: -1..+1
#   B5-R6 chosen settings, ARC-Easy validation, % right: 47..55
#   B5-R7 chosen settings, ARC-Challenge validation, % right: 22..32
#   B5-R8 chosen settings, MMLU validation, % right: 26..33
#   B5-R9 chosen settings, OpenBookQA validation, % right: 25..35
#   B5-T10 the cell, building the indexes: <= 900 s
#   B5-T11 the cell, loading the indexes: <= 300 s
# =====================================================================
import re, time
from array import array
from collections import Counter

import numpy as np
import scipy.sparse as sp

print("=" * 64); print("MC AI BRAIN 5 - CELL 6: QUESTIONS - FORMS, CHAPTERS, OPPOSITES, PINS"); print("=" * 64)
T0 = time.time()
for need in ("SOLVED_TRAIN", "SOLVED", "TRAIN5", "QSETS", "STOPW", "LETTERS", "SPLIT", "MAX_LEN", "SYNLIST",
             "credit", "z", "arc", "table_rows"):
    if need not in globals():
        raise RuntimeError(f"run Cells 0-2 and 5 first ({need} is missing)")
from mcai.book import read_encyclopedia
from mcai.core import at_least, at_most, between
from mcai.sense import Gloss

for key, claim, rule in (
        ("B5-R1", "base forms alone, ARC train, points vs Cell 5 SEARCH", between(-1, 2)),
        ("B5-R2", "flip alone, ARC train questions it flips, points", at_least(5)),
        ("B5-R3", "chapters alone (best wc), ARC-Challenge train, points", between(0, 4)),
        ("B5-R4", "open world alone, ARC train, points", between(-1, 1)),
        ("B5-R5", "contradiction alone (best wx), ARC train, points", between(-1, 1)),
        ("B5-R6", "chosen settings, ARC-Easy validation, % right", between(47, 55)),
        ("B5-R7", "chosen settings, ARC-Challenge validation, % right", between(22, 32)),
        ("B5-R8", "chosen settings, MMLU validation, % right", between(26, 33)),
        ("B5-R9", "chosen settings, OpenBookQA validation, % right", between(25, 35)),
        ("B5-T10", "Cell 6 seconds, building the indexes", at_most(900)),
        ("B5-T11", "Cell 6 seconds, loading the indexes", at_most(300))):
    LEDGER.predict(key, claim, rule)
GLOSS, FORM = Gloss(WN), {}


def base(t: str) -> str:
    b = FORM.get(t)
    if b is None:
        b = FORM[t] = GLOSS.lemma(t)
    return b


def forms(text: str) -> list:
    out = []
    for t in text.lower().translate(LETTERS).split():
        if t.isascii() and len(t) <= MAX_LEN and t not in STOPW:
            b = base(t)
            if b not in STOPW:
                out.append(b)
    return out


def bm25(rows, cols, n_rows: int, n_terms: int):
    """(BM25 weight of each word in each entry as a CSC matrix, the number of entries holding each word)"""
    tf = sp.csr_matrix((np.ones(len(rows), np.float32), (rows, cols)), shape=(n_rows, n_terms))
    tf.sum_duplicates()
    length = np.asarray(tf.sum(1)).ravel()
    df = np.bincount(tf.indices, minlength=n_terms)
    idf = np.log(1 + (n_rows - df + 0.5) / (df + 0.5)).astype(np.float32)
    norm = (K1 * (1 - B + B * length / length[length > 0].mean())).astype(np.float32)
    tf.data = (idf[tf.indices] * tf.data * (K1 + 1) / (tf.data + np.repeat(norm, np.diff(tf.indptr))))
    tf.data = tf.data.astype(np.float32)
    return tf.tocsc(), df


def build6() -> dict:
    t1 = time.time()
    enc = read_encyclopedia(sorted((DATA / "encyclopedia").rglob("*.parquet"))[0])
    term, cols, lens, art, sen = {}, [], array("i"), array("i"), array("i")

    def add(ws: list, a: int, s: int, out: list) -> None:
        out.extend(term.setdefault(x, len(term)) for x in ws)
        lens.append(len(ws)); art.append(a); sen.append(s)

    for a_i, a in enumerate(enc.articles()):
        ids = []
        for s_i, s in enumerate(SPLIT.split(a.text)):
            w = forms(s)
            if len(w) >= MIN_WORDS:
                add(w, a_i, s_i, ids)
        cols.append(np.array(ids, np.int32))
    n_wiki, n_art = len(lens), enc.groups[-1]
    ids = []
    for si, syn in enumerate(SYNLIST):
        w = forms(" ".join(x.replace("_", " ") for x in syn.words) + " " + syn.definition + " "
                  + " ".join(syn.examples))
        if w:
            add(w, -1, si, ids)
    cols.append(np.array(ids, np.int32))
    c = np.concatenate(cols)
    del cols, ids
    n, ln, a_of = len(lens), np.frombuffer(lens, np.int32), np.frombuffer(art, np.int32)
    r = np.repeat(np.arange(n, dtype=np.int32), ln)
    k = int(ln[:n_wiki].sum())
    S, df = bm25(r, c, n, len(term))
    A, _ = bm25(a_of[r[:k]], c[:k], n_art, len(term))
    del r, c
    keep = np.flatnonzero(df >= MIN_DF)
    S, A = S[:, keep], A[:, keep]
    terms = sorted(term, key=term.get)
    print(f"  INDEXES ({time.time() - t1:.0f}s): {n_wiki:,} sentences and {n - n_wiki:,} WordNet entries "
          f"({S.nnz:,} entries); {n_art:,} chapters ({A.nnz:,}); {len(term):,} base forms, {len(keep):,} kept "
          f"(in >= {MIN_DF} sentences)")
    out = dict(terms=np.array([terms[j] for j in keep]), n=np.array(n), n_wiki=np.array(n_wiki),
               n_art=np.array(n_art), src=np.stack([a_of, np.frombuffer(sen, np.int32)], 1))
    for name, M in (("s", S), ("a", A)):
        out.update({f"{name}_data": M.data, f"{name}_indices": M.indices,
                    f"{name}_indptr": M.indptr.astype(np.int64)})
    return out


DEPENDS6 = dict(what="base-form BM25 indexes of sentences and chapters, Brain 5 Cell 6", version=1,
                forms="mcai.sense.Gloss.lemma (morphy; n, v, a, r)", k1=K1, b=B, stopwords=sorted(STOPW),
                max_len=MAX_LEN, min_words=MIN_WORDS, min_df=MIN_DF, split=SPLIT.pattern,
                data={r["file"]: r["sha256"] for r in RECORD if r["source"] in ("encyclopedia", "wordnet")})
IX6, BUILT6 = ARTS.get("form_indexes", DEPENDS6, build6)
TERM6 = {str(w): i for i, w in enumerate(IX6["terms"].tolist())}
MATS = {"s": (IX6["s_indices"], IX6["s_indptr"], IX6["s_data"], int(IX6["n"])),
        "a": (IX6["a_indices"], IX6["a_indptr"], IX6["a_data"], int(IX6["n_art"]))}
ANT = {}
for syn in SYNLIST:
    for p in syn.pointers:
        if p.symbol == "!" and p.source_word > 0 and p.target_word > 0:
            a = syn.words[p.source_word - 1].split("(")[0].lower()
            b = WN.synsets[p.target].words[p.target_word - 1].split("(")[0].lower()
            if "_" not in a + b:
                ANT.setdefault(a, set()).add(b)
print(f"\n  INDEXES {'built and saved' if BUILT6 else 'loaded'}: {MATS['s'][3]:,} sentences and entries, "
      f"{MATS['a'][3]:,} chapters, {len(TERM6):,} base forms; {len(ANT):,} words with WordNet antonyms "
      f"({time.time() - T0:.0f}s)")
EMPTY = (np.zeros(0, np.int64), np.zeros(0, np.float32))


def post(m: str, ids: list):
    xi, xp, xd, _ = MATS[m]
    if not ids:
        return EMPTY
    return (np.concatenate([xi[xp[t]:xp[t + 1]] for t in ids]), np.concatenate([xd[xp[t]:xp[t + 1]] for t in ids]))


def best(sq, r, v) -> tuple:
    """(best score of an entry holding a question word and one of these words, its row), or (0, -1)"""
    if sq is None or not len(r):
        return 0.0, -1
    u, inv = np.unique(r, return_inverse=True)
    so = np.bincount(inv, weights=v)
    ok = sq[u] > 0
    if not ok.any():
        return 0.0, -1
    k = int(np.argmax(np.where(ok, so + sq[u], -1)))
    return float(so[k] + sq[u[k]]), int(u[k])


def solve6(question: str, choices: list) -> np.ndarray:
    """3 x choices: S, C, X"""
    qw = list(dict.fromkeys(forms(question)))
    q_ids = [TERM6[w] for w in qw if w in TERM6]
    sq = {}
    for m in "sa":
        r, v = post(m, q_ids)
        sq[m] = np.bincount(r, weights=v, minlength=MATS[m][3]) if len(r) else None
    out = np.zeros((3, len(choices)))
    for j, ch in enumerate(choices):
        cw = [w for w in dict.fromkeys(forms(ch)) if w not in qw]
        c_ids = [TERM6[w] for w in cw if w in TERM6]
        out[0, j] = best(sq["s"], *post("s", c_ids))[0]
        out[1, j] = best(sq["a"], *post("a", c_ids))[0]
        anti = sorted({TERM6[a] for w in cw for a in ANT.get(w, ()) if a in TERM6 and a not in qw})
        if anti:
            out[2, j] = best(sq["s"], *post("s", anti))[0]
    return out


NEG = re.compile(r"\b(not|except|least)\b", re.I)


def asks_not(q: str) -> bool:
    last = ([p for p in re.split(r"(?<=[.!?])\s+", q.strip()) if p] or [q])[-1]
    return bool(NEG.search(re.sub(r"\bat least\b", " ", last, flags=re.I)))


def score6(s: np.ndarray, wc: float, wx: float, open_world: bool, flip: bool, neg: bool) -> np.ndarray:
    f = z(s[0]) + wc * z(s[1]) - wx * z(s[2])
    pinned = (s[0] == 0) & (s[1] == 0)
    if open_world and pinned.any() and (~pinned).any():
        f = f.copy()
        f[pinned] = f[~pinned].mean()
    return -f if flip and neg else f


t1 = time.time()
SOL6_TRAIN = [solve6(q, ch) for q, ch, _ in TRAIN5]
NEG_TRAIN = [asks_not(q) for q, _, _ in TRAIN5]
SOL6 = {name: [solve6(q, ch) for q, ch, _ in qs] for name, qs in QSETS.items()}
print(f"  SOLVED ({time.time() - t1:.0f}s): {len(TRAIN5) + sum(len(v) for v in QSETS.values()):,} questions")
N_EASY = len(arc("train", "ARC-Easy"))
EASY, CHAL, EVERY = range(N_EASY), range(N_EASY, len(TRAIN5)), range(len(TRAIN5))


def acc(st: tuple, idx=EVERY) -> float:
    return 100 * np.mean([credit(score6(SOL6_TRAIN[i], *st, NEG_TRAIN[i]), TRAIN5[i][2]) for i in idx])


def acc5(idx=EVERY) -> float:
    return 100 * np.mean([credit(SOLVED_TRAIN[i][0][0], TRAIN5[i][2]) for i in idx])


BASE = (0, 0, False, False)
GRID6 = {(wc, wx, o, f): acc((wc, wx, o, f)) for wc in (0, 0.25, 0.5, 1) for wx in (0, 0.5, 1)
         for o in (False, True) for f in (False, True)}
CHOSEN = max(GRID6, key=lambda k: (round(GRID6[k], 6), -((k[0] > 0) + (k[1] > 0) + k[2] + k[3])))
WC1 = max((0.25, 0.5, 1), key=lambda w: (round(GRID6[w, 0, False, False], 6), -w))
WX1 = max((0.5, 1), key=lambda w: (round(GRID6[0, w, False, False], 6), -w))
FLIPPED = [i for i in EVERY if NEG_TRAIN[i]]
print("\n  ARC TRAIN, % right (ties share credit)              all    Easy  Chall.")
for name, st in (("Cell 5 SEARCH (words as written)", None), ("base forms", BASE),
                 ("base forms + flip", (0, 0, False, True)),
                 (f"base forms + chapters (wc {WC1:g})", (WC1, 0, False, False)),
                 ("base forms + open world", (0, 0, True, False)),
                 (f"base forms + contradiction (wx {WX1:g})", (0, WX1, False, False)),
                 (f"chosen: wc {CHOSEN[0]:g}, wx {CHOSEN[1]:g}, open world {'on' if CHOSEN[2] else 'off'}, "
                  f"flip {'on' if CHOSEN[3] else 'off'}", CHOSEN)):
    vals = [acc5(ix) if st is None else acc(st, ix) for ix in (EVERY, EASY, CHAL)]
    print(f"    {name:<46}" + "".join(f"{v:>8.1f}" for v in vals))
print(f"    flip changes {len(FLIPPED)} questions (NOT, EXCEPT, LEAST): {acc(BASE, FLIPPED):.1f}% -> "
      f"{acc((0, 0, False, True), FLIPPED):.1f}%")

ACC6, SURE6 = {}, {}
print("\n  VALIDATION, chosen settings           % right  chance  Cell 5 SEARCH  30% most sure")
for name, qs in QSETS.items():
    sc = [score6(s, *CHOSEN, asks_not(q)) for s, (q, _, _) in zip(SOL6[name], qs)]
    right = np.array([credit(f, g) for f, (_, _, g) in zip(sc, qs)])
    margin = np.array([np.diff(np.sort(f)[-2:])[0] if len(f) > 1 else 0 for f in sc])
    top = np.argsort(-margin, kind="stable")[:max(1, int(round(0.3 * len(qs))))]
    ACC6[name], SURE6[name] = 100 * right.mean(), 100 * right[top].mean()
    chance = 100 * np.mean([1 / len(ch) for _, ch, _ in qs])
    old = 100 * np.mean([credit(s[0], g) for (s, _), (_, _, g) in zip(SOLVED[name], qs)])
    print(f"    {name:<34}{ACC6[name]:>9.1f}{chance:>8.1f}{old:>15.1f}{SURE6[name]:>15.1f}")

# ---------------- pin log ----------------
MMLU_SUBJ = [r["subject"] for r in table_rows(DATA / "mmlu" / "all" / "validation-00000-of-00001.parquet")]
lines = ["MC AI Brain 5 Cell 6 pin log: right answers with no evidence (S = C = 0), and their words in no index",
         "entry (unknown words). Words only; no question text.", ""]
for name, qs, sols in (("ARC train", TRAIN5, SOL6_TRAIN), ("MMLU validation", QSETS["MMLU validation"],
                                                         SOL6["MMLU validation"])):
    pins = [i for i, ((_, ch, g), s) in enumerate(zip(qs, sols)) if s[0, g] == 0 and s[1, g] == 0 and forms(ch[g])]
    unknown = Counter(w for i in pins for w in set(forms(qs[i][1][qs[i][2]])) if w not in TERM6)
    has_unknown = sum(any(w not in TERM6 for w in forms(qs[i][1][qs[i][2]])) for i in pins)
    head = (f"{name}: {len(pins)} of {len(qs)} right answers pinned; {has_unknown} hold an unknown word, "
            f"{len(pins) - has_unknown} have only known words with no link found")
    lines += [head, "  unknown words, most often first: " + ", ".join(f"{w} {k}" for w, k in unknown.most_common(40))]
    print(f"\n  PINS {head}")
    print("    unknown words: " + ", ".join(w for w, _ in unknown.most_common(15)))
    if name.startswith("MMLU"):
        by = Counter(MMLU_SUBJ[i] for i in pins)
        tot = Counter(MMLU_SUBJ)
        worst = sorted(by, key=lambda s: -by[s] / tot[s])[:8]
        lines.append("  subjects with the most pinned right answers: "
                     + ", ".join(f"{s} {by[s]}/{tot[s]}" for s in worst))
        print("    most pinned subjects: " + ", ".join(f"{s} {by[s]}/{tot[s]}" for s in worst))
    lines.append("")
(WORK / "reports").mkdir(exist_ok=True)
(WORK / "reports" / "cell06_pins.txt").write_text("\n".join(lines) + "\n")

print("\n  PREDICTIONS vs RESULTS")
LEDGER.result("B5-R1", round(acc(BASE) - acc5(), 2))
LEDGER.result("B5-R2", round(acc((0, 0, False, True), FLIPPED) - acc(BASE, FLIPPED), 2))
LEDGER.result("B5-R3", round(acc((WC1, 0, False, False), CHAL) - acc(BASE, CHAL), 2))
LEDGER.result("B5-R4", round(acc((0, 0, True, False)) - acc(BASE), 2))
LEDGER.result("B5-R5", round(acc((0, WX1, False, False)) - acc(BASE), 2))
LEDGER.result("B5-R6", round(ACC6["ARC-Easy validation"], 2))
LEDGER.result("B5-R7", round(ACC6["ARC-Challenge validation"], 2))
LEDGER.result("B5-R8", round(ACC6["MMLU validation"], 2))
LEDGER.result("B5-R9", round(ACC6["OpenBookQA validation"], 2))
LEDGER.result("B5-T10" if BUILT6 else "B5-T11", round(time.time() - T0))
if "sync" in globals():
    sync("Cell 6")
print(f"\n  ({time.time() - T0:.0f}s)")
print("  LAST LINE OF MC AI BRAIN 5 CELL 6")


MC AI BRAIN 5 - CELL 6: QUESTIONS - FORMS, CHAPTERS, OPPOSITES, PINS
  INDEXES (55s): 3,353,322 sentences and 117,659 WordNet entries (23,002,333 entries); 241,787 chapters (12,980,610); 472,951 base forms, 238,109 kept (in >= 2 sentences)

  INDEXES built and saved: 3,470,981 sentences and entries, 241,787 chapters, 238,109 base forms; 6,159 words with WordNet antonyms (56s)
  SOLVED (69s): 6,270 questions

  ARC TRAIN, % right (ties share credit)              all    Easy  Chall.
    Cell 5 SEARCH (words as written)                  41.8    48.8    27.6
    base forms                                        42.3    49.8    27.3
    base forms + flip                                 42.5    49.8    27.9
    base forms + chapters (wc 0.5)                    45.0    53.2    28.4
    base forms + open world                           42.3    49.8    27.2
    base forms + contradiction (wx 0.5)               41.2    48.8    26.0
    chosen: wc 0.5, wx 0, open world off, flip on     45.2    53

In [10]:
# =====================================================================
# MC AI BRAIN 5 - CELL 7A: FEVER - PAGES FOR EACH CLAIM, BY TITLE
# Needs Cells 0-2, 1B, 5 and 6. No internet, nothing trained. Test files
# (paper_test, shared_task_dev) are not read.
# CLAIMS  a train sample (3,000 per label, seed 7; fit) and paper_dev
#   (9,999; practice)
# PASS 1  every page title of wiki-pages.zip (June 2017, 5,416,537 pages)
#   as a key: lower case, letters and digits only, a closing
#   "(disambiguation)" dropped; one key may name several pages
# PAGES PER CLAIM  every run of 1-8 words of the claim that is a key; a
#   run of only common words is left out, and so is a run inside a longer
#   matched run; longer runs first, then earlier; up to 6 pages per run
#   (the page without a parenthesis first), up to 12 per claim. The
#   choice reads the claim's words only, never its label or evidence
# PASS 2  the sentences of every chosen page (the "lines" of the dump),
#   saved with Artifacts
# PREDICTIONS (written before running)
#   B5-F1 paper_dev SUPPORTS/REFUTES claims with every page of one gold
#         evidence set among their pages, %: 70-95
#   B5-T12 Cells 7A + 7B, building: <= 1500 s; B5-T13 loading: <= 300 s
# =====================================================================
import json, re, time, zipfile
from array import array

import numpy as np

print("=" * 64); print("MC AI BRAIN 5 - CELL 7A: FEVER - PAGES FOR EACH CLAIM, BY TITLE"); print("=" * 64)
T7 = time.time()
for need in ("FEVER", "FEVER_PINS", "fever_parts", "forms", "STOPW", "ARTS", "LEDGER"):
    if need not in globals():
        raise RuntimeError(f"run Cells 0-2, 1B, 5 and 6 first ({need} is missing)")
from mcai.core import at_most, between

for key, claim, rule in (("B5-F1", "paper_dev S/R claims with a whole gold evidence set's pages found, %",
                          between(70, 95)),
                         ("B5-T12", "Cells 7A + 7B seconds, building", at_most(1500)),
                         ("B5-T13", "Cells 7A + 7B seconds, loading", at_most(300))):
    LEDGER.predict(key, claim, rule)
FV_SEED, FV_PER, MAX_N, PER_RUN, PER_CLAIM = 7, 3000, 8, 6, 12
WORD = re.compile(r"[^\W_]+")
LABELS = ("SUPPORTS", "REFUTES", "NOT ENOUGH INFO")


def tkey(s: str) -> str:
    return " ".join(WORD.findall(s.casefold()))


def unbracket(s: str) -> str:
    return (s.replace("_", " ").replace("-LRB-", "(").replace("-RRB-", ")").replace("-LSB-", "[")
            .replace("-RSB-", "]").replace("-COLON-", ":"))


def claims_of(name: str) -> list:
    with open(FEVER[name], encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


def raw_pages(z: zipfile.ZipFile):
    for n in fever_parts(z):
        with z.open(n) as f:
            for line in f:
                if line.strip():
                    yield line


def page_id(line: bytes) -> str:
    if line.startswith(b'{"id": "'):
        e = line.find(b'", "', 8)
        if e > 0:
            raw = line[8:e]
            return json.loads(b'"' + raw + b'"') if b"\\" in raw else raw.decode("utf-8")
    return json.loads(line)["id"]


def gold_sets(c: dict) -> list:
    """Each evidence set as a list of (page, sentence) pairs; [] for NOT ENOUGH INFO"""
    return [[(e[2], e[3]) for e in s] for s in c["evidence"] if all(e[2] is not None for e in s)]


def build7() -> dict:
    t1 = time.time()
    train = claims_of("train.jsonl")
    rng = np.random.default_rng(FV_SEED)
    pick = sorted(int(i) for lab in LABELS
                  for i in rng.choice([j for j, c in enumerate(train) if c["label"] == lab], FV_PER, replace=False))
    dev = claims_of("paper_dev.jsonl")
    claims = [train[i]["claim"] for i in pick] + [c["claim"] for c in dev]
    keys, multi, paren = {}, [], array("b")
    with zipfile.ZipFile(FEVER["wiki-pages.zip"]) as z:
        for i, line in enumerate(raw_pages(z)):
            t = page_id(line)
            b = unbracket(t)
            p = bool(re.search(r"\([^()]*\)\s*$", b))
            paren.append(p)
            k = tkey(re.sub(r"\s*\([^()]*\)\s*$", "", b)) if p else tkey(b)
            if not k:
                continue
            v = keys.get(k)
            if v is None:
                keys[k] = i
            elif v >= 0:
                multi.append([v, i])
                keys[k] = -len(multi)
            else:
                multi[-v - 1].append(i)
    n_pages = len(paren)
    paren = np.frombuffer(paren, np.int8).astype(bool)
    print(f"  PASS 1 ({time.time() - t1:.0f}s): {n_pages:,} pages, {len(keys):,} title keys "
          f"({len(multi):,} name several pages)")
    t1 = time.time()
    cand_lists = []
    for text in claims:
        tok = WORD.findall(text.casefold())
        runs = [(i, j) for i in range(len(tok)) for j in range(i + 1, min(len(tok), i + MAX_N) + 1)
                if " ".join(tok[i:j]) in keys and not all(t in STOPW for t in tok[i:j])]
        runs = [(i, j) for i, j in runs if not any(a <= i and j <= b and (a, b) != (i, j) for a, b in runs)]
        got = []
        for i, j in sorted(runs, key=lambda r: (r[0] - r[1], r[0])):
            v = keys[" ".join(tok[i:j])]
            pages = [v] if v >= 0 else sorted(multi[-v - 1], key=lambda p: (paren[p], p))
            for p in pages[:PER_RUN]:
                if p not in got and len(got) < PER_CLAIM:
                    got.append(p)
        cand_lists.append(got)
    del keys, multi
    kept = sorted({p for g in cand_lists for p in g})
    where = {p: k for k, p in enumerate(kept)}
    keep = np.zeros(n_pages, bool)
    keep[kept] = True
    titles, s_page, s_id, texts = [], array("i"), array("i"), []
    with zipfile.ZipFile(FEVER["wiki-pages.zip"]) as z:
        for i, line in enumerate(raw_pages(z)):
            if not keep[i]:
                continue
            d = json.loads(line)
            titles.append(d["id"])
            for row in d.get("lines", "").split("\n"):
                parts = row.split("\t")
                if len(parts) > 1 and parts[0].isdigit() and parts[1].strip():
                    s_page.append(len(titles) - 1); s_id.append(int(parts[0])); texts.append(parts[1])
    blob = "\n".join(texts).encode("utf-8")
    lens = np.array([len(t.encode("utf-8")) for t in texts], np.int64)
    off = np.concatenate([[0], np.cumsum(lens + 1)])
    print(f"  PAGES PER CLAIM + PASS 2 ({time.time() - t1:.0f}s): {len(claims):,} claims, {len(kept):,} pages "
          f"kept, {len(texts):,} sentences ({len(blob) / 1e6:.0f} MB)")
    flat = np.array([where[p] for g in cand_lists for p in g], np.int32)
    c_off = np.concatenate([[0], np.cumsum([len(g) for g in cand_lists])]).astype(np.int64)
    return dict(titles=np.array(titles), paren=paren[kept], s_page=np.frombuffer(s_page, np.int32),
                s_id=np.frombuffer(s_id, np.int32), blob=np.frombuffer(blob, np.uint8), off=off,
                train_pick=np.array(pick, np.int64), cand=flat, cand_off=c_off, n_pages=np.array(n_pages))


DEPENDS7 = dict(what="FEVER pages chosen by title for a train sample and paper_dev, Brain 5 Cell 7A", version=1,
                seed=FV_SEED, per_label=FV_PER, max_n=MAX_N, per_run=PER_RUN, per_claim=PER_CLAIM,
                stopwords=sorted(STOPW), word=WORD.pattern,
                data={n: FEVER_PINS[n][0] for n in ("wiki-pages.zip", "train.jsonl", "paper_dev.jsonl")})
FV, BUILT7 = ARTS.get("fever_pages", DEPENDS7, build7)
TRAIN_FV_ALL = claims_of("train.jsonl")
FV_CLAIMS = [TRAIN_FV_ALL[int(i)] for i in FV["train_pick"]] + claims_of("paper_dev.jsonl")
del TRAIN_FV_ALL
N_TR = len(FV["train_pick"])
FV_TITLES = FV["titles"].tolist()
SENT_BY_PAGE = np.concatenate([[0], np.cumsum(np.bincount(FV["s_page"], minlength=len(FV_TITLES)))])
BLOB, OFF = FV["blob"].tobytes(), FV["off"]


def sentence_text(k: int) -> str:
    return BLOB[OFF[k]:OFF[k + 1] - 1].decode("utf-8")


def cands(ci: int) -> list:
    return FV["cand"][FV["cand_off"][ci]:FV["cand_off"][ci + 1]].tolist()


found = whole = 0
for ci in range(N_TR, len(FV_CLAIMS)):
    sets = gold_sets(FV_CLAIMS[ci])
    if sets:
        have = {FV_TITLES[p] for p in cands(ci)}
        found += 1
        whole += any(all(pg in have for pg, _ in s) for s in sets)
PAGE_RECALL = 100 * whole / found
print(f"\n  FEVER PAGES {'built and saved' if BUILT7 else 'loaded'}: {len(FV_CLAIMS):,} claims ({N_TR:,} train sample, "
      f"{len(FV_CLAIMS) - N_TR:,} paper_dev), {len(FV_TITLES):,} pages, {len(FV['s_id']):,} sentences; "
      f"pages per claim {np.mean(np.diff(FV['cand_off'])):.1f} ({time.time() - T7:.0f}s)")
print(f"  paper_dev SUPPORTS/REFUTES claims with a whole gold evidence set's pages found: {whole:,} of {found:,} "
      f"({PAGE_RECALL:.1f}%)")
print("\n  PREDICTIONS vs RESULTS")
LEDGER.result("B5-F1", round(PAGE_RECALL, 2))
print("  LAST LINE OF MC AI BRAIN 5 CELL 7A")


MC AI BRAIN 5 - CELL 7A: FEVER - PAGES FOR EACH CLAIM, BY TITLE
  PASS 1 (83s): 5,416,537 pages, 4,865,621 title keys (247,813 name several pages)
  PAGES PER CLAIM + PASS 2 (53s): 18,999 claims, 31,218 pages kept, 193,248 sentences (23 MB)

  FEVER PAGES built and saved: 18,999 claims (9,000 train sample, 9,999 paper_dev), 31,218 pages, 193,248 sentences; pages per claim 8.9 (137s)
  paper_dev SUPPORTS/REFUTES claims with a whole gold evidence set's pages found: 5,629 of 6,666 (84.4%)

  PREDICTIONS vs RESULTS
    B5-F1 paper_dev S/R claims with a whole gold evidence set's pages found, %: 84.44 (predicted between 70 and 95) -> HELD
  LAST LINE OF MC AI BRAIN 5 CELL 7A


In [11]:
# =====================================================================
# MC AI BRAIN 5 - CELL 7B: FEVER - EVIDENCE, CONTRADICTION, VERDICT
# Needs Cell 7A. Settings are chosen on the train sample (fit); paper_dev
# (practice) is scored once, with the chosen settings.
# EVIDENCE  each sentence of the claim's pages scores the summed IDF
#   (over the kept sentences) of the claim's words (Cell 6's base forms)
#   it holds, its page title's words counting as its own; numbers count
#   too. The 5 best sentences are the evidence; COVER = share of the
#   claim's summed IDF held by the best 2 together
# CONTRADICTION (in the best 2 sentences; from Mark's FEVER notebook):
#   NUM  the claim's number is not there, another number is
#   NEG  one of claim and best sentence is negated (not, never, no,
#        cannot, n't), the other is not
#   ANT  a claim word is missing and its WordNet antonym is there
#   SIB  a claim word is missing and a word that shares a WordNet
#        hypernym with it (within 3 steps, the hypernym 5 or more steps
#        below the top) is there
#   ONLY the claim says only, exclusively or solely
# VERDICT  no page: NOT ENOUGH INFO (unknown entity); COVER < t1: NOT
#   ENOUGH INFO (missing fact); a chosen contradiction: REFUTES; COVER >=
#   t2: SUPPORTS; otherwise NOT ENOUGH INFO (below threshold)
#   t1 0.1-0.6, t2 0.4-1.0, any set of the 5 contradictions: the most
#   right on the train sample is chosen (ties: fewer contradictions)
# SCORES  label accuracy; FEVER score (Thorne et al. 2018): label right
#   and, for SUPPORTS/REFUTES, one whole gold evidence set in the 5.
#   Published (test, trained classifier): 50.91 label, 31.87 FEVER score
# PREDICTIONS (written before running)
#   B5-F2 paper_dev S/R claims with a whole gold set in the 5, %: 35-75
#   B5-F3 paper_dev label accuracy, %: 38-50
#   B5-F4 paper_dev FEVER score, %: 20-40
#   B5-F5 train sample, contradictions chosen vs none, points: 1-10
# =====================================================================
import re, time

import numpy as np

print("=" * 64); print("MC AI BRAIN 5 - CELL 7B: FEVER - EVIDENCE, CONTRADICTION, VERDICT"); print("=" * 64)
for need in ("FV", "FV_CLAIMS", "N_TR", "cands", "sentence_text", "gold_sets", "unbracket", "forms", "ANT", "WN"):
    if need not in globals():
        raise RuntimeError(f"run Cell 7A first ({need} is missing)")
for key, claim, rule in (("B5-F2", "paper_dev S/R claims with a whole gold set in the 5 sentences, %",
                          between(35, 75)),
                         ("B5-F3", "paper_dev label accuracy, %", between(38, 50)),
                         ("B5-F4", "paper_dev FEVER score, %", between(20, 40)),
                         ("B5-F5", "train sample, contradictions chosen vs none, points", between(1, 10))):
    LEDGER.predict(key, claim, rule)
t1 = time.time()
NUMS = re.compile(r"\d+(?:[.,]\d+)*")
NEGW = re.compile(r"\b(not|never|no|cannot|none|nor)\b|n't\b", re.I)
ONLYW = re.compile(r"\b(only|exclusively|solely)\b", re.I)
KINDS = ("NUM", "NEG", "ANT", "SIB", "ONLY")
S_WORDS = [frozenset(forms(unbracket(sentence_text(k)))) for k in range(len(FV["s_id"]))]
S_NUMS = [frozenset(NUMS.findall(unbracket(sentence_text(k)))) for k in range(len(FV["s_id"]))]
T_WORDS = [frozenset(forms(unbracket(t))) for t in FV["titles"].tolist()]
DF7 = {}
for ws in S_WORDS:
    for w in ws:
        DF7[w] = DF7.get(w, 0) + 1
N7 = len(S_WORDS)


def idf7(w: str) -> float:
    d = DF7.get(w, 0)
    return float(np.log(1 + (N7 - d + 0.5) / (d + 0.5)))


NUM_IDF = idf7("")
UP = {}


def ups(w: str) -> set:
    """Hypernyms within 3 steps of the word's noun senses, keeping those 5+ steps below the top"""
    if w not in UP:
        out = set()
        for syn in WN.synsets_of(w, "n")[:3]:
            path = WN.path_up(syn.id)
            out |= {s for s in path[1:4] if len(WN.path_up(s)) >= 6}
        UP[w] = out
    return UP[w]


def read_claim(ci: int):
    """(predicted evidence [(page, sentence)], COVER, has page, contradictions as 0/1 by KINDS)"""
    c = FV_CLAIMS[ci]
    text = unbracket(c["claim"])
    cw, cn = set(forms(text)), set(NUMS.findall(text))
    pages = cands(ci)
    if not pages:
        return [], 0.0, False, np.zeros(len(KINDS), bool)
    total = sum(idf7(w) for w in cw) + NUM_IDF * len(cn)
    scored = []
    for p in pages:
        for k in range(SENT_BY_PAGE[p], SENT_BY_PAGE[p + 1]):
            ws = S_WORDS[k] | T_WORDS[p]
            scored.append((sum(idf7(w) for w in cw & ws) + NUM_IDF * len(cn & S_NUMS[k]), -len(scored), k))
    scored.sort(reverse=True)
    top = [k for _, _, k in scored[:5]]
    best2 = top[:2]
    ws2 = set().union(*(S_WORDS[k] | T_WORDS[FV["s_page"][k]] for k in best2))
    ns2 = set().union(*(S_NUMS[k] for k in best2))
    cover = (sum(idf7(w) for w in cw & ws2) + NUM_IDF * len(cn & ns2)) / total if total else 0.0
    missing = cw - ws2
    s_neg = bool(NEGW.search(unbracket(sentence_text(top[0])))) if top else False
    clash = np.array([bool(cn) and not cn <= ns2 and bool(ns2 - cn),
                      bool(NEGW.search(text)) != s_neg,
                      any(a in ws2 for w in missing for a in ANT.get(w, ())),
                      any(ups(w) & ups(v) for w in missing for v in ws2 - cw if v.isalpha()),
                      bool(ONLYW.search(text))])
    ev = [(FV_TITLES[FV["s_page"][k]], int(FV["s_id"][k])) for k in top]
    return ev, cover, True, clash


READ = [read_claim(ci) for ci in range(len(FV_CLAIMS))]
EV = [r[0] for r in READ]
COVER = np.array([r[1] for r in READ])
HASP = np.array([r[2] for r in READ])
CLASH = np.stack([r[3] for r in READ])
GOLD = np.array([LABELS.index(c["label"]) for c in FV_CLAIMS])
print(f"  READ ({time.time() - t1:.0f}s): {len(READ):,} claims; {N7:,} sentences give IDF; "
      f"contradictions found on the train sample: " + ", ".join(f"{k} {CLASH[:N_TR, i].mean():.0%}"
                                                              for i, k in enumerate(KINDS)))


def verdict(t1_: float, t2_: float, use: tuple, idx: slice) -> np.ndarray:
    """0 SUPPORTS, 1 REFUTES, 2 NOT ENOUGH INFO"""
    cov, cl = COVER[idx], CLASH[idx][:, list(use)].any(1) if use else np.zeros(len(COVER[idx]), bool)
    out = np.full(len(cov), 2)
    ok = HASP[idx] & (cov >= t1_)
    out[ok & cl] = 1
    out[ok & ~cl & (cov >= t2_)] = 0
    return out


TR, DV = slice(0, N_TR), slice(N_TR, len(FV_CLAIMS))
USES = [tuple(i for i in range(len(KINDS)) if m >> i & 1) for m in range(2 ** len(KINDS))]
GRID7 = {(a, b, u): 100 * np.mean(verdict(a, b, u, TR) == GOLD[TR])
         for a in (0.1, 0.2, 0.3, 0.4, 0.5, 0.6) for b in (0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0) if b >= a
         for u in USES}
CH7 = max(GRID7, key=lambda k: (round(GRID7[k], 6), -len(k[2])))
NONE7 = max((k for k in GRID7 if not k[2]), key=lambda k: round(GRID7[k], 6))
print(f"\n  TRAIN SAMPLE ({N_TR:,} claims, a third of each label), label accuracy %:")
rows7 = [(f"no contradiction (t1 {NONE7[0]}, t2 {NONE7[1]})", GRID7[NONE7])]
for i, k in enumerate(KINDS):
    top7 = max((g for g in GRID7 if g[2] == (i,)), key=lambda g: round(GRID7[g], 6))
    rows7.append((f"{k} alone (t1 {top7[0]}, t2 {top7[1]})", GRID7[top7]))
rows7.append((f"chosen: t1 {CH7[0]}, t2 {CH7[1]}, {'+'.join(KINDS[i] for i in CH7[2]) or 'none'}", GRID7[CH7]))
for label, value in rows7:
    print(f"    {label:<48}{value:>7.1f}")


def fever_scores(pred: np.ndarray, idx: range) -> tuple:
    """(label accuracy %, FEVER score %, S/R claims with a whole gold set in the 5 %)"""
    right = strict = sr = inside = 0
    for p, ci in zip(pred, idx):
        g = GOLD[ci]
        sets = gold_sets(FV_CLAIMS[ci])
        got = set(EV[ci])
        whole = any(all(e in got for e in s) for s in sets)
        if g < 2:
            sr += 1
            inside += whole
        right += p == g
        strict += p == g and (g == 2 or whole)
    return 100 * right / len(idx), 100 * strict / len(idx), 100 * inside / max(sr, 1)


PRED = verdict(*CH7, DV)
DV_IDX = range(N_TR, len(FV_CLAIMS))
ACC7, FS7, EV7 = fever_scores(PRED, DV_IDX)
print(f"\n  PAPER_DEV ({len(DV_IDX):,} claims), chosen settings: label accuracy {ACC7:.1f}%, FEVER score {FS7:.1f}%; "
      f"S/R claims with a whole gold set in the 5 sentences {EV7:.1f}% (chance label accuracy 33.3%)")
print("    gold \\ said          SUPPORTS   REFUTES       NEI")
for g, lab in enumerate(LABELS):
    row = [int(((GOLD[DV] == g) & (PRED == p)).sum()) for p in range(3)]
    print(f"    {lab:<18}" + "".join(f"{x:>10,}" for x in row))
why = np.where(~HASP[DV], "unknown entity (no page)", np.where(COVER[DV] < CH7[0], "missing fact (cover < t1)",
               "below threshold (t1 <= cover < t2)"))
said_nei = PRED == 2
print("    NOT ENOUGH INFO said, by reason: " + ", ".join(
    f"{r} {int((said_nei & (why == r)).sum()):,}" for r in ("unknown entity (no page)", "missing fact (cover < t1)",
                                                             "below threshold (t1 <= cover < t2)")))

print("\n  PREDICTIONS vs RESULTS")
LEDGER.result("B5-F2", round(EV7, 2))
LEDGER.result("B5-F3", round(ACC7, 2))
LEDGER.result("B5-F4", round(FS7, 2))
LEDGER.result("B5-F5", round(GRID7[CH7] - GRID7[NONE7], 2))
LEDGER.result("B5-T12" if BUILT7 else "B5-T13", round(time.time() - T7))
if "sync" in globals():
    sync("Cell 7")
print(f"\n  ({time.time() - T7:.0f}s since Cell 7A started)")
print("  LAST LINE OF MC AI BRAIN 5 CELL 7B")


MC AI BRAIN 5 - CELL 7B: FEVER - EVIDENCE, CONTRADICTION, VERDICT
  READ (10s): 18,999 claims; 193,248 sentences give IDF; contradictions found on the train sample: NUM 5%, NEG 8%, ANT 1%, SIB 35%, ONLY 4%

  TRAIN SAMPLE (9,000 claims, a third of each label), label accuracy %:
    no contradiction (t1 0.1, t2 0.8)                  46.5
    NUM alone (t1 0.5, t2 0.8)                         47.7
    NEG alone (t1 0.1, t2 0.8)                         50.5
    ANT alone (t1 0.1, t2 0.8)                         47.0
    SIB alone (t1 0.6, t2 0.8)                         44.4
    ONLY alone (t1 0.1, t2 0.8)                        49.8
    chosen: t1 0.1, t2 0.8, NUM+NEG+ANT+ONLY           55.2

  PAPER_DEV (9,999 claims), chosen settings: label accuracy 54.2%, FEVER score 50.0%; S/R claims with a whole gold set in the 5 sentences 79.3% (chance label accuracy 33.3%)
    gold \ said          SUPPORTS   REFUTES       NEI
    SUPPORTS               2,267       140       926
    REFUTES        

In [12]:
# =====================================================================
# MC AI BRAIN 5 - CELL 8: FEVER - THE CLAIM ALONE, AND LINKS COUNTED
# FROM TRAIN CLAIMS AND THEIR EVIDENCE
# Needs Cells 7A and 7B. Train claims outside Cell 7A's sample (fit) are
# counted; settings are chosen on the sample; paper_dev (practice) is
# scored once. Test files are not read.
# CLAIM ONLY  multinomial naive Bayes over the claim's words (every word,
#   lower case; add-one smoothing; equal label priors): a reader that
#   never sees evidence
# LINKS  for each counted SUPPORTS or REFUTES claim and its first gold
#   evidence set (sentences read from wiki-pages.zip, saved with
#   Artifacts): C = the claim's base forms, E = the evidence's (page
#   title words too). Each pair (w in C not in E, v in E not in C) is
#   counted under the claim's label. For a pair seen 3 or more times:
#   clash(w, v) = log((R + 1) / (S + 1)) - log(R claims / S claims)
# USE  in Cell 7B's best 2 sentences, for each claim word they miss:
#   LEARNED  a sixth contradiction: some word there gives clash >= c
#   SAME     the word counts as found: some word there gives clash <= -c
#   t1, t2, the 6 contradictions, c (0.5, 1, 1.5, 2) and SAME (off, on)
#   are chosen on the sample: most right (ties: fewer contradictions)
# PREDICTIONS (written before running)
#   B5-G1 claim only, paper_dev label accuracy, %: 45-60
#   B5-G2 links chosen vs Cell 7B chosen, sample label accuracy, points: 1-8
#   B5-G3 paper_dev label accuracy with links, %: 55-62
#   B5-G4 paper_dev REFUTES claims said REFUTES, %: 38-55
#   B5-T14 Cell 8 seconds, building: <= 900; B5-T15 loading: <= 300
# =====================================================================
import json, math, time, zipfile
from array import array
from collections import Counter

import numpy as np

print("=" * 64); print("MC AI BRAIN 5 - CELL 8: FEVER - THE CLAIM ALONE, AND COUNTED LINKS"); print("=" * 64)
T8 = time.time()
for need in ("FV", "FV_CLAIMS", "N_TR", "GOLD", "COVER", "CLASH", "HASP", "GRID7", "CH7", "fever_scores", "idf7",
             "S_WORDS", "T_WORDS", "S_NUMS", "SENT_BY_PAGE", "raw_pages", "page_id", "claims_of", "WORD"):
    if need not in globals():
        raise RuntimeError(f"run Cells 7A and 7B first ({need} is missing)")
for key, claim, rule in (("B5-G1", "claim only, paper_dev label accuracy, %", between(45, 60)),
                         ("B5-G2", "links chosen vs Cell 7B chosen, sample label accuracy, points", between(1, 8)),
                         ("B5-G3", "paper_dev label accuracy with links, %", between(55, 62)),
                         ("B5-G4", "paper_dev REFUTES claims said REFUTES, %", between(38, 55)),
                         ("B5-T14", "Cell 8 seconds, building", at_most(900)),
                         ("B5-T15", "Cell 8 seconds, loading", at_most(300))):
    LEDGER.predict(key, claim, rule)
PICKED = {int(i) for i in FV["train_pick"]}
COUNTED = [c for i, c in enumerate(claims_of("train.jsonl")) if i not in PICKED]

# ---------------- the claim alone ----------------
NBC = [Counter() for _ in LABELS]
for c in COUNTED:
    NBC[LABELS.index(c["label"])].update(WORD.findall(c["claim"].casefold()))
NB_V, NB_T = len(set().union(*NBC)), [sum(k.values()) for k in NBC]


def claim_only(text: str) -> int:
    t = WORD.findall(text.casefold())
    return int(np.argmax([sum(math.log((NBC[l][w] + 1) / (NB_T[l] + NB_V)) for w in t) for l in range(3)]))


NB_DEV = np.array([claim_only(FV_CLAIMS[ci]["claim"]) for ci in DV_IDX])
NB_ACC = 100 * np.mean(NB_DEV == GOLD[DV])
print(f"  CLAIM ONLY ({len(COUNTED):,} train claims counted, {NB_V:,} words): paper_dev label accuracy "
      f"{NB_ACC:.1f}%; said REFUTES for {100 * np.mean(NB_DEV[GOLD[DV] == 1] == 1):.1f}% of REFUTES claims")


# ---------------- gold evidence of the counted claims ----------------
def build8() -> dict:
    t1 = time.time()
    need = {}
    for c in COUNTED:
        sets = gold_sets(c)
        if c["label"] != "NOT ENOUGH INFO" and sets:
            for pg, sid in sets[0]:
                need.setdefault(pg, set()).add(sid)
    pages, sids, texts = [], array("i"), []
    with zipfile.ZipFile(FEVER["wiki-pages.zip"]) as z:
        for line in raw_pages(z):
            t = page_id(line)
            if t in need:
                for row in json.loads(line).get("lines", "").split("\n"):
                    parts = row.split("\t")
                    if len(parts) > 1 and parts[0].isdigit() and int(parts[0]) in need[t]:
                        pages.append(t); sids.append(int(parts[0])); texts.append(parts[1])
    blob = "\n".join(texts).encode("utf-8")
    off = np.concatenate([[0], np.cumsum([len(x.encode("utf-8")) + 1 for x in texts])]).astype(np.int64)
    print(f"  GOLD EVIDENCE ({time.time() - t1:.0f}s): {len(texts):,} sentences from {len(need):,} pages")
    return dict(pages=np.array(pages), sids=np.frombuffer(sids, np.int32), blob=np.frombuffer(blob, np.uint8), off=off)


DEPENDS8 = dict(what="gold evidence sentences of the counted FEVER train claims, Brain 5 Cell 8", version=1,
                seed=FV_SEED, per_label=FV_PER, set="first gold evidence set",
                data={n: FEVER_PINS[n][0] for n in ("wiki-pages.zip", "train.jsonl")})
GE, BUILT8 = ARTS.get("fever_train_evidence", DEPENDS8, build8)
gb, go = GE["blob"].tobytes(), GE["off"]
LOOK = {(p, int(s)): gb[go[k]:go[k + 1] - 1].decode("utf-8") for k, (p, s) in
        enumerate(zip(GE["pages"].tolist(), GE["sids"]))}
del gb

# ---------------- links ----------------
t1 = time.time()
PAIR, N_LAB = {}, [0, 0]
for c in COUNTED:
    if c["label"] == "NOT ENOUGH INFO":
        continue
    sets = gold_sets(c)
    if not sets or any((pg, sid) not in LOOK for pg, sid in sets[0]):
        continue
    E = set()
    for pg, sid in sets[0]:
        E |= set(forms(unbracket(LOOK[pg, sid]))) | set(forms(unbracket(pg)))
    C = set(forms(unbracket(c["claim"])))
    k = 0 if c["label"] == "SUPPORTS" else 1
    N_LAB[k] += 1
    for w in C - E:
        for v in E - C:
            PAIR.setdefault((w, v), [0, 0])[k] += 1
PRIOR = math.log(N_LAB[1] / N_LAB[0])
CL = {wv: math.log((r + 1) / (s + 1)) - PRIOR for wv, (s, r) in PAIR.items() if s + r >= 3}
TOPC = sorted(((r, s, wv) for wv, (s, r) in PAIR.items() if s + r >= 3 and CL[wv] >= 1.5), reverse=True)[:12]
TOPS = sorted(((s, r, wv) for wv, (s, r) in PAIR.items() if s + r >= 3 and CL[wv] <= -1.5), reverse=True)[:12]
del PAIR
print(f"  LINKS ({time.time() - t1:.0f}s): {N_LAB[0]:,} SUPPORTS and {N_LAB[1]:,} REFUTES claims lined up with their "
      f"evidence; {len(CL):,} word pairs seen 3+ times")
print("    clash (claim word / evidence word, REFUTES-SUPPORTS): "
      + ", ".join(f"{w}/{v} {r}-{s}" for r, s, (w, v) in TOPC))
print("    same: " + ", ".join(f"{w}/{v} {r}-{s}" for s, r, (w, v) in TOPS))


def best2(ci: int):
    """(claim base forms, the words of Cell 7B's best 2 sentences): the same scoring and order as Cell 7B"""
    text = unbracket(FV_CLAIMS[ci]["claim"])
    cw, cn = set(forms(text)), set(NUMS.findall(text))
    scored = []
    for p in cands(ci):
        for k in range(SENT_BY_PAGE[p], SENT_BY_PAGE[p + 1]):
            ws = S_WORDS[k] | T_WORDS[p]
            scored.append((sum(idf7(w) for w in cw & ws) + NUM_IDF * len(cn & S_NUMS[k]), -len(scored), k))
    scored.sort(reverse=True)
    top = [k for _, _, k in scored[:2]]
    total = sum(idf7(w) for w in cw) + NUM_IDF * len(cn)
    return cw, set().union(*(S_WORDS[k] | T_WORDS[FV["s_page"][k]] for k in top)) if top else set(), total


CS = (0.5, 1.0, 1.5, 2.0)
LEARN = {c: np.zeros(len(FV_CLAIMS), bool) for c in CS}
COV8 = {(c, s): COVER.copy() for c in CS for s in (False, True)}
for ci in range(len(FV_CLAIMS)):
    cw, ws2, total = best2(ci)
    extra = ws2 - cw
    for w in cw - ws2:
        vals = [CL[w, v] for v in extra if (w, v) in CL]
        if not vals:
            continue
        hi, lo = max(vals), min(vals)
        for c in CS:
            LEARN[c][ci] |= hi >= c
            if lo <= -c and total:
                COV8[c, True][ci] += idf7(w) / total


CLM = {c: np.concatenate([CLASH, LEARN[c][:, None]], 1) for c in CS}


def verdict8(t1_, t2_, use, c, same, idx):
    cov = COV8[c, same][idx]
    cl = CLM[c][idx][:, list(use)].any(1) if use else np.zeros(len(cov), bool)
    out = np.full(len(cov), 2)
    ok = HASP[idx] & (cov >= t1_)
    out[ok & cl] = 1
    out[ok & ~cl & (cov >= t2_)] = 0
    return out


USES8 = [tuple(i for i in range(6) if m >> i & 1) for m in range(64)]
GRID8 = {(a, b, u, c, s): 100 * np.mean(verdict8(a, b, u, c, s, TR) == GOLD[TR])
         for a in (0.1, 0.2, 0.3, 0.4, 0.5, 0.6) for b in (0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0) if b >= a
         for u in USES8 for c in CS for s in (False, True)}
CH8 = max(GRID8, key=lambda k: (round(GRID8[k], 6), -len(k[2]), -k[4], k[3]))
NAMES8 = KINDS + ("LEARNED",)
print(f"\n  SAMPLE ({N_TR:,} claims), label accuracy %: Cell 7B chosen {GRID7[CH7]:.1f}; with links "
      f"{GRID8[CH8]:.1f} (t1 {CH8[0]}, t2 {CH8[1]}, {'+'.join(NAMES8[i] for i in CH8[2]) or 'none'}, c {CH8[3]}, "
      f"SAME {'on' if CH8[4] else 'off'})")
PRED8 = verdict8(*CH8, DV)
ACC8, FS8, _ = fever_scores(PRED8, DV_IDX)
REF8 = 100 * np.mean(PRED8[GOLD[DV] == 1] == 1)
print(f"\n  PAPER_DEV ({len(DV_IDX):,} claims)          label accuracy  FEVER score  REFUTES said REFUTES")
NB_REF, REF7 = 100 * np.mean(NB_DEV[GOLD[DV] == 1] == 1), 100 * np.mean(PRED[GOLD[DV] == 1] == 1)
print(f"    claim only (no evidence)          {NB_ACC:>10.1f}            -  {NB_REF:>14.1f}")
print(f"    Cell 7B (evidence, cues)          {ACC7:>10.1f}  {FS7:>11.1f}  {REF7:>14.1f}")
print(f"    Cell 8 (evidence, cues, links)    {ACC8:>10.1f}  {FS8:>11.1f}  {REF8:>14.1f}")
print("    gold \\ said          SUPPORTS   REFUTES       NEI")
for g, lab in enumerate(LABELS):
    print(f"    {lab:<18}" + "".join(f"{int(((GOLD[DV] == g) & (PRED8 == p)).sum()):>10,}" for p in range(3)))

print("\n  PREDICTIONS vs RESULTS")
LEDGER.result("B5-G1", round(NB_ACC, 2))
LEDGER.result("B5-G2", round(GRID8[CH8] - GRID7[CH7], 2))
LEDGER.result("B5-G3", round(ACC8, 2))
LEDGER.result("B5-G4", round(REF8, 2))
LEDGER.result("B5-T14" if BUILT8 else "B5-T15", round(time.time() - T8))
if "sync" in globals():
    sync("Cell 8")
print(f"\n  ({time.time() - T8:.0f}s)")
print("  LAST LINE OF MC AI BRAIN 5 CELL 8")


MC AI BRAIN 5 - CELL 8: FEVER - THE CLAIM ALONE, AND COUNTED LINKS
  CLAIM ONLY (136,449 train claims counted, 21,099 words): paper_dev label accuracy 49.3%; said REFUTES for 61.8% of REFUTES claims
  GOLD EVIDENCE (57s): 17,254 sentences from 7,847 pages
  LINKS (9s): 76,371 SUPPORTS and 26,547 REFUTES claims lined up with their evidence; 116,068 word pairs seen 3+ times
    clash (claim word / evidence word, REFUTES-SUPPORTS): exclusively/american 162-4, canadian/american 155-2, refuse/born 140-1, incapable/american 132-2, die/born 130-73, refuse/american 130-3, ever/american 127-2, book/film 121-38, book/american 117-52, ever/born 116-2, exclusively/film 110-2, television/film 109-38
    same: person/born 10-796, person/american 36-674, person/actor 5-376, person/producer 7-301, acts/born 7-291, person/actress 3-252, person/singer 3-250, person/know 5-245, state/american 13-217, unite/american 13-212, acts/american 8-210, person/songwriter 3-172

  SAMPLE (9,000 claims), label accur

In [13]:
# =====================================================================
# MC AI BRAIN 5 - CELL 9A: FEVER - NATURAL-LOGIC ALIGNMENT (NO TRAINING)
# Needs Cells 7A, 7B and 8; Cell 9B joins. Settings are chosen on Cell
# 7A's train sample (fit); paper_dev (practice) is scored once.
# FEVER-SYMMETRIC (Schuster et al. 2019) v0.2, CC BY-SA 3.0: each claim
#   comes with evidence that supports it and with evidence that refutes
#   it, so the claim alone gets 50% by construction. dev (708 pairs) is
#   practice; test (712) is pinned, not read. Only pairs built from
#   paper_dev claims are scored; the rest are counted and held back.
# ALIGN (MacCartney & Manning 2009, natural logic): each claim content
#   word (Cell 6's base forms; operators and single letters left out)
#   gets one relation to the evidence words (Cell 7B's best 2 sentences
#   with page titles; FEVER-Symmetric's sentence with the gold page names,
#   brackets cut, of its source claim):
#   EQ   the same word, or a WordNet synonym (a shared synset)
#   FWD  an evidence word is a kind of it (WordNet hypernym, <= 4 steps)
#   REV  it is a kind of an evidence word: the claim says more
#   INS  none of these: the claim adds what the evidence does not say
#   SAME on: INS or REV counts as found if a link (LINKS) is <= -c
# EXCLUSION (on a word not found, or on the whole claim):
#   ANT  its WordNet antonym is there
#   SIB  an evidence word shares a WordNet hypernym with it (<= 2 steps
#        each, 6+ steps below the top); neither is a kind of the other
#   NUM  the claim's number is not there and another number is
#   LINK a link (LINKS) to an evidence word is >= c
#   TYPE the kind the claim names after is/was (head noun) matches no
#        kind the evidence names after is/was (same, synonym, or a kind
#        of the other within 12 steps); two kinds of person are let pass
# LINKS  v1 (written first): Cell 8's clash(w, v) minus w's own lean,
#   log((R + 1) / (S + 1)) - log(R / S claims) over counted claims whose
#   evidence lacks w. v2 (after lab/look_cell09.py, on fit and practice
#   data, found the lean removes evidence signal): Cell 8's clash as is.
#   Each version gets its own settings chosen on the sample (Cell 9B)
# OPERATORS  not, never, no, none, nor, cannot, n't and the negative
#   implicatives refuse/fail/decline/neglect to, unable to, incapable of
#   ("refused to X" entails "did not X") each flip polarity; only,
#   exclusively, solely make the claim say more
# PREDICTIONS (written before running)
#   B5-N0 Symmetric dev pairs built from paper_dev claims, %: 30-70
#   B5-N1 claim only (Cell 8), Symmetric scored, two-way, %: 49.9-50.1
#   B5-N2 Cell 8's rule, Symmetric scored, two-way, %: 50-62
#   B5-N3 the gate, Symmetric scored, two-way, %: 55-72
#   B5-N4 the gate minus Cell 8's rule, Symmetric, points: 2-15
#   B5-N5 the gate, paper_dev label accuracy, %: 55-61
#   B5-N6 Cells 9A + 9B seconds: <= 600
#   (In the lab N0, N2, N4 and N5 failed and N3 held; N3-N5 score v1.
#   Written after the lab ablation, which showed values near these for
#   v2's form:)
#   B5-N7 v2 gate, Symmetric scored, two-way, %: 66-70
#   B5-N8 v2 gate, paper_dev label accuracy, %: 54-57
# =====================================================================
import json, math, re, time, urllib.request

import numpy as np

print("=" * 64); print("MC AI BRAIN 5 - CELL 9A: FEVER - NATURAL-LOGIC ALIGNMENT (NO TRAINING)"); print("=" * 64)
T9 = time.time()
for need in ("CL", "COUNTED", "LOOK", "CH8", "GRID8", "NBC", "NB_T", "PRIOR", "ACC8", "FS8", "ups", "NEGW", "EV"):
    if need not in globals():
        raise RuntimeError(f"run Cells 7A, 7B and 8 first ({need} is missing)")
for key, claim, rule in (("B5-N0", "Symmetric dev pairs from paper_dev claims, %", between(30, 70)),
                         ("B5-N1", "claim only, Symmetric scored, two-way, %", between(49.9, 50.1)),
                         ("B5-N2", "Cell 8's rule, Symmetric scored, two-way, %", between(50, 62)),
                         ("B5-N3", "the gate, Symmetric scored, two-way, %", between(55, 72)),
                         ("B5-N4", "gate minus Cell 8's rule, Symmetric, points", between(2, 15)),
                         ("B5-N5", "the gate, paper_dev label accuracy, %", between(55, 61)),
                         ("B5-N6", "Cells 9A + 9B seconds", at_most(600)),
                         ("B5-N7", "v2 gate, Symmetric scored, two-way, %", between(66, 70)),
                         ("B5-N8", "v2 gate, paper_dev label accuracy, %", between(54, 57))):
    LEDGER.predict(key, claim, rule)

# ---------------- FEVER-Symmetric, pinned ----------------
SYM_COMMIT = "fa25d17e73ea0694edc78133ec654b9f32f6db73"
SYM_URL = f"https://raw.githubusercontent.com/TalSchuster/FeverSymmetric/{SYM_COMMIT}/symmetric_v0.2/"
SYM_PINS = {"fever_symmetric_dev.jsonl": ("1d43d016da9264de2139cd10edd459ffbeac84f1fd85d60b70ab6ef63d283f81",
                                          466_743, "practice"),
            "fever_symmetric_test.jsonl": ("b2ff560615bd3337c858bc5b09256d04f39aaa034aaae2351bb2d6345c110f83",
                                           485_280, "test")}


def sym_get(name: str):
    want, nbytes, _ = SYM_PINS[name]
    here = DATA / "feversym" / name
    if here.exists() and here.stat().st_size == nbytes and sha256(here) == want:
        return here
    for p in sorted(INPUT.rglob(name)) if INPUT.exists() else []:
        if p.stat().st_size == nbytes and sha256(p) == want:
            return p
    here.parent.mkdir(parents=True, exist_ok=True)
    with urllib.request.urlopen(urllib.request.Request(SYM_URL + name, headers={"User-Agent": "MC_AI"}),
                                timeout=60) as r:
        here.write_bytes(r.read())
    if here.stat().st_size != nbytes or sha256(here) != want:
        here.unlink()
        raise RuntimeError(f"{name}: other bytes than pinned")
    return here


SYM_PATH = sym_get("fever_symmetric_dev.jsonl")
kept = str(SYM_PATH.relative_to(DATA)) if SYM_PATH.is_relative_to(DATA) else str(SYM_PATH)
RECORD[:] = [r for r in RECORD if r["source"] != "feversym"]
RECORD.append(dict(source="feversym", url=SYM_URL + SYM_PATH.name, repo="TalSchuster/FeverSymmetric",
                   commit=SYM_COMMIT, file=kept, sha256=SYM_PINS[SYM_PATH.name][0], bytes=SYM_PATH.stat().st_size))
ROLES[kept], FROM[kept] = "practice", "feversym"
ROLES["feversym/fever_symmetric_test.jsonl (pinned, not fetched)"] = "test"
(DATA / "sources.json").write_text(json.dumps(RECORD, indent=1))
(DATA / "roles.json").write_text(json.dumps(dict(roles=ROLES, sealed=SEALED, TEST4=[k for k, v in SETS.items()
                                                                                    if v == "test"]), indent=1))
SYM_ALL = [json.loads(line) for line in open(SYM_PATH, encoding="utf-8") if line.strip()]
DEV_CI = {int(FV_CLAIMS[ci]["id"]): ci for ci in range(N_TR, len(FV_CLAIMS))}
SEEN_IDS = {int(c["id"]) for c in COUNTED} | {int(FV_CLAIMS[ci]["id"]) for ci in range(N_TR)}


def base_id(r: dict) -> int:
    return int(re.sub(r"000000[234]$", "", str(r["id"])))


SYM = [r for r in SYM_ALL if base_id(r) in DEV_CI]
GROUPS = {base_id(r) for r in SYM_ALL}
n_dev, n_seen = sum(g in DEV_CI for g in GROUPS), sum(g in SEEN_IDS for g in GROUPS)
print(f"  FEVER-SYMMETRIC dev: {len(SYM_ALL)} pairs from {len(GROUPS)} FEVER claims; {n_dev} of them in paper_dev "
      f"({len(SYM)} pairs scored), {n_seen} in train (held back), {len(GROUPS) - n_dev - n_seen} elsewhere (held back)")

# ---------------- the lexicon ----------------
NEG_LEAN = {}
for c in COUNTED:
    sets = gold_sets(c)
    if c["label"] == "NOT ENOUGH INFO" or not sets or any((pg, sid) not in LOOK for pg, sid in sets[0]):
        continue
    E = set()
    for pg, sid in sets[0]:
        E |= set(forms(unbracket(LOOK[pg, sid]))) | set(forms(unbracket(pg)))
    for w in set(forms(unbracket(c["claim"]))) - E:
        NEG_LEAN.setdefault(w, [0, 0])[0 if c["label"] == "SUPPORTS" else 1] += 1
LEAN = {w: math.log((r + 1) / (s + 1)) - PRIOR for w, (s, r) in NEG_LEAN.items()}
CLC = {wv: v - LEAN.get(wv[0], 0.0) for wv, v in CL.items()}
del NEG_LEAN
SYN9, UP9, DEEP9, NOUN9 = {}, {}, {}, {}
POL9 = re.compile(r"\b(?:not|never|no|none|nor|cannot)\b|n't\b|\b(?:refus|fail|declin|neglect)\w* to\b"
                  r"|\bunable to\b|\bincapable of\b", re.I)
ONLY9 = re.compile(r"\b(?:only|exclusively|solely)\b", re.I)
OPS9 = set("never none nor cannot refuse fail decline neglect unable incapable exclusively solely".split())
COP9 = re.compile(r"\b(?:is|was|are|were)\s+(?:(?:not|n't|never)\s+)?(?:(?:a|an|the)\s+)?(.*)", re.I)
TSTOP = set("of that which who whom whose from in on at by for with to as since where when born ( ) [ ] ; : . -- "
            "known based".split())
TSKIP = set("one part member kind type form sort example name term number set piece unit".split())


def syns(w: str) -> set:
    if w not in SYN9:
        SYN9[w] = {s.id for p in "nvar" for s in WN.synsets_of(w, p)[:3]}
    return SYN9[w]


def up_to(w: str, k: int) -> set:
    if (w, k) not in UP9:
        seen, front = set(), {s.id for p in "nv" for s in WN.synsets_of(w, p)[:3]}
        for _ in range(k):
            front = {q.target for s in front for q in WN.synsets[s].pointers if q.symbol in ("@", "@i")} - seen
            seen |= front
        UP9[w, k] = seen
    return UP9[w, k]


def deep(shared: set) -> bool:
    for s in shared:
        if s not in DEEP9:
            DEEP9[s] = len(WN.path_up(s)) >= 7
        if DEEP9[s]:
            return True
    return False


def kin(h: str, g: str) -> bool:
    return h == g or bool(syns(h) & syns(g)) or bool(syns(h) & up_to(g, 12)) or bool(syns(g) & up_to(h, 12))


def person(w: str) -> bool:
    return any(s.lexname == "noun.person" for s in WN.synsets_of(w, "n")[:2])


def heads(text: str) -> frozenset:
    m, out, last = COP9.search(text), set(), ""
    for t in (m.group(1).split()[:16] if m else []):
        t = t.lower()
        if t in TSTOP:
            break
        if t in (",", "and", "or"):
            out.add(last); last = ""
            continue
        if t not in NOUN9:
            NOUN9[t] = WN.base_forms(t, "n") if t.isascii() and t.isalpha() else []
        n = NOUN9[t]
        if n:
            last = n[0]
        elif t.endswith(("ed", "ing")):
            break
    out.add(last)
    return frozenset(out - TSKIP - {""})


def type_clash(H: frozenset, G: frozenset, E: set) -> bool:
    if not H or not G or H & E or any(kin(h, g) for h in H for g in G):
        return False
    return not (any(person(h) for h in H) and any(person(g) for g in G))


def relate(w: str, E: set, others: list):
    sw, rel, part = syns(w), ("EQ" if w in E else ""), w
    for kind in ("EQ", "FWD", "REV"):
        if rel:
            break
        uw = up_to(w, 4) if kind == "REV" else None
        for v in others:
            if (kind == "EQ" and sw & syns(v)) or (kind == "FWD" and sw & up_to(v, 4)) or (uw and syns(v) & uw):
                rel, part = kind, v
                break
    if rel in ("EQ", "FWD"):
        return rel, part, False, False, (0.0, 0.0, 0.0, 0.0)
    u2 = up_to(w, 2)
    sib = not rel and any(deep(u2 & up_to(v, 2)) for v in others)
    vc, vr = [CLC[w, v] for v in others if (w, v) in CLC], [CL[w, v] for v in others if (w, v) in CL]
    return (rel or "INS", part if rel else "", any(a in E for a in ANT.get(w, ())), sib,
            (min(vc, default=0.0), max(vc, default=0.0), min(vr, default=0.0), max(vr, default=0.0)))


def read9(claim: str, E: set, nums: set, ev_pol: int, kinds_e: frozenset):
    """([(word, share, relation, evidence word, ANT, SIB, links)], NUM, TYPE, number share, swap, only)"""
    text = unbracket(claim)
    C = {w for w in forms(text) if len(w) > 1 and w not in OPS9}
    cn = set(NUMS.findall(text))
    total = sum(idf7(w) for w in C) + NUM_IDF * len(cn)
    others = [v for v in E if v not in C]
    words = [(w, idf7(w) / total if total else 0.0, *relate(w, E, others)) for w in sorted(C)]
    return (words, bool(cn) and not cn <= nums and bool(nums - cn), type_clash(heads(text), kinds_e, E),
            NUM_IDF * len(cn & nums) / total if total else 0.0, len(POL9.findall(text)) % 2 != ev_pol,
            bool(ONLY9.search(text)))


def top2(ci: int) -> list:
    text = unbracket(FV_CLAIMS[ci]["claim"])
    cw, cn, scored = set(forms(text)), set(NUMS.findall(text)), []
    for p in cands(ci):
        for k in range(SENT_BY_PAGE[p], SENT_BY_PAGE[p + 1]):
            ws = S_WORDS[k] | T_WORDS[p]
            scored.append((sum(idf7(w) for w in cw & ws) + NUM_IDF * len(cn & S_NUMS[k]), -len(scored), k))
    scored.sort(reverse=True)
    return [k for _, _, k in scored[:2]]


def sym_title(r: dict) -> set:
    c = FV_CLAIMS[DEV_CI[base_id(r)]]
    return set().union(*(set(forms(unbracket(pg).split("(")[0])) for s in gold_sets(c) for pg, _ in s))


t1 = time.time()
R9 = []
for ci in range(len(FV_CLAIMS)):
    top = top2(ci)
    E = set().union(*(S_WORDS[k] | T_WORDS[FV["s_page"][k]] for k in top)) if top else set()
    texts = [unbracket(sentence_text(k)) for k in top]
    R9.append(read9(FV_CLAIMS[ci]["claim"], E, set().union(*(S_NUMS[k] for k in top)) if top else set(),
                    len(POL9.findall(texts[0])) % 2 if top else 0, frozenset().union(*map(heads, texts))))
ESYM = [set(forms(unbracket(r["evidence"]))) | sym_title(r) for r in SYM]
RS = [read9(r["claim"], E, set(NUMS.findall(unbracket(r["evidence"]))), len(POL9.findall(r["evidence"])) % 2,
            heads(unbracket(r["evidence"]))) for r, E in zip(SYM, ESYM)]
print(f"  ALIGNED ({time.time() - t1:.0f}s): {len(R9):,} FEVER claims, {len(RS)} Symmetric pairs, {len(CLC):,} links")
print("\n  PREDICTIONS vs RESULTS")
LEDGER.result("B5-N0", round(100 * len(SYM) / len(SYM_ALL), 2))
if "sync" in globals():
    sync("Cell 9A")
print("  LAST LINE OF MC AI BRAIN 5 CELL 9A")


MC AI BRAIN 5 - CELL 9A: FEVER - NATURAL-LOGIC ALIGNMENT (NO TRAINING)
  FEVER-SYMMETRIC dev: 708 pairs from 177 FEVER claims; 171 of them in paper_dev (684 pairs scored), 0 in train (held back), 6 elsewhere (held back)
  ALIGNED (9s): 18,999 FEVER claims, 684 Symmetric pairs, 116,068 links

  PREDICTIONS vs RESULTS
    B5-N0 Symmetric dev pairs from paper_dev claims, %: 96.61 (predicted between 30 and 70) -> DID NOT HOLD
  sync: pushed 5 files to kaggle/ on main as commit f2f6a48e84f8; GitHub confirms it (468 entries, head 9c8e3ec1cbe83569) (4s)
  LAST LINE OF MC AI BRAIN 5 CELL 9A


In [14]:
# =====================================================================
# MC AI BRAIN 5 - CELL 9B: FEVER - THE NATURAL-LOGIC GATE (NO TRAINING)
# Needs Cell 9A (its relations, exclusions and operators). Predictions
# B5-N1 to B5-N8 are written in Cell 9A.
# JOIN  no page: N. Words not found hold <= t_ins of the claim's IDF: S,
#   else N. A chosen exclusion with found words >= t_anc: R. Only: S
#   becomes N. Claim and best sentence of opposite polarity: S and R swap.
#   t_ins, t_anc 0.1-0.6, any set of the 5 exclusions, c (1, 1.5, 2),
#   SAME (off, on): the most right on the sample (ties: fewer exclusions)
# FEVER-Symmetric is two-way: N counts as SUPPORTS (also shown wrong)
# CUES  each exclusion and operator: share of claims it fires on, and of
#   those, the share that are REFUTES, on the sample and on Symmetric
# =====================================================================
import math, time

import numpy as np

print("=" * 64); print("MC AI BRAIN 5 - CELL 9B: FEVER - THE NATURAL-LOGIC GATE (NO TRAINING)"); print("=" * 64)
if "R9" not in globals() or "RS" not in globals():
    raise RuntimeError("run Cell 9A first")
CS9, SRC = (1.0, 1.5, 2.0), ("ANT", "SIB", "NUM", "LINK", "TYPE")


def features(rows: list, raw: bool) -> dict:
    out = {"swap": np.array([r[4] for r in rows], bool), "only": np.array([r[5] for r in rows], bool)}
    for c in CS9:
        for same in (False, True):
            eqf, alt = np.zeros(len(rows)), np.zeros((len(rows), 5), bool)
            for i, (words, nmiss, tcl, nshare, _, _) in enumerate(rows):
                eqf[i] = nshare
                for _, share, rel, _, ant, sib, links in words:
                    lo, hi = links[2:] if raw else links[:2]
                    if rel in ("EQ", "FWD") or (same and lo <= -c):
                        eqf[i] += share
                    else:
                        alt[i, 0] |= ant; alt[i, 1] |= sib; alt[i, 3] |= hi >= c
                alt[i, 2], alt[i, 4] = nmiss, tcl
            out[c, same] = (eqf, alt)
    return out


def gate(F: dict, t_ins: float, t_anc: float, use: tuple, c: float, same: bool, hasp=None) -> np.ndarray:
    eqf, alt = F[c, same]
    st = np.where(1 - eqf <= t_ins + 1e-9, 0, 2)
    if use:
        st = np.where(alt[:, list(use)].any(1) & (eqf >= t_anc - 1e-9), 1, st)
    st = np.where(F["only"] & (st == 0), 2, st)
    st = np.where(F["swap"] & (st < 2), 1 - st, st)
    return np.where(hasp, st, 2) if hasp is not None else st


TT, USE9 = (0.1, 0.2, 0.3, 0.4, 0.5, 0.6), [tuple(i for i in range(5) if m >> i & 1) for m in range(32)]
GOLD_S, V9 = np.array([int(r["label"] == "REFUTES") for r in SYM]), {}
for ver, raw in (("v1", False), ("v2", True)):
    F, FS = features(R9, raw), features(RS, raw)
    FT = {k: ((v[0][:N_TR], v[1][:N_TR]) if isinstance(k, tuple) else v[:N_TR]) for k, v in F.items()}
    grid = {(a, b, u, c, s): 100 * np.mean(gate(FT, a, b, u, c, s, HASP[:N_TR]) == GOLD[:N_TR])
            for a in TT for b in TT for u in USE9 for c in CS9 for s in (False, True)}
    ch = max(grid, key=lambda k: (round(grid[k], 6), -len(k[2]), -k[4]))
    p, ps = gate(F, *ch, HASP), gate(FS, *ch)
    V9[ver] = dict(F=F, FS=FS, ch=ch, sample=grid[ch], P=p, PS=ps, scores=fever_scores(p[DV], DV_IDX),
                   sym=100 * np.mean((ps == 1) == (GOLD_S == 1)) if len(SYM) else float("nan"))
P9, P9S, CH9, ACC9, FSC9 = V9["v2"]["P"], V9["v2"]["PS"], V9["v2"]["ch"], *V9["v2"]["scores"][:2]
F9, FS9 = V9["v2"]["F"], V9["v2"]["FS"]
# ---------------- FEVER-Symmetric: claim only, Cell 8's rule, the gate ----------------
c8, s8 = CH8[3], CH8[4]
NB2, P8S = np.zeros(len(SYM), int), np.zeros(len(SYM), int)
for i, (r, E) in enumerate(zip(SYM, ESYM)):
    cwords = WORD.findall(r["claim"].casefold())
    NB2[i] = int(np.argmax([sum(math.log((NBC[l][w] + 1) / (NB_T[l] + NB_V)) for w in cwords) for l in (0, 1)]))
    text, ev = unbracket(r["claim"]), unbracket(r["evidence"])
    cw, cn, ns = set(forms(text)), set(NUMS.findall(text)), set(NUMS.findall(ev))
    total, extra, miss = sum(idf7(w) for w in cw) + NUM_IDF * len(cn), E - cw, cw - E
    same_w = {w for w in miss if any(CL.get((w, v), 0) <= -c8 for v in extra)} if s8 else set()
    cov = (sum(idf7(w) for w in (cw & E) | same_w) + NUM_IDF * len(cn & ns)) / total if total else 0.0
    cues = [bool(cn) and not cn <= ns and bool(ns - cn), bool(NEGW.search(text)) != bool(NEGW.search(ev)),
            any(a in E for w in miss for a in ANT.get(w, ())),
            any(ups(w) & ups(v) for w in miss for v in extra if v.isalpha()), bool(ONLYW.search(text)),
            any(CL.get((w, v), 0) >= c8 for w in miss for v in extra)]
    clash = any(cues[j] for j in CH8[2])
    P8S[i] = 1 if cov >= CH8[0] and clash else (0 if cov >= CH8[1] else 2)
print(f"\n  FEVER-SYMMETRIC dev, scored pairs ({len(SYM)}, {GOLD_S.sum()} REFUTES)   two-way   NEI wrong   said NEI")
SYMR = {}
for label, p in (("claim only (Cell 8's naive Bayes)", NB2), ("Cell 8's rule, same evidence", P8S),
                 ("gate v1 (corrected links)", V9["v1"]["PS"]), ("gate v2 (Cell 8's links)", P9S)):
    SYMR[label] = 100 * np.mean((p == 1) == (GOLD_S == 1)) if len(SYM) else float("nan")
    wrong = 100 * np.mean(p == GOLD_S) if len(SYM) else float("nan")
    print(f"    {label:<38}{SYMR[label]:>8.1f}{wrong:>12.1f}{int((p == 2).sum()):>11}")
print(f"\n  {'':<28}{'sample':>8}{'paper_dev':>10}{'FEVER score':>12}  chosen (t_ins, t_anc, exclusions, c, SAME)")
print(f"  {'Cell 8':<28}{GRID8[CH8]:>8.1f}{ACC8:>10.1f}{FS8:>12.1f}")
for ver, d in V9.items():
    a, b, u, c, sm = d["ch"]
    print(f"  {'gate ' + ver:<28}{d['sample']:>8.1f}{d['scores'][0]:>10.1f}{d['scores'][1]:>12.1f}  {a}, {b}, "
          f"{'+'.join(SRC[i] for i in u) or 'none'}, {c}, {'on' if sm else 'off'}")
print("  gate v2 on paper_dev:")
print("    gold \\ said          SUPPORTS   REFUTES       NEI")
for g, lab in enumerate(LABELS):
    print(f"    {lab:<18}" + "".join(f"{int(((GOLD[DV] == g) & (P9[DV] == p)).sum()):>10,}" for p in range(3)))
print(f"\n  CUES (v2, c {CH9[3]})   fires, sample   then REFUTES   fires, Symm.   then REFUTES")
alt_t, alt_s = F9[CH9[3], CH9[4]][1][:N_TR], FS9[CH9[3], CH9[4]][1]
for name, a, b in zip(SRC + ("POLARITY", "ONLY"), [*alt_t.T, F9["swap"][:N_TR], F9["only"][:N_TR]],
                      [*alt_s.T, FS9["swap"], FS9["only"]]):
    print(f"    {name:<17}{a.mean():>14.1%}{(GOLD[:N_TR][a] == 1).mean() if a.any() else 0:>15.1%}"
          f"{b.mean():>15.1%}{(GOLD_S[b] == 1).mean() if b.any() else 0:>15.1%}")


def proof(row) -> str:
    words, nmiss, tcl, _, swap, only = row
    steps = [f"{w}:{rel}" + (f"({p})" if p and p != w else "") + ("+ANT" if a else "") + ("+SIB" if s else "")
             for w, _, rel, p, a, s, _ in words]
    flags = [n for n, on in (("NUM-clash", nmiss), ("TYPE-clash", tcl), ("polarity-swap", swap), ("only", only)) if on]
    return " ".join(steps + flags)


print("\n  PROOFS (paper_dev: the first claim said SUPPORTS, REFUTES, NOT ENOUGH INFO)")
for want_p in range(3):
    hit = np.flatnonzero(P9[DV] == want_p)
    if len(hit):
        ci = N_TR + int(hit[0])
        print(f"    {FV_CLAIMS[ci]['claim'][:100]}\n      {proof(R9[ci])[:220]} -> {LABELS[P9[ci]]} "
              f"(gold {FV_CLAIMS[ci]['label']})")
if SYM:
    first = base_id(SYM[0])
    print(f"  PROOFS (FEVER-Symmetric, the pairs built from claim {first})")
    for i in (j for j, r in enumerate(SYM) if base_id(r) == first):
        print(f"    {SYM[i]['claim'][:70]} | {SYM[i]['evidence'][:70]}\n      {proof(RS[i])[:220]} -> "
              f"{LABELS[P9S[i]]} (gold {SYM[i]['label']})")

print("\n  PREDICTIONS vs RESULTS")
LEDGER.result("B5-N1", round(SYMR["claim only (Cell 8's naive Bayes)"], 2))
LEDGER.result("B5-N2", round(SYMR["Cell 8's rule, same evidence"], 2))
LEDGER.result("B5-N3", round(SYMR["gate v1 (corrected links)"], 2))
LEDGER.result("B5-N4", round(SYMR["gate v1 (corrected links)"] - SYMR["Cell 8's rule, same evidence"], 2))
LEDGER.result("B5-N5", round(V9["v1"]["scores"][0], 2))
LEDGER.result("B5-N6", round(time.time() - T9))
LEDGER.result("B5-N7", round(SYMR["gate v2 (Cell 8's links)"], 2))
LEDGER.result("B5-N8", round(ACC9, 2))
if "sync" in globals():
    sync("Cell 9")
print(f"\n  ({time.time() - T9:.0f}s since Cell 9A started)")
print("  LAST LINE OF MC AI BRAIN 5 CELL 9B")


MC AI BRAIN 5 - CELL 9B: FEVER - THE NATURAL-LOGIC GATE (NO TRAINING)

  FEVER-SYMMETRIC dev, scored pairs (684, 342 REFUTES)   two-way   NEI wrong   said NEI
    claim only (Cell 8's naive Bayes)         50.0        50.0          0
    Cell 8's rule, same evidence              65.4        54.8        206
    gate v1 (corrected links)                 61.3        47.5        278
    gate v2 (Cell 8's links)                  68.1        54.8        233

                                sample paper_dev FEVER score  chosen (t_ins, t_anc, exclusions, c, SAME)
  Cell 8                          59.9      58.4        52.6
  gate v1                         54.3      52.4        48.6  0.2, 0.1, ANT+NUM+TYPE, 2.0, on
  gate v2                         57.4      55.3        50.2  0.2, 0.1, ANT+NUM+LINK, 2.0, on
  gate v2 on paper_dev:
    gold \ said          SUPPORTS   REFUTES       NEI
    SUPPORTS               2,429       275       629
    REFUTES                  693     1,612     1,028
    NO

In [15]:
# =====================================================================
# MC AI BRAIN 5 - CELL 10A: FEVER - WHAT A CLAIM HAS LEFT OVER, AND THE
# DEFINITION NETWORK
# Needs Cells 7A-9B. Cell 10B counts what each reading does and scores.
# The readings below were designed after lab/look_cell10.py and
# lab/look_cell10b.py (fit data; look_cell10 also printed one table of
# FEVER-Symmetric dev, practice). Their weights are counted in Cell 10B
# on fit data only.
# LEFTOVER  a claim word (Cell 6's base forms, or a number) that is not in
#   the evidence, is no WordNet synonym of an evidence word, has no
#   evidence word that is a kind of it, and is not FAMILY
# FAMILY  the word and an evidence word are WordNet "derivationally
#   related forms" or pertainyms (nominate / nomination, Venezuela /
#   Venezuelan): the same word in another part of speech; counts as found
# DEFINITION NETWORK (Mark's; spreading activation, Collins & Loftus 1975)
#   a word's neighbours: the content words of the definitions of its
#   first 2 WordNet senses per part of speech, and its hypernyms' words.
#   A word's weight is its IDF over all WordNet definitions; a path is as
#   strong as its weakest stop (the honeycomb's bottleneck rule)
#   LINK1 one is in the other's definition; MEET1 their definitions share
#   a word; LINK2, MEET2 the same through one middle word of weight >= 4
# EIGHT READINGS of a claim against its evidence
#   KIND     of the strongest leftover: opposite (WordNet antonym there),
#            number (another number there), kind-of (a kind of an evidence
#            word), sibling (Cell 9's SIB), name (not in WordNet), word;
#            or family only; or nothing left
#   SLOT     claim and evidence compared in word order (difflib): the
#            leftover's gap holds 1-3 evidence words (tight swap), more
#            (loose swap), none (added)
#   DEFINED  a leftover has a LINK1 of strength >= 5 to an evidence word
#   GAP      to the evidence words in its own gap: LINK1 (linked), another
#            relation (related), none, at strength >= 5
#   LINKS    Cell 8's counted links, leftover against evidence words:
#            clash >= 2, same <= -2, both, neither. For the train claims
#            the links are counted again on the other half of the train
#            claims (cross-fitting: a count is not applied to the claims
#            it was counted on)
#   SHARE    of the claim's IDF held by the leftovers
#   POLARITY claim and first evidence sentence differ in polarity (Cell 9)
#   TYPE     Cell 9's kind clash after is/was
#   Read for: every 2nd train claim outside the sample that has gold
#   evidence (fit); the sample and paper_dev with Cell 7B's best 2
#   sentences; FEVER-Symmetric dev's scored pairs. Saved with Artifacts.
# PREDICTIONS (written before running)
#   B5-W0 train claims (gold evidence) whose only leftovers are FAMILY, %: 1.5-8
#   B5-W1 counted reader, train (gold evidence), balanced accuracy, %: 66-73
#   B5-W2 counted reader, Symmetric scored, two-way, %: 67-74
#   B5-W3 counted reader minus Cell 9's gate v2, Symmetric, points: 0-6
#   B5-W4 with minus without the definition network, Symmetric, points: -1.5-2
#   B5-W5 counted reader, three labels, paper_dev label accuracy, %: 55-61
#   B5-W6 its paper_dev FEVER score, %: 49-55
#   B5-T16 Cells 10A + 10B seconds, building: <= 900; B5-T17 loading: <= 300
#   (The first lab run applied Cell 8's links to the claims they were
#   counted on: train 83.9, Symmetric 75.3; without the links 79.1 and
#   76.5. W1-W3 did not hold there. Written after it, for this version:)
#   B5-W7 counted reader, train, balanced accuracy, %: 78-82
#   B5-W8 counted reader, Symmetric scored, two-way, %: 74-78
# =====================================================================
import difflib, math, re, time
from collections import Counter

import numpy as np

print("=" * 64); print("MC AI BRAIN 5 - CELL 10A: FEVER - LEFTOVERS AND THE DEFINITION NETWORK"); print("=" * 64)
T10 = time.time()
for need in ("V9", "ESYM", "SYM", "syns", "up_to", "deep", "heads", "type_clash", "POL9", "OPS9", "CL", "LOOK",
             "COUNTED", "top2", "base", "ANT"):
    if need not in globals():
        raise RuntimeError(f"run Cells 7A to 9B first ({need} is missing)")
for key, claim, rule in (("B5-W0", "train claims (gold evidence) whose only leftovers are FAMILY, %", between(1.5, 8)),
                         ("B5-W1", "counted reader, train (gold evidence), balanced accuracy, %", between(66, 73)),
                         ("B5-W2", "counted reader, Symmetric scored, two-way, %", between(67, 74)),
                         ("B5-W3", "counted reader minus gate v2, Symmetric, points", between(0, 6)),
                         ("B5-W4", "with minus without the definition network, Symmetric, points", between(-1.5, 2)),
                         ("B5-W5", "counted reader, three labels, paper_dev label accuracy, %", between(55, 61)),
                         ("B5-W6", "counted reader, three labels, paper_dev FEVER score, %", between(49, 55)),
                         ("B5-W7", "counted reader, cross-fitted links, train, balanced accuracy, %", between(78, 82)),
                         ("B5-W8", "counted reader, cross-fitted links, Symmetric, two-way, %", between(74, 78)),
                         ("B5-T16", "Cells 10A + 10B seconds, building", at_most(900)),
                         ("B5-T17", "Cells 10A + 10B seconds, loading", at_most(300))):
    LEDGER.predict(key, claim, rule)
F10 = (("KIND", ("opposite", "number", "kind-of", "sibling", "name", "word", "family only", "nothing left")),
       ("SLOT", ("tight swap", "loose swap", "added")),
       ("DEFINED", ("linked", "not linked", "not in WordNet")),
       ("GAP", ("linked", "related", "unrelated", "not in WordNet")),
       ("LINKS", ("clash", "same", "both", "neither")),
       ("SHARE", ("to 0.2", "to 0.4", "to 0.6", "over 0.6")),
       ("POLARITY", ("same", "swapped")),
       ("TYPE", ("no clash", "clash")))
COND10, T_DEF, HUB10 = (1, 2, 3, 4, 5), 5.0, 4.0
TOK10 = re.compile(r"\d+(?:[.,]\d+)*|[^\W\d_]+")

# ---------------- word families and the definition network ----------------
FAM, GDF, NEAR, TWO, NOFAM = {}, Counter(), {}, {}, frozenset()
for s in WN.synsets.values():
    GDF.update(set(forms(s.definition)))
    for q in s.pointers:
        if q.symbol in ("+", "\\") and q.source_word > 0 and q.target_word > 0:
            a = s.words[q.source_word - 1].split("(")[0].lower()
            b = WN.synsets[q.target].words[q.target_word - 1].split("(")[0].lower()
            if "_" not in a + b and a != b:
                FAM.setdefault(a, set()).add(b)
                FAM.setdefault(b, set()).add(a)
N_G = len(WN.synsets)


def gidf(x: str) -> float:
    d = GDF.get(x, 0)
    return math.log(1 + (N_G - d + 0.5) / (d + 0.5))


def near(w: str) -> frozenset:
    if w not in NEAR:
        out = set()
        for p in "nvar":
            for s in WN.synsets_of(w, p)[:2]:
                out |= set(forms(s.definition))
                for q in s.pointers:
                    if q.symbol in ("@", "@i"):
                        out |= set(forms(" ".join(WN.synsets[q.target].words).replace("_", " ")))
        NEAR[w] = frozenset(x for x in out if len(x) > 1 and x != w)
    return NEAR[w]


def two(w: str) -> dict:
    """word reached in 2 steps -> the weight of the strongest middle word (middle words of weight >= HUB10)"""
    if w not in TWO:
        out = {}
        for a in near(w):
            ga = gidf(a)
            if ga >= HUB10:
                for x in near(a):
                    if x != w and out.get(x, 0) < ga:
                        out[x] = ga
        TWO[w] = out
    return TWO[w]


def link1(w: str, v: str) -> float:
    return max(gidf(v) if v in near(w) else 0.0, gidf(w) if w in near(v) else 0.0)


def relate10(w: str, v: str) -> tuple:
    """strength of LINK1, and of the strongest of MEET1, LINK2, MEET2 with the stop it goes through"""
    a, b, a2, b2 = near(w), near(v), two(w), two(v)
    rest = [(gidf(m), m) for m in a & b]
    rest += [(min(a2[v], gidf(v)), v)] if v in a2 else []
    rest += [(min(b2[w], gidf(w)), w)] if w in b2 else []
    rest += [(min(gidf(m), b2[m]), m) for m in a if m in b2] + [(min(gidf(m), a2[m]), m) for m in b if m in a2]
    return link1(w, v), max(rest, default=(0.0, ""))


def seq10(text: str) -> list:
    out = []
    for t in TOK10.findall(text.lower()):
        if t[0].isdigit():
            out.append(t)
        elif t.isascii() and len(t) <= MAX_LEN and t not in STOPW:
            b = base(t)
            if len(b) > 1 and b not in STOPW and b not in OPS9:
                out.append(b)
    return out


def kind10(w: str, E: set, V: list, other_number: bool) -> int:
    if w[0].isdigit():
        return 1 if other_number else 5
    if any(a in E for a in ANT.get(w, ())):
        return 0
    if any(syns(v) & up_to(w, 4) for v in V):
        return 2
    u2 = up_to(w, 2)
    if any(deep(u2 & up_to(v, 2)) for v in V):
        return 3
    return 5 if syns(w) else 4


def read10(claim: str, E: set, sents: list, tell: bool = False, links: dict = None):
    """The 8 readings as numbers; with tell, also (leftovers, family words, the strongest network path)"""
    links = CL if links is None else links
    text, ev = unbracket(claim), [unbracket(s) for s in sents]
    c_seq, e_seq = seq10(text), seq10(" ".join(ev))
    C = set(c_seq)
    V = [v for v in E if v not in C and v.isalpha()]
    e_nums = {t for t in e_seq if t[0].isdigit()}
    left, fam = [], []
    for w in dict.fromkeys(c_seq):
        if w[0].isdigit():
            if w not in e_nums:
                left.append(w)
        elif w in E or any(syns(w) & syns(v) or syns(w) & up_to(v, 4) for v in V):
            continue
        elif FAM.get(w, NOFAM) & E:
            fam.append(w)
        else:
            left.append(w)
    pol = int(len(POL9.findall(text)) % 2 != (len(POL9.findall(ev[0])) % 2 if ev else 0))
    typ = int(type_clash(heads(text), frozenset().union(*map(heads, ev)), E))
    if not left:
        out = (6 if fam else 7, 0, 0, 0, 0, 0, pol, typ)
        return (out, [], fam, None) if tell else out
    total = sum(idf7(w) if w.isalpha() else NUM_IDF for w in C)
    share = sum(idf7(w) if w.isalpha() else NUM_IDF for w in left) / total
    kind = min(kind10(w, E, V, bool(e_nums - C)) for w in left)
    L, Vs, slot, gap = set(left), set(V), 2, {w: set() for w in left}
    for tag, i1, i2, j1, j2 in difflib.SequenceMatcher(None, e_seq, c_seq, autojunk=False).get_opcodes():
        if tag == "replace":
            hit = L.intersection(c_seq[j1:j2])
            if hit:
                slot = min(slot, 0 if i2 - i1 <= 3 else 1)
                for w in hit:
                    gap[w].update(v for v in e_seq[i1:i2] if v in Vs)
    lw = [w for w in left if w.isalpha() and syns(w)]
    d1, d2, path = 2, 3, None
    if lw:
        d1 = int(not any(link1(w, v) >= T_DEF for w in lw for v in V if syns(v)))
        best_link, best_rest = 0.0, (0.0, "", "", "")
        for w in lw:
            for v in gap[w]:
                if syns(v):
                    lk, (st, stop) = relate10(w, v)
                    best_link = max(best_link, lk)
                    best_rest = max(best_rest, (st, w, stop, v))
        d2 = 0 if best_link >= T_DEF else 1 if best_rest[0] >= T_DEF else 2
        path = best_rest if best_rest[0] >= T_DEF else None
    vals = [links[w, v] for w in left if w.isalpha() for v in V if (w, v) in links]
    hi, lo = max(vals, default=0.0) >= 2, min(vals, default=0.0) <= -2
    out = (kind, slot, d1, d2, 2 if hi and lo else 0 if hi else 1 if lo else 3,
           max(0, min(3, math.ceil(share * 5 - 1e-9) - 1)), pol, typ)
    return (out, left, fam, path) if tell else out


def gold_evidence(c: dict):
    """(evidence words, evidence sentences) of a counted train claim's first gold set; None without one"""
    sets = gold_sets(c)
    if c["label"] == "NOT ENOUGH INFO" or not sets or any((pg, sid) not in LOOK for pg, sid in sets[0]):
        return None
    E = set()
    for pg, sid in sets[0]:
        E |= set(forms(unbracket(LOOK[pg, sid]))) | set(forms(unbracket(pg)))
    return E, [LOOK[pg, sid] for pg, sid in sets[0]]


def found_evidence(ci: int):
    top = top2(ci)
    E = set().union(*(S_WORDS[k] | T_WORDS[FV["s_page"][k]] for k in top)) if top else set()
    return E, [sentence_text(k) for k in top]


def build10() -> dict:
    t1 = time.time()
    rows, labs, half, n_lab, n = [], [], {}, [0, 0], 0
    for c in COUNTED:
        got = gold_evidence(c)
        if got is None:
            continue
        n += 1
        if n % 2:
            C, k = set(forms(unbracket(c["claim"]))), int(c["label"] == "REFUTES")
            n_lab[k] += 1
            for w in C - got[0]:
                for v in got[0] - C:
                    half.setdefault((w, v), [0, 0])[k] += 1
    prior = math.log(n_lab[1] / n_lab[0])
    half = {wv: math.log((r + 1) / (s + 1)) - prior for wv, (s, r) in half.items() if s + r >= 3}
    n = 0
    for c in COUNTED:
        got = gold_evidence(c)
        if got is None:
            continue
        n += 1
        if n % 2 == 0:
            rows.append(read10(c["claim"], *got, links=half))
            labs.append(int(c["label"] == "REFUTES"))
    fv = [read10(FV_CLAIMS[ci]["claim"], *found_evidence(ci)) for ci in range(len(FV_CLAIMS))]
    sym = [read10(r["claim"], E, [r["evidence"]]) for r, E in zip(SYM, ESYM)]
    print(f"  READ ({time.time() - t1:.0f}s): {len(rows):,} train claims with gold evidence ({len(half):,} links "
          f"counted on the other half), {len(fv):,} sample and paper_dev claims, {len(sym)} Symmetric pairs")
    return dict(train=np.array(rows, np.int8), train_y=np.array(labs, np.int8), fv=np.array(fv, np.int8),
                sym=np.array(sym, np.int8).reshape(-1, 8))


DEPENDS10 = dict(what="8 readings of FEVER claims against evidence, Brain 5 Cell 10A", version=2, every=2,
                 strength=T_DEF, hub=HUB10, senses=2, seed=FV_SEED, per_label=FV_PER, synsets=N_G, pairs=len(SYM),
                 data={n: FEVER_PINS[n][0] for n in ("wiki-pages.zip", "train.jsonl", "paper_dev.jsonl")},
                 symmetric=SYM_PINS["fever_symmetric_dev.jsonl"][0])
FEAT10, BUILT10 = ARTS.get("fever_readings", DEPENDS10, build10)
X_TR, Y_TR = FEAT10["train"].astype(int), FEAT10["train_y"].astype(int)
X_FV, X_SY = FEAT10["fv"].astype(int), FEAT10["sym"].astype(int)
FAM_ONLY = 100 * np.mean(X_TR[:, 0] == 6)
print(f"  READINGS {'built and saved' if BUILT10 else 'loaded'}: {len(X_TR):,} train claims; word families of "
      f"{len(FAM):,} words; {len(GDF):,} words weighed over {N_G:,} definitions")
print(f"  KIND of leftover, train claims with gold evidence ({100 * Y_TR.mean():.1f}% REFUTES):")
for k, name in enumerate(F10[0][1]):
    rows = X_TR[:, 0] == k
    print(f"    {name:<14}{rows.mean():>7.1%} of claims, {Y_TR[rows].mean() if rows.any() else 0:>6.1%} REFUTES")
print("\n  PREDICTIONS vs RESULTS")
LEDGER.result("B5-W0", round(FAM_ONLY, 2))
if "sync" in globals():
    sync("Cell 10A")
print("  LAST LINE OF MC AI BRAIN 5 CELL 10A")


MC AI BRAIN 5 - CELL 10A: FEVER - LEFTOVERS AND THE DEFINITION NETWORK
  READ (31s): 51,459 train claims with gold evidence (55,241 links counted on the other half), 18,999 sample and paper_dev claims, 684 Symmetric pairs
  READINGS built and saved: 51,459 train claims; word families of 30,670 words; 32,777 words weighed over 117,659 definitions
  KIND of leftover, train claims with gold evidence (25.8% REFUTES):
    opposite         0.7% of claims,  77.9% REFUTES
    number           5.0% of claims,  62.0% REFUTES
    kind-of          3.6% of claims,  38.4% REFUTES
    sibling          7.5% of claims,  52.3% REFUTES
    name             3.1% of claims,  59.4% REFUTES
    word            33.4% of claims,  31.5% REFUTES
    family only      3.3% of claims,   8.4% REFUTES
    nothing left    43.4% of claims,   9.7% REFUTES

  PREDICTIONS vs RESULTS
    B5-W0 train claims (gold evidence) whose only leftovers are FAMILY, %: 3.34 (predicted between 1.5 and 8) -> HELD
  sync: pushed 5 files 

In [16]:
# =====================================================================
# MC AI BRAIN 5 - CELL 10B: FEVER - A READER THAT COUNTS WHAT EACH
# READING DOES
# Needs Cell 10A (its 8 readings). Predictions B5-W1 to B5-W8 are written
# in Cell 10A.
# COUNT  naive Bayes over the readings (counting; add-one smoothing; equal
#   label priors): P(reading's value | label) is counted, the label with
#   the largest product wins. SLOT, DEFINED, GAP, LINKS and SHARE are
#   counted and used only for claims with something left over
#   two labels   counted on train claims outside the sample with their
#                gold evidence (fit): SUPPORTS or REFUTES, the reading
#                task FEVER-Symmetric scores
#   three labels counted on Cell 7A's sample with the evidence Cell 7B
#                found (fit); a claim with no page is NOT ENOUGH INFO
# WEIGHT of a value = log P(value | REFUTES) - log P(value | SUPPORTS)
#   from the two-label count: above 0 speaks for REFUTES. A verdict's
#   weights add up, so each verdict prints with its reasons
# HUD CHECK  next to each weight, the share of REFUTES among FEVER-
#   Symmetric pairs with that value (practice; shown, not used)
# WITHOUT  the count repeated without the definition network (DEFINED and
#   GAP), without Cell 8's links, and with KIND alone
# =====================================================================
import time

import numpy as np

print("=" * 64); print("MC AI BRAIN 5 - CELL 10B: FEVER - A READER THAT COUNTS"); print("=" * 64)
if "X_TR" not in globals() or "read10" not in globals():
    raise RuntimeError("run Cell 10A first")
NV10, ALL10 = [len(v) for _, v in F10], tuple(range(len(F10)))


def count10(X: np.ndarray, y: np.ndarray, n_labels: int, use: tuple) -> dict:
    """reading -> log P(value | label), add-one smoothing"""
    has, model = X[:, 0] < 6, {}
    for f in use:
        rows = has if f in COND10 else np.ones(len(X), bool)
        c = np.zeros((n_labels, NV10[f]))
        np.add.at(c, (y[rows], X[rows, f]), 1)
        model[f] = np.log((c + 1) / (c.sum(1, keepdims=True) + NV10[f]))
    return model


def score10(model: dict, X: np.ndarray) -> np.ndarray:
    """(labels, claims) summed log P, equal priors"""
    has, s = X[:, 0] < 6, 0
    for f, m in model.items():
        s = s + (m[:, X[:, f]] * has if f in COND10 else m[:, X[:, f]])
    return s


def balanced(pred: np.ndarray, y: np.ndarray) -> float:
    return 100 * np.mean([np.mean(pred[y == g] == g) for g in sorted(set(y.tolist()))])


# ---------------- two labels: reading with the evidence given ----------------
GOLD_Y = np.array([int(r["label"] == "REFUTES") for r in SYM])
SETS10 = (("all 8 readings", ALL10), ("without the definition network", (0, 1, 4, 5, 6, 7)),
          ("without Cell 8's links", (0, 1, 2, 3, 5, 6, 7)), ("KIND alone", (0,)))
READ10 = {}
print(f"  TWO LABELS, counted on {len(X_TR):,} train claims with gold evidence ({100 * Y_TR.mean():.1f}% REFUTES)")
print(f"    {'':<34}{'train, balanced':>16}{'Symmetric, two-way':>20}")
for label, use in SETS10:
    m = count10(X_TR, Y_TR, 2, use)
    p_tr, p_sy = score10(m, X_TR).argmax(0), score10(m, X_SY).argmax(0)
    READ10[label] = (m, balanced(p_tr, Y_TR), 100 * np.mean(p_sy == GOLD_Y) if len(SYM) else float("nan"), p_sy)
    print(f"    {label:<34}{READ10[label][1]:>16.1f}{READ10[label][2]:>20.1f}")
M2, BAL10, SYM10, P_SY = READ10["all 8 readings"]
R8S = SYMR["Cell 8's rule, same evidence"]
print(f"    Cell 9's gate v2 {V9['v2']['sym']:.1f}; Cell 8's rule {R8S:.1f}; the claim alone 50.0")

print("\n  WHAT EACH READING DOES (weight above 0 speaks for REFUTES)")
print(f"    {'':<26}{'of train':>9}{'REFUTES':>9}{'weight':>8}   {'Symmetric: pairs':>17}{'REFUTES':>9}")
for f, (name, values) in enumerate(F10):
    tr = X_TR[:, 0] < 6 if f in COND10 else np.ones(len(X_TR), bool)
    sy = X_SY[:, 0] < 6 if f in COND10 else np.ones(len(X_SY), bool)
    for k, value in enumerate(values):
        a, b = tr & (X_TR[:, f] == k), sy & (X_SY[:, f] == k)
        if a.any():
            print(f"    {name + ' ' + value:<26}{a.sum() / tr.sum():>9.1%}{Y_TR[a].mean():>9.1%}"
                  f"{M2[f][1, k] - M2[f][0, k]:>+8.2f}   {int(b.sum()):>17}"
                  f"{GOLD_Y[b].mean() if b.any() else 0:>9.1%}")

# ---------------- three labels: the whole pipeline ----------------
HAS_TR = HASP[:N_TR]
THREE10 = {}
for label, use in SETS10[:2]:
    m3 = count10(X_FV[:N_TR][HAS_TR], GOLD[:N_TR][HAS_TR], 3, use)
    p3 = np.where(HASP, score10(m3, X_FV).argmax(0), 2)
    THREE10[label] = (p3, 100 * np.mean(p3[:N_TR] == GOLD[:N_TR]), *fever_scores(p3[DV], DV_IDX)[:2])
P10, SAMPLE10, ACC10, FS10 = THREE10["all 8 readings"]
print(f"\n  THREE LABELS, counted on the sample ({int(HAS_TR.sum()):,} claims with a page)")
print(f"    {'':<40}{'sample':>6}{'paper_dev':>11}{'FEVER score':>13}")
print(f"    {'Cell 8 (cues and links)':<40}{GRID8[CH8]:>6.1f}{ACC8:>11.1f}{FS8:>13.1f}")
G2 = V9["v2"]
print(f"    {'Cell 9 gate v2':<40}{G2['sample']:>6.1f}{G2['scores'][0]:>11.1f}{G2['scores'][1]:>13.1f}")
for label, (_, a, b, c) in THREE10.items():
    print(f"    {'counted, ' + label:<40}{a:>6.1f}{b:>11.1f}{c:>13.1f}")
print(f"    REFUTES said REFUTES {100 * np.mean(P10[DV][GOLD[DV] == 1] == 1):.1f}%; counted reader on paper_dev:")
print("    gold \\ said          SUPPORTS   REFUTES       NEI")
for g, lab in enumerate(LABELS):
    print(f"    {lab:<18}" + "".join(f"{int(((GOLD[DV] == g) & (P10[DV] == p)).sum()):>10,}" for p in range(3)))


def reasons(x: np.ndarray) -> str:
    parts = [(M2[f][1, x[f]] - M2[f][0, x[f]], f"{F10[f][0]} {F10[f][1][x[f]]}") for f in ALL10
             if f not in COND10 or x[0] < 6]
    return ", ".join(f"{name} {w:+.2f}" for w, name in sorted(parts, key=lambda t: -abs(t[0]))[:5])


if SYM:
    first = base_id(SYM[0])
    print(f"\n  VERDICTS WITH REASONS (FEVER-Symmetric, the pairs built from claim {first}; two labels)")
    for i in (j for j, r in enumerate(SYM) if base_id(r) == first):
        _, left, fam, path = read10(SYM[i]["claim"], ESYM[i], [SYM[i]["evidence"]], tell=True)
        net = f"; network: {path[1]} -> {path[2]} <- {path[3]} ({path[0]:.1f})" if path else ""
        print(f"    {SYM[i]['claim'][:66]} | {SYM[i]['evidence'][:66]}\n      left over: {' '.join(left) or '-'}"
              f"{'; family: ' + ' '.join(fam) if fam else ''}{net}\n      {reasons(X_SY[i])} -> "
              f"{LABELS[P_SY[i]]} (gold {SYM[i]['label']})")

print("\n  PREDICTIONS vs RESULTS")
LEDGER.result("B5-W1", round(BAL10, 2))
LEDGER.result("B5-W2", round(SYM10, 2))
LEDGER.result("B5-W3", round(SYM10 - V9["v2"]["sym"], 2))
LEDGER.result("B5-W4", round(SYM10 - READ10["without the definition network"][2], 2))
LEDGER.result("B5-W5", round(ACC10, 2))
LEDGER.result("B5-W6", round(FS10, 2))
LEDGER.result("B5-W7", round(BAL10, 2))
LEDGER.result("B5-W8", round(SYM10, 2))
LEDGER.result("B5-T16" if BUILT10 else "B5-T17", round(time.time() - T10))
if "sync" in globals():
    sync("Cell 10")
print(f"\n  ({time.time() - T10:.0f}s since Cell 10A started)")
print("  LAST LINE OF MC AI BRAIN 5 CELL 10B")


MC AI BRAIN 5 - CELL 10B: FEVER - A READER THAT COUNTS
  TWO LABELS, counted on 51,459 train claims with gold evidence (25.8% REFUTES)
                                       train, balanced  Symmetric, two-way
    all 8 readings                                81.5                77.5
    without the definition network                80.9                76.0
    without Cell 8's links                        79.1                76.5
    KIND alone                                    69.8                70.0
    Cell 9's gate v2 68.1; Cell 8's rule 65.4; the claim alone 50.0

  WHAT EACH READING DOES (weight above 0 speaks for REFUTES)
                               of train  REFUTES  weight    Symmetric: pairs  REFUTES
    KIND opposite                  0.7%    77.9%   +2.31                  10    90.0%
    KIND number                    5.0%    62.0%   +1.54                  36    80.6%
    KIND kind-of                   3.6%    38.4%   +0.58                  21    61.9%
    KIND sibling

In [17]:
# =====================================================================
# MC AI BRAIN 5 - CELL 11A: FEVER - WHO DID WHAT, LENT LINKS, AND PRACTICE
# MADE FROM BEACONS
# Needs Cells 7A-10B. Cell 11B counts and scores.
# Written after lab/redteam10.py (fit data; its map was also laid over
# FEVER-Symmetric dev, practice): Cell 10's reader kept SUPPORTS for 99.9%
# of claims whose two names had traded places, and was right on 65% of
# Symmetric pairs whose leftover was a word with no counted link.
# TWO MORE READINGS of a claim against its evidence (9th and 10th)
#   ROLE  names: capitalized runs of the claim that the evidence also has
#         (a leading The/A/An dropped; of, the, de, la, del, di, von, van
#         allowed inside). An evidence sentence that does not hold its
#         page's name (it says He, She, It) gets that name put before it.
#         With two or more names, take the claim's first and last. MARKER
#         of a name: the word right before it if it is one of by in of at
#         on for with to from as than into against (the, a, an, and, or
#         skipped), else none. Compare the claim's second name with
#         whichever of the two comes later in the evidence:
#         same order, same marker    same roles
#         other order, same marker   traded places
#         same order, other marker   (as named)
#         other order, other marker  (as named: reworded)
#   LENT  for a claim whose leftovers have no counted link of their own
#         (Cell 8's links): links of NEIGHBOUR words are lent. Neighbours:
#         WordNet family forms, synonyms of the first sense, and the other
#         kinds under the first noun sense's hypernym when that hypernym
#         has 60 kinds or fewer. (w', v) and (w, v') are looked up; lent
#         clash >= 2, lent same <= -2 (similarity smoothing, Dagan, Lee &
#         Pereira 1999)
#   For the train claims, links are counted on the other half of the train
#   claims, as in Cell 10A (cross-fitting).
# PRACTICE FROM BEACONS (counterfactual data augmentation, Kaushik et al.
#   2020): train claims that are SUPPORTS with nothing left over, polarity
#   the same and no type clash (Cell 10's largest beacon), up to 8,000.
#   Each is changed in two ways whose effect is known:
#   OTHER NAME  the page's name -> the next such claim's page name
#   ROLE SWAP   the page's name and another name of the claim trade places
#   Both are REFUTES. The beacons take turns three ways: the changes of
#   the 1st are practice (Cell 11B may add them to the counts), of the 2nd
#   choose how much the practice weighs, of the 3rd are held out to score
#   on. The unchanged beacons are train claims and stay in the counts.
# PREDICTIONS (written before running, for the first draft. Before any
#   run on real data the draft was changed, after reading it and a run on
#   made-up data: ROLE is given the page names, and the practice weight is
#   chosen and scored on different beacons. The ranges are as first
#   written)
#   B5-X1 train claims with two names the evidence also has, %: 15-60
#   B5-X2 of those that are SUPPORTS, read "traded places", %: 1-15
#   B5-X3 train claims with a leftover and no own link that are lent one, %: 5-40
#   B5-X4 held-out ROLE SWAP called REFUTES, chosen reader, %: 40-95
#   B5-X5 held-out OTHER NAME called REFUTES, chosen reader, %: 88-100
#   B5-X6 held-out unchanged claims still SUPPORTS, chosen reader, %: 90-100
#   B5-X7 chosen reader minus Cell 10's, train balanced accuracy, points: -0.5-1
#   B5-X8 chosen reader minus Cell 10's, Symmetric two-way, points: -1.5-2.5
#   B5-X9 three labels, 10 readings, paper_dev label accuracy, %: 55-59
#   B5-T18 Cells 11A + 11B seconds: <= 600
# =====================================================================
import math, re, time

import numpy as np

print("=" * 64); print("MC AI BRAIN 5 - CELL 11A: FEVER - ROLES, LENT LINKS, PRACTICE FROM BEACONS"); print("=" * 64)
T11 = time.time()
for need in ("read10", "score10", "M2", "X_TR", "Y_TR", "X_SY", "X_FV", "gold_evidence", "gold_sets", "FAM", "seq10",
             "SYM10", "top2", "sentence_text", "FV_TITLES", "DEV_CI", "base_id"):
    if need not in globals():
        raise RuntimeError(f"run Cells 7A to 10B first ({need} is missing)")
for key, claim, rule in (("B5-X1", "train claims with two names the evidence also has, %", between(15, 60)),
                         ("B5-X2", "SUPPORTS train claims with two names read 'traded places', %", between(1, 15)),
                         ("B5-X3", "train claims with a leftover and no own link that are lent one, %", between(5, 40)),
                         ("B5-X4", "held-out ROLE SWAP called REFUTES, chosen reader, %", between(40, 95)),
                         ("B5-X5", "held-out OTHER NAME called REFUTES, chosen reader, %", between(88, 100)),
                         ("B5-X6", "held-out unchanged claims still SUPPORTS, chosen reader, %", between(90, 100)),
                         ("B5-X7", "chosen reader minus Cell 10's, train balanced accuracy, points", between(-0.5, 1)),
                         ("B5-X8", "chosen reader minus Cell 10's, Symmetric two-way, points", between(-1.5, 2.5)),
                         ("B5-X9", "three labels, 10 readings, paper_dev label accuracy, %", between(55, 59)),
                         ("B5-T18", "Cells 11A + 11B seconds", at_most(600))):
    LEDGER.predict(key, claim, rule)
F11 = F10 + (("ROLE", ("fewer than two names", "same roles", "traded places", "same order, other marker",
                       "other order, other marker")),
             ("LENT", ("own link", "lent clash", "lent same", "lent both", "nothing lent")))
COND11 = COND10 + (9,)
NAME11 = re.compile(r"[A-Z][\w'-]*(?:(?: (?:of|the|de|la|del|di|von|van))* [A-Z][\w'-]*)*")
MARK11 = frozenset("by in of at on for with to from as than into against".split())
SKIP11 = frozenset("the a an and or".split())
LEAD11 = ("The ", "An ", "A ")
HUB11, MAX_BASES, NEIGH11 = 60, 8000, {}
PARTS11, KINDS11 = ("practice", "choose", "score"), ("OTHER NAME", "ROLE SWAP", "unchanged")


def page_name(title: str) -> str:
    return unbracket(title).split(" (")[0]


def led11(sents: list, pages: list) -> list:
    """each sentence with the names of its pages put before it, when it does not hold them"""
    out = []
    for s, titles in zip(sents, pages):
        s = unbracket(s)
        out.append(" . ".join([n for n in dict.fromkeys(map(page_name, titles)) if n and n not in s] + [s]))
    return out


def names11(text: str, ev: str) -> list:
    """(position, name) of the capitalized runs of text that ev also has; a leading The, A, An is dropped"""
    out = []
    for m in NAME11.finditer(text):
        s, at = m.group(), m.start()
        for lead in LEAD11:
            if s.startswith(lead):
                s, at = s[len(lead):], at + len(lead)
        if len(s) > 2 and s.lower() not in STOPW and s in ev and all(s != n for _, n in out):
            out.append((at, s))
    return out


def marker11(text: str, at: int) -> str:
    for t in reversed(text[:at].replace(",", " ").split()[-3:]):
        if t.lower() not in SKIP11:
            return t.lower() if t.lower() in MARK11 else ""
    return ""


def role11(claim: str, sents: list) -> int:
    text, ev = unbracket(claim), " ".join(unbracket(s) for s in sents)
    found = names11(text, ev)
    if len(found) < 2:
        return 0
    a, (b_at, b) = found[0][1], found[-1]
    ea, eb = ev.find(a), ev.find(b)
    same_mark = marker11(text, b_at) == marker11(ev, max(ea, eb))
    return (1 if same_mark else 3) if ea < eb else (2 if same_mark else 4)


def neighbours11(w: str) -> frozenset:
    if w not in NEIGH11:
        out = set(FAM.get(w, ()))
        for p in "nvar":
            for s in WN.synsets_of(w, p)[:1]:
                out.update(x.lower() for x in s.words if x.isalpha())
                for q in s.pointers if p == "n" else ():
                    if q.symbol in ("@", "@i"):
                        kids = [d.target for d in WN.synsets[q.target].pointers if d.symbol in ("~", "~i")]
                        if len(kids) <= HUB11:
                            for k in kids:
                                out.update(x.lower() for x in WN.synsets[k].words if x.isalpha())
        out.discard(w)
        NEIGH11[w] = frozenset(out)
    return NEIGH11[w]


def index11(links: dict) -> tuple:
    by_w, by_v = {}, {}
    for (w, v), x in links.items():
        by_w.setdefault(w, {})[v] = x
        by_v.setdefault(v, {})[w] = x
    return by_w, by_v


def lent11(left: list, V: list, links: dict, index: tuple) -> int:
    words = [w for w in left if w.isalpha()]
    if any((w, v) in links for w in words for v in V):
        return 0
    by_w, by_v = index
    got = []
    for w in words:
        near_w, row_w = neighbours11(w), by_w.get(w)
        for v in V:
            row_v = by_v.get(v)
            if row_v:
                got += [row_v[x] for x in near_w if x in row_v]
            if row_w:
                got += [row_w[x] for x in neighbours11(v) if x in row_w]
    hi, lo = max(got, default=0.0) >= 2, min(got, default=0.0) <= -2
    return 3 if hi and lo else 1 if hi else 2 if lo else 4


def read11(claim: str, E: set, sents: list, pages: list, links: dict = None, index: tuple = None) -> tuple:
    """Cell 10A's 8 readings, then ROLE and LENT; pages: for each sentence, the titles of its pages"""
    links, index = (CL, INDEX_CL) if links is None else (links, index)
    out, left, _, _ = read10(claim, E, sents, tell=True, links=links)
    C = set(seq10(unbracket(claim)))
    V = [v for v in E if v not in C and v.isalpha()]
    return out + (role11(claim, led11(sents, pages)), lent11(left, V, links, index) if left else 0)


def found11(ci: int) -> tuple:
    """Cell 10A's found_evidence, and each sentence's page title"""
    top = top2(ci)
    E = set().union(*(S_WORDS[k] | T_WORDS[FV["s_page"][k]] for k in top)) if top else set()
    return E, [sentence_text(k) for k in top], [(FV_TITLES[FV["s_page"][k]],) for k in top]


def sym_pages(r: dict) -> tuple:
    """the gold pages of the paper_dev claim a Symmetric pair was built from (the pages of Cell 9A's sym_title)"""
    return tuple(dict.fromkeys(pg for s in gold_sets(FV_CLAIMS[DEV_CI[base_id(r)]]) for pg, _ in s))


def with_lead(text: str, at: int, s: str) -> str:
    """the name at that place, with a capitalized The, An or A right before it"""
    for a in LEAD11:
        if text[:at].endswith(a) and (at == len(a) or text[at - len(a) - 1] == " "):
            return a + s
    return s


# ---------------- links counted on the other half of the train claims (as in Cell 10A) ----------------
t1 = time.time()
INDEX_CL = index11(CL)
HALF11, n_lab, n = {}, [0, 0], 0
for c in COUNTED:
    got = gold_evidence(c)
    if got is None:
        continue
    n += 1
    if n % 2:
        C, k = set(forms(unbracket(c["claim"]))), int(c["label"] == "REFUTES")
        n_lab[k] += 1
        for w in C - got[0]:
            for v in got[0] - C:
                HALF11.setdefault((w, v), [0, 0])[k] += 1
prior = math.log(n_lab[1] / n_lab[0])
HALF11 = {wv: math.log((r + 1) / (s + 1)) - prior for wv, (s, r) in HALF11.items() if s + r >= 3}
INDEX_HALF = index11(HALF11)

# ---------------- the 10 readings; beacons kept as bases ----------------
rows, BASES, n = [], [], 0
for c in COUNTED:
    got = gold_evidence(c)
    if got is None:
        continue
    n += 1
    if n % 2 == 0:
        pages = [(pg,) for pg, _ in gold_sets(c)[0]]
        x = read11(c["claim"], *got, pages, links=HALF11, index=INDEX_HALF)
        rows.append(x)
        if c["label"] == "SUPPORTS" and x[0] == 7 and x[6] == 0 and x[7] == 0 and len(BASES) < MAX_BASES:
            BASES.append((c["claim"], got[0], got[1], pages, page_name(pages[0][0]), x))
X11_TR = np.array(rows, int)
X11_FV = np.array([read11(FV_CLAIMS[ci]["claim"], *found11(ci)) for ci in range(len(FV_CLAIMS))], int)
X11_SY = np.array([read11(r["claim"], E, [r["evidence"]], [sym_pages(r)]) for r, E in zip(SYM, ESYM)],
                  int).reshape(-1, 10)
SAME8 = X11_TR.shape == (len(X_TR), 10) and np.array_equal(X11_TR[:, :8], X_TR)
print(f"  READ ({time.time() - t1:.0f}s): {len(X11_TR):,} train claims, {len(X11_FV):,} sample and paper_dev claims, "
      f"{len(X11_SY)} Symmetric pairs; the first 8 readings equal Cell 10A's: {'yes' if SAME8 else 'NO'}")
if not SAME8:
    raise RuntimeError("Cell 11A's first 8 readings differ from Cell 10A's: rerun Cell 10A, then this cell")

# ---------------- changes made from the bases: practice, to choose the weight, held out ----------------
t1 = time.time()
MADE = {part: {kind: [] for kind in KINDS11} for part in PARTS11}
SHOW11, UNSEEN11 = [], 0
for i, (claim, E, sents, pages, name, x) in enumerate(BASES):
    part, other, text = PARTS11[i % 3], BASES[(i + 1) % len(BASES)][4], unbracket(claim)
    made = {"unchanged": None}
    if name and name in text:
        if other and other != name and other not in text:
            made["OTHER NAME"] = text.replace(name, other, 1)
        second = [(at, s) for at, s in names11(text, " ".join(led11(sents, pages))) if s not in name and name not in s]
        if second:
            a, b = with_lead(text, text.find(name), name), with_lead(text, *second[0])
            made["ROLE SWAP"] = text.replace(a, "\x00", 1).replace(b, a, 1).replace("\x00", b, 1)
    for kind, edited in made.items():
        y = x if edited is None else read11(edited, E, sents, pages, links=HALF11, index=INDEX_HALF)
        MADE[part][kind].append(y)
        if kind == "ROLE SWAP":
            UNSEEN11 += y[8] == 0
            if part == "score" and len(SHOW11) < 3:
                SHOW11.append((text, edited, y))


def made11(part: str, kinds: tuple) -> np.ndarray:
    return np.array([y for kind in kinds for y in MADE[part][kind]], int).reshape(-1, 10)


PRACT11 = made11("practice", KINDS11[:2])
CHOOSE11 = {kind: made11("choose", (kind,)) for kind in KINDS11}
HELD11 = {kind: made11("score", (kind,)) for kind in KINDS11}
N_SWAP = sum(len(MADE[part]["ROLE SWAP"]) for part in PARTS11)
print(f"  CHANGED CLAIMS ({time.time() - t1:.0f}s) from {len(BASES):,} beacon claims, as other names + role swaps: "
      f"practice {len(MADE['practice']['OTHER NAME']):,} + {len(MADE['practice']['ROLE SWAP']):,}; to choose the "
      f"weight {len(CHOOSE11['OTHER NAME']):,} + {len(CHOOSE11['ROLE SWAP']):,}; held out "
      f"{len(HELD11['OTHER NAME']):,} + {len(HELD11['ROLE SWAP']):,}")
print(f"    role swaps that ROLE reads as fewer than two names: {UNSEEN11:,} of {N_SWAP:,}")

two = X11_TR[:, 8] > 0
left_no_link = (X11_TR[:, 0] < 6) & (X11_TR[:, 9] > 0)
print(f"\n  ROLE on train claims with gold evidence ({two.mean():.1%} have two names the evidence also has):")
for k, name in enumerate(F11[8][1]):
    r = X11_TR[:, 8] == k
    print(f"    {name:<28}{r.mean():>7.1%} of claims, {Y_TR[r].mean() if r.any() else 0:>6.1%} REFUTES")
print(f"  LENT on train claims with a leftover ({left_no_link.sum():,} have no link of their own):")
for k, name in enumerate(F11[9][1]):
    r = (X11_TR[:, 0] < 6) & (X11_TR[:, 9] == k)
    print(f"    {name:<28}{r.sum():>7,} claims, {Y_TR[r].mean() if r.any() else 0:>6.1%} REFUTES")
sup2 = two & (Y_TR == 0)
print("\n  PREDICTIONS vs RESULTS")
LEDGER.result("B5-X1", round(100 * two.mean(), 2))
LEDGER.result("B5-X2", round(100 * np.mean(X11_TR[sup2, 8] == 2) if sup2.any() else 0.0, 2))
LEDGER.result("B5-X3", round(100 * np.mean(X11_TR[left_no_link, 9] < 4) if left_no_link.any() else 0.0, 2))
if "sync" in globals():
    sync("Cell 11A")
print("  LAST LINE OF MC AI BRAIN 5 CELL 11A")


MC AI BRAIN 5 - CELL 11A: FEVER - ROLES, LENT LINKS, PRACTICE FROM BEACONS
  READ (33s): 51,459 train claims, 18,999 sample and paper_dev claims, 684 Symmetric pairs; the first 8 readings equal Cell 10A's: yes
  CHANGED CLAIMS (4s) from 8,000 beacon claims, as other names + role swaps: practice 2,606 + 1,231; to choose the weight 2,598 + 1,231; held out 2,610 + 1,256
    role swaps that ROLE reads as fewer than two names: 118 of 3,718

  ROLE on train claims with gold evidence (41.2% have two names the evidence also has):
    fewer than two names          58.8% of claims,  30.6% REFUTES
    same roles                    24.7% of claims,  20.7% REFUTES
    traded places                  1.5% of claims,   7.8% REFUTES
    same order, other marker      12.1% of claims,  18.9% REFUTES
    other order, other marker      2.9% of claims,  10.4% REFUTES
  LENT on train claims with a leftover (7,272 have no link of their own):
    own link                     20,152 claims,  35.5% REFUTES
    l

In [18]:
# =====================================================================
# MC AI BRAIN 5 - CELL 11B: FEVER - THE READER COUNTS AGAIN, WITH ROLES,
# LENT LINKS AND PRACTICE
# Needs Cell 11A (10 readings; the changed claims for practice, for
# choosing the weight, and held out). Predictions B5-X4 to B5-X9 are
# written in Cell 11A.
# COUNT  Cell 10B's naive Bayes (add-one smoothing, equal label priors),
#   now over 10 readings; LENT, like SLOT to SHARE, is counted and used
#   only for claims with something left over. Counts can carry a weight.
# READINGS ALONE  two labels counted on the train claims (fit): Cell 10's
#   8, with ROLE, with LENT, all 10
# PRACTICE  the practice rows (all REFUTES) are added to the train counts
#   with weight L each, L in 0, 0.05, 0.1, 0.25, 0.5, 1. For each L: train
#   balanced accuracy on the real claims, and on the changed claims that
#   choose the weight the share called REFUTES (OTHER NAME, ROLE SWAP) and
#   the share of their unchanged beacons still SUPPORTS
#   CHOSEN L: among the L whose real train balanced accuracy is within 0.5
#   points of L = 0, the highest mean of those three shares (ties: the
#   smaller L). FEVER-Symmetric is shown, not used
#   HELD OUT  the same three shares, for the chosen L only, on the changed
#   claims that were neither practice nor used to choose L. The unchanged
#   beacons are train claims: they are in the counts (fit)
# THREE LABELS  counted on Cell 7A's sample (fit) with all 10 readings and
#   no practice rows; a claim with no page is NOT ENOUGH INFO
# =====================================================================
import time

import numpy as np

print("=" * 64); print("MC AI BRAIN 5 - CELL 11B: FEVER - ROLES, LENT LINKS AND PRACTICE, COUNTED"); print("=" * 64)
if "X11_TR" not in globals() or "PRACT11" not in globals() or "CHOOSE11" not in globals():
    raise RuntimeError("run Cell 11A first")
NV11, ALL11 = [len(v) for _, v in F11], tuple(range(len(F11)))


def count11(X: np.ndarray, y: np.ndarray, w: np.ndarray, n_labels: int, use: tuple) -> dict:
    """reading -> log P(value | label) from weighted counts, add-one smoothing"""
    has, model = X[:, 0] < 6, {}
    for f in use:
        rows = has if f in COND11 else np.ones(len(X), bool)
        c = np.zeros((n_labels, NV11[f]))
        np.add.at(c, (y[rows], X[rows, f]), w[rows])
        model[f] = np.log((c + 1) / (c.sum(1, keepdims=True) + NV11[f]))
    return model


def score11(model: dict, X: np.ndarray) -> np.ndarray:
    has, s = X[:, 0] < 6, np.zeros((len(next(iter(model.values()))), len(X)))
    for f, m in model.items():
        s = s + (m[:, X[:, f]] * has if f in COND11 else m[:, X[:, f]])
    return s


def said(model: dict, X: np.ndarray) -> np.ndarray:
    return score11(model, X).argmax(0) if len(X) else np.zeros(0, int)


def share(pred: np.ndarray, label: int) -> float:
    return 100 * float(np.mean(pred == label)) if len(pred) else 0.0


def shares11(model: dict, rows: dict) -> tuple:
    """(OTHER NAME called REFUTES, ROLE SWAP called REFUTES, unchanged still SUPPORTS), %"""
    return (share(said(model, rows["OTHER NAME"]), 1), share(said(model, rows["ROLE SWAP"]), 1),
            share(said(model, rows["unchanged"]), 0))


def sym11(model: dict) -> float:
    return 100 * float(np.mean(said(model, X11_SY) == GOLD_Y)) if len(X11_SY) else 0.0


ONES = np.ones(len(X11_TR))
print(f"  READINGS ALONE, two labels, counted on {len(X11_TR):,} train claims with gold evidence")
print(f"    {'':<26}{'train, balanced':>16}{'Symmetric, two-way':>20}")
ALONE = {}
for label, use in (("Cell 10's 8 readings", ALL11[:8]), ("with ROLE", ALL11[:9]),
                   ("with LENT", ALL11[:8] + (9,)), ("all 10", ALL11)):
    m = count11(X11_TR, Y_TR, ONES, 2, use)
    ALONE[label] = (balanced(said(m, X11_TR), Y_TR), sym11(m))
    print(f"    {label:<26}{ALONE[label][0]:>16.1f}{ALONE[label][1]:>20.1f}")
print(f"    (Cell 10B printed {BAL10:.1f} and {SYM10:.1f} for its 8 readings)")

# ---------------- practice from beacons: how much weight ----------------
X_ALL = np.vstack([X11_TR, PRACT11])
Y_ALL = np.concatenate([Y_TR, np.ones(len(PRACT11), int)])
TRY11 = {}
print(f"\n  PRACTICE, all 10 readings: {len(PRACT11):,} changed claims added as REFUTES with weight L each")
print(f"    on the claims that choose L:\n    {'L':>5}{'train, balanced':>17}{'OTHER NAME':>12}{'ROLE SWAP':>11}"
      f"{'unchanged':>11}{'Symmetric':>11}")
for lam in (0.0, 0.05, 0.1, 0.25, 0.5, 1.0):
    m = count11(X_ALL, Y_ALL, np.concatenate([ONES, np.full(len(PRACT11), lam)]), 2, ALL11)
    TRY11[lam] = (m, balanced(said(m, X11_TR), Y_TR), shares11(m, CHOOSE11), sym11(m))
    a, b, c = TRY11[lam][2]
    print(f"    {lam:>5}{TRY11[lam][1]:>17.1f}{a:>12.1f}{b:>11.1f}{c:>11.1f}{TRY11[lam][3]:>11.1f}")
FLOOR = TRY11[0.0][1] - 0.5
LAM11 = max((lam for lam in TRY11 if TRY11[lam][1] >= FLOOR),
            key=lambda lam: (round(float(np.mean(TRY11[lam][2])), 6), -lam))
M11, BAL11, _, SYM11 = TRY11[LAM11]
HELD_C, HELD_0 = shares11(M11, HELD11), shares11(TRY11[0.0][0], HELD11)
P11_SY = said(M11, X11_SY)
print(f"    chosen L = {LAM11} (real train balanced accuracy must stay at or above {FLOOR:.1f})")
print(f"    HELD OUT ({len(HELD11['OTHER NAME']):,} other names, {len(HELD11['ROLE SWAP']):,} role swaps, "
      f"{len(HELD11['unchanged']):,} unchanged): OTHER NAME {HELD_C[0]:.1f}, ROLE SWAP {HELD_C[1]:.1f}, unchanged "
      f"{HELD_C[2]:.1f} (with L = 0: {HELD_0[0]:.1f}, {HELD_0[1]:.1f}, {HELD_0[2]:.1f})")
print(f"    chosen reader: train balanced {BAL11:.1f} (Cell 10: {BAL10:.1f}); Symmetric two-way {SYM11:.1f} "
      f"(Cell 10: {SYM10:.1f})")

print("\n  WHAT ROLE AND LENT DO (weight above 0 speaks for REFUTES; counted without, then with, the practice)")
print(f"    {'':<32}{'of train':>9}{'REFUTES':>9}{'weight':>8}{'with':>7}   {'Symmetric: pairs':>17}{'REFUTES':>9}")
M0 = TRY11[0.0][0]
for f in (8, 9):
    name, values = F11[f]
    tr = X11_TR[:, 0] < 6 if f in COND11 else np.ones(len(X11_TR), bool)
    sy = X11_SY[:, 0] < 6 if f in COND11 else np.ones(len(X11_SY), bool)
    for k, value in enumerate(values):
        a, b = tr & (X11_TR[:, f] == k), sy & (X11_SY[:, f] == k)
        if a.any():
            print(f"    {name + ' ' + value:<32}{a.sum() / tr.sum():>9.1%}{Y_TR[a].mean():>9.1%}"
                  f"{M0[f][1, k] - M0[f][0, k]:>+8.2f}{M11[f][1, k] - M11[f][0, k]:>+7.2f}   {int(b.sum()):>17}"
                  f"{GOLD_Y[b].mean() if b.any() else 0:>9.1%}")
print(f"\n  ROLE AS READ, % of the held-out claims{'under 2':>10}{'same':>7}{'traded':>8}{'o=, m/':>8}{'o/, m/':>8}"
      f"   (o order, m marker; = same, / other)")
for kind in ("unchanged", "ROLE SWAP"):
    if len(HELD11[kind]):
        print(f"    {kind:<35}" + "".join(f"{100 * np.mean(HELD11[kind][:, 8] == k):>{w}.0f}"
                                         for k, w in zip(range(NV11[8]), (10, 7, 8, 8, 8))))

# ---------------- three labels: the whole pipeline ----------------
HAS11 = HASP[:N_TR]
M3_11 = count11(X11_FV[:N_TR][HAS11], GOLD[:N_TR][HAS11], np.ones(int(HAS11.sum())), 3, ALL11)
P3_11 = np.where(HASP, said(M3_11, X11_FV), 2)
ACC11, FS11 = fever_scores(P3_11[DV], DV_IDX)[:2]
print(f"\n  THREE LABELS, 10 readings, counted on the sample: sample {100 * np.mean(P3_11[:N_TR] == GOLD[:N_TR]):.1f}, "
      f"paper_dev label accuracy {ACC11:.1f}, FEVER score {FS11:.1f} (Cell 10: {ACC10:.1f}, {FS10:.1f}; "
      f"Cell 8: {ACC8:.1f}, {FS8:.1f})")


def reasons11(x: np.ndarray) -> str:
    parts = [(M11[f][1, x[f]] - M11[f][0, x[f]], f"{F11[f][0]} {F11[f][1][x[f]]}") for f in ALL11
             if f not in COND11 or x[0] < 6]
    return ", ".join(f"{name} {w:+.2f}" for w, name in sorted(parts, key=lambda t: -abs(t[0]))[:4])


print("\n  HELD-OUT ROLE SWAPS WITH REASONS (train claims; claim -> changed claim)")
for claim, edited, x in SHOW11:
    v = int(said(M11, np.array([x], int))[0])
    print(f"    {claim[:76]}\n      -> {edited[:76]}\n      {reasons11(np.array(x, int))} -> {LABELS[v]} "
          f"(calls for REFUTES)")

print("\n  PREDICTIONS vs RESULTS")
LEDGER.result("B5-X4", round(HELD_C[1], 2))
LEDGER.result("B5-X5", round(HELD_C[0], 2))
LEDGER.result("B5-X6", round(HELD_C[2], 2))
LEDGER.result("B5-X7", round(BAL11 - BAL10, 2))
LEDGER.result("B5-X8", round(SYM11 - SYM10, 2))
LEDGER.result("B5-X9", round(ACC11, 2))
LEDGER.result("B5-T18", round(time.time() - T11))
if "sync" in globals():
    sync("Cell 11")
print(f"\n  ({time.time() - T11:.0f}s since Cell 11A started)")
print("  LAST LINE OF MC AI BRAIN 5 CELL 11B")


MC AI BRAIN 5 - CELL 11B: FEVER - ROLES, LENT LINKS AND PRACTICE, COUNTED
  READINGS ALONE, two labels, counted on 51,459 train claims with gold evidence
                               train, balanced  Symmetric, two-way
    Cell 10's 8 readings                  81.5                77.5
    with ROLE                             81.7                78.1
    with LENT                             82.8                77.8
    all 10                                82.3                78.9
    (Cell 10B printed 81.5 and 77.5 for its 8 readings)

  PRACTICE, all 10 readings: 3,837 changed claims added as REFUTES with weight L each
    on the claims that choose L:
        L  train, balanced  OTHER NAME  ROLE SWAP  unchanged  Symmetric
      0.0             82.3        87.7        0.0      100.0       78.9
     0.05             82.3        87.9        0.0      100.0       78.9
      0.1             82.3        87.9        0.0      100.0       78.9
     0.25             82.5        89.0        0

In [19]:
# =====================================================================
# MC AI BRAIN 5 - CELL 12A: MMLU - LOOK THE WORDS UP IN WIKIPEDIA
# Needs Cells 0-7A (QSETS, TRAIN5, forms, the FEVER Wikipedia pages and
# 7A's page readers). Cell 12B reads the pages and answers.
# DRAFT: written without a lab run.
# QUESTION SETS  ARC train (fit: weights are counted on it in Cell 12B);
#   practice: Cell 5's four validation sets, MMLU dev (285), MMLU-Pro
#   validation (70). No test split is read.
# LOOKUP (the dictionary step, on 5.4 million Wikipedia pages)
#   keys: every run of 1-5 words of a question or of a choice (single
#   words: 3+ letters, no stop word, no number)
#   a page is kept when its title, or its title before a bracket, is a key
#   ("Mercury (planet)" is kept for the key "mercury"); its first 8
#   sentences are kept. One pass over the pages; saved with Artifacts.
#   Each key leads to at most 12 pages (the first in file order).
# PREDICTIONS (written before running)
#   B5-M1 pages kept: 20,000-400,000
#   B5-T19 Cell 12A seconds, building: <= 1800; B5-T20 loading: <= 600
# =====================================================================
import hashlib, json, math, time, zipfile

import numpy as np

print("=" * 64); print("MC AI BRAIN 5 - CELL 12A: MMLU - LOOK THE WORDS UP IN WIKIPEDIA"); print("=" * 64)
T12 = time.time()
for need in ("QSETS", "TRAIN5", "table_rows", "forms", "STOPW", "FEVER", "FEVER_PINS", "raw_pages", "page_id",
             "unbracket", "tkey", "WORD"):
    if need not in globals():
        raise RuntimeError(f"run Cells 0 to 7A first ({need} is missing)")
for key, claim, rule in (("B5-M1", "Wikipedia pages kept for the question sets", between(20_000, 400_000)),
                         ("B5-T19", "Cell 12A seconds, building", at_most(1800)),
                         ("B5-T20", "Cell 12A seconds, loading", at_most(600))):
    LEDGER.predict(key, claim, rule)
MAX_N12, MAX_S12, PER_KEY12 = 5, 8, 12


def pro_row(r) -> tuple:
    opts = r["options"] if "options" in r else r["choices"]
    return r["question"], list(opts), int(r["answer_index"] if "answer_index" in r else r["answer"])


MMLU_VAL = table_rows(DATA / "mmlu" / "all" / "validation-00000-of-00001.parquet")
SUBJ12 = [r["subject"] for r in MMLU_VAL]
SETS12 = {"ARC train": list(TRAIN5)}
SETS12.update({name: list(qs) for name, qs in QSETS.items()})
SETS12["MMLU dev"] = [(r["question"], list(r["choices"]), int(r["answer"]))
                      for r in table_rows(DATA / "mmlu" / "all" / "dev-00000-of-00001.parquet")]
SETS12["MMLU-Pro validation"] = [pro_row(r) for r in
                                 table_rows(DATA / "mmlupro" / "data" / "validation-00000-of-00001.parquet")]
ROLE12 = {name: "fit" if name == "ARC train" else "practice" for name in SETS12}
print("  QUESTION SETS: " + ", ".join(f"{k} {len(v):,} ({ROLE12[k]})" for k, v in SETS12.items()))


def keys12(text: str) -> set:
    t, out = WORD.findall(text.casefold()), set()
    for n in range(1, MAX_N12 + 1):
        for i in range(len(t) - n + 1):
            if n > 1 or (len(t[i]) > 2 and t[i] not in STOPW and not t[i].isdigit()):
                out.add(" ".join(t[i:i + n]))
    return out


WANT12 = set()
for qs in SETS12.values():
    for q, ch, _ in qs:
        WANT12 |= keys12(q)
        for c in ch:
            WANT12 |= keys12(str(c))


def pack12(items: list) -> np.ndarray:
    return np.frombuffer("\n".join(items).encode("utf-8"), np.uint8)


def unpack12(a: np.ndarray) -> list:
    return bytes(a).decode("utf-8").split("\n")


def build12() -> dict:
    t1 = time.time()
    titles, heads, sents, s_page, seen = [], [], [], [], 0
    with zipfile.ZipFile(FEVER["wiki-pages.zip"]) as z:
        for line in raw_pages(z):
            pid = page_id(line)
            if not pid:
                continue
            seen += 1
            name = unbracket(pid)
            head, full = tkey(name.split(" (")[0]), tkey(name)
            key = head if head in WANT12 else full if full in WANT12 else ""
            if key:
                rows = [r.split("\t") for r in json.loads(line)["lines"].split("\n")]
                text = [unbracket(r[1]).replace("\n", " ") for r in rows if len(r) > 1 and r[1].strip()][:MAX_S12]
                if text:
                    titles.append(name.replace("\n", " "))
                    heads.append(key)
                    s_page += [len(titles) - 1] * len(text)
                    sents += text
    print(f"  LOOKED UP ({time.time() - t1:.0f}s): {seen:,} pages passed, {len(titles):,} kept,",
          f"{len(sents):,} sentences")
    return dict(titles=pack12(titles), heads=pack12(heads), sents=pack12(sents), s_page=np.array(s_page, np.int32))


DEPENDS12 = dict(what="Wikipedia pages whose titles are words of the question sets, Brain 5 Cell 12A", version=1,
                 max_n=MAX_N12, max_s=MAX_S12, wiki=FEVER_PINS["wiki-pages.zip"][0],
                 sets={k: len(v) for k, v in SETS12.items()},
                 keys=hashlib.sha256("\n".join(sorted(WANT12)).encode("utf-8")).hexdigest())
LOOK12, BUILT12 = ARTS.get("mmlu_lookup", DEPENDS12, build12)
TITLES12, HEADS12, SENTS12 = unpack12(LOOK12["titles"]), unpack12(LOOK12["heads"]), unpack12(LOOK12["sents"])
S_PAGE12 = LOOK12["s_page"]
P_START12 = np.searchsorted(S_PAGE12, np.arange(len(TITLES12) + 1))
KEY2PAGES = {}
for p, key in enumerate(HEADS12):
    got = KEY2PAGES.setdefault(key, [])
    if len(got) < PER_KEY12:
        got.append(p)
t1 = time.time()
S_FORMS12 = [frozenset(forms(s)) for s in SENTS12]
T_FORMS12 = [frozenset(forms(t)) for t in TITLES12]
DF12 = {}
for ws in S_FORMS12:
    for w in ws:
        DF12[w] = DF12.get(w, 0) + 1
N12 = max(1, len(S_FORMS12))


def idf12(w: str) -> float:
    d = DF12.get(w, 0)
    return math.log(1 + (N12 - d + 0.5) / (d + 0.5))


print(f"  LOOKUP {'built and saved' if BUILT12 else 'loaded'}: {len(TITLES12):,} pages, {len(SENTS12):,} sentences, "
      f"{len(KEY2PAGES):,} keys lead to a page (of {len(WANT12):,} asked); base forms in {time.time() - t1:.0f}s")
for q, ch, g in SETS12["MMLU dev"][:1]:
    hit = sorted(k for k in keys12(q) if k in KEY2PAGES)
    print(f"  EXAMPLE (MMLU dev): {q[:90]}\n    keys with a page: {', '.join(hit[:12])}")
print("\n  PREDICTIONS vs RESULTS")
LEDGER.result("B5-M1", len(TITLES12))
LEDGER.result("B5-T19" if BUILT12 else "B5-T20", round(time.time() - T12))
if "sync" in globals():
    sync("Cell 12A")
print("  LAST LINE OF MC AI BRAIN 5 CELL 12A")


MC AI BRAIN 5 - CELL 12A: MMLU - LOOK THE WORDS UP IN WIKIPEDIA
  QUESTION SETS: ARC train 3,370 (fit), ARC-Easy validation 570 (practice), ARC-Challenge validation 299 (practice), OpenBookQA validation 500 (practice), MMLU validation 1,531 (practice), MMLU dev 285 (practice), MMLU-Pro validation 70 (practice)
  LOOKED UP (75s): 5,416,536 pages passed, 71,253 kept, 315,821 sentences
  LOOKUP built and saved: 71,253 pages, 315,821 sentences, 18,160 keys lead to a page (of 695,754 asked); base forms in 5s
  EXAMPLE (MMLU dev): Find all c in Z_3 such that Z_3[x]/(x^2 + c) is a field.
    keys with a page: field, find, is a, x x

  PREDICTIONS vs RESULTS
    B5-M1 Wikipedia pages kept for the question sets: 7.125e+04 (predicted between 20000 and 400000) -> HELD
    B5-T19 Cell 12A seconds, building: 81 (predicted <= 1800) -> HELD
  sync: pushed 5 files to kaggle/ on main as commit a28d2ab2bb0b; GitHub confirms it (522 entries, head dbe0f4b90258bf64) (4s)
  LAST LINE OF MC AI BRAIN 5 CELL 1

In [20]:
# =====================================================================
# MC AI BRAIN 5 - CELL 12B: MMLU - READ THE PAGES, COUNT WHAT EACH READING
# SAYS, ANSWER
# Needs Cell 12A (the looked-up pages) and Cell 6 (solve6, score6,
# asks_not, CHOSEN, SOL6, SOL6_TRAIN). DRAFT: written without a lab run.
# FOR ONE QUESTION  pages: those its own keys lead to, and those each
#   choice's keys lead to. Q = the question's base forms; a choice's words
#   = its base forms that the question does not have. IDF over the kept
#   sentences. For every kept sentence of those pages, per choice:
#   JOINT    (share of Q's IDF in the sentence or its page title) x (share
#            of the choice's IDF there): one sentence ties the question to
#            the choice
#   DEFINED  the page is one the CHOICE leads to: share of Q's IDF in the
#            sentence (the answer's own page talks about the question)
#   REVERSE  the page is one the QUESTION leads to: share of the choice's
#            IDF in the sentence (the question's page names the answer)
#   Each choice keeps its best value of each.
# FIVE READINGS per choice
#   JOINT, DEFINED, REVERSE as a place among the question's choices: alone
#   first, tied first, not first, nothing found (for NOT / EXCEPT / LEAST
#   questions, Cell 6's asks_not, the lowest is first)
#   LEVEL  of its JOINT value: 0, to 0.15, to 0.35, to 0.6, over 0.6
#   CELL 6 its pick, not its pick, no pick (Cell 6's chosen settings)
# COUNT  naive Bayes (add-one smoothing): P(value | the choice is right)
#   and P(value | it is wrong), counted on ARC train (fit). A choice's
#   score is the sum of its log ratios; the highest wins; ties share the
#   credit. The same weights answer every practice set.
# PREDICTIONS (written before running)
#   B5-M2 MMLU validation accuracy, %: 27-36
#   B5-M3 minus Cell 6 on MMLU validation, points: -1-6
#   B5-M4 ARC-Challenge validation accuracy, %: 28-36
#   B5-M5 MMLU-Pro validation accuracy (70 questions, up to 10 choices), %: 6-24
#   B5-M6 MMLU validation questions where no reading found anything, %: 5-45
#   B5-T21 Cell 12B seconds: <= 1500
# =====================================================================
import time

import numpy as np

print("=" * 64); print("MC AI BRAIN 5 - CELL 12B: MMLU - READ, COUNT, ANSWER"); print("=" * 64)
T12B = time.time()
for need in ("SETS12", "KEY2PAGES", "S_FORMS12", "idf12", "solve6", "score6", "asks_not", "CHOSEN", "SOL6",
             "SOL6_TRAIN"):
    if need not in globals():
        raise RuntimeError(f"run Cells 6 and 12A first ({need} is missing)")
if not callable(globals().get("best")):
    raise RuntimeError("Cell 6's best() was overwritten by a name in a later cell; run the fixed Cell 7B")
for key, claim, rule in (("B5-M2", "MMLU validation accuracy, %", between(27, 36)),
                         ("B5-M3", "Cell 12 minus Cell 6, MMLU validation, points", between(-1, 6)),
                         ("B5-M4", "ARC-Challenge validation accuracy, %", between(28, 36)),
                         ("B5-M5", "MMLU-Pro validation accuracy, %", between(6, 24)),
                         ("B5-M6", "MMLU validation questions where no reading found anything, %", between(5, 45)),
                         ("B5-T21", "Cell 12B seconds", at_most(1500))):
    LEDGER.predict(key, claim, rule)
F12 = (("JOINT", ("alone first", "tied first", "not first", "nothing found")),
       ("LEVEL", ("0", "to 0.15", "to 0.35", "to 0.6", "over 0.6")),
       ("DEFINED", ("alone first", "tied first", "not first", "nothing found")),
       ("REVERSE", ("alone first", "tied first", "not first", "nothing found")),
       ("CELL 6", ("its pick", "not its pick", "no pick")))
NV12 = [len(v) for _, v in F12]


def pages12(text: str) -> set:
    out = set()
    for k in keys12(text):
        out.update(KEY2PAGES.get(k, ()))
    return out


def share12(words: set, total: float) -> float:
    return sum(idf12(w) for w in words) / total if words else 0.0


def raw12(question: str, choices: list) -> np.ndarray:
    """(choices, 3): the best JOINT, DEFINED and REVERSE value of each choice"""
    Q = set(forms(question))
    qt = sum(idf12(w) for w in Q) or 1.0
    C = [set(forms(str(c))) - Q for c in choices]
    ct = [sum(idf12(w) for w in c) or 1.0 for c in C]
    qp, cp = pages12(question), [pages12(str(c)) for c in choices]
    out = np.zeros((len(choices), 3))
    for p in qp.union(*cp):
        tw, in_q = T_FORMS12[p], p in qp
        for k in range(P_START12[p], P_START12[p + 1]):
            sw = S_FORMS12[k]
            both = sw | tw
            q_s, q_all = share12(Q & sw, qt), share12(Q & both, qt)
            for j, c in enumerate(C):
                if c:
                    c_s, c_all = share12(c & sw, ct[j]), share12(c & both, ct[j])
                    out[j, 0] = max(out[j, 0], q_all * c_all)
                    if p in cp[j]:
                        out[j, 1] = max(out[j, 1], q_s)
                    if in_q:
                        out[j, 2] = max(out[j, 2], c_s)
    return out


def place12(x: np.ndarray, neg: bool) -> list:
    if not x.any():
        return [3] * len(x)
    y = -x if neg else x
    top = y.max()
    alone = int((y == top).sum() > 1)
    return [alone if v == top else 2 for v in y]


def level12(v: float) -> int:
    return 0 if v <= 0 else 1 if v <= 0.15 else 2 if v <= 0.35 else 3 if v <= 0.6 else 4


def pick6(name: str, i: int, q: str, ch: list) -> tuple:
    """Cell 6's reading per choice, and its credit-ready scores"""
    s = SOL6_TRAIN[i] if name == "ARC train" else SOL6[name][i] if name in SOL6 else solve6(q, ch)
    sc = np.asarray(score6(s, *CHOSEN, asks_not(q)), float)
    top = sc.max()
    if (sc == top).all():
        return [2] * len(sc), sc
    return [int(v != top) for v in sc], sc


def credit12(scores: np.ndarray, gold: int) -> float:
    top = scores.max()
    return float(scores[gold] == top) / float((scores == top).sum())


def read12(name: str, i: int, q: str, ch: list) -> tuple:
    """((choices, 5) readings, Cell 6's scores, whether anything was found)"""
    neg = asks_not(q)
    raw = raw12(q, ch)
    six, sc = pick6(name, i, q, ch)
    x = np.array([place12(raw[:, 0], neg), [level12(v) for v in raw[:, 0]], place12(raw[:, 1], neg),
                  place12(raw[:, 2], neg), six], int).T
    return x, sc, bool(raw.any())


t1 = time.time()
READ12 = {name: [read12(name, i, q, ch) for i, (q, ch, _) in enumerate(qs)] for name, qs in SETS12.items()}
print(f"  READ ({time.time() - t1:.0f}s): " + ", ".join(f"{k} {len(v):,}" for k, v in READ12.items()))

# ---------------- count on ARC train (fit) ----------------
COUNT12 = [np.zeros((2, n)) for n in NV12]
for (x, _, _), (_, _, gold) in zip(READ12["ARC train"], SETS12["ARC train"]):
    for j in range(len(x)):
        for f in range(len(F12)):
            COUNT12[f][int(j == gold), x[j, f]] += 1
W12 = []
for c in COUNT12:
    lp = np.log((c + 1) / (c.sum(1, keepdims=True) + c.shape[1]))
    W12.append(lp[1] - lp[0])


def scores12(x: np.ndarray) -> np.ndarray:
    return sum(W12[f][x[:, f]] for f in range(len(F12)))


print("\n  WHAT EACH READING SAYS (ARC train; weight above 0 speaks for the choice being right)")
print(f"    {'':<26}{'of choices':>11}{'right':>8}{'weight':>8}")
for f, (name, values) in enumerate(F12):
    for k, value in enumerate(values):
        n = COUNT12[f][:, k].sum()
        if n:
            share, right = n / COUNT12[f].sum(), COUNT12[f][1, k] / n
            print(f"    {name + ' ' + value:<26}{share:>11.1%}{right:>8.1%}{W12[f][k]:>+8.2f}")

# ---------------- answer every set ----------------
ACC12, ACC6, NONE12 = {}, {}, {}
print(f"\n  {'':<28}{'role':>9}{'questions':>10}{'Cell 6':>8}{'Cell 12':>9}{'nothing found':>15}")
for name, qs in SETS12.items():
    got = [(credit12(scores12(x), g), credit12(sc, g), found) for (x, sc, found), (_, _, g) in zip(READ12[name], qs)]
    ACC12[name], ACC6[name] = 100 * np.mean([a for a, _, _ in got]), 100 * np.mean([b for _, b, _ in got])
    NONE12[name] = 100 * np.mean([not f for _, _, f in got])
    print(f"  {name:<28}{ROLE12[name]:>9}{len(qs):>10,}{ACC6[name]:>8.1f}{ACC12[name]:>9.1f}{NONE12[name]:>14.1f}%")
print("  (ARC train is where the weights were counted; MMLU-Pro has up to 10 choices, so chance is near 10%)")

by_subject = {}
for (x, _, _), (_, _, g), s in zip(READ12["MMLU validation"], SETS12["MMLU validation"], SUBJ12):
    by_subject.setdefault(s, []).append(credit12(scores12(x), g))
ranked = sorted(by_subject.items(), key=lambda kv: -np.mean(kv[1]))
print("\n  MMLU validation by subject, 6 best and 6 worst (questions, accuracy):")
for s, v in ranked[:6] + ranked[-6:]:
    print(f"    {s:<40}{len(v):>5}{100 * np.mean(v):>8.1f}")

print("\n  TWO ANSWERS WITH REASONS (MMLU dev)")
for i, (q, ch, g) in enumerate(SETS12["MMLU dev"][:2]):
    x = READ12["MMLU dev"][i][0]
    s = scores12(x)
    j = int(s.argmax())
    why = ", ".join(f"{F12[f][0]} {F12[f][1][x[j, f]]} {W12[f][x[j, f]]:+.2f}" for f in range(len(F12)))
    print(f"    {q[:90]}\n      said: {str(ch[j])[:60]} (gold: {str(ch[g])[:60]})\n      {why}")

print("\n  PREDICTIONS vs RESULTS")
LEDGER.result("B5-M2", round(ACC12["MMLU validation"], 2))
LEDGER.result("B5-M3", round(ACC12["MMLU validation"] - ACC6["MMLU validation"], 2))
LEDGER.result("B5-M4", round(ACC12["ARC-Challenge validation"], 2))
LEDGER.result("B5-M5", round(ACC12["MMLU-Pro validation"], 2))
LEDGER.result("B5-M6", round(NONE12["MMLU validation"], 2))
LEDGER.result("B5-T21", round(time.time() - T12B))
if "sync" in globals():
    sync("Cell 12B")
print(f"\n  ({time.time() - T12B:.0f}s)")
print("  LAST LINE OF MC AI BRAIN 5 CELL 12B")


MC AI BRAIN 5 - CELL 12B: MMLU - READ, COUNT, ANSWER
  READ (28s): ARC train 3,370, ARC-Easy validation 570, ARC-Challenge validation 299, OpenBookQA validation 500, MMLU validation 1,531, MMLU dev 285, MMLU-Pro validation 70

  WHAT EACH READING SAYS (ARC train; weight above 0 speaks for the choice being right)
                               of choices   right  weight
    JOINT alone first               21.1%   35.7%   +0.51
    JOINT tied first                 6.8%   28.8%   +0.19
    JOINT not first                 67.5%   21.3%   -0.21
    JOINT nothing found              4.5%   25.0%   +0.01
    LEVEL 0                         17.7%   21.7%   -0.18
    LEVEL to 0.15                   58.9%   23.3%   -0.10
    LEVEL to 0.35                   18.2%   29.7%   +0.24
    LEVEL to 0.6                     4.3%   36.3%   +0.54
    LEVEL over 0.6                   1.0%   52.7%   +1.20
    DEFINED alone first             14.9%   32.2%   +0.35
    DEFINED tied first              23.2%   25.5

In [21]:
# =====================================================================
# MC AI BRAIN 5 - CELL 13: NOISE MAP OF THE QUESTION SETS
# Needs Cells 12A and 12B. Counts only: nothing is fitted or chosen here.
# Error analysis by slices: each question gets yes/no features; for every
# slice the share of questions and Cell 6 / Cell 12 accuracy, with a
# Wilson 95% interval. A slice whose interval is above chance is a beacon;
# one whose interval holds chance is fog.
# SIX PARTS
#   1 MEASUREMENT  how wide the interval of a set or a subject is
#   2 KEYS         keys that lead to a page, counted by how many questions
#                  use them (document frequency); a key used by 1% or more
#                  of all questions is COMMON; share of a question's pages
#                  that only common keys bring
#   3 AGREEMENT    JOINT, DEFINED, REVERSE: how often two of them pick the
#                  same choice, and the phi correlation of their "alone
#                  first" marks over all choices of ARC train
#   4 SHAPES       per set: digits, math symbols, a number with a unit, all
#                  choices numbers, NOT/EXCEPT, Statement 1 | 2, an "all /
#                  none of the above" choice, over 60 words, nothing found
#   5 SLICES       MMLU validation by shape: Cell 6, Cell 12, interval
#   6 ARTIFACTS    choice-only check (the longest choice); exact duplicate
#                  questions inside and across sets; repeated choices
# PREDICTIONS (written before running)
#   B5-Y1 MMLU validation questions with digits or math symbols, %: 15-35
#   B5-Y2 MMLU-Pro validation questions with digits or math symbols, %: 30-70
#   B5-Y3 MMLU validation questions using a common key, %: 40-90
#   B5-Y4 ARC train, JOINT and DEFINED pick the same choice (both alone
#         first), %: 40-85
#   B5-Y5 MMLU validation, Cell 12 with digits or symbols minus without,
#         points: -12-0
#   B5-Y6 MMLU validation, the longest choice is right, %: 25-40
#   B5-Y7 question texts found in two different sets: 0-60
#   B5-T22 Cell 13 seconds: <= 300
# =====================================================================
import re
import time
from collections import Counter

import numpy as np

print("=" * 64); print("MC AI BRAIN 5 - CELL 13: NOISE MAP OF THE QUESTION SETS"); print("=" * 64)
T13 = time.time()
for need in ("SETS12", "READ12", "W12", "scores12", "credit12", "keys12", "KEY2PAGES", "SUBJ12", "ROLE12", "tkey",
             "asks_not"):
    if need not in globals():
        raise RuntimeError(f"run Cells 12A and 12B first ({need} is missing)")
for key, claim, rule in (
        ("B5-Y1", "MMLU validation questions with digits or math symbols, %", between(15, 35)),
        ("B5-Y2", "MMLU-Pro validation questions with digits or math symbols, %", between(30, 70)),
        ("B5-Y3", "MMLU validation questions using a common key, %", between(40, 90)),
        ("B5-Y4", "ARC train, JOINT and DEFINED pick the same choice, %", between(40, 85)),
        ("B5-Y5", "MMLU validation, Cell 12 with digits or symbols minus without, points", between(-12, 0)),
        ("B5-Y6", "MMLU validation, the longest choice is right, %", between(25, 40)),
        ("B5-Y7", "question texts found in two different sets", between(0, 60)),
        ("B5-T22", "Cell 13 seconds", at_most(300))):
    LEDGER.predict(key, claim, rule)
SHORT13 = {"ARC train": "ARC tr", "ARC-Easy validation": "ARC-E", "ARC-Challenge validation": "ARC-C",
           "OpenBookQA validation": "OBQA", "MMLU validation": "MMLU v", "MMLU dev": "MMLU d",
           "MMLU-Pro validation": "Pro v"}


def wilson(k: float, n: int, z: float = 1.96) -> tuple:
    """Wilson score interval for k right of n, in %"""
    if not n:
        return 0.0, 100.0
    p = k / n
    d = 1 + z * z / n
    mid = (p + z * z / (2 * n)) / d
    half = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return 100 * max(0.0, mid - half), 100 * min(1.0, mid + half)


# per question: Cell 12 credit, Cell 6 credit, chance
GOT13 = {}
for name, qs in SETS12.items():
    GOT13[name] = np.array([(credit12(scores12(x), g), credit12(sc, g), 1 / len(ch))
                            for (x, sc, _), (_, ch, g) in zip(READ12[name], qs)])

# ---------------- 1 measurement ----------------
print("\n  1 MEASUREMENT: how wide a 95% interval is (Cell 12 accuracy)")
for name in SETS12:
    a = GOT13[name][:, 0]
    lo, hi = wilson(a.sum(), len(a))
    print(f"    {name:<28}{len(a):>6,} questions  {100 * a.mean():>5.1f}  [{lo:5.1f}, {hi:5.1f}]",
          f" +-{(hi - lo) / 2:.1f}")
subj13 = {}
for (c12, _, _), s in zip(GOT13["MMLU validation"], SUBJ12):
    subj13.setdefault(s, []).append(c12)
halves = [(lambda lh: (lh[1] - lh[0]) / 2)(wilson(sum(v), len(v))) for v in subj13.values()]
holds = sum(wilson(sum(v), len(v))[0] <= 25 <= wilson(sum(v), len(v))[1] for v in subj13.values())
print(f"    MMLU validation subjects: {len(subj13)}, median {np.median([len(v) for v in subj13.values()]):.0f}",
      f"questions, median interval +-{np.median(halves):.1f} points; {holds} of {len(subj13)} hold chance (25)")

# ---------------- 2 keys ----------------
QKEYS13 = {name: [(set(k for k in keys12(q) if k in KEY2PAGES),
                   [set(k for k in keys12(str(c)) if k in KEY2PAGES) for c in ch]) for q, ch, _ in qs]
           for name, qs in SETS12.items()}
DF13 = Counter()
for rows in QKEYS13.values():
    for qk, cks in rows:
        DF13.update(qk.union(*cks))
N13 = sum(len(v) for v in QKEYS13.values())
COMMON13 = {k for k, n in DF13.items() if n >= 0.01 * N13}
print(f"\n  2 KEYS: {len(DF13):,} keys lead to a page; {len(COMMON13)} are COMMON (used by 1% or more of {N13:,}",
      "questions)")
print("    most used:", ", ".join(f"{k} {n}" for k, n in DF13.most_common(15)))
print(f"    {'':<28}{'uses common':>12}{'pages (median)':>16}{'pages only common keys bring':>30}")
COMMON_Q13 = {}
for name, rows in QKEYS13.items():
    uses, sizes, only = [], [], []
    for qk, cks in rows:
        keys = qk.union(*cks)
        uses.append(bool(keys & COMMON13))
        pool = set().union(*(KEY2PAGES[k] for k in keys)) if keys else set()
        rare = set().union(*(KEY2PAGES[k] for k in keys - COMMON13)) if keys - COMMON13 else set()
        sizes.append(len(pool))
        if pool:
            only.append(len(pool - rare) / len(pool))
    COMMON_Q13[name] = 100 * np.mean(uses)
    print(f"    {name:<28}{COMMON_Q13[name]:>11.1f}%{np.median(sizes):>16.0f}{100 * np.mean(only or [0]):>29.1f}%")

# ---------------- 3 agreement ----------------
print("\n  3 AGREEMENT of the three lookup readings, ARC train (both alone first; phi over all choices)")
PAIRS13 = ((0, 2, "JOINT-DEFINED"), (0, 3, "JOINT-REVERSE"), (2, 3, "DEFINED-REVERSE"))
SAME13 = {}
for a, b, label in PAIRS13:
    same = both = 0
    xa, xb = [], []
    for x, _, _ in READ12["ARC train"]:
        ia, ib = np.flatnonzero(x[:, a] == 0), np.flatnonzero(x[:, b] == 0)
        if len(ia) and len(ib):
            both += 1
            same += int(ia[0] == ib[0])
        xa += list(x[:, a] == 0)
        xb += list(x[:, b] == 0)
    xa, xb = np.array(xa, float), np.array(xb, float)
    phi = float(np.corrcoef(xa, xb)[0, 1]) if xa.std() and xb.std() else float("nan")
    SAME13[label] = 100 * same / max(both, 1)
    print(f"    {label:<18} same choice {SAME13[label]:5.1f}% of {both:,} questions; phi {phi:+.2f}")

# ---------------- 4 shapes ----------------
MATH13 = re.compile(r"[=+*/^<>\\]|[A-Za-z]_\w|\d\s*-\s*\d")
UNIT13 = re.compile(r"\$\s*\d|\d(?:\.\d+)?\s*(?:%|kg|mg|cm|mm|km|ms|hz|khz|mhz|kj|kw|mol|kpa|atm|ml|ev|nm|mph|ohms?|"
                    r"years?|dollars?|meters?|seconds?|joules?|newtons?|watts?|volts?|grams?|liters?|m/s|g|m|s|j|n|w|v|"
                    r"k|l)\b", re.I)
NUMBER13 = re.compile(r"^\s*[-+$]?\s*\d[\d,]*(?:\.\d+)?\s*\S{0,12}\s*$")
ABOVE13 = re.compile(r"\b(?:all|none|both|neither) of (?:the )?(?:above|these|them)\b", re.I)


def shape13(q: str, ch: list, found: bool) -> dict:
    text = q + " " + " ".join(str(c) for c in ch)
    return {"digits": bool(re.search(r"\d", text)), "math symbols": bool(MATH13.search(text)),
            "digits or symbols": bool(re.search(r"\d", text) or MATH13.search(text)),
            "number with a unit": bool(UNIT13.search(text)),
            "all choices numbers": all(NUMBER13.match(str(c)) for c in ch),
            "NOT / EXCEPT": asks_not(q), "Statement 1 | 2": "statement 1" in q.lower(),
            "all/none of the above": any(ABOVE13.search(str(c)) for c in ch),
            "over 60 words": len(q.split()) > 60, "nothing found": not found}


SHAPES13 = {name: [shape13(q, ch, r[2]) for (q, ch, _), r in zip(qs, READ12[name])] for name, qs in SETS12.items()}
FEAT13 = list(SHAPES13["MMLU validation"][0])
print("\n  4 SHAPES: share of questions, %")
print(f"    {'':<24}" + "".join(f"{SHORT13.get(n, n[:7]):>8}" for n in SETS12))
for f in FEAT13:
    print(f"    {f:<24}" + "".join(f"{100 * np.mean([s[f] for s in SHAPES13[n]]):>8.1f}" for n in SETS12))

# ---------------- 5 slices ----------------
print("\n  5 SLICES, MMLU validation (chance 25): accuracy with and without each shape")
print(f"    {'':<24}{'with':>6}{'Cell 6':>8}{'Cell 12':>8}{'95% interval':>16}{'':>7}{'without':>9}{'Cell 12':>8}")
G, S = GOT13["MMLU validation"], SHAPES13["MMLU validation"]
for f in FEAT13:
    m = np.array([s[f] for s in S])
    if m.sum() == 0:
        continue
    lo, hi = wilson(G[m, 0].sum(), int(m.sum()))
    mark = "beacon" if lo > 25 else "fog" if hi >= 25 else "below"
    rest = 100 * G[~m, 0].mean() if (~m).any() else float("nan")
    print(f"    {f:<24}{int(m.sum()):>6}{100 * G[m, 1].mean():>8.1f}{100 * G[m, 0].mean():>8.1f}",
          f"  [{lo:5.1f}, {hi:5.1f}]  {mark:<7}{int((~m).sum()):>8}{rest:>8.1f}")
m_ds = np.array([s["digits or symbols"] for s in S])
GAP13 = 100 * (G[m_ds, 0].mean() - G[~m_ds, 0].mean())

# ---------------- 6 artifacts ----------------
print("\n  6 ARTIFACTS")
print(f"    {'':<28}{'longest choice':>15}{'chance':>8}{'Cell 12':>9}{'repeated choices':>18}")
LONG13 = {}
for name, qs in SETS12.items():
    longest = [credit12(np.array([len(str(c)) for c in ch], float), g) for _, ch, g in qs]
    rep = sum(len({tkey(str(c)) for c in ch}) < len(ch) for _, ch, _ in qs)
    LONG13[name] = 100 * np.mean(longest)
    print(f"    {name:<28}{LONG13[name]:>14.1f}%{100 * GOT13[name][:, 2].mean():>7.1f}%",
          f"{100 * GOT13[name][:, 0].mean():>8.1f}{rep:>18}")
seen13 = {}
for name, qs in SETS12.items():
    for q, ch, _ in qs:
        seen13.setdefault(tkey(q), []).append(name)
inside = sum(len(v) - len(set(v)) for v in seen13.values())
across = [v for v in seen13.values() if len(set(v)) > 1]
pairs = Counter(" + ".join(sorted(set(v))) for v in across)
print(f"    exact duplicate question texts: {inside} repeats inside a set; {len(across)} texts in two or more sets")
for p, n in pairs.most_common(5):
    print(f"      {p}: {n}")

print("\n  PREDICTIONS vs RESULTS")
LEDGER.result("B5-Y1", round(100 * np.mean(m_ds), 2))
LEDGER.result("B5-Y2", round(100 * np.mean([s["digits or symbols"] for s in SHAPES13["MMLU-Pro validation"]]), 2))
LEDGER.result("B5-Y3", round(COMMON_Q13["MMLU validation"], 2))
LEDGER.result("B5-Y4", round(SAME13["JOINT-DEFINED"], 2))
LEDGER.result("B5-Y5", round(GAP13, 2))
LEDGER.result("B5-Y6", round(LONG13["MMLU validation"], 2))
LEDGER.result("B5-Y7", len(across))
LEDGER.result("B5-T22", round(time.time() - T13))
if "sync" in globals():
    sync("Cell 13")
print(f"\n  ({time.time() - T13:.0f}s)")
print("  LAST LINE OF MC AI BRAIN 5 CELL 13")


MC AI BRAIN 5 - CELL 13: NOISE MAP OF THE QUESTION SETS

  1 MEASUREMENT: how wide a 95% interval is (Cell 12 accuracy)
    ARC train                    3,370 questions   43.0  [ 41.4,  44.7]  +-1.7
    ARC-Easy validation            570 questions   51.1  [ 47.0,  55.1]  +-4.1
    ARC-Challenge validation       299 questions   30.6  [ 25.6,  36.0]  +-5.2
    OpenBookQA validation          500 questions   24.8  [ 21.2,  28.8]  +-3.8
    MMLU validation              1,531 questions   31.2  [ 29.0,  33.6]  +-2.3
    MMLU dev                       285 questions   30.1  [ 25.0,  35.6]  +-5.3
    MMLU-Pro validation             70 questions   14.7  [  8.2,  24.8]  +-8.3
    MMLU validation subjects: 57, median 21 questions, median interval +-18.7 points; 50 of 57 hold chance (25)

  2 KEYS: 18,160 keys lead to a page; 297 are COMMON (used by 1% or more of 6,625 questions)
    most used: following 1015, the following 1005, water 875, likely 689, best 679, one 527, energy 516, two 454, earth 4

In [22]:
# =====================================================================
# MC AI BRAIN 5 - CELL 14: MMLU - DROP THE JUNK KEYS
# Needs Cells 12A and 12B (Cell 13 is not needed).
# Cell 13 found that about 30% of a question's pages come only through
# keys like "following", "the following", "best", "what": words of the
# question form, not of the subject.
# JUNK KEYS  document frequency counted on ARC train questions and choices
#   only (fit); a key is junk when it is used by at least a share t of ARC
#   train questions. The practice sets are never used to count or choose.
# CHOICE OF t  t in 0.5%, 1%, 2%, 5%, and "keep all": for each, Cell 12B's
#   readings and naive Bayes weights on ARC train; the t with the best ARC
#   train accuracy is kept (ties: the larger t, which drops fewer keys).
#   The same weights then answer every practice set.
# PREDICTIONS (written before running)
#   B5-Z1 junk keys at t = 1%: 100-600
#   B5-Z2 MMLU validation accuracy with the chosen t, %: 30-35
#   B5-Z3 chosen t minus Cell 12, MMLU validation, points: -1-3
#   B5-Z4 MMLU validation questions with nothing found, chosen t, %: 20-35
#   B5-Z5 ARC-Challenge validation accuracy, chosen t, %: 28-35
#   B5-T23 Cell 14 seconds: <= 900
# =====================================================================
import time
from collections import Counter

import numpy as np

print("=" * 64); print("MC AI BRAIN 5 - CELL 14: MMLU - DROP THE JUNK KEYS"); print("=" * 64)
T14 = time.time()
for need in ("SETS12", "READ12", "KEY2PAGES", "keys12", "read12", "credit12", "F12", "NV12", "ACC12"):
    if need not in globals():
        raise RuntimeError(f"run Cells 12A and 12B first ({need} is missing)")
for key, claim, rule in (("B5-Z1", "junk keys at t = 1%", between(100, 600)),
                         ("B5-Z2", "MMLU validation accuracy with the chosen t, %", between(30, 35)),
                         ("B5-Z3", "chosen t minus Cell 12, MMLU validation, points", between(-1, 3)),
                         ("B5-Z4", "MMLU validation questions with nothing found, chosen t, %", between(20, 35)),
                         ("B5-Z5", "ARC-Challenge validation accuracy, chosen t, %", between(28, 35)),
                         ("B5-T23", "Cell 14 seconds", at_most(900))):
    LEDGER.predict(key, claim, rule)
KEY2PAGES12 = KEY2PAGES

# ---------------- document frequency on ARC train only ----------------
DF14 = Counter()
for q, ch, _ in SETS12["ARC train"]:
    used = set(keys12(q)).union(*(keys12(str(c)) for c in ch))
    DF14.update(k for k in used if k in KEY2PAGES12)
N14 = len(SETS12["ARC train"])
print(f"  ARC train: {N14:,} questions; {len(DF14):,} of the page keys appear there")
print("  most used:", ", ".join(f"{k} {n}" for k, n in DF14.most_common(20)))


def junk14(t: float) -> set:
    return {k for k, n in DF14.items() if n >= t * N14}


def weights14(rows: list, golds: list) -> list:
    count = [np.zeros((2, n)) for n in NV12]
    for (x, _, _), g in zip(rows, golds):
        for j in range(len(x)):
            for f in range(len(F12)):
                count[f][int(j == g), x[j, f]] += 1
    out = []
    for c in count:
        lp = np.log((c + 1) / (c.sum(1, keepdims=True) + c.shape[1]))
        out.append(lp[1] - lp[0])
    return out


def accuracy14(rows: list, qs: list, w: list) -> float:
    return 100 * np.mean([credit12(sum(w[f][x[:, f]] for f in range(len(F12))), g)
                          for (x, _, _), (_, _, g) in zip(rows, qs)])


def read_with14(drop: set, names: list) -> dict:
    """Cell 12B's readings with the keys in drop removed from the lookup"""
    global KEY2PAGES
    KEY2PAGES = {k: v for k, v in KEY2PAGES12.items() if k not in drop}
    try:
        return {n: [read12(n, i, q, ch) for i, (q, ch, _) in enumerate(SETS12[n])] for n in names}
    finally:
        KEY2PAGES = KEY2PAGES12


# ---------------- choose t on ARC train ----------------
TS14 = (None, 0.05, 0.02, 0.01, 0.005)
GOLD_TR14 = [g for _, _, g in SETS12["ARC train"]]
FIT14 = {}
print("\n  CHOOSING t ON ARC TRAIN (fit; weights counted and scored on the same questions)")
print(f"    {'t':<10}{'junk keys':>10}{'ARC train':>11}{'nothing found':>15}")
for t in TS14:
    drop = junk14(t) if t else set()
    rows = READ12["ARC train"] if t is None else read_with14(drop, ["ARC train"])["ARC train"]
    acc = accuracy14(rows, SETS12["ARC train"], weights14(rows, GOLD_TR14))
    FIT14[t] = acc
    none = 100 * np.mean([not r[2] for r in rows])
    label = "keep all" if t is None else f"{100 * t:g}%"
    print(f"    {label:<10}{len(drop):>10,}{acc:>11.2f}{none:>14.1f}%")
T_CHOSEN14 = max(TS14, key=lambda t: (round(FIT14[t], 6), 1.0 if t is None else t))
DROP14 = junk14(T_CHOSEN14) if T_CHOSEN14 else set()
print(f"  chosen: {'keep all' if T_CHOSEN14 is None else f'{100 * T_CHOSEN14:g}%'} ({len(DROP14):,} junk keys)")
if DROP14:
    print("  junk keys, most used first:", ", ".join(sorted(DROP14, key=lambda k: -DF14[k])[:30]))

# ---------------- answer every set ----------------
READ14 = read_with14(DROP14, list(SETS12)) if DROP14 else dict(READ12)
W14 = weights14(READ14["ARC train"], GOLD_TR14)
ACC14, NONE14 = {}, {}
print(f"\n  {'':<28}{'role':>9}{'questions':>10}{'Cell 12':>9}{'Cell 14':>9}{'nothing found':>15}")
for name, qs in SETS12.items():
    ACC14[name] = accuracy14(READ14[name], qs, W14)
    NONE14[name] = 100 * np.mean([not r[2] for r in READ14[name]])
    print(f"  {name:<28}{ROLE12[name]:>9}{len(qs):>10,}{ACC12[name]:>9.1f}{ACC14[name]:>9.1f}{NONE14[name]:>14.1f}%")
print("  (MMLU-Pro validation: 68 of its 70 questions are MMLU dev questions, so it is not a separate check)")

print("\n  PREDICTIONS vs RESULTS")
LEDGER.result("B5-Z1", len(junk14(0.01)))
LEDGER.result("B5-Z2", round(ACC14["MMLU validation"], 2))
LEDGER.result("B5-Z3", round(ACC14["MMLU validation"] - ACC12["MMLU validation"], 2))
LEDGER.result("B5-Z4", round(NONE14["MMLU validation"], 2))
LEDGER.result("B5-Z5", round(ACC14["ARC-Challenge validation"], 2))
LEDGER.result("B5-T23", round(time.time() - T14))
if "sync" in globals():
    sync("Cell 14")
print(f"\n  ({time.time() - T14:.0f}s)")
print("  LAST LINE OF MC AI BRAIN 5 CELL 14")


MC AI BRAIN 5 - CELL 14: MMLU - DROP THE JUNK KEYS
  ARC train: 3,370 questions; 7,817 of the page keys appear there
  most used: water 614, best 432, likely 423, energy 369, following 353, the following 349, earth 317, food 238, is a 222, one 216, light 211, change 201, sun 196, use 189, air 187, the sun 184, used 180, different 179, what is 178, two 175

  CHOOSING t ON ARC TRAIN (fit; weights counted and scored on the same questions)
    t          junk keys  ARC train  nothing found
    keep all           0      43.02           4.3%
    5%                24      42.90           4.7%
    2%               113      43.47           6.0%
    1%               318      44.35          11.2%
    0.5%             729      45.18          19.8%
  chosen: 0.5% (729 junk keys)
  junk keys, most used first: water, best, likely, energy, following, the following, earth, food, is a, one, light, change, sun, use, air, the sun, used, different, what is, two, temperature, chemical, heat, soil, mass, bo

In [23]:
# =====================================================================
# MC AI BRAIN 5 - CELL 15: THE NUMBER LAYER
# Needs Cells 12A and 12B (SETS12, READ12, scores12, credit12, asks_not).
# Nothing here is fitted: every rule is fixed before running.
# 1 NUMBER WORDS  inverse text normalization: "twenty-five" -> 25, "one
#   hundred and five" -> 105, "a dozen" -> 12, "half" -> 1/2, "twice" -> 2.
#   Digits too: 1,200  3.5  2/3  4.1 x 10^-3  25%  $30
# 2 FHRR CROSS-ASSOCIATION  fractional power encoding: number x is the
#   phasor vector exp(i * theta * x) (theta random per dimension). Binding
#   two numbers multiplies phasors, which adds the numbers. A memory bundles
#   word (x) number for every number word; unbinding a word recalls its
#   number by the nearest encoded value.
# 3 EXACT ARITHMETIC  a span of digits and operators (+ - * / ^ x and the
#   signs for times, divide, minus) is parsed with Python's ast module into
#   a tree; only numbers and + - * / ** are allowed; evaluated with
#   fractions.Fraction (exact).
# 4 GATE AND ANSWER  a question goes to the layer when every choice reads
#   as one number and it does not ask NOT / EXCEPT. In order:
#   EXPR   a choice equals the value of an arithmetic span of the question
#          (only one choice may match)
#   PAIRS  the numbers of the question (first 8): every pair under + - x /;
#          the choice hit by the most (pair, operation) wins if alone
#   else   Cell 12's answer
#   A match: |a - b| <= 0.5% of the larger (or both 0).
# PREDICTIONS (written before running)
#   B5-A1 the 12 number-word checks right: 12
#   B5-A2 FHRR: bound 3 and 4 against encoded 7, similarity: >= 0.999
#   B5-A3 FHRR: number words recalled from one bundle of all of them, %: >= 95
#   B5-A4 MMLU validation questions through the gate: 90-220
#   B5-A5 MMLU validation gated questions where EXPR fires, %: 3-30
#   B5-A6 MMLU validation gated questions where PAIRS fires, %: 15-60
#   B5-A7 MMLU validation, layer minus Cell 12 on the gated questions,
#         points: -3-10
#   B5-T24 Cell 15 seconds: <= 300
# =====================================================================
import ast
import operator
import re
import time
from fractions import Fraction

import numpy as np

print("=" * 64); print("MC AI BRAIN 5 - CELL 15: THE NUMBER LAYER"); print("=" * 64)
T15 = time.time()
for need in ("SETS12", "READ12", "scores12", "credit12", "asks_not", "ROLE12"):
    if need not in globals():
        raise RuntimeError(f"run Cells 12A and 12B first ({need} is missing)")
for key, claim, rule in (
        ("B5-A1", "the 12 number-word checks right", between(12, 12)),
        ("B5-A2", "FHRR: bound 3 and 4 against encoded 7, similarity", at_least(0.999)),
        ("B5-A3", "FHRR: number words recalled from one bundle, %", at_least(95)),
        ("B5-A4", "MMLU validation questions through the gate", between(90, 220)),
        ("B5-A5", "MMLU validation gated questions where EXPR fires, %", between(3, 30)),
        ("B5-A6", "MMLU validation gated questions where PAIRS fires, %", between(15, 60)),
        ("B5-A7", "MMLU validation, layer minus Cell 12 on the gated questions, points", between(-3, 10)),
        ("B5-T24", "Cell 15 seconds", at_most(300))):
    LEDGER.predict(key, claim, rule)

# ---------------- 1 number words ----------------
UNITS15 = {w: i for i, w in enumerate("zero one two three four five six seven eight nine ten eleven twelve thirteen "
                                      "fourteen fifteen sixteen seventeen eighteen nineteen".split())}
TENS15 = {w: 10 * (i + 2) for i, w in enumerate("twenty thirty forty fifty sixty seventy eighty ninety".split())}
SCALES15 = {"hundred": 100, "thousand": 1000, "million": 10 ** 6, "billion": 10 ** 9}
ALONE15 = {"single": 1, "pair": 2, "dozen": 12, "half": Fraction(1, 2), "quarter": Fraction(1, 4), "twice": 2,
           "double": 2, "thrice": 3, "triple": 3}
OPWORDS15 = {"plus": "+", "sum": "+", "added": "+", "more": "+", "increase": "+", "minus": "-", "less": "-",
             "difference": "-", "subtract": "-", "decrease": "-", "times": "*", "product": "*", "multiplied": "*",
             "of": "*", "divided": "/", "per": "/", "quotient": "/", "ratio": "/", "over": "/", "squared": "^2",
             "cubed": "^3", "power": "^"}
TOK15 = re.compile(r"[a-z]+|\S")


def word_numbers(text: str) -> list:
    """number words in order, each as a Fraction"""
    out, cur, total, live, scaled = [], 0, 0, False, False
    for w in TOK15.findall(text.lower().replace("-", " ")):
        if w in UNITS15 or w in TENS15:
            if live and not scaled and (w in TENS15 or cur % 10 or cur == 0 or cur < 20):
                out.append(Fraction(total + cur))
                cur, total = 0, 0
            cur += UNITS15.get(w, 0) + TENS15.get(w, 0)
            live, scaled = True, False
        elif w in SCALES15 and live:
            if w == "hundred":
                cur *= 100
            else:
                total += cur * SCALES15[w]
                cur = 0
            scaled = True
        elif w == "and" and live and scaled:
            continue
        else:
            if live:
                out.append(Fraction(total + cur))
                cur, total, live, scaled = 0, 0, False, False
            if w in ALONE15:
                out.append(Fraction(ALONE15[w]))
    if live:
        out.append(Fraction(total + cur))
    return out


SIGNS15 = str.maketrans({"\u00d7": "*", "\u00b7": "*", "\u2219": "*", "\u00f7": "/", "\u2212": "-", "\u2013": "-"})
SCI15 = re.compile(r"(\d+(?:\.\d+)?)\s*[x*]\s*10\s*\^\s*\{?\s*(-?\d+)\s*\}?")
DIGIT15 = re.compile(r"(?<![\w.])-?\d[\d,]*(?:\.\d+)?(?:\s*/\s*\d+(?![\d.]))?")


def to_fraction(s: str) -> Fraction:
    s = s.replace(",", "").replace(" ", "")
    if "/" in s:
        a, b = s.split("/")
        return Fraction(a) / Fraction(b) if Fraction(b) else None
    return Fraction(s)


def digit_numbers(text: str) -> list:
    text = SCI15.sub(lambda m: str(Fraction(m.group(1)) * Fraction(10) ** int(m.group(2))), text.translate(SIGNS15))
    out = []
    for m in DIGIT15.finditer(text):
        try:
            v = to_fraction(m.group())
        except (ValueError, ZeroDivisionError):
            v = None
        if v is not None:
            out.append(v)
    return out


CHOICE15 = re.compile(r"^\s*[-+$]?\s*\d[\d,]*(?:\.\d+)?(?:\s*/\s*\d+)?\s*(?:[x*]\s*10\s*\^\s*\{?-?\d+\}?)?\s*"
                      r"(?:%|[a-zA-Z/^0-9.]{0,10})?\s*\.?\s*$")


def choice_value(c: str):
    """the one number a choice states, or None"""
    s = str(c).translate(SIGNS15).strip()
    if CHOICE15.match(s):
        v = digit_numbers(s.replace("$", ""))
        return v[0] if v else None
    words = word_numbers(s)
    if len(words) == 1 and len(s.split()) <= 4:
        return words[0]
    return None


CHECKS15 = (("twenty-five", 25), ("one hundred and five", 105), ("a dozen eggs", 12), ("half", Fraction(1, 2)),
            ("three thousand two hundred", 3200), ("twice", 2), ("a single cell", 1), ("ninety-nine", 99),
            ("two million", 2 * 10 ** 6), ("1,200", 1200), ("4.1 x 10^-3", Fraction(41, 10000)),
            ("2/3", Fraction(2, 3)))
OK15 = 0
print("\n  1 NUMBER WORDS")
for text, want in CHECKS15:
    got = (word_numbers(text) + digit_numbers(text))[:1]
    ok = bool(got) and got[0] == want
    OK15 += ok
    print(f"    {text:<28} -> {str(got[0]) if got else 'nothing':<12}", 'right' if ok else f'WRONG (want {want})')
print(f"    operation words in the table: {len(OPWORDS15)} (plus -> +, per -> /, squared -> ^2, ...)")

# ---------------- 2 FHRR cross-association ----------------
D15 = 2048
RNG15 = np.random.default_rng(15)
THETA15 = RNG15.uniform(-np.pi, np.pi, D15)


def enc15(x: float) -> np.ndarray:
    return np.exp(1j * THETA15 * float(x))


def sim15(u: np.ndarray, v: np.ndarray) -> float:
    return float(np.mean(np.real(u * np.conj(v))))


S_BIND15 = sim15(enc15(3) * enc15(4), enc15(7))
print("\n  2 FHRR CROSS-ASSOCIATION (fractional power encoding, d = 2048)")
print(f"    bind(3, 4) against 7: {S_BIND15:.6f}; against 8: {sim15(enc15(3) * enc15(4), enc15(8)):+.4f}")
print("    similarity of 5 to 5 + delta:", ", ".join(f"{d:g}: {sim15(enc15(5), enc15(5 + d)):+.3f}"
                                                   for d in (0, 0.25, 0.5, 1, 2, 10)))
NWORDS15 = dict(list(UNITS15.items()) + list(TENS15.items()) + [(k, float(v)) for k, v in ALONE15.items()])
WVEC15 = {w: np.exp(1j * RNG15.uniform(-np.pi, np.pi, D15)) for w in NWORDS15}
MEM15 = sum(WVEC15[w] * enc15(v) for w, v in NWORDS15.items())
VALUES15 = sorted(set(float(v) for v in NWORDS15.values()))
VENC15 = np.array([enc15(v) for v in VALUES15])
recall = {w: VALUES15[int(np.argmax(np.real(VENC15.conj() @ (MEM15 * np.conj(WVEC15[w]))) / D15))] for w in NWORDS15}
RECALL15 = 100 * np.mean([recall[w] == float(v) for w, v in NWORDS15.items()])
print(f"    one bundle of {len(NWORDS15)} pairs (word bound to its number): {RECALL15:.1f}% recalled;",
      "dozen ->", f"{recall['dozen']:g},", f"half -> {recall['half']:g}, single -> {recall['single']:g}")

# ---------------- 3 exact arithmetic ----------------
OPS15 = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv,
         ast.Pow: operator.pow}
SPAN15 = re.compile(r"[\d.(][\d.\s()+\-*/^]*[\d)]")


def evaluate(node):
    if isinstance(node, ast.Expression):
        return evaluate(node.body)
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)) and not isinstance(node.value, bool):
        return Fraction(str(node.value))
    if isinstance(node, ast.UnaryOp) and isinstance(node.op, (ast.USub, ast.UAdd)):
        v = evaluate(node.operand)
        return -v if isinstance(node.op, ast.USub) else v
    if isinstance(node, ast.BinOp) and type(node.op) in OPS15:
        a, b = evaluate(node.left), evaluate(node.right)
        if isinstance(node.op, ast.Pow) and (b.denominator != 1 or abs(b) > 64 or abs(a) > 10 ** 6):
            raise ValueError("power out of range")
        return OPS15[type(node.op)](a, b)
    raise ValueError("not arithmetic")


def expressions(text: str) -> list:
    """(span, exact value) for every arithmetic span with an operator between numbers"""
    text = SCI15.sub(lambda m: f"({m.group(1)}*10**{m.group(2)})", text.translate(SIGNS15))
    text = re.sub(r"(?<=\d)\s*x\s*(?=\d)", "*", text)
    out = []
    for m in SPAN15.finditer(text):
        s = m.group().strip()
        if not re.search(r"\d\s*[-+*/^]\s*[\d(]", s):
            continue
        try:
            out.append((s, evaluate(ast.parse(s.replace("^", "**"), mode="eval"))))
        except (SyntaxError, ValueError, ZeroDivisionError, OverflowError, TypeError):
            pass
    return out


def same15(a, b) -> bool:
    try:
        a, b = float(a), float(b)
    except OverflowError:
        return a == b
    return a == b or abs(a - b) <= 0.005 * max(abs(a), abs(b))


print("\n  3 EXACT ARITHMETIC (ast tree, Fraction values)")
for text in ("What is 3 + 4 * 2?", "Compute (2^5 - 7) / 5.", "Find 1/3 + 1/6.", "In 1990-2000 the rate was 3%."):
    print(f"    {text:<34} -> " + (", ".join(f"{s} = {v}" for s, v in expressions(text)) or "no span"))


# ---------------- 4 gate and answer ----------------
def layer15(q: str, ch: list):
    """(path, the chosen choice or None, detail)"""
    vals = [choice_value(c) for c in ch]
    if any(v is None for v in vals) or asks_not(q):
        return "not gated", None, ""
    for s, v in expressions(q):
        hit = [j for j, c in enumerate(vals) if same15(c, v)]
        if len(hit) == 1:
            return "EXPR", hit[0], f"{s} = {v}"
    nums = list(dict.fromkeys(word_numbers(q) + digit_numbers(q)))[:8]
    hits = np.zeros(len(ch))
    for a in range(len(nums)):
        for b in range(len(nums)):
            if a == b:
                continue
            x, y = nums[a], nums[b]
            cand = [x - y] + ([x / y] if y else []) + ([x + y, x * y] if a < b else [])
            for v in cand:
                for j, c in enumerate(vals):
                    hits[j] += same15(c, v)
    if hits.max() > 0 and (hits == hits.max()).sum() == 1:
        return "PAIRS", int(hits.argmax()), f"numbers {', '.join(str(n) for n in nums)}"
    return "Cell 12", None, ""


def wilson15(k: float, n: int, z: float = 1.96) -> tuple:
    if not n:
        return 0.0, 100.0
    p, d = k / n, 1 + z * z / n
    mid, half = (p + z * z / (2 * n)) / d, z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return 100 * max(0.0, mid - half), 100 * min(1.0, mid + half)


RES15, SHOW15 = {}, []
print("\n  4 GATE AND ANSWER")
print(f"    {'':<26}{'gated':>6}{'EXPR':>6}{'PAIRS':>7}{'layer':>7}{'Cell 12':>9}{'chance':>8}{'whole set':>11}"
      f"{'was':>6}")
for name, qs in SETS12.items():
    rows = []
    for (q, ch, g), (x, _, _) in zip(qs, READ12[name]):
        c12 = credit12(scores12(x), g)
        path, j, why = layer15(q, ch)
        mine = c12 if j is None else float(j == g)
        rows.append((path, mine, c12, 1 / len(ch)))
        if name == "MMLU dev" and path in ("EXPR", "PAIRS") and len(SHOW15) < 4:
            SHOW15.append((q, ch, g, path, j, why))
    gated = [r for r in rows if r[0] != "not gated"]
    n = len(gated)
    RES15[name] = dict(n=n, expr=sum(r[0] == "EXPR" for r in gated), pairs=sum(r[0] == "PAIRS" for r in gated),
                       layer=100 * np.mean([r[1] for r in gated]) if n else float("nan"),
                       c12=100 * np.mean([r[2] for r in gated]) if n else float("nan"),
                       chance=100 * np.mean([r[3] for r in gated]) if n else float("nan"),
                       whole=100 * np.mean([r[1] for r in rows]), was=100 * np.mean([r[2] for r in rows]),
                       paths={p: [r for r in gated if r[0] == p] for p in ("EXPR", "PAIRS")})
    r = RES15[name]
    print(f"    {name:<26}{n:>6}{r['expr']:>6}{r['pairs']:>7}{r['layer']:>7.1f}{r['c12']:>9.1f}{r['chance']:>8.1f}"
          f"{r['whole']:>11.1f}{r['was']:>6.1f}")
print("    (layer and Cell 12 are accuracy % on the gated questions; whole set: the layer's answers in place of"
      " Cell 12's)")
print("\n    each path, all practice sets together: fired, right %, 95% interval, Cell 12 on the same questions")
for p in ("EXPR", "PAIRS"):
    rs = [x for name in SETS12 if ROLE12[name] == "practice" for x in RES15[name]["paths"][p]]
    k = sum(x[1] for x in rs)
    lo, hi = wilson15(k, len(rs))
    c12 = 100 * np.mean([x[2] for x in rs]) if rs else float("nan")
    print(f"      {p:<6}{len(rs):>5}{100 * k / max(len(rs), 1):>8.1f}  [{lo:5.1f}, {hi:5.1f}]{c12:>8.1f}")

print("\n    EXAMPLES (MMLU dev)")
for q, ch, g, path, j, why in SHOW15:
    print(f"      {q[:100]}\n        {path}: {why[:70]}; said {str(ch[j])[:30]}, gold {str(ch[g])[:30]}")

print("\n  PREDICTIONS vs RESULTS")
M15 = RES15["MMLU validation"]
LEDGER.result("B5-A1", OK15)
LEDGER.result("B5-A2", round(S_BIND15, 6))
LEDGER.result("B5-A3", round(RECALL15, 2))
LEDGER.result("B5-A4", M15["n"])
LEDGER.result("B5-A5", round(100 * M15["expr"] / max(M15["n"], 1), 2))
LEDGER.result("B5-A6", round(100 * M15["pairs"] / max(M15["n"], 1), 2))
LEDGER.result("B5-A7", round(M15["layer"] - M15["c12"], 2))
LEDGER.result("B5-T24", round(time.time() - T15))
if "sync" in globals():
    sync("Cell 15")
print(f"\n  ({time.time() - T15:.0f}s)")
print("  LAST LINE OF MC AI BRAIN 5 CELL 15")


MC AI BRAIN 5 - CELL 15: THE NUMBER LAYER

  1 NUMBER WORDS
    twenty-five                  -> 25           right
    one hundred and five         -> 105          right
    a dozen eggs                 -> 12           right
    half                         -> 1/2          right
    three thousand two hundred   -> 3200         right
    twice                        -> 2            right
    a single cell                -> 1            right
    ninety-nine                  -> 99           right
    two million                  -> 2000000      right
    1,200                        -> 1200         right
    4.1 x 10^-3                  -> 41/10000     right
    2/3                          -> 2/3          right
    operation words in the table: 22 (plus -> +, per -> /, squared -> ^2, ...)

  2 FHRR CROSS-ASSOCIATION (fractional power encoding, d = 2048)
    bind(3, 4) against 7: 1.000000; against 8: +0.0047
    similarity of 5 to 5 + delta: 0: +1.000, 0.25: +0.901, 0.5: +0.640, 1: +0.00

In [24]:
# =====================================================================
# MC AI BRAIN 5 - CELL 16: A DEPENDENCY PARSER LEARNED BY COUNTING
# Needs Cells 0-2 (DATA, WN, LEDGER). Uses EWT train (fit) and EWT dev
# (practice); EWT test is not read. No gradient, no iteration: one pass of
# counting, as in the generative parsers of Eisner (1996) and Collins (2003).
# WORD CLASSES  Brain 4's TnT tagger (mcai.count, Brants 2000), counted on
#   EWT train, with WordNet's possible tags for unseen words (eps 0.0001)
#   and lower-case fallback: the setting Brain 4 called MC AI
# BASE FORMS  WordNet's base form for the word's class (n, v, a, r), else
#   the word in lower case
# ARC COUNTS  for every ordered pair (head h, dependent d) of a train
#   sentence (h = 0 is ROOT): how often h is d's head, out of how often the
#   pair occurs, under eight keys:
#   T   tags of h and d, direction, distance (1, 2, 3-4, 5-7, 8+)
#   TB  T + a verb between them? + a punctuation mark between them?
#   CD  TB + the tags left and right of d;  CH  TB + those of h
#   C   TB + all four neighbour tags (distance capped at 3-4)
#   HL  h's base form + both tags;  DL  d's base form + both tags
#   LL  both base forms, direction, verb between
#   Keys are kept as 64-bit hashes (Python's hash, fixed within a session)
# SCORE  backoff chain, each level p = (attached + k * p_lower) / (pairs + k),
#   k = 5: T (lower 0.001) -> TB -> mean(CD, CH) -> C -> mean(HL, DL) -> LL;
#   arc score = log odds. The chain and k were chosen on EWT dev in a local
#   test (UAS with gold tags: T alone 72.6; this chain 78.1)
# TREE  Eisner's (1996) algorithm: the highest-scoring projective tree
# LABELS  the most frequent relation for (h tag, d tag, direction), else
#   for (d tag, direction), else "dep"
# SCORED  heads right (UAS) and heads + relation right (LAS), punctuation
#   left out (UPOS PUNCT), on EWT dev
# PREDICTIONS (written before the first run, a local one on the same files;
#   B5-P2 is a CHECK: its chain was chosen on dev, local 78.1)
#   B5-P1 tagger XPOS accuracy on EWT dev, %: 92-95
#   B5-P2 dev UAS with gold tags, %: 77-79.5
#   B5-P3 dev UAS with the tagger's tags, %: 72-77
#   B5-P4 dev LAS with the tagger's tags, %: 62-72
#   B5-P5 Brain 1's perceptron (dev UAS 83.4, its own tags) minus B5-P3,
#         points: 5-11
#   B5-T25 Cell 16 seconds: <= 900
# =====================================================================
import math
import re
import sys
import time
from collections import Counter, defaultdict

import numpy as np

print("=" * 64); print("MC AI BRAIN 5 - CELL 16: A DEPENDENCY PARSER LEARNED BY COUNTING"); print("=" * 64)
T16 = time.time()
for need in ("DATA", "WN", "LEDGER"):
    if need not in globals():
        raise RuntimeError(f"run Cells 0-2 first ({need} is missing)")
from mcai.core import at_most, between
from mcai.count import TrigramTagger, wordnet_tags
from mcai.read import read_conllu

for key, claim, rule in (("B5-P1", "tagger XPOS accuracy on EWT dev, %", between(92, 95)),
                         ("B5-P2", "dev UAS with gold tags, % (check)", between(77, 79.5)),
                         ("B5-P3", "dev UAS with the tagger's tags, %", between(72, 77)),
                         ("B5-P4", "dev LAS with the tagger's tags, %", between(62, 72)),
                         ("B5-P5", "Brain 1's dev UAS 83.4 minus B5-P3, points", between(5, 11)),
                         ("B5-T25", "Cell 16 seconds", at_most(900))):
    LEDGER.predict(key, claim, rule)
EWT16 = DATA / "ewt"
TRAIN16 = read_conllu(EWT16 / "en_ewt-ud-train.conllu")
DEV16 = read_conllu(EWT16 / "en_ewt-ud-dev.conllu")
print(f"  EWT train {len(TRAIN16):,} sentences, dev {len(DEV16):,} (test not read)")

# ---------------- word classes ----------------
t1 = time.time()
TAGGER16 = TrigramTagger.count([[(w.form, w.xpos) for w in s.words] for s in TRAIN16])
WNTAGS16 = {}


def lexicon16(word: str) -> set:
    if word not in WNTAGS16:
        WNTAGS16[word] = wordnet_tags(WN, word)
    return WNTAGS16[word]


def tag16(words: list) -> list:
    return TAGGER16.tag(words, lexicon16, eps=1e-4, lower=True)


DEV_TAGS16 = [tag16([w.form for w in s.words]) for s in DEV16]
TAG_ACC16 = 100 * np.mean([t == w.xpos for s, ts in zip(DEV16, DEV_TAGS16) for w, t in zip(s.words, ts)])
print(f"  TAGGER counted and run on dev ({time.time() - t1:.0f}s): XPOS {TAG_ACC16:.2f}%")

# ---------------- base forms ----------------
POS16 = {"N": "n", "V": "v", "J": "a", "R": "r"}
BASE16 = {}


def base16(word: str, tag: str) -> str:
    k = (word, tag[:1])
    if k not in BASE16:
        low, p = word.lower(), POS16.get(tag[:1])
        b = WN.base_forms(low, p) if p and low.isalpha() else []
        BASE16[k] = b[0] if b else low
    return BASE16[k]


# ---------------- arc counts ----------------
PUNCT16 = {",", ":", ".", "-LRB-", "-RRB-", "HYPH", "``", "''", "NFP"}


def dist16(d: int) -> int:
    d = abs(d)
    return 1 if d == 1 else 2 if d == 2 else 3 if d <= 4 else 4 if d <= 7 else 5


def keys16(t: list, l: list, h: int, d: int) -> tuple:
    """the eight keys of the pair (head h, dependent d), as hashes, in the order T TB CD CH C HL DL LL"""
    n, dr, b = len(t), int(h > d), dist16(h - d)
    bc = min(b, 3)
    a, z = min(h, d), max(h, d)
    mid = t[a + 1:z]
    v = int(any(x[:1] == "V" or x == "MD" for x in mid))
    pu = int(any(x in PUNCT16 for x in mid))
    dl, dn = (t[d - 1] if d > 1 else "<"), (t[d + 1] if d + 1 < n else ">")
    hl, hn = (t[h - 1] if h > 0 else "<"), (t[h + 1] if h + 1 < n else ">")
    th, td = t[h], t[d]
    return (hash(("T", th, td, dr, b)), hash(("TB", th, td, dr, b, v, pu)),
            hash(("CD", th, td, dr, b, v, pu, dl, dn)), hash(("CH", th, td, dr, b, v, pu, hl, hn)),
            hash(("C", th, td, dr, bc, v, pu, hl, hn, dl, dn)), hash(("HL", l[h], th, td, dr, bc, v)),
            hash(("DL", th, l[d], td, dr, bc, v)), hash(("LL", l[h], l[d], dr, v)))


def view16(forms: list, tags: list) -> tuple:
    return ["ROOT"] + list(tags), ["ROOT"] + [base16(f, t) for f, t in zip(forms, tags)]


t1 = time.time()
PAIRS16, ATTACH16 = Counter(), Counter()
LABEL16, LABEL_D16 = defaultdict(Counter), defaultdict(Counter)
for s in TRAIN16:
    t, l = view16([w.form for w in s.words], [w.xpos for w in s.words])
    gold = [0] + [w.head for w in s.words]
    for d in range(1, len(t)):
        for h in range(len(t)):
            if h != d:
                ks = keys16(t, l, h, d)
                PAIRS16.update(ks)
                if gold[d] == h:
                    ATTACH16.update(ks)
        w = s.words[d - 1]
        dr = int(w.head > d)
        LABEL16[(t[w.head], t[d], dr)][w.deprel] += 1
        LABEL_D16[(t[d], dr)][w.deprel] += 1
print(f"  ARC COUNTS ({time.time() - t1:.0f}s): {len(PAIRS16):,} keys, {len(ATTACH16):,} with an attachment")

K16 = 5.0


def score16(t: list, l: list, h: int, d: int) -> float:
    T, TB, CD, CH, C, HL, DL, LL = keys16(t, l, h, d)

    def p(k, lower):
        return (ATTACH16[k] + K16 * lower) / (PAIRS16[k] + K16)

    pb = p(TB, p(T, 1e-3))
    pc = p(C, (p(CD, pb) + p(CH, pb)) / 2)
    x = p(LL, (p(HL, pc) + p(DL, pc)) / 2)
    x = min(max(x, 1e-6), 1 - 1e-6)
    return math.log(x / (1 - x))


def eisner16(S: np.ndarray) -> list:
    """head of every position 1..n-1 in the best projective tree; S[h, d] = score of the arc h -> d"""
    n, NEG = S.shape[0], -1e18
    C, I = np.full((n, n, 2), NEG), np.full((n, n, 2), NEG)
    CB, IB = np.zeros((n, n, 2), int), np.zeros((n, n, 2), int)
    for s in range(n):
        C[s, s] = 0.0
    for k in range(1, n):
        for s in range(n - k):
            e = s + k
            v = C[s, s:e, 1] + C[s + 1:e + 1, e, 0]
            r = int(np.argmax(v))
            I[s, e, 0], I[s, e, 1], IB[s, e] = v[r] + S[e, s], v[r] + S[s, e], s + r
            v = C[s, s:e, 0] + I[s:e, e, 0]
            r = int(np.argmax(v))
            C[s, e, 0], CB[s, e, 0] = v[r], s + r
            v = I[s, s + 1:e + 1, 1] + C[s + 1:e + 1, e, 1]
            r = int(np.argmax(v))
            C[s, e, 1], CB[s, e, 1] = v[r], s + 1 + r
    head, todo = [0] * n, [("C", 0, n - 1, 1)]
    while todo:
        kind, s, e, dr = todo.pop()
        if s == e:
            continue
        if kind == "C":
            r = CB[s, e, dr]
            todo += [("C", s, r, 0), ("I", r, e, 0)] if dr == 0 else [("I", s, r, 1), ("C", r, e, 1)]
        else:
            r = IB[s, e, dr]
            if dr == 0:
                head[s] = e
            else:
                head[e] = s
            todo += [("C", s, r, 1), ("C", r + 1, e, 0)]
    return head


def label16(t: list, h: int, d: int) -> str:
    dr = int(h > d)
    c = LABEL16.get((t[h], t[d], dr)) or LABEL_D16.get((t[d], dr))
    return c.most_common(1)[0][0] if c else "dep"


def parse16(forms: list, tags: list = None) -> list:
    """[(form, tag, head, relation)] for one tokenised sentence; tags from the tagger when not given"""
    tags = tags or tag16(forms)
    t, l = view16(forms, tags)
    n = len(t)
    S = np.full((n, n), -1e9)
    for d in range(1, n):
        for h in range(n):
            if h != d:
                S[h, d] = score16(t, l, h, d)
    heads = eisner16(S)
    return [(forms[i - 1], tags[i - 1], heads[i], label16(t, heads[i], i)) for i in range(1, n)]


TOKEN16 = re.compile(r"\d+(?:[.,]\d+)*|\w+(?:'\w+)?|n't|[^\w\s]")


def read16(text: str) -> list:
    """parse16 of raw text, split into words by a regular expression"""
    return parse16(TOKEN16.findall(text))


# ---------------- scores on EWT dev ----------------
t1 = time.time()
OUT16 = {}
for mode in ("gold tags", "tagger's tags"):
    right = labelled = total = 0
    for s, ts in zip(DEV16, DEV_TAGS16):
        got = parse16([w.form for w in s.words], [w.xpos for w in s.words] if mode == "gold tags" else ts)
        for w, (_, _, h, rel) in zip(s.words, got):
            if w.upos == "PUNCT":
                continue
            total += 1
            right += h == w.head
            labelled += h == w.head and rel == w.deprel
    OUT16[mode] = (100 * right / total, 100 * labelled / total)
    print(f"  DEV, {mode:<14} UAS {OUT16[mode][0]:.2f}  LAS {OUT16[mode][1]:.2f}  ({total:,} words)")
print(f"  ({time.time() - t1:.0f}s); Brain 1's perceptron parser printed dev UAS 83.4 with its own tags")

print("\n  EXAMPLE")
for text in ("The eye-piece lens has a focal length of 20 cm.",
             "Which of the following best describes the role of DNA?"):
    print(f"    {text}")
    print("      " + "  ".join(f"{f}/{t}<-{h}:{r}" for f, t, h, r in read16(text)))

print("\n  PREDICTIONS vs RESULTS")
LEDGER.result("B5-P1", round(TAG_ACC16, 2))
LEDGER.result("B5-P2", round(OUT16["gold tags"][0], 2))
LEDGER.result("B5-P3", round(OUT16["tagger's tags"][0], 2))
LEDGER.result("B5-P4", round(OUT16["tagger's tags"][1], 2))
LEDGER.result("B5-P5", round(83.4 - OUT16["tagger's tags"][0], 2))
LEDGER.result("B5-T25", round(time.time() - T16))
if "sync" in globals():
    sync("Cell 16")
print(f"\n  ({time.time() - T16:.0f}s)")
print("  LAST LINE OF MC AI BRAIN 5 CELL 16")


MC AI BRAIN 5 - CELL 16: A DEPENDENCY PARSER LEARNED BY COUNTING
  EWT train 12,544 sentences, dev 2,001 (test not read)
  TAGGER counted and run on dev (9s): XPOS 92.49%
  ARC COUNTS (38s): 8,865,214 keys, 509,655 with an attachment
  DEV, gold tags      UAS 78.11  LAS 68.52  (22,073 words)
  DEV, tagger's tags  UAS 74.71  LAS 63.52  (22,073 words)
  (17s); Brain 1's perceptron parser printed dev UAS 83.4 with its own tags

  EXAMPLE
    The eye-piece lens has a focal length of 20 cm.
      The/DT<-2:det  eye/NN<-4:compound  -/HYPH<-2:punct  piece/NN<-5:compound  lens/NN<-6:nsubj  has/VBZ<-0:root  a/DT<-9:det  focal/JJ<-9:amod  length/NN<-6:obj  of/IN<-12:case  20/CD<-12:nummod  cm/NNS<-9:nmod  ./.<-6:punct
    Which of the following best describes the role of DNA?
      Which/WDT<-4:nsubj  of/IN<-4:mark  the/DT<-4:det  following/VBG<-0:root  best/JJS<-6:amod  describes/NNS<-4:obj  the/DT<-8:det  role/NN<-4:obj  of/IN<-10:case  DNA/NN<-8:nmod  ?/.<-4:punct

  PREDICTIONS vs RESULTS
  

In [25]:
# =====================================================================
# MC AI BRAIN 5 - CELL 17: DOES SENTENCE STRUCTURE HELP? A TEST ON MMLU
# Needs Cells 12A, 12B (lookup, readings, Cell 12's scores) and Cell 16
# (the counted parser). Nothing is fitted or chosen here: one fixed rule,
# scored on practice sets only (MMLU validation, MMLU dev, ARC-Challenge
# validation). MMLU-Pro validation is left out: 68 of its 70 questions are
# MMLU dev questions.
# CANDIDATES  for each question and choice, of the looked-up sentences
#   holding at least one question word and one of the choice's own words
#   (Cell 6's base forms; the choice's words = its words the question does
#   not have), the 2 with the highest JOINT (Cell 12B); at most 60 tokens
# PARSED  each candidate once by Cell 16 (tags by its tagger); the heads
#   are saved as an artifact
# LINK  in a parsed sentence, a question word and a choice word are linked
#   when the tree path between them is at most 2 arcs (a direct arc, or
#   through one word such as "of" or "is")
# READING  per choice: the share of the question's IDF held by the
#   question words linked to one of its words, best over its candidates
# RULE  the choice with the highest LINK share alone answers; when no
#   choice has a link, or the best is tied, Cell 12's answer stands
#   (NOT / EXCEPT questions: Cell 12's answer stands)
# PREDICTIONS (written before running)
#   B5-J1 sentences parsed: 8,000-25,000
#   B5-J2 MMLU validation questions where LINK picks one choice, %: 15-45
#   B5-J3 on those questions, LINK minus Cell 12, points: -3-5
#   B5-J4 MMLU validation, rule minus Cell 12, whole set, points: -1.5-2
#   B5-J5 ARC-Challenge validation, rule minus Cell 12, points: -2-3
#   B5-T26 Cell 17 seconds, parsing: <= 2400; B5-T27 loading: <= 300
# =====================================================================
import hashlib
import time

import numpy as np

print("=" * 64); print("MC AI BRAIN 5 - CELL 17: DOES SENTENCE STRUCTURE HELP? A TEST ON MMLU"); print("=" * 64)
T17 = time.time()
for need in ("SETS12", "READ12", "scores12", "credit12", "pages12", "SENTS12", "S_FORMS12", "T_FORMS12", "P_START12",
             "idf12", "parse16", "tag16", "forms", "asks_not", "ARTS"):
    if need not in globals():
        raise RuntimeError(f"run Cells 12A, 12B and 16 first ({need} is missing)")
for key, claim, rule in (("B5-J1", "sentences parsed", between(8000, 25000)),
                         ("B5-J2", "MMLU validation questions where LINK picks one choice, %", between(15, 45)),
                         ("B5-J3", "on those questions, LINK minus Cell 12, points", between(-3, 5)),
                         ("B5-J4", "MMLU validation, rule minus Cell 12, whole set, points", between(-1.5, 2)),
                         ("B5-J5", "ARC-Challenge validation, rule minus Cell 12, points", between(-2, 3)),
                         ("B5-T26", "Cell 17 seconds, parsing", at_most(2400)),
                         ("B5-T27", "Cell 17 seconds, loading", at_most(300))):
    LEDGER.predict(key, claim, rule)
NAMES17 = ("MMLU validation", "MMLU dev", "ARC-Challenge validation")
PER17, LONG17 = 2, 60


def share17(words: set, total: float) -> float:
    return sum(idf12(w) for w in words) / total if words else 0.0


def candidates17(q: str, ch: list) -> list:
    """per choice: (its words, up to 2 sentence numbers)"""
    Q = set(forms(q))
    qt = sum(idf12(w) for w in Q) or 1.0
    C = [set(forms(str(c))) - Q for c in ch]
    pool = pages12(q).union(*(pages12(str(c)) for c in ch))
    best = [[] for _ in ch]
    for p in pool:
        tw = T_FORMS12[p]
        for k in range(P_START12[p], P_START12[p + 1]):
            sw = S_FORMS12[k]
            if not (Q & sw) or len(SENTS12[k].split()) > LONG17:
                continue
            q_all = share17(Q & (sw | tw), qt)
            for j, c in enumerate(C):
                if c & sw:
                    best[j].append((q_all * share17(c & (sw | tw), sum(idf12(w) for w in c) or 1.0), k))
    return [(C[j], [k for _, k in sorted(b, reverse=True)[:PER17]]) for j, b in enumerate(best)]


t1 = time.time()
CAND17 = {n: [candidates17(q, ch) for q, ch, _ in SETS12[n]] for n in NAMES17}
WANT17 = sorted({k for n in NAMES17 for row in CAND17[n] for _, ks in row for k in ks})
print(f"  CANDIDATES ({time.time() - t1:.0f}s): {len(WANT17):,} sentences for {sum(len(SETS12[n]) for n in NAMES17):,}"
      " questions")


def build17() -> dict:
    heads, start, t2 = [], [0], time.time()
    for i, k in enumerate(WANT17):
        words = SENTS12[k].split()
        heads += [h for _, _, h, _ in parse16(words, tag16(words))]
        start.append(len(heads))
        if (i + 1) % 2000 == 0:
            print(f"    parsed {i + 1:,} of {len(WANT17):,} ({time.time() - t2:.0f}s)")
    return dict(ids=np.array(WANT17, np.int64), heads=np.array(heads, np.int16), start=np.array(start, np.int64))


DEPENDS17 = dict(what="Cell 16's heads for Cell 17's candidate sentences", version=1, per=PER17, longest=LONG17,
                 ids=hashlib.sha256(np.array(WANT17, np.int64).tobytes()).hexdigest(),
                 parser=hashlib.sha256(repr((K16, sorted(PUNCT16))).encode()).hexdigest(),
                 ewt=sha256(DATA / "ewt" / "en_ewt-ud-train.conllu") if "sha256" in globals() else "")
PARSED17, BUILT17 = ARTS.get("mmlu_parses", DEPENDS17, build17)
ROW17 = {int(k): i for i, k in enumerate(PARSED17["ids"])}
print(f"  PARSES {'built and saved' if BUILT17 else 'loaded'}: {len(ROW17):,} sentences ({time.time() - t1:.0f}s)")


def tree17(k: int) -> tuple:
    """(base form of each token or None, parent of each token 0..n-1, -1 = root)"""
    i = ROW17[k]
    h = PARSED17["heads"][PARSED17["start"][i]:PARSED17["start"][i + 1]]
    words = SENTS12[k].split()
    base = [(forms(w) or [None])[0] for w in words]
    return base, [int(x) - 1 for x in h]


def path17(par: list, a: int, b: int) -> int:
    """arcs between tokens a and b in the tree"""
    up, d, x = {}, 0, a
    while x >= 0 and x not in up:
        up[x], d, x = d, d + 1, par[x]
    d, x, seen = 0, b, set()
    while x >= 0 and x not in seen:
        if x in up:
            return up[x] + d
        seen.add(x)
        d, x = d + 1, par[x]
    return 99


TREES17 = {}


def link17(q: str, row: list) -> np.ndarray:
    Q = set(forms(q))
    qt = sum(idf12(w) for w in Q) or 1.0
    out = np.zeros(len(row))
    for j, (C, ks) in enumerate(row):
        for k in ks:
            if k not in TREES17:
                TREES17[k] = tree17(k)
            base, par = TREES17[k]
            qpos = [i for i, b in enumerate(base) if b in Q]
            cpos = [i for i, b in enumerate(base) if b in C]
            linked = {base[a] for a in qpos if any(path17(par, a, c) <= 2 for c in cpos)}
            out[j] = max(out[j], share17(linked, qt))
    return out


# ---------------- the rule, against Cell 12 ----------------
RES17 = {}
print(f"\n  {'':<26}{'questions':>10}{'LINK picks':>12}{'LINK':>7}{'Cell 12':>9}{'rule':>7}{'Cell 12':>9}")
print(f"  {'':<26}{'':>10}{'one, %':>12}{'on those':>16}{'whole set':>16}")
for n in NAMES17:
    rows = []
    for (q, ch, g), row, (x, _, _) in zip(SETS12[n], CAND17[n], READ12[n]):
        c12 = credit12(scores12(x), g)
        lk = link17(q, row)
        top = lk.max()
        alone = top > 0 and (lk == top).sum() == 1 and not asks_not(q)
        mine = float(int(lk.argmax()) == g) if alone else c12
        rows.append((alone, mine, c12))
    r = np.array(rows, float)
    a = r[:, 0] > 0
    RES17[n] = dict(n=len(r), share=100 * a.mean(), link=100 * r[a, 1].mean() if a.any() else float("nan"),
                    c12_on=100 * r[a, 2].mean() if a.any() else float("nan"), rule=100 * r[:, 1].mean(),
                    c12=100 * r[:, 2].mean(), fired=int(a.sum()))
    v = RES17[n]
    print(f"  {n:<26}{v['n']:>10,}{v['share']:>11.1f}%{v['link']:>7.1f}{v['c12_on']:>9.1f}{v['rule']:>7.1f}"
          f"{v['c12']:>9.1f}")

print("\n  EXAMPLES (MMLU dev, where LINK picked one choice)")
shown = 0
for (q, ch, g), row in zip(SETS12["MMLU dev"], CAND17["MMLU dev"]):
    lk = link17(q, row)
    if lk.max() > 0 and (lk == lk.max()).sum() == 1 and shown < 3:
        j = int(lk.argmax())
        k = row[j][1][0] if row[j][1] else None
        print(f"    {q[:90]}\n      LINK: {str(ch[j])[:40]} (gold {str(ch[g])[:40]}); from: {SENTS12[k][:100]}")
        shown += 1

print("\n  PREDICTIONS vs RESULTS")
M17 = RES17["MMLU validation"]
LEDGER.result("B5-J1", len(ROW17))
LEDGER.result("B5-J2", round(M17["share"], 2))
LEDGER.result("B5-J3", round(M17["link"] - M17["c12_on"], 2))
LEDGER.result("B5-J4", round(M17["rule"] - M17["c12"], 2))
LEDGER.result("B5-J5", round(RES17["ARC-Challenge validation"]["rule"] - RES17["ARC-Challenge validation"]["c12"], 2))
LEDGER.result("B5-T26" if BUILT17 else "B5-T27", round(time.time() - T17))
if "sync" in globals():
    sync("Cell 17")
print(f"\n  ({time.time() - T17:.0f}s)")
print("  LAST LINE OF MC AI BRAIN 5 CELL 17")


MC AI BRAIN 5 - CELL 17: DOES SENTENCE STRUCTURE HELP? A TEST ON MMLU
  CANDIDATES (4s): 6,957 sentences for 2,115 questions
    parsed 2,000 of 6,957 (50s)
    parsed 4,000 of 6,957 (101s)
    parsed 6,000 of 6,957 (151s)
  PARSES built and saved: 6,957 sentences (179s)

                             questions  LINK picks   LINK  Cell 12   rule  Cell 12
                                            one, %        on those       whole set
  MMLU validation                1,531       38.9%   27.7     31.5   29.8     31.2
  MMLU dev                         285       35.4%   30.7     30.7   30.1     30.1
  ARC-Challenge validation         299       51.8%   23.9     30.3   27.2     30.6

  EXAMPLES (MMLU dev, where LINK picked one choice)
    Which of the following is the body cavity that contains the pituitary gland?
      LINK: Cranial (gold Cranial); from: The hypophysis rests upon the hypophysial fossa of the sphenoid bone in the center of the middle cra
    Where do most short-period come

In [26]:
# =====================================================================
# MC AI BRAIN 5 - CELL 18A: TEXTBOOKS AS KNOWLEDGE - FETCHED, READ, INDEXED
# Needs Cells 0, 2 and 6 (DATA, ARTS, forms, sha256) and git with the
# internet on. Role: knowledge (read, never scored).
# SOURCE  25 OpenStax college textbooks (Rice University), GitHub
#   openstax/osbooks-*, each pinned to a commit; only the text of every
#   module (modules/*/index.cnxml) is fetched (git sparse checkout). A
#   book's fingerprint = SHA-256 of its sorted "path<TAB>SHA-256" lines; a
#   book whose fingerprint differs is refused. Licences as the books state
#   them (mostly CC BY-NC-SA 4.0): local research use with attribution;
#   never copied into a dataset, a product or anything sold
# READ  every <para> outside exercises (MathML left out) of 6+ words, and
#   every glossary definition as "term: meaning"; each with its module's
#   title. Saved as an artifact (text, book, title)
# INDEX  Cell 6's base forms of title + text; BM25 (k1 1.2, b 0.75), with
#   the base-form counts saved in the artifact
# PREDICTIONS (written before running; B5-O1 is a CHECK: a local run of
#   the same reading, with a plain tokenizer, gave 138,006 paragraphs)
#   B5-O1 paragraphs and definitions: 125,000-150,000
#   B5-O2 words indexed (base forms kept), millions: 3-7
#   B5-T28 Cell 18A seconds, building: <= 1800; B5-T29 loading: <= 300
# =====================================================================
import hashlib
import re
import subprocess
import time
import xml.etree.ElementTree as ET

import numpy as np
import scipy.sparse as sp
from mcai.core import at_most, between

print("=" * 64); print("MC AI BRAIN 5 - CELL 18A: TEXTBOOKS AS KNOWLEDGE"); print("=" * 64)
T18 = time.time()
for need in ("DATA", "ARTS", "forms", "sha256", "pack12", "unpack12"):
    if need not in globals():
        raise RuntimeError(f"run Cells 0, 2, 6 and 12A first ({need} is missing)")
for key, claim, rule in (("B5-O1", "paragraphs and definitions (check)", between(125000, 150000)),
                         ("B5-O2", "words indexed, millions", between(3, 7)),
                         ("B5-T28", "Cell 18A seconds, building", at_most(1800)),
                         ("B5-T29", "Cell 18A seconds, loading", at_most(300))):
    LEDGER.predict(key, claim, rule)
BOOKS18 = {  # book: (commit, fingerprint, modules, licence)
    "american-government": ("a2677245241954039c7ebc996e82a8f3b0b4376f",
        "7297a4644b077273d3d793d077a76e55e2f5662102028c3bf58360d4bda5fe6e", 100, "by-nc-sa/4.0"),
    "anatomy-physiology": ("5ae32b3f4bc24ed003e91dc38bf47dba80751044",
        "473078433b5161e291669e0957caab7caf3cfbbf39e99a585de296b3b07ca654", 198, "by-nc-sa/4.0"),
    "astronomy": ("dff6acf8df597ceda73985308f838ed38417a606",
        "dec9a26286225cf912e7917ba74145dd5e316d0ddd24442e135988d23d44082c", 199, "by-nc-sa/4.0"),
    "biology-bundle": ("5ae2f953cfe924469a6618578a00b3160439fc43",
        "d98d56b89a01c25b8626f687d41564c7d52013ad669204facd4f3b7e3983d1f2", 574, "by-nc-sa/4.0"),
    "business-ethics": ("37c6dbe07bd93b5ead10942092a348d60d5dfd30",
        "8a66764f774f017f868dda998162b5b6882cc9bd2702cd3d984631575f8f9092", 61, "by-nc-sa/4.0"),
    "calculus-bundle": ("018da4da252bb9c56759e34396efea91852bcf5a",
        "137645507a190364b88cfe838c93dbada1cd3bd6267948b1947d298c68bb3965", 133, "by-nc-sa/4.0"),
    "chemistry-bundle": ("78b742a7fed1379705c5d882cee1ebfab7f58dc9",
        "09d1179ef32e653b1cb12b00fb7e51918a1d7adcc5adf1b7d481c8810392c2b9", 176, "by-nc-sa/4.0"),
    "college-algebra-bundle": ("709c7417a68293f40a42531c727fbb41877afed5",
        "f652da3caa419e15bb086faf2f903dd28968b96fd72e0406e8c17644c811c4c0", 138, "by-nc-sa/4.0"),
    "college-physics-bundle": ("66fcef29596e3a32818162b837dd0c85b754cbad",
        "901c9f9645dd14939e8ade2c5b75b44b324ac7c799a7fe8bb4082a061d1a254e", 318, "by-nc-sa/4.0"),
    "introduction-anthropology": ("e466666414f1900a783141339dc5ef3106dba5dd",
        "e72301ab06f3e0c7b36aa447d75bfb63ccb9059ccc7f7dba5b44a121e02d6038", 121, "by-nc-sa/4.0"),
    "introduction-business": ("2a839877ab89a207c9b9e501df438173013fa45c",
        "5d317d27cc6ea433c2e552e60457f2014719ebba010f74e7989b90b8fe3a7d19", 153, "by-nc-sa/4.0"),
    "introduction-philosophy": ("a0dbb585086d0302a9e509c0aa50f2ef436c3613",
        "7292031548d7895d1471d689c7031dfa80bb1f461ae5c56b14b4684fb8e46786", 76, "by-nc-sa/4.0"),
    "introduction-political-science": ("61afbf50d27ba833feb64b66ba06a5384e9f371c",
        "231d285f132bf97bc661f1f1f3365ae55dfe37adf851ccc3a8045d2c5b1b7593", 109, "by-nc-sa/4.0"),
    "introduction-sociology": ("ab8839c5c5340cb4dffbc310699da450549deb20",
        "2cd71bf8edc2b0decdca09e2102267cf4ed7fce9c7069997b95660073e2fd359", 96, "by-nc-sa/4.0"),
    "introductory-statistics-bundle": ("24e9884724b13baba2b8879d03103cccafc7e25d",
        "49b8c788528d1b7985b344ab297cbfc6d87a18895f48324b4b93ab4de9e59293", 179, "by-nc-sa/4.0"),
    "life-liberty-and-pursuit-happiness": ("acb6fbe2744dd112187b6aff5946cfe969410fca",
        "2404926a1f3caa97f8dcb8ceeb53571a6eeea5fde4acb568437240098a99ba9b", 532, "by/4.0"),
    "lifespan-development": ("ebed4ae43c261041030f752eb49d92d0411608db",
        "c9a451362a3762fb5145556099552554293f17cb6e4b8491bc48f34879b4f3b6", 100, "by-nc-sa/4.0"),
    "microbiology": ("f91d7fcdc7ebbafa36ec33099f78ff0a494f1c2b",
        "d6e35525a4959848e8306e4cbe4efdd0db1c4e657d7666549e4a633bfd08e260", 159, "by-nc-sa/4.0"),
    "principles-economics-bundle": ("cf93bf19bbbca7d1324c2d31af0e81740ae3e5f5",
        "287976baddbcc644638019a08c03763773d7d2f49bb50c0b5cbd7e66ee3e05de", 190, "by-nc-sa/4.0"),
    "principles-finance": ("e3e5135ae36e21282d7afb4cccd5ce09f421d830",
        "9246a8aaa253168dab4b994455027f858764cc3360ab29e886b643bedd962bbc", 132, "by-nc-sa/4.0"),
    "principles-marketing": ("a6d7372579d13e0a733954953c4613867f018a48",
        "68818e2786c12602c6c81c5742821e79f285e0145bd13d42bdb16644c21b11e9", 139, "by-nc-sa/4.0"),
    "psychology": ("43035bcc5dd238f2e236ef532a01f11d6a59fc9d",
        "a0f76b0636aaa83a3417d588a778c4bd8967480a707e406ee3b01d5c897e2f2f", 105, "by-nc-sa/4.0"),
    "university-physics-bundle": ("2300d5b8ca2571dacc6ec25ebef56ba8d49000a2",
        "a3d5dca2a7b6f6284eab8733517b82687cbe84327cfa0a42a9bdd135847b54fd", 322, "by-nc-sa/4.0"),
    "us-history": ("c05b860a035ff59a2ca1fd44339ef86694e09b57",
        "082b95e92ad1dc3f2335534dc2a5120dbb72b9dd1c3d64faa7b2ae33d354a976", 177, "by-nc-sa/4.0"),
    "world-history": ("c0dd82697bc2c7cd355c942e778f215762768c43",
        "2d3f79ed6165aea5a71802373b2f8d7054643b5885bfa9f4ae150d77a59711ec", 162, "by-nc-sa/4.0"),
}
CNX18, MML18 = "{http://cnx.rice.edu/cnxml}", "{http://www.w3.org/1998/Math/MathML}"
SHELF18 = DATA / "openstax"


def manifest18(folder) -> tuple:
    files = sorted(folder.glob("modules/*/index.cnxml"))
    lines = "".join(f"{f.relative_to(folder).as_posix()}\t{sha256(f)}\n" for f in files)
    return hashlib.sha256(lines.encode()).hexdigest(), len(files)


def fetch18(book: str) -> str:
    commit, want, n, _ = BOOKS18[book]
    folder = SHELF18 / book
    if folder.exists() and manifest18(folder)[0] == want:
        return "already here"
    url = f"https://github.com/openstax/osbooks-{book}"
    git = ["git", "-c", "advice.detachedHead=false"]
    if not (folder / ".git").exists():
        folder.parent.mkdir(parents=True, exist_ok=True)
        r = subprocess.run(git + ["clone", "-q", "--filter=blob:none", "--no-checkout", url, str(folder)],
                           capture_output=True, text=True, timeout=600)
        if r.returncode:
            raise RuntimeError(f"git clone {url} failed: {r.stderr.strip()[:300]}. Is the internet on?")
    for args in (["sparse-checkout", "set", "--no-cone", "/modules/*/index.cnxml", "/collections/*"],
                 ["checkout", "-q", commit]):
        r = subprocess.run(git + ["-C", str(folder)] + args, capture_output=True, text=True, timeout=600)
        if r.returncode:
            raise RuntimeError(f"git {args[0]} in {book} failed: {r.stderr.strip()[:300]}")
    got, k = manifest18(folder)
    if got != want:
        raise RuntimeError(f"{book}: fingerprint {got[:16]}... ({k} modules), not {want[:16]}... ({n}): refused")
    return "fetched"


def text18(e) -> str:
    out = []

    def walk(x, top):
        if x.tag.startswith(MML18):
            out.append(" ")
        else:
            out.append(x.text or "")
            for c in x:
                walk(c, False)
        if not top:
            out.append(x.tail or "")
    walk(e, True)
    return re.sub(r"\s+", " ", "".join(out)).strip()


def build18() -> dict:
    t2, how = time.time(), {}
    for book in BOOKS18:
        how[book] = fetch18(book)
    print(f"    books ({time.time() - t2:.0f}s): " + ", ".join(f"{v} {sum(1 for x in how.values() if x == v)}"
                                                         for v in sorted(set(how.values()))))
    texts, book_of, titles = [], [], []
    for b, book in enumerate(BOOKS18):
        for f in sorted((SHELF18 / book).glob("modules/*/index.cnxml")):
            try:
                root = ET.parse(f).getroot()
            except ET.ParseError:
                continue
            title = (root.findtext(f"{CNX18}title") or "").strip()
            skip = {id(x) for ex in root.iter(f"{CNX18}exercise") for x in ex.iter()}
            for p in root.iter(f"{CNX18}para"):
                if id(p) not in skip:
                    t = text18(p)
                    if len(t.split()) >= 6:
                        texts.append(t); book_of.append(b); titles.append(title)
            for d in root.iter(f"{CNX18}definition"):
                term, mean = d.find(f"{CNX18}term"), d.find(f"{CNX18}meaning")
                if term is not None and mean is not None:
                    texts.append(text18(term) + ": " + text18(mean)); book_of.append(b); titles.append(title)
    print(f"    read ({time.time() - t2:.0f}s): {len(texts):,} paragraphs and definitions")
    vocab, rows, cols = {}, [], []
    for i, (title, t) in enumerate(zip(titles, texts)):
        for w in forms(title + " " + t):
            rows.append(i)
            cols.append(vocab.setdefault(w, len(vocab)))
    tf = sp.csr_matrix((np.ones(len(rows), np.int32), (np.array(rows, np.int64), np.array(cols, np.int64))),
                       shape=(len(texts), len(vocab)))
    tf.sum_duplicates()
    print(f"    indexed ({time.time() - t2:.0f}s): {len(rows):,} words, {len(vocab):,} base forms")
    return dict(texts=pack12(texts), titles=pack12(titles), book=np.array(book_of, np.int16),
                vocab=pack12(list(vocab)), data=tf.data, indices=tf.indices, indptr=tf.indptr,
                words=np.array([len(rows)], np.int64))


DEPENDS18 = dict(what="OpenStax textbook paragraphs and their base-form counts, Brain 5 Cell 18A", version=1,
                 books={b: v[1] for b, v in BOOKS18.items()}, min_words=6,
                 forms=hashlib.sha256(" ".join(forms("The cells were dividing rapidly in 1998")).encode()).hexdigest())
BOOK18, BUILT18 = ARTS.get("textbooks", DEPENDS18, build18)
TEXTS18, TITLES18, VOCAB18 = unpack12(BOOK18["texts"]), unpack12(BOOK18["titles"]), unpack12(BOOK18["vocab"])
BOOK_OF18 = BOOK18["book"]
TERM18 = {w: j for j, w in enumerate(VOCAB18)}
TF18 = sp.csr_matrix((BOOK18["data"], BOOK18["indices"], BOOK18["indptr"]), shape=(len(TEXTS18), len(VOCAB18)))
LEN18 = np.asarray(TF18.sum(1)).ravel().astype(np.float64)
DF18 = np.diff(TF18.tocsc().indptr)
IDF18 = np.log(1 + (len(TEXTS18) - DF18 + 0.5) / (DF18 + 0.5))
C18 = TF18.tocoo()
W18 = sp.csc_matrix(((C18.data * 2.2 / (C18.data + 1.2 * (0.25 + 0.75 * LEN18[C18.row] / LEN18.mean())))
                     * IDF18[C18.col], (C18.row, C18.col)), shape=TF18.shape, dtype=np.float32)
HAS18 = (TF18 > 0).astype(np.float32).tocsc()
del C18
print(f"  TEXTBOOKS {'built and saved' if BUILT18 else 'loaded'}: {len(TEXTS18):,} paragraphs and definitions from "
      f"{len(BOOKS18)} books, {int(BOOK18['words'][0]):,} words, {len(VOCAB18):,} base forms")
per = np.bincount(BOOK_OF18, minlength=len(BOOKS18))
print("  per book: " + ", ".join(f"{b} {n:,}" for b, n in zip(BOOKS18, per)))
print("  credit: OpenStax, Rice University (openstax.org); licences as stated in each book")
print("\n  PREDICTIONS vs RESULTS")
LEDGER.result("B5-O1", len(TEXTS18))
LEDGER.result("B5-O2", round(int(BOOK18["words"][0]) / 1e6, 3))
LEDGER.result("B5-T28" if BUILT18 else "B5-T29", round(time.time() - T18))
if "sync" in globals():
    sync("Cell 18A")
print(f"\n  ({time.time() - T18:.0f}s)")
print("  LAST LINE OF MC AI BRAIN 5 CELL 18A")


MC AI BRAIN 5 - CELL 18A: TEXTBOOKS AS KNOWLEDGE
    books (63s): fetched 25
    read (70s): 138,006 paragraphs and definitions
    indexed (79s): 5,304,289 words, 71,054 base forms
  TEXTBOOKS built and saved: 138,006 paragraphs and definitions from 25 books, 5,304,289 words, 71,054 base forms
  per book: american-government 3,195, anatomy-physiology 7,386, astronomy 5,490, biology-bundle 16,718, business-ethics 1,508, calculus-bundle 5,085, chemistry-bundle 6,384, college-algebra-bundle 6,242, college-physics-bundle 7,411, introduction-anthropology 3,713, introduction-business 3,076, introduction-philosophy 2,505, introduction-political-science 2,792, introduction-sociology 4,132, introductory-statistics-bundle 4,934, life-liberty-and-pursuit-happiness 10,921, lifespan-development 7,891, microbiology 3,409, principles-economics-bundle 4,900, principles-finance 3,203, principles-marketing 3,780, psychology 5,057, university-physics-bundle 8,716, us-history 4,155, world-history 5,403
 

In [27]:
# =====================================================================
# MC AI BRAIN 5 - CELL 18B: TEXTBOOKS AS KNOWLEDGE - ANSWERING
# Needs Cells 12A, 12B (SETS12, READ12, Cell 12's scores, SUBJ12) and 18A.
# Nothing is fitted or chosen here: fixed rules, scored on practice sets.
# IR ANSWER  the information-retrieval solver of Clark et al. (2016): for
#   each choice, the best BM25 score of a paragraph for the question's
#   words + the choice's own words (those the question does not have),
#   counting only paragraphs holding at least one word of each. The
#   highest choice wins; ties share the credit; no paragraph: all tied
# COVERAGE  the share of questions whose right choice has such a paragraph
# RULE  IR answers when it picks one choice alone and the question does not
#   ask NOT / EXCEPT; otherwise Cell 12's answer stands
# GROUPS  MMLU validation by Hendrycks et al.'s four groups: STEM,
#   humanities, social sciences, other
# PREDICTIONS (written before running)
#   B5-O3 MMLU validation, IR alone, %: 27-36
#   B5-O4 MMLU validation, rule minus Cell 12, points: -1-4
#   B5-O5 ARC-Challenge validation, IR alone, %: 20-32
#   B5-O6 ARC-Easy validation, IR alone, %: 45-65
#   B5-O7 MMLU validation coverage, %: 50-90
#   B5-O8 MMLU validation STEM, rule minus Cell 12, points: -2-6
#   B5-T30 Cell 18B seconds: <= 900
# =====================================================================
import time

import numpy as np
from mcai.core import at_most, between

print("=" * 64); print("MC AI BRAIN 5 - CELL 18B: TEXTBOOKS AS KNOWLEDGE - ANSWERING"); print("=" * 64)
T18B = time.time()
for need in ("W18", "HAS18", "TERM18", "TEXTS18", "SETS12", "READ12", "scores12", "credit12", "asks_not", "SUBJ12"):
    if need not in globals():
        raise RuntimeError(f"run Cells 12A, 12B and 18A first ({need} is missing)")
for key, claim, rule in (("B5-O3", "MMLU validation, IR alone, %", between(27, 36)),
                         ("B5-O4", "MMLU validation, rule minus Cell 12, points", between(-1, 4)),
                         ("B5-O5", "ARC-Challenge validation, IR alone, %", between(20, 32)),
                         ("B5-O6", "ARC-Easy validation, IR alone, %", between(45, 65)),
                         ("B5-O7", "MMLU validation coverage, %", between(50, 90)),
                         ("B5-O8", "MMLU validation STEM, rule minus Cell 12, points", between(-2, 6)),
                         ("B5-T30", "Cell 18B seconds", at_most(900))):
    LEDGER.predict(key, claim, rule)
GROUP18 = {}
for g, subjects in (
        ("STEM", "abstract_algebra astronomy college_biology college_chemistry college_computer_science "
                 "college_mathematics college_physics computer_security conceptual_physics electrical_engineering "
                 "elementary_mathematics high_school_biology high_school_chemistry high_school_computer_science "
                 "high_school_mathematics high_school_physics high_school_statistics machine_learning"),
        ("humanities", "formal_logic high_school_european_history high_school_us_history high_school_world_history "
                       "international_law jurisprudence logical_fallacies moral_disputes moral_scenarios philosophy "
                       "prehistory professional_law world_religions"),
        ("social sciences", "econometrics high_school_geography high_school_government_and_politics "
                            "high_school_macroeconomics high_school_microeconomics high_school_psychology "
                            "human_sexuality professional_psychology public_relations security_studies sociology "
                            "us_foreign_policy"),
        ("other", "anatomy business_ethics clinical_knowledge college_medicine global_facts human_aging management "
                  "marketing medical_genetics miscellaneous nutrition professional_accounting professional_medicine "
                  "virology")):
    for s in subjects.split():
        GROUP18[s] = g


def ir18(q: str, ch: list) -> tuple:
    """(best paragraph score per choice, best paragraph number per choice or -1)"""
    Q = set(forms(q))
    qi = [TERM18[w] for w in Q if w in TERM18]
    out, at = np.zeros(len(ch)), np.full(len(ch), -1)
    if not qi:
        return out, at
    sq = np.asarray(W18[:, qi].sum(1)).ravel()
    hq = np.asarray(HAS18[:, qi].sum(1)).ravel() > 0
    for j, c in enumerate(ch):
        ci = [TERM18[w] for w in set(forms(str(c))) - Q if w in TERM18]
        if not ci:
            continue
        ok = hq & (np.asarray(HAS18[:, ci].sum(1)).ravel() > 0)
        if ok.any():
            s = np.where(ok, sq + np.asarray(W18[:, ci].sum(1)).ravel(), -1.0)
            at[j] = int(s.argmax())
            out[j] = s[at[j]]
    return out, at


t1 = time.time()
IR18 = {n: [ir18(q, ch) for q, ch, _ in qs] for n, qs in SETS12.items()}
print(f"  IR read ({time.time() - t1:.0f}s): " + ", ".join(f"{n} {len(v):,}" for n, v in IR18.items()))
RES18 = {}
print(f"\n  {'':<26}{'coverage':>9}{'Cell 12':>9}{'IR':>7}{'rule':>7}{'IR picks':>10}{'IR':>7}{'Cell 12':>9}")
print(f"  {'':<26}{'':>9}{'':>9}{'alone':>7}{'':>7}{'one, %':>10}{'on those':>16}")
for n, qs in SETS12.items():
    rows = []
    for (q, ch, g), (x, _, _), (s, _) in zip(qs, READ12[n], IR18[n]):
        c12, cir = credit12(scores12(x), g), credit12(s, g)
        top = s.max()
        alone = top > 0 and (s == top).sum() == 1 and not asks_not(q)
        rows.append((s[g] > 0, c12, cir, cir if alone else c12, alone))
    r = np.array(rows, float)
    a = r[:, 4] > 0
    RES18[n] = dict(cover=100 * r[:, 0].mean(), c12=100 * r[:, 1].mean(), ir=100 * r[:, 2].mean(),
                    rule=100 * r[:, 3].mean(), fired=100 * a.mean(), rows=r,
                    ir_on=100 * r[a, 2].mean() if a.any() else float("nan"),
                    c12_on=100 * r[a, 1].mean() if a.any() else float("nan"))
    v = RES18[n]
    print(f"  {n:<26}{v['cover']:>8.1f}%{v['c12']:>9.1f}{v['ir']:>7.1f}{v['rule']:>7.1f}{v['fired']:>9.1f}%"
          f"{v['ir_on']:>7.1f}{v['c12_on']:>9.1f}")
print("  (ARC train is fit data elsewhere; nothing here is fitted on it. MMLU-Pro validation: 68 of 70 are MMLU dev)")

print("\n  MMLU validation by group (questions, Cell 12, IR alone, rule, rule minus Cell 12)")
R = RES18["MMLU validation"]["rows"]
G18 = {}
for g in ("STEM", "humanities", "social sciences", "other"):
    m = np.array([GROUP18.get(s) == g for s in SUBJ12])
    G18[g] = 100 * (R[m, 3].mean() - R[m, 1].mean())
    print(f"    {g:<18}{int(m.sum()):>6}{100 * R[m, 1].mean():>8.1f}{100 * R[m, 2].mean():>8.1f}"
          f"{100 * R[m, 3].mean():>8.1f}{G18[g]:>+8.1f}")

print("\n  EXAMPLES (MMLU dev: the rule's answer and the paragraph behind it)")
for (q, ch, g), (s, at) in list(zip(SETS12["MMLU dev"], IR18["MMLU dev"]))[3:6]:
    j = int(s.argmax())
    src = TEXTS18[at[j]][:110] if at[j] >= 0 else "(no paragraph)"
    print(f"    {q[:90]}\n      IR: {str(ch[j])[:40]} (gold {str(ch[g])[:40]})\n      from: {src}")

print("\n  PREDICTIONS vs RESULTS")
M18 = RES18["MMLU validation"]
LEDGER.result("B5-O3", round(M18["ir"], 2))
LEDGER.result("B5-O4", round(M18["rule"] - M18["c12"], 2))
LEDGER.result("B5-O5", round(RES18["ARC-Challenge validation"]["ir"], 2))
LEDGER.result("B5-O6", round(RES18["ARC-Easy validation"]["ir"], 2))
LEDGER.result("B5-O7", round(M18["cover"], 2))
LEDGER.result("B5-O8", round(G18["STEM"], 2))
LEDGER.result("B5-T30", round(time.time() - T18B))
if "sync" in globals():
    sync("Cell 18B")
print(f"\n  ({time.time() - T18B:.0f}s)")
print("  LAST LINE OF MC AI BRAIN 5 CELL 18B")


MC AI BRAIN 5 - CELL 18B: TEXTBOOKS AS KNOWLEDGE - ANSWERING
  IR read (18s): ARC train 3,370, ARC-Easy validation 570, ARC-Challenge validation 299, OpenBookQA validation 500, MMLU validation 1,531, MMLU dev 285, MMLU-Pro validation 70

                             coverage  Cell 12     IR   rule  IR picks     IR  Cell 12
                                                alone           one, %        on those
  ARC train                     96.9%     43.0   41.5   42.0     88.2%   43.2     44.3
  ARC-Easy validation           98.6%     51.1   51.0   52.1     90.2%   52.9     51.8
  ARC-Challenge validation      96.0%     30.6   27.8   29.6     78.9%   28.8     30.1
  OpenBookQA validation         92.8%     24.8   28.7   28.1     95.0%   28.6     25.2
  MMLU validation               77.3%     31.2   30.7   30.1     62.5%   32.8     34.7
  MMLU dev                      79.3%     30.1   31.2   30.3     62.5%   34.8     34.5
  MMLU-Pro validation           71.4%     14.7   15.7   16.8     4

In [28]:
# =====================================================================
# MC AI BRAIN 5 - CELL 19: EXACT SOLVERS I - LOGIC AND ARITHMETIC
# Needs Cells 12A and 12B (SETS12, READ12, scores12, credit12, SUBJ12).
# Nothing is learned or fitted. A solver answers only when it can work the
# question out exactly and exactly one choice agrees; otherwise it stays
# silent and Cell 12's answer stands. Designed on the 308 practice
# questions printed by lab/look_cell19.py (MMLU dev and validation of 13
# subjects); the test split is not read.
# LOGIC  formulas of propositional logic (~ not; . or the dot, and; v or,
#   written as the wedge; > if-then, written as the horseshoe; = iff,
#   written as the triple bar; brackets ( [ {), parsed by precedence
#   ~ > and > or > if-then > iff; every row of the truth table checked:
#   PAIR   "pairs of propositions": equivalent (same column), contradictory
#          (opposite columns), else consistent (a row with both true), else
#          inconsistent
#   VALID  premises / conclusion: valid when no row makes every premise
#          true and the conclusion false; a counterexample choice is right
#          when its valuation is such a row
#   SET    "consistent" sets: consistent when a row makes all true; a
#          valuation choice is right when it makes all true
# ARITHMETIC  exact fractions (Python's fractions and ast); unicode signs,
#   x between numbers, "over", |..| for absolute value; implicit times (3y)
#   EQUATION  one letter, linear: solved exactly; the choice equal to it
#   VALUE     the question's one expression; the choice equal to its value
#   EQUIVALENT  choices that are expressions: the one equal to the
#             question's expression
#   TRUE      choices that are comparisons (< > <= >= =): the one true one
#   NAMED     square / cube / square root of N; greatest common divisor /
#             least common multiple of a list
#   SILENT    on questions with LaTeX formulas holding letters ($...x...$)
#             or function notation such as h(4x - 1)
# PREDICTIONS (written before running; the solvers were built on these
# practice questions, so B5-I2 is a CHECK of the building, not a test)
#   B5-I1 MMLU validation questions a solver answers: 15-60
#   B5-I2 of those, right, %: >= 85
#   B5-I3 MMLU validation, solvers in place of Cell 12, minus Cell 12,
#         points: 0.3-3
#   B5-I4 formal_logic validation questions answered: 5-12 (of 14)
#   B5-T31 Cell 19 seconds: <= 300
# =====================================================================
import ast
import itertools
import math
import operator
import re
import time
from fractions import Fraction

import numpy as np
from mcai.core import at_least, at_most, between

print("=" * 64); print("MC AI BRAIN 5 - CELL 19: EXACT SOLVERS I - LOGIC AND ARITHMETIC"); print("=" * 64)
T19 = time.time()
for need in ("SETS12", "READ12", "scores12", "credit12", "SUBJ12"):
    if need not in globals():
        raise RuntimeError(f"run Cells 12A and 12B first ({need} is missing)")
for key, claim, rule in (("B5-I1", "MMLU validation questions a solver answers", between(15, 60)),
                         ("B5-I2", "of those, right, % (check)", at_least(85)),
                         ("B5-I3", "MMLU validation, solvers in place of Cell 12, minus Cell 12, points",
                          between(0.3, 3)),
                         ("B5-I4", "formal_logic validation questions answered", between(5, 12)),
                         ("B5-T31", "Cell 19 seconds", at_most(300))):
    LEDGER.predict(key, claim, rule)

# ---------------- logic ----------------
SIGN19 = {"\u223c": "~", "\u00ac": "~", "\u2283": ">", "\u2192": ">", "\u00b7": "&", "\u2022": "&", "\u2219": "&",
          "\u2227": "&", "\u2228": "|", "\u2261": "=", "\u2194": "="}
OPEN19, CLOSE19 = "([{", ")]}"
LINE19 = re.compile(r"^[A-Z~&|>=()\[\]{}\s/]+$")


def norm19(s: str) -> str:
    return "".join(SIGN19.get(c, c) for c in s)


def parse19(s: str):
    """formula -> nested tuples: ("v", A) ("~", f) (op, f, g)"""
    toks = [c for c in s if not c.isspace()]
    pos = [0]

    def peek():
        return toks[pos[0]] if pos[0] < len(toks) else None

    def atom():
        c = peek()
        if c is None:
            raise ValueError("end")
        pos[0] += 1
        if c == "~":
            return ("~", atom())
        if c in OPEN19:
            f = level(0)
            if peek() not in CLOSE19:
                raise ValueError("bracket")
            pos[0] += 1
            return f
        if c.isalpha() and c.isupper():
            return ("v", c)
        raise ValueError(f"token {c}")

    ops = ("=", ">", "|", "&")

    def level(i):
        if i == len(ops):
            return atom()
        left = level(i + 1)
        while peek() == ops[i]:
            pos[0] += 1
            right = level(i) if ops[i] == ">" else level(i + 1)
            left = (ops[i], left, right)
        return left
    f = level(0)
    if pos[0] != len(toks):
        raise ValueError("left over")
    return f


def value19(f, row: dict) -> bool:
    k = f[0]
    if k == "v":
        return row[f[1]]
    if k == "~":
        return not value19(f[1], row)
    a, b = value19(f[1], row), value19(f[2], row)
    return {"&": a and b, "|": a or b, ">": (not a) or b, "=": a == b}[k]


def names19(f) -> set:
    return {f[1]} if f[0] == "v" else set().union(*(names19(x) for x in f[1:]))


def rows19(fs: list):
    names = sorted(set().union(*(names19(f) for f in fs)))
    for bits in itertools.product((True, False), repeat=len(names)):
        yield dict(zip(names, bits))


def valuation19(choice: str):
    """{letter: True/False} from "... when A and B are true and C is false", else None"""
    m = re.search(r"when (.*)$", choice)
    if not m:
        return None
    out = {}
    for part in re.finditer(r"((?:[A-Z][, ]*(?:and )?)+)\s*(?:is|are) (true|false)", m.group(1)):
        for v in re.findall(r"[A-Z]", part.group(1)):
            out[v] = part.group(2) == "true"
    return out or None


def logic19(q: str, ch: list):
    """(kind, index of the one right choice) or None"""
    lines = [norm19(l).strip() for l in q.replace("\t", " ").split("\n")]
    body = [l for l in lines[1:] if l and LINE19.match(l.replace(" and ", " "))]
    if not body:
        return None
    low = q.lower()
    try:
        if "pairs of propositions" in low and len(body) == 1:
            a, b = (parse19(x) for x in body[0].split(" and "))
            rs = list(rows19([a, b]))
            va, vb = [value19(a, r) for r in rs], [value19(b, r) for r in rs]
            if va == vb:
                want = "logically equivalent"
            elif all(x != y for x, y in zip(va, vb)):
                want = "contradictory"
            elif any(x and y for x, y in zip(va, vb)):
                want = "consistent"
            else:
                want = "inconsistent"
            hit = [j for j, c in enumerate(ch) if c.strip().lower() == want
                   or (want == "consistent" and c.lower().strip().endswith("but consistent"))]
            return ("PAIR", hit[0]) if len(hit) == 1 else None
        if "valid" in low and "/" in body[-1]:
            prem, concl = body[:-1] + [body[-1].split("/")[0]], body[-1].split("/")[1]
            ps, c = [parse19(p) for p in prem if p.strip()], parse19(concl)
            bad = [r for r in rows19(ps + [c]) if all(value19(p, r) for p in ps) and not value19(c, r)]
            ok = []
            for j, x in enumerate(ch):
                if x.strip().lower() == "valid":
                    ok.append(j) if not bad else None
                elif x.lower().startswith("invalid"):
                    v = valuation19(x)
                    if bad and v and any(all(r[k] == b for k, b in v.items() if k in r) for r in bad):
                        ok.append(j)
            return ("VALID", ok[0]) if len(ok) == 1 else None
        if "consistent" in low:
            fs = [parse19(l) for l in body]
            good = [r for r in rows19(fs) if all(value19(f, r) for f in fs)]
            ok = []
            for j, x in enumerate(ch):
                if x.strip().lower() == "inconsistent":
                    ok.append(j) if not good else None
                elif x.lower().startswith("consistent"):
                    v = valuation19(x)
                    if good and v and any(all(r[k] == b for k, b in v.items() if k in r) for r in good):
                        ok.append(j)
            return ("SET", ok[0]) if len(ok) == 1 else None
    except (ValueError, KeyError, IndexError):
        return None
    return None


# ---------------- arithmetic ----------------
DASH19 = {"\u2212": "-", "\u2013": "-", "\u2014": "-", "\u00d7": "*", "\u22c5": "*", "\u00b7": "*", "\u2022": "*",
          "\u00f7": "/", "\u2264": "<=", "\u2265": ">=", "\u2009": " ", "\u00a0": " "}
OPS19 = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv,
         ast.Pow: operator.pow}
CMP19 = {ast.Lt: operator.lt, ast.Gt: operator.gt, ast.LtE: operator.le, ast.GtE: operator.ge, ast.Eq: operator.eq}


def clean19(s: str) -> str:
    s = "".join(DASH19.get(c, c) for c in s)
    s = re.sub(r"(?<=\d),(?=\d{3}\b)", "", s)
    s = re.sub(r"\bover\b", "/", s)
    s = re.sub(r"(?<=\d)\s*[xX]\s*(?=[\d(])", "*", s)
    s = re.sub(r"(?<=\))\s*[xX]\s*(?=[\d(])", "*", s)
    s = re.sub(r"\|([^|]+)\|", r"abs(\1)", s)
    return s.replace("$", "").replace("^", "**")


def calc19(node, env: dict):
    if isinstance(node, ast.Expression):
        return calc19(node.body, env)
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)) and not isinstance(node.value, bool):
        return Fraction(str(node.value))
    if isinstance(node, ast.Name) and node.id in env:
        return env[node.id]
    if isinstance(node, ast.UnaryOp) and isinstance(node.op, (ast.USub, ast.UAdd)):
        v = calc19(node.operand, env)
        return -v if isinstance(node.op, ast.USub) else v
    if isinstance(node, ast.BinOp) and type(node.op) in OPS19:
        a, b = calc19(node.left, env), calc19(node.right, env)
        if isinstance(node.op, ast.Pow) and (b.denominator != 1 or abs(b) > 12 or abs(a) > 10 ** 6):
            raise ValueError("power")
        return OPS19[type(node.op)](a, b)
    if isinstance(node, ast.Call) and isinstance(node.func, ast.Name) and node.func.id == "abs" and len(node.args) == 1:
        return abs(calc19(node.args[0], env))
    if isinstance(node, ast.Compare):
        vals = [calc19(node.left, env)] + [calc19(c, env) for c in node.comparators]
        return all(CMP19[type(o)](a, b) for o, a, b in zip(node.ops, vals, vals[1:]) if type(o) in CMP19)
    raise ValueError("not arithmetic")


def expr19(s: str, env: dict = None):
    """exact value (Fraction, or bool for a comparison) of a math string, or None"""
    s = re.sub(r"(?<=\d)\s*(?=[a-z(])", "*", s.strip().rstrip(".?"))
    s = re.sub(r"(?<=\))\s*(?=[\d(a-z])", "*", s).replace("abs*(", "abs(")
    if not s or not re.search(r"\d", s):
        return None
    try:
        return calc19(ast.parse(s.replace("=", "==") if re.search(r"(?<![<>=])=(?!=)", s) else s, mode="eval"),
                      env or {})
    except (SyntaxError, ValueError, ZeroDivisionError, OverflowError, TypeError, RecursionError):
        return None


MATH19 = re.compile(r"(?:[-+*/()<>=.\d\s]|abs\(|\b[a-z]\b)+")


def spans19(text: str) -> list:
    """maths-looking stretches of a question: digits, operators, single letters"""
    out = []
    for m in MATH19.finditer(clean19(text)):
        s = m.group().strip().strip(".")
        if re.search(r"\d", s) and re.search(r"[-+*/=<>]", s.lstrip("-")) or re.search(r"abs\(", s):
            out.append(s)
    return out


NUM19 = re.compile(r"^\s*[-+]?\d+(?:\.\d+)?(?:\s*/\s*\d+)?\s*%?\s*(?:[a-zA-Z]+\.?)?\s*$")


def number19(c: str):
    s = clean19(str(c)).strip()
    if not NUM19.match(s):
        return None
    m = re.match(r"\s*([-+]?\d+(?:\.\d+)?(?:\s*/\s*\d+)?)", s)
    try:
        return Fraction(m.group(1).replace(" ", ""))
    except (ValueError, ZeroDivisionError):
        return None


def one19(vals: list, want) -> int:
    hit = [j for j, v in enumerate(vals) if v is not None and not isinstance(v, bool) and v == want]
    return hit[0] if len(hit) == 1 else None


def arith19(q: str, ch: list):
    if re.search(r"\$[^$]*[a-zA-Z][^$]*\$", q) or re.search(r"\b[a-z]\(\s*[-\d a-z]", q):
        return None
    nums = [number19(c) for c in ch]
    low = clean19(q.lower())
    m = re.search(r"(square root|square|cube) of (-?\d+(?:\.\d+)?)", low)
    if m and all(v is not None for v in nums):
        n = Fraction(m.group(2))
        want = n ** 2 if m.group(1) == "square" else n ** 3 if m.group(1) == "cube" else None
        if m.group(1) == "square root":
            r = math.isqrt(int(n)) if n.denominator == 1 and n >= 0 else -1
            want = Fraction(r) if r >= 0 and r * r == n else None
        j = one19(nums, want) if want is not None else None
        if j is not None:
            return ("NAMED", j)
    m = re.search(r"(greatest common (?:divisor|factor)|least common multiple) of ([\d,\sand]+)", low)
    if m and all(v is not None for v in nums):
        xs = [int(x) for x in re.findall(r"\d+", m.group(2))]
        if len(xs) >= 2:
            g = math.gcd(*xs) if m.group(1).startswith("greatest") else math.lcm(*xs)
            j = one19(nums, Fraction(g))
            if j is not None:
                return ("NAMED", j)
    cmp = [expr19(clean19(str(c))) if re.search(r"[<>]", clean19(str(c))) else None for c in ch]
    if all(isinstance(v, bool) for v in cmp):
        hit = [j for j, v in enumerate(cmp) if v]
        return ("TRUE", hit[0]) if len(hit) == 1 else None
    spans = spans19(q)
    eqs = [s for s in spans if "=" in s and s.count("=") == 1 and "<" not in s and ">" not in s]
    letters = {v for s in eqs for v in re.findall(r"\b[a-z]\b", s)}
    if len(eqs) == 1 and len(letters) == 1 and all(v is not None for v in nums):
        v = letters.pop()
        lhs, rhs = eqs[0].split("=")
        try:
            f = [expr19(f"({lhs})-({rhs})", {v: Fraction(k)}) for k in (0, 1, 2)]
        except Exception:
            f = [None]
        if all(isinstance(x, Fraction) for x in f) and f[2] - f[1] == f[1] - f[0] and f[1] != f[0]:
            j = one19(nums, -f[0] / (f[1] - f[0]))
            if j is not None:
                return ("EQUATION", j)
    vals = [s for s in spans if "=" not in s and not re.search(r"\b[a-z]\b", s.replace("abs(", "("))]
    known = [x for x in (expr19(s) for s in vals) if isinstance(x, Fraction)]
    if len(known) == 1:
        if all(v is not None for v in nums):
            j = one19(nums, known[0])
            if j is not None:
                return ("VALUE", j)
        cv = [expr19(clean19(str(c))) if re.search(r"\d\s*[-+*/]", clean19(str(c))) else None for c in ch]
        if all(isinstance(x, Fraction) for x in cv):
            j = one19(cv, known[0])
            if j is not None:
                return ("EQUIVALENT", j)
    return None


def solve19(q: str, ch: list):
    return logic19(q, ch) or arith19(q, ch)


# ---------------- scored on the practice sets ----------------
RES19, KINDS19 = {}, {}
print(f"\n  {'':<26}{'questions':>10}{'answered':>10}{'right':>7}{'Cell 12':>9}{'with solvers':>14}{'Cell 12':>9}")
print(f"  {'':<26}{'':>10}{'':>10}{'':>7}{'same qs':>9}{'whole set':>14}{'':>9}")
for n, qs in SETS12.items():
    rows = []
    for i, ((q, ch, g), (x, _, _)) in enumerate(zip(qs, READ12[n])):
        c12 = credit12(scores12(x), g)
        got = solve19(q, list(ch))
        if got:
            KINDS19.setdefault((n, got[0]), []).append(got[1] == g)
        rows.append((got is not None, float(got[1] == g) if got else c12, c12,
                     SUBJ12[i] if n == "MMLU validation" else ""))
    a = np.array([r[0] for r in rows])
    mine, c12 = np.array([r[1] for r in rows]), np.array([r[2] for r in rows])
    RES19[n] = dict(n=len(rows), answered=int(a.sum()), right=100 * mine[a].mean() if a.any() else float("nan"),
                    c12_on=100 * c12[a].mean() if a.any() else float("nan"), whole=100 * mine.mean(),
                    c12=100 * c12.mean(), subj=[r[3] for r in rows], a=a)
    v = RES19[n]
    print(f"  {n:<26}{v['n']:>10,}{v['answered']:>10}{v['right']:>7.1f}{v['c12_on']:>9.1f}{v['whole']:>14.2f}"
          f"{v['c12']:>9.2f}")
print("\n  by solver (practice sets together: answered, right)")
for kind in ("PAIR", "VALID", "SET", "EQUATION", "VALUE", "EQUIVALENT", "TRUE", "NAMED"):
    hits = [x for (n, k), xs in KINDS19.items() if k == kind and n != "ARC train" for x in xs]
    if hits:
        print(f"    {kind:<12}{len(hits):>5}{sum(hits):>6}")
V19 = RES19["MMLU validation"]
FL19 = sum(1 for s, a in zip(V19["subj"], V19["a"]) if a and s == "formal_logic")
print(f"  formal_logic validation answered: {FL19} of {sum(1 for s in V19['subj'] if s == 'formal_logic')}")

print("\n  PREDICTIONS vs RESULTS")
LEDGER.result("B5-I1", V19["answered"])
LEDGER.result("B5-I2", round(V19["right"], 2) if V19["answered"] else 0)
LEDGER.result("B5-I3", round(V19["whole"] - V19["c12"], 2))
LEDGER.result("B5-I4", FL19)
LEDGER.result("B5-T31", round(time.time() - T19))
if "sync" in globals():
    sync("Cell 19")
print(f"\n  ({time.time() - T19:.0f}s)")
print("  LAST LINE OF MC AI BRAIN 5 CELL 19")


MC AI BRAIN 5 - CELL 19: EXACT SOLVERS I - LOGIC AND ARITHMETIC

                             questions  answered  right  Cell 12  with solvers  Cell 12
                                                         same qs     whole set         
  ARC train                      3,370         1  100.0     25.0         43.05    43.02
  ARC-Easy validation              570         0    nan      nan         51.05    51.05
  ARC-Challenge validation         299         0    nan      nan         30.57    30.57
  OpenBookQA validation            500         0    nan      nan         24.82    24.82
  MMLU validation                1,531        21  100.0     22.2         32.29    31.23
  MMLU dev                         285         3  100.0     25.0         30.85    30.06
  MMLU-Pro validation               70         0    nan      nan         14.67    14.67

  by solver (practice sets together: answered, right)
    PAIR            3     3
    VALID           3     3
    SET             4     4
    

In [29]:
# =====================================================================
# MC AI BRAIN 5 - CELL 20: THE NOISE FLOOR OF EVERY READING
# Needs Cells 12A, 12B; uses Cells 18B (IR18) and 19 (solve19) when run.
# Two ways of measuring each reading against noise of a known kind:
# 1 EXACT NOISE FLOOR  if a reading's values were shuffled among a
#   question's choices, the right choice would get a uniformly random
#   value: its expected credit is exactly 1 / choices and its variance is
#   known. Summed over a set, this gives the noise floor's mean and spread
#   exactly, with no simulation (a permutation test in closed form). A
#   reading beats noise when z = (accuracy - floor) / spread >= 1.96.
#   Each reading answers alone with its Cell 12B weight (JOINT, LEVEL,
#   DEFINED, REVERSE, CELL 6), plus IR (Cell 18B's best paragraph score)
# 2 KNOCKOFFS  (Barber and Candes 2015) for each of Cell 12B's five
#   readings, a fake copy: the same values shuffled among each ARC train
#   question's choices, so it has the same spread and no tie to the answer.
#   Naive Bayes weights are counted for the real and the fake readings
#   together; a reading's statistic is the spread of its weights (max - min)
#   minus its fake's. 50 fakes; a reading is KEPT when its statistic is
#   above 0 in at least 95% of them
# ANSWER  Cell 12B's rule with only the kept readings, against Cell 12
#   (all five), on the practice sets; Cell 19's solvers on top where they
#   answer
# PREDICTIONS (written before running)
#   B5-E1 of the six readings, beating noise on ARC train: 4-6
#   B5-E2 of the six, beating noise on MMLU validation: 1-4
#   B5-E3 of Cell 12B's five readings, kept by the knockoffs: 2-5
#   B5-E4 MMLU validation, kept readings minus all five, points: -1.5-1.5
#   B5-T32 Cell 20 seconds: <= 600
# =====================================================================
import time

import numpy as np
from mcai.core import at_most, between

print("=" * 64); print("MC AI BRAIN 5 - CELL 20: THE NOISE FLOOR OF EVERY READING"); print("=" * 64)
T20 = time.time()
for need in ("SETS12", "READ12", "W12", "F12", "NV12", "credit12", "scores12"):
    if need not in globals():
        raise RuntimeError(f"run Cells 12A and 12B first ({need} is missing)")
for key, claim, rule in (("B5-E1", "of the six readings, beating noise on ARC train", between(4, 6)),
                         ("B5-E2", "of the six readings, beating noise on MMLU validation", between(1, 4)),
                         ("B5-E3", "of Cell 12B's five readings, kept by the knockoffs", between(2, 5)),
                         ("B5-E4", "MMLU validation, kept readings minus all five, points", between(-1.5, 1.5)),
                         ("B5-T32", "Cell 20 seconds", at_most(600))):
    LEDGER.predict(key, claim, rule)
NAMES20 = [f for f, _ in F12] + (["IR"] if "IR18" in globals() else [])
SETS20 = [n for n in SETS12 if n != "MMLU-Pro validation"]


def floor20(scores: np.ndarray, gold: int) -> tuple:
    """(credit, expected credit under shuffling, its variance) for one question"""
    k = len(scores)
    c = np.array([credit12(scores, j) for j in range(k)])
    return c[gold], c.mean(), c.var()


def reading20(n: str, i: int, f: str) -> np.ndarray:
    x = READ12[n][i][0]
    if f == "IR":
        return np.asarray(IR18[n][i][0], float)
    j = [name for name, _ in F12].index(f)
    return np.asarray(W12[j][x[:, j]], float)


# ---------------- 1 exact noise floor ----------------
Z20 = {}
print("\n  1 EXACT NOISE FLOOR: accuracy alone / noise floor / z  (z >= 1.96 beats noise)")
print(f"  {'':<26}" + "".join(f"{f:>17}" for f in NAMES20))
for n in SETS20:
    cells = []
    for f in NAMES20:
        r = np.array([floor20(reading20(n, i, f), g) for i, (_, _, g) in enumerate(SETS12[n])])
        acc, flo, sd = r[:, 0].mean(), r[:, 1].mean(), np.sqrt(r[:, 2].sum()) / len(r)
        Z20[(n, f)] = (acc - flo) / sd if sd > 0 else 0.0
        cells.append(f"{100 * acc:5.1f}/{100 * flo:4.1f}/{Z20[(n, f)]:+5.1f}")
    print(f"  {n:<26}" + "".join(f"{c:>17}" for c in cells))
BEAT_TR20 = sum(Z20[("ARC train", f)] >= 1.96 for f in NAMES20)
BEAT_MM20 = sum(Z20[("MMLU validation", f)] >= 1.96 for f in NAMES20)
print(f"  beating noise: ARC train {BEAT_TR20} of {len(NAMES20)}, MMLU validation {BEAT_MM20} of {len(NAMES20)}")

# ---------------- 2 knockoffs ----------------
RNG20 = np.random.default_rng(20)
TR20 = [(x, g) for (x, _, _), (_, _, g) in zip(READ12["ARC train"], SETS12["ARC train"])]
NF20 = len(F12)


def weights20(rows: list, cats: list) -> list:
    count = [np.zeros((2, c)) for c in cats]
    for x, g in rows:
        for j in range(len(x)):
            for f in range(len(cats)):
                count[f][int(j == g), x[j, f]] += 1
    out = []
    for c in count:
        lp = np.log((c + 1) / (c.sum(1, keepdims=True) + c.shape[1]))
        out.append(lp[1] - lp[0])
    return out


STAT20 = np.zeros((50, NF20))
for rep in range(50):
    rows = []
    for x, g in TR20:
        fake = np.stack([x[RNG20.permutation(len(x)), f] for f in range(NF20)], 1)
        rows.append((np.concatenate([x, fake], 1), g))
    w = weights20(rows, list(NV12) * 2)
    STAT20[rep] = [np.ptp(w[f]) - np.ptp(w[NF20 + f]) for f in range(NF20)]
KEPT20 = [f for f in range(NF20) if (STAT20[:, f] > 0).mean() >= 0.95]
print("\n  2 KNOCKOFFS on ARC train: weight spread, real minus fake (mean over 50 fakes; share above 0)")
for f, (name, _) in enumerate(F12):
    print(f"    {name:<10}{STAT20[:, f].mean():>+8.3f}{100 * (STAT20[:, f] > 0).mean():>8.0f}%"
          f"  {'KEPT' if f in KEPT20 else 'dropped'}")

# ---------------- answer with the kept readings ----------------
print("\n  ANSWER: Cell 12 (all five) / kept readings only / kept + Cell 19's solvers")
RES20 = {}
for n in SETS20:
    rows = []
    for (q, ch, g), (x, _, _) in zip(SETS12[n], READ12[n]):
        kept = sum(W12[f][x[:, f]] for f in KEPT20) if KEPT20 else np.zeros(len(ch))
        c12, ck = credit12(scores12(x), g), credit12(np.asarray(kept, float), g)
        got = solve19(q, list(ch)) if "solve19" in globals() else None
        rows.append((c12, ck, float(got[1] == g) if got else ck))
    r = 100 * np.array(rows).mean(0)
    RES20[n] = r
    print(f"    {n:<26}{r[0]:>7.1f}{r[1]:>7.1f}{r[2]:>7.1f}")

print("\n  PREDICTIONS vs RESULTS")
LEDGER.result("B5-E1", BEAT_TR20)
LEDGER.result("B5-E2", BEAT_MM20)
LEDGER.result("B5-E3", len(KEPT20))
LEDGER.result("B5-E4", round(RES20["MMLU validation"][1] - RES20["MMLU validation"][0], 2))
LEDGER.result("B5-T32", round(time.time() - T20))
if "sync" in globals():
    sync("Cell 20")
print(f"\n  ({time.time() - T20:.0f}s)")
print("  LAST LINE OF MC AI BRAIN 5 CELL 20")


MC AI BRAIN 5 - CELL 20: THE NOISE FLOOR OF EVERY READING

  1 EXACT NOISE FLOOR: accuracy alone / noise floor / z  (z >= 1.96 beats noise)
                                        JOINT            LEVEL          DEFINED          REVERSE           CELL 6               IR
  ARC train                   34.5/25.0/+13.6  31.6/25.0/+15.4  29.5/25.0/ +7.4  31.4/25.0/+10.4  45.2/25.0/+27.8  41.5/25.0/+23.0
  ARC-Easy validation         43.5/25.0/+10.9  37.9/25.0/+11.6  32.9/25.0/ +5.3  37.5/25.0/ +8.3  50.3/25.0/+14.2  51.0/25.0/+14.8
  ARC-Challenge validation    21.2/25.1/ -1.6  28.2/25.1/ +2.3  22.8/25.1/ -1.2  21.3/25.1/ -1.7  30.1/25.1/ +2.1  27.8/25.1/ +1.2
  OpenBookQA validation       22.6/25.0/ -1.4  23.8/25.0/ -1.0  25.4/25.0/ +0.2  24.2/25.0/ -0.5  25.6/25.0/ +0.3  28.7/25.0/ +2.0
  MMLU validation             29.8/25.0/ +5.1  29.3/25.0/ +8.4  28.3/25.0/ +4.2  29.0/25.0/ +4.6  29.5/25.0/ +4.7  30.7/25.0/ +6.1
  MMLU dev                    26.3/25.0/ +0.6  27.4/25.0/ +1.9  26.5/25.0/

In [30]:
# =====================================================================
# MC AI BRAIN 5 - CELL 21: REFINING - JOINT COUNTS AND KNOWING HOW SURE
# Needs Cells 12A, 12B (READ12, W12) and 19 (solve19).
# 1 JOINT COUNTS  Cell 12B's naive Bayes adds the five readings as if they
#   were separate evidence; Cell 20 showed they overlap. Here a choice's
#   five readings are counted together as one key on ARC train (fit): of the
#   choices with this key, how many were right. Rare keys lean on naive
#   Bayes: p = (right + K * p_nb) / (seen + K), K = 10, where p_nb =
#   sigmoid(logit(base rate) + Cell 12B's score). The choice with the
#   highest p answers
# 2 HOW SURE  a question's confidence = the top choice's p over the sum of
#   its choices' p. Cell 19's exact answers get confidence 1. Questions
#   are sorted by confidence: accuracy of the surest 10%, 25%, 50% and of
#   all (selective prediction; risk-coverage). A calibration table compares
#   stated confidence with measured accuracy on MMLU validation
# PREDICTIONS (written before running)
#   B5-C1 MMLU validation, joint counts minus Cell 12, points: -1-1.5
#   B5-C2 ARC-Easy validation, joint counts minus Cell 12, points: -1-2
#   B5-C3 MMLU validation, the surest 10% (exact answers first), right, %:
#         40-70
#   B5-C4 MMLU validation, the surest 25%, right, %: 33-50
#   B5-T33 Cell 21 seconds: <= 300
# =====================================================================
import time
from collections import Counter

import numpy as np
from mcai.core import at_most, between

print("=" * 64); print("MC AI BRAIN 5 - CELL 21: REFINING - JOINT COUNTS AND KNOWING HOW SURE"); print("=" * 64)
T21 = time.time()
for need in ("SETS12", "READ12", "W12", "F12", "credit12", "scores12", "solve19"):
    if need not in globals():
        raise RuntimeError(f"run Cells 12A, 12B and 19 first ({need} is missing)")
for key, claim, rule in (("B5-C1", "MMLU validation, joint counts minus Cell 12, points", between(-1, 1.5)),
                         ("B5-C2", "ARC-Easy validation, joint counts minus Cell 12, points", between(-1, 2)),
                         ("B5-C3", "MMLU validation, the surest 10%, right, %", between(40, 70)),
                         ("B5-C4", "MMLU validation, the surest 25%, right, %", between(33, 50)),
                         ("B5-T33", "Cell 21 seconds", at_most(300))):
    LEDGER.predict(key, claim, rule)
K21 = 10.0
SEEN21, RIGHT21 = Counter(), Counter()
for (x, _, _), (_, _, g) in zip(READ12["ARC train"], SETS12["ARC train"]):
    for j in range(len(x)):
        k = tuple(int(v) for v in x[j])
        SEEN21[k] += 1
        RIGHT21[k] += int(j == g)
BASE21 = sum(RIGHT21.values()) / sum(SEEN21.values())
print(f"  JOINT KEYS on ARC train: {len(SEEN21):,} of {int(np.prod([len(v) for _, v in F12])):,} possible; "
      f"base rate {BASE21:.3f}; keys seen 10+ times: {sum(n >= 10 for n in SEEN21.values())}")


def joint21(x: np.ndarray) -> np.ndarray:
    nb = np.log(BASE21 / (1 - BASE21)) + scores12(x)
    p_nb = 1 / (1 + np.exp(-nb))
    out = np.empty(len(x))
    for j in range(len(x)):
        k = tuple(int(v) for v in x[j])
        out[j] = (RIGHT21[k] + K21 * p_nb[j]) / (SEEN21[k] + K21)
    return out


# ---------------- 1 joint counts against Cell 12 ----------------
RES21, ROWS21 = {}, {}
print(f"\n  {'':<26}{'Cell 12':>9}{'joint':>8}{'joint + exact':>15}")
for n, qs in SETS12.items():
    rows = []
    for (q, ch, g), (x, _, _) in zip(qs, READ12[n]):
        p = joint21(x)
        got = solve19(q, list(ch))
        c12, cj = credit12(scores12(x), g), credit12(p, g)
        conf = 1.0 if got else float(p.max() / p.sum())
        rows.append((c12, cj, float(got[1] == g) if got else cj, conf))
    r = np.array(rows)
    ROWS21[n] = r
    RES21[n] = 100 * r[:, :3].mean(0)
    print(f"  {n:<26}{RES21[n][0]:>9.1f}{RES21[n][1]:>8.1f}{RES21[n][2]:>15.1f}")

# ---------------- 2 how sure ----------------
print("\n  HOW SURE: accuracy of the surest share of questions (joint + exact)")
print(f"  {'':<26}{'10%':>7}{'25%':>7}{'50%':>7}{'100%':>7}{'chance':>8}")
SURE21 = {}
for n in SETS12:
    if n == "ARC train":
        continue
    r = ROWS21[n]
    order = np.argsort(-r[:, 3], kind="stable")
    cells = []
    for share in (0.10, 0.25, 0.50, 1.0):
        top = order[:max(1, int(round(share * len(r))))]
        cells.append(100 * r[top, 2].mean())
    SURE21[n] = cells
    chance = 100 * np.mean([1 / len(ch) for _, ch, _ in SETS12[n]])
    print(f"  {n:<26}" + "".join(f"{c:>7.1f}" for c in cells) + f"{chance:>8.1f}")

print("\n  CALIBRATION, MMLU validation: stated confidence -> measured accuracy (questions)")
r = ROWS21["MMLU validation"]
for lo, hi in ((0.0, 0.3), (0.3, 0.35), (0.35, 0.45), (0.45, 0.999), (0.999, 1.01)):
    m = (r[:, 3] >= lo) & (r[:, 3] < hi)
    if m.any():
        print(f"    {lo:.2f}-{min(hi, 1):.2f}: stated {100 * r[m, 3].mean():5.1f}%,",
              f"measured {100 * r[m, 2].mean():5.1f}% ({int(m.sum())})")

print("\n  PREDICTIONS vs RESULTS")
LEDGER.result("B5-C1", round(RES21["MMLU validation"][1] - RES21["MMLU validation"][0], 2))
LEDGER.result("B5-C2", round(RES21["ARC-Easy validation"][1] - RES21["ARC-Easy validation"][0], 2))
LEDGER.result("B5-C3", round(SURE21["MMLU validation"][0], 2))
LEDGER.result("B5-C4", round(SURE21["MMLU validation"][1], 2))
LEDGER.result("B5-T33", round(time.time() - T21))
if "sync" in globals():
    sync("Cell 21")
print(f"\n  ({time.time() - T21:.0f}s)")
print("  LAST LINE OF MC AI BRAIN 5 CELL 21")


MC AI BRAIN 5 - CELL 21: REFINING - JOINT COUNTS AND KNOWING HOW SURE
  JOINT KEYS on ARC train: 345 of 960 possible; base rate 0.250; keys seen 10+ times: 143

                              Cell 12   joint  joint + exact
  ARC train                      43.0    46.0           46.0
  ARC-Easy validation            51.1    51.9           51.9
  ARC-Challenge validation       30.6    29.2           29.2
  OpenBookQA validation          24.8    25.3           25.3
  MMLU validation                31.2    30.7           31.7
  MMLU dev                       30.1    30.4           31.2
  MMLU-Pro validation            14.7    13.2           13.2

  HOW SURE: accuracy of the surest share of questions (joint + exact)
                                10%    25%    50%   100%  chance
  ARC-Easy validation          78.9   73.2   59.3   51.9    25.0
  ARC-Challenge validation     30.0   30.7   26.7   29.2    25.1
  OpenBookQA validation        26.0   23.2   22.8   25.3    25.0
  MMLU validation   

In [31]:
# =====================================================================
# MC AI BRAIN 5 - CELL 22: BRAIN 2'S WORD-PROBLEM READER, LOADED AS IT WAS
# Needs Cells 0 (fetch, DATA), 12A, 12B and 19 (solve19, number19).
# DATA  (Cell 0's fetch; its BASE is set again first: Cell 6 reused the name)
#   GSM8K train and test (openai/grade-school-math), ASDiv
#   (chaochun/nlu-asdiv-dataset) and SVAMP (arkilpatel/SVAMP), each pinned
#   to a commit and checked by SHA-256 (the files Brain 2 used, same bytes)
# CODE  Brain 2's math cells 22-30 (MATH I-IX), unchanged, from this repo's
#   brain2/ folder at a pinned commit, each checked by SHA-256: from the
#   local checkout in the lab, from GitHub's API with the Kaggle secret
#   GITHUB_TOKEN on Kaggle. They run in their own namespace (B2), so no
#   Brain 5 name can be overwritten; four helpers they expect from Brain 2's
#   Cell 0 are given (check, sha_file, mf_say, NCHK); their search of
#   /kaggle/input for ASDiv and SVAMP is pointed at this cell's folder.
#   Their printing is kept aside; the lines with their test results are shown
# ANSWER  mc_math(story, question) on every practice question whose choices
#   are all numbers and that Cell 19 does not answer: the story is every
#   sentence but the last question sentence. A choice equal to mc_math's
#   exact answer, alone, answers; else Cell 12's answer stands. Why mc_math
#   stayed silent is counted, as the list of what to extend next
# PREDICTIONS (written before running; B5-B1 and B5-B2 are CHECKS: a local
#   run gave 20 checks, and Brain 2 printed 50 of 53)
#   B5-B1 Brain 2's own checks passed (it stops at the first that fails): 20
#   B5-B2 ASDiv test, grades 1-2, right of answered: 50 of 53 (50)
#   B5-B3 MMLU validation questions mc_math answers: 0-5
#   B5-B4 practice answers by mc_math that are wrong: <= 1
#   B5-T34 Cell 22 seconds: <= 900
# =====================================================================
import contextlib
import hashlib
import io
import json
import os
import re
import textwrap
import time
import urllib.request
from collections import Counter
from fractions import Fraction
from pathlib import Path

import numpy as np
from mcai.core import at_least, at_most, between

print("=" * 64); print("MC AI BRAIN 5 - CELL 22: BRAIN 2'S WORD-PROBLEM READER, LOADED AS IT WAS"); print("=" * 64)
T22 = time.time()
for need in ("fetch", "DATA", "GITHUB", "SETS12", "READ12", "scores12", "credit12", "solve19", "number19"):
    if need not in globals():
        raise RuntimeError(f"run Cells 0, 12A, 12B and 19 first ({need} is missing)")
for key, claim, rule in (("B5-B1", "Brain 2's own checks passed (check)", between(20, 20)),
                         ("B5-B2", "ASDiv test, grades 1-2, right of answered (check)", between(50, 50)),
                         ("B5-B3", "MMLU validation questions mc_math answers", between(0, 5)),
                         ("B5-B4", "practice answers by mc_math that are wrong", at_most(1)),
                         ("B5-T34", "Cell 22 seconds", at_most(900))):
    LEDGER.predict(key, claim, rule)
WORDS22 = {  # source: (repo, commit, {file: SHA-256})
    "gsm8k": ("openai/grade-school-math", "3101c7d5072418e28b9008a6636bde82a006892c", {
        "grade_school_math/data/train.jsonl": "17f347dc51477c50d4efb83959dbb7c56297aba886e5544ee2aaed3024813465",
        "grade_school_math/data/test.jsonl": "3730d312f6e3440559ace48831e51066acaca737f6eabec99bccb9e4b3c39d14"}),
    "asdiv": ("chaochun/nlu-asdiv-dataset", "883f90a9a65bf00304ba8f37423910fe743abc47", {
        "dataset/ASDiv.xml": "ef8904068482919ac48c8eeaaf6df344b8a308ba66d048c2d4d87eab82dc4929"}),
    "svamp": ("arkilpatel/SVAMP", "78e727689e1c1bebfc4be39c446898e8e10b0518", {
        "SVAMP.json": "5be77703a6d891ae476d7c082787ad361392aa02453b132516cdd5f4e7934e3e"})}
BASE = os.environ.get("MC_AI_FETCH_BASE", "")  # Cell 0's fetch base: Cell 6 reuses the name BASE for its setting
GOT22 = {}
for source, (repo, commit, files) in WORDS22.items():
    for rel, want in files.items():
        path, how, _ = fetch(source, repo, commit, rel, want, None, [GITHUB])
        if how.startswith("REFUSED"):
            raise RuntimeError(f"{rel}: {how}")
        GOT22[rel.rsplit("/", 1)[-1]] = path
print("  data: " + ", ".join(f"{k} ({p.stat().st_size // 1024:,} KB)" for k, p in GOT22.items()))

CODE22_COMMIT = "17df14a710133d73d8e5fec0a616edb9964a18bf"
CODE22 = {22: "196938008098e8164381aabad680bc2d4f970cf25dbd9465f365e56947d6fdf9",
          23: "be8f726aabc4aa938e2268df93acf63eae30612c93e086d33b219228d25b9b41",
          24: "f219249413aa6945c47d35108ceda7ed2c7c4d1a7f91cc64c6aec9e8bf9c91dd",
          25: "fc655ccfaeca3f659f908d4ee7571eb5d607311a8f3faea4cf4c672f92c365fe",
          26: "b1588a778c1c709d6750c4798703e1d01ee997c40112baa0e89b33a5ad5625f4",
          27: "3007a78ceeef0dc8c59277c26698340e136ba1180795e79469cce64c8955810a",
          28: "0cd453c75ec57ebe73f2f0834ca5492143e0f55acc2e7ee45e97f07b317ed097",
          29: "e01874d0825b78954a2fb9da6608b34dfff80d7d2952e74d52cfabd3797ca1fc",
          30: "21e07d1dcd823b7fc12e0405f6391e3e5535161b914c6c8288dd6483a8b82294"}


def token22() -> str:
    if "_token" in globals():
        return _token()
    from kaggle_secrets import UserSecretsClient
    return UserSecretsClient().get_secret("GITHUB_TOKEN")


def source22(cell: int) -> str:
    """Brain 2's math cell, checked by SHA-256: the local checkout first, else GitHub's API"""
    name = f"brain2/b2_math_cell{cell}.py"
    local = Path(name)
    data = local.read_bytes() if local.exists() else None
    if data is None or hashlib.sha256(data).hexdigest() != CODE22[cell]:
        req = urllib.request.Request(
            f"https://api.github.com/repos/Coffey333/mc-ai-brain5/contents/{name}?ref={CODE22_COMMIT}",
            headers={"Accept": "application/vnd.github.raw", "User-Agent": "MC-AI-Brain5",
                     "Authorization": f"Bearer {token22()}"})
        with urllib.request.urlopen(req, timeout=60) as r:
            data = r.read()
    got = hashlib.sha256(data).hexdigest()
    if got != CODE22[cell]:
        raise RuntimeError(f"{name}: SHA-256 {got[:16]}..., not {CODE22[cell][:16]}...: refused")
    return data.decode("utf-8")


def b2_check(ok, msg):
    if not ok:
        raise AssertionError(msg)
    B2["NCHK"][0] += 1
    print(f"  [PASS] {msg}")


B2 = dict(__name__="brain2_math", NCHK=[0], check=b2_check,
          sha_file=lambda p: hashlib.sha256(Path(p).read_bytes()).hexdigest(),
          mf_say=lambda s, ind="    ": print(textwrap.fill(s, 64, initial_indent=ind, subsequent_indent=ind + "  ")),
          GSM={"train": GOT22["train.jsonl"], "test": GOT22["test.jsonl"]})
B2["check"] = b2_check
SHELF22 = repr(str(GOT22["ASDiv.xml"].parent.parent))
OUT22 = io.StringIO()
t1 = time.time()
with contextlib.redirect_stdout(OUT22):
    for cell in sorted(CODE22):
        exec(compile(source22(cell).replace('"/kaggle/input"', SHELF22), f"brain2_math_cell{cell}", "exec"), B2)
LOG22 = OUT22.getvalue().splitlines()
print(f"  BRAIN 2 MATH loaded and run ({time.time() - t1:.0f}s): {B2['NCHK'][0]} of its own checks passed")
for line in LOG22:
    if re.match(r"\s+(DEV|TEST)\s+grades", line) and "problems;" in line:
        print("   " + line.strip())
ASDIV22 = next((int(m.group(1)) for line in LOG22 for m in [re.search(r"TEST\s+grades 1-2:.*right\s+(\d+) of", line)]
                if m), -1)


def value22(x):
    if x is None:
        return None
    if hasattr(x, "n") and hasattr(x, "d"):
        return Fraction(x.n, x.d)
    try:
        return Fraction(str(x))
    except (ValueError, ZeroDivisionError):
        return None


def mc22(q: str, ch: list):
    """(index of the one choice equal to mc_math's answer or None, why silent)"""
    if solve19(q, ch) or any(number19(c) is None for c in ch):
        return None, None
    parts = re.split(r"(?<=[.!?])\s+(?=[A-Z])", q.strip())
    asks = [i for i, p in enumerate(parts) if p.rstrip().endswith("?")]
    k = asks[-1] if asks else len(parts) - 1
    body, ask = " ".join(parts[:k] + parts[k + 1:]), parts[k]
    try:
        out = B2["mc_math"](body, ask)
    except Exception as err:
        return None, f"error {type(err).__name__}"
    x = value22(out.get("answer"))
    if x is None:
        return None, str(out.get("why") or "no answer")[:60]
    hit = [j for j, c in enumerate(ch) if number19(c) == x]
    return (hit[0], None) if len(hit) == 1 else (None, f"answer {x} matches {len(hit)} choices")


RES22, WHY22, WRONG22 = {}, Counter(), 0
print(f"\n  {'':<26}{'numbers only':>13}{'answered':>10}{'right':>7}{'whole set':>11}{'was':>7}")
for n, qs in SETS12.items():
    rows, numeric = [], 0
    for (q, ch, g), (x, _, _) in zip(qs, READ12[n]):
        c12 = credit12(scores12(x), g)
        got19 = solve19(q, list(ch))
        base = float(got19[1] == g) if got19 else c12
        j, why = mc22(q, list(ch))
        numeric += why is not None or j is not None
        if why and n == "MMLU validation":
            WHY22[why] += 1
        if j is not None and n != "ARC train":
            WRONG22 += j != g
        rows.append((j is not None, float(j == g) if j is not None else base, base))
    r = np.array(rows, float)
    RES22[n] = dict(numeric=numeric, answered=int(r[:, 0].sum()), right=int((r[:, 0] * r[:, 1]).sum()),
                    whole=100 * r[:, 1].mean(), was=100 * r[:, 2].mean())
    v = RES22[n]
    print(f"  {n:<26}{v['numeric']:>13}{v['answered']:>10}{v['right']:>7}{v['whole']:>11.2f}{v['was']:>7.2f}")
print("  (was = Cell 12 with Cell 19's exact answers)")
print("\n  WHY mc_math stayed silent, MMLU validation (what to extend next):")
for why, c in WHY22.most_common(10):
    print(f"    {c:>4}  {why}")

print("\n  PREDICTIONS vs RESULTS")
LEDGER.result("B5-B1", B2["NCHK"][0])
LEDGER.result("B5-B2", ASDIV22)
LEDGER.result("B5-B3", RES22["MMLU validation"]["answered"])
LEDGER.result("B5-B4", WRONG22)
LEDGER.result("B5-T34", round(time.time() - T22))
if "sync" in globals():
    sync("Cell 22")
print(f"\n  ({time.time() - T22:.0f}s)")
print("  LAST LINE OF MC AI BRAIN 5 CELL 22")


MC AI BRAIN 5 - CELL 22: BRAIN 2'S WORD-PROBLEM READER, LOADED AS IT WAS
  data: train.jsonl (4,068 KB), test.jsonl (732 KB), ASDiv.xml (925 KB), SVAMP.json (332 KB)
  BRAIN 2 MATH loaded and run (23s): 20 of its own checks passed
   DEV  grades 1-2:  432 problems; answers 60.2%; right 260 of 260 = 100.0%; right of all problems 60.2%; internal errors 0
   DEV  grades 3-6: 1,412 problems; answers  8.5%; right 120 of 120 = 100.0%; right of all problems  8.5%; internal errors 0
   TEST grades 1-2:  103 problems; answers 51.5%; right  50 of  53 = 94.3%; right of all problems 48.5%; internal errors 0
   TEST grades 3-6:  358 problems; answers 11.2%; right  36 of  40 = 90.0%; right of all problems 10.1%; internal errors 0

                             numbers only  answered  right  whole set    was
  ARC train                            35         2      1      43.06  43.05
  ARC-Easy validation                   4         0      0      51.05  51.05
  ARC-Challenge validation              4 

In [32]:
# =====================================================================
# MC AI BRAIN 5 - CELL 23: TOOLBOX I - EXACT TOOLS FOR NAMED PROBLEM KINDS
# Needs Cells 12A, 12B, 19 (clean19, number19, expr19, solve19) and 22.
# Each tool knows one kind of problem, reads it with a fixed pattern,
# computes the answer exactly (fractions; square roots only when exact) and
# says which numbers of the question it used. Built on the 136 practice
# questions printed by lab/look_cell23.py (MMLU dev and validation); the
# test split is not read. Nothing is learned.
# TOOLS
#   GROUP    cyclic subgroup of Z_n made by k (n / gcd); factor groups Z_n/<k>
#            and (Z_a x Z_b)/(<c> x <d>) (gcds); largest element order in
#            Z_a x Z_b (lcm) and in S_n (Landau's function); characteristic
#            of nZ (0) and of Z_n (n)
#   CHANCE   Bayes' rule for a test (prevalence, true and false positive
#            rates); Hardy-Weinberg (2pq, p^2 from q^2 or from 1 in N);
#            share of variation explained, r^2
#   SHARE    P% of a number is X -> Q% of it; a whole from its other part
#            (either ... or); a part's percent of a two-part total; a / b of n
#   COUNT    choices of one from each of several lists (product); divisors
#            of N between a and b; money multiplier 1 / reserve ratio;
#            grating resolving power lambda / (difference)
#   CIRCUIT  V = I R; C = Q / V; P = V I; parallel wires F / L = 2e-7 I^2 / d;
#            telescope magnification (L - f) / f
#   CODE     a one-line Python lambda applied to numbers (exact arithmetic)
# CHECK  (the round trip) a tool answers only when every number in the
#   question is one it used, and exactly one choice equals its answer; a
#   choice written to k decimals equals the answer rounded to k decimals;
#   percent answers are matched only to percent choices, and other answers
#   only to choices without a percent sign (when the choices have both)
# PREDICTIONS (written before running; the tools were built on these
#   practice questions, so B5-TB2..K4 are CHECKS of the building)
#   B5-TB1 MMLU validation questions a tool answers: 10-30
#   B5-TB2 practice answers that disagree with the answer key: <= 1
#   B5-TB3 MMLU validation, Cells 19 + 23 in place of Cell 12, minus Cell 12,
#         points: 1.5-4
#   B5-TB4 MMLU dev questions a tool answers: 5-20
#   B5-T35 Cell 23 seconds: <= 300
# =====================================================================
import math
import re
import time
from collections import Counter
from fractions import Fraction

import numpy as np
from mcai.core import at_most, between

print("=" * 64); print("MC AI BRAIN 5 - CELL 23: TOOLBOX I - EXACT TOOLS FOR NAMED PROBLEM KINDS"); print("=" * 64)
T23 = time.time()
for need in ("SETS12", "READ12", "scores12", "credit12", "SUBJ12", "clean19", "number19", "expr19", "solve19"):
    if need not in globals():
        raise RuntimeError(f"run Cells 12A, 12B and 19 first ({need} is missing)")
for key, claim, rule in (("B5-TB1", "MMLU validation questions a tool answers", between(10, 30)),
                         ("B5-TB2", "practice answers that disagree with the answer key (check)", at_most(1)),
                         ("B5-TB3", "MMLU validation, Cells 19 + 23 in place of Cell 12, minus Cell 12, points",
                          between(1.5, 4)),
                         ("B5-TB4", "MMLU dev questions a tool answers (check)", between(5, 20)),
                         ("B5-T35", "Cell 23 seconds", at_most(300))):
    LEDGER.predict(key, claim, rule)
F23 = r"(\d+(?:\.\d+)?)"
NUMS23 = re.compile(r"(?<![A-Za-z_\d.])\d+(?:,\d{3})*(?:\.\d+)?")


def numbers23(q: str) -> list:
    return [Fraction(x.replace(",", "")) for x in NUMS23.findall(clean19(q))]


def F(x) -> Fraction:
    return Fraction(str(x).replace(",", ""))


def sqrt23(x: Fraction):
    """exact square root of a fraction, or None"""
    if x < 0:
        return None
    a, b = math.isqrt(x.numerator), math.isqrt(x.denominator)
    return Fraction(a, b) if a * a == x.numerator and b * b == x.denominator else None


def landau23(n: int) -> int:
    """largest lcm of a partition of n (the largest order of an element of S_n)"""
    best = {0: 1}
    for part in range(1, n + 1):
        new = dict(best)
        for s, l in best.items():
            k = s + part
            while k <= n:
                v = math.lcm(l, part)
                if v > new.get(k, 0):
                    new[k] = v
                k += part
        best = new
    return max(best.values())


def group23(q: str):
    s = clean19(q)
    m = re.search(r"cyclic subgroup of Z_(\d+) generated by (\d+) has order", s)
    if m:
        n, k = int(m.group(1)), int(m.group(2))
        return Fraction(n // math.gcd(n, k)), [n, k]
    m = re.search(r"order of the factor group \(Z_(\d+) x Z_(\d+)\)\s*/\s*\(<(\d+)> x <(\d+)>\)", s)
    if m:
        a, b, c, d = (int(x) for x in m.groups())
        return Fraction(math.gcd(a, c) * math.gcd(b, d)), [a, b, c, d]
    m = re.search(r"order of the factor group Z_(\d+)\s*/\s*<\s*(\d+)\s*>", s)
    if m:
        n, k = int(m.group(1)), int(m.group(2))
        return Fraction(math.gcd(n, k)), [n, k]
    m = re.search(r"maximum possible order for (?:some|an) element of Z_(\d+) (?:x|\*) Z_(\d+)", s)
    if m:
        a, b = int(m.group(1)), int(m.group(2))
        return Fraction(math.lcm(a, b)), [a, b]
    m = re.search(r"maximum possible order for (?:some|an) element of S_n for n = (\d+)", s)
    if m:
        return Fraction(landau23(int(m.group(1)))), [int(m.group(1))]
    m = re.search(r"characteristic of the ring (\d+)Z\b", s)
    if m and int(m.group(1)) > 0:
        return Fraction(0), [int(m.group(1))]
    m = re.search(r"characteristic of the ring Z_(\d+)\b", s)
    if m:
        return Fraction(int(m.group(1))), [int(m.group(1))]
    return None


def chance23(q: str):
    s = re.sub(r"\s*percent", "%", clean19(q))
    m = re.search(F23 + r"% of the population have .{0,60}?positive reading for " + F23 + r"% of people who have"
                  r".{0,40}?" + F23 + r"% of people who do not", s)
    if m and re.search(r"tests? positive, what is the probability", s):
        p, a, b = (F(x) / 100 for x in m.groups())
        return p * a / (p * a + (1 - p) * b), [x * 100 for x in (p, a, b)]
    m = re.search(F23 + r"% of the gametes .{0,60}?contain the \w+ allele", s)
    if m and re.search(r"carry both|heterozygous", s):
        p = F(m.group(1)) / 100
        return 2 * p * (1 - p), [p * 100]
    m = re.search(r"recessive phenotype present in " + F23 + r"% of", s)
    if m:
        q2 = F(m.group(1)) / 100
        r = sqrt23(q2)
        rest = [F(x) for x in re.findall(F23 + r"%", s)]
        if r is not None and all(x in (q2 * 100, 100 - q2 * 100) for x in rest):
            if "homozygous dominant" in s:
                return (1 - r) ** 2 * 100, rest
            if "heterozygous" in s:
                return 2 * r * (1 - r) * 100, rest
    m = re.search(r"\b1 out of every ([\d,]+) people has .{0,60}?recessive allele", s)
    if m and re.search(r"carry the \w+ allele but (?:are )?not", s):
        r = sqrt23(1 / F(m.group(1)))
        if r is not None:
            return 2 * r * (1 - r), [Fraction(1), F(m.group(1))]
    m = re.search(r"correlation coefficient r = (-?\d*\.\d+)", s)
    if m and "variation" in s and "%" in s:
        return F(m.group(1)) ** 2 * 100, [abs(F(m.group(1)))]
    return None


def share23(q: str):
    s = re.sub(r"\s*percent", "%", clean19(q))
    m = re.search(r"[Ii]f " + F23 + r"% of a number is " + F23 + r", what is " + F23 + r"% of the number", s)
    if m:
        p, x, r = (F(v) for v in m.groups())
        return x * r / p, [p, x, r]
    m = re.search(r"\beither\b.{0,120}?\bor\b.{0,120}?\. " + F23 +
                  r"% of the (\w+) (\w+).{0,60}?\. ([\d,]+) \2 (\w+)", s)
    if m and re.search(r"How many " + m.group(2) + r" are (?:in|there)", s):
        p, n = F(m.group(1)) / 100, F(m.group(4))
        return n / (1 - p), [p * 100, n]
    m = re.search(r"in favor of .{0,40}?was ([\d,]+)\. The number of votes against .{0,40}?was ([\d,]+)", s)
    if m and "percent of the voters were in favor" in clean19(q):
        a, b = F(m.group(1)), F(m.group(2))
        return a / (a + b) * 100, [a, b]
    m = (re.search(r"\\frac\{(\d+)\}\{(\d+)\}\$? of the \$?(\d+)\$?", q)
         or re.search(r"\b(\d+)/(\d+) of the (\d+)\b", s))
    if m and re.search(r"How many", s):
        a, b, n = (F(v) for v in m.groups())
        return a / b * n, [a, b, n]
    return None


def count23(q: str):
    s = clean19(q)
    if re.search(r"one each from", s) and re.search(r"How many different", s):
        m = re.search(r"There are ([^.]*?)(?:listed|available|in all)?\.", s[s.index("one each from"):])
        if m:
            ns = [F(x) for x in re.findall(r"\b(\d+) [a-z]", m.group(1))]
            if len(ns) >= 2:
                return math.prod(ns), ns
    m = re.search(r"How many integers (\d+) ?- ?(\d+) are divisors of the (?:[\w-]+ )?number ([\d,]+)", s)
    if m:
        a, b, n = int(m.group(1)), int(m.group(2)), int(m.group(3).replace(",", ""))
        return Fraction(sum(1 for d in range(a, b + 1) if d and n % d == 0)), [F(a), F(b), F(n)]
    m = re.search(r"reserve requirement is " + F23 + r" percent", s)
    if m and "money multiplier" in s:
        return 100 / F(m.group(1)), [F(m.group(1))]
    m = re.search(r"resolve two wavelengths of " + F23 + r" nm and " + F23 + r" nm", s)
    if m and "resolving power" in s:
        a, b = F(m.group(1)), F(m.group(2))
        return a / abs(b - a), [a, b]
    return None


def circuit23(q: str):
    s = clean19(q)
    R = re.search(r"resistance of " + F23 + r" ohms", s)
    I = re.search(r"current of " + F23 + r" amp(?:ere)?s", s) or re.search(r"draws a current of " + F23 + r" amps?", s)
    if R and I and re.search(r"potential difference|voltage", s):
        return F(R.group(1)) * F(I.group(1)), [F(R.group(1)), F(I.group(1))]
    m = re.search(r"charge of " + F23 + r" C at " + F23 + r" V\b", s)
    if m and "capacitance" in s:
        return F(m.group(1)) / F(m.group(2)), [F(m.group(1)), F(m.group(2))]
    V = re.search(r"\b" + F23 + r" V\b", s)
    if V and I and re.search(r"rate is energy being used|power", s):
        return F(V.group(1)) * F(I.group(1)), [F(V.group(1)), F(I.group(1))]
    m = re.search(r"parallel conductors carry " + F23 + r" A\b.{0,40}?separated by " + F23 + r" (mm|cm|m)\b", s)
    if m and "force per meter" in s:
        d = F(m.group(2)) * {"mm": Fraction(1, 1000), "cm": Fraction(1, 100), "m": 1}[m.group(3)]
        return Fraction(2, 10 ** 7) * F(m.group(1)) ** 2 / d, [F(m.group(1)), F(m.group(2))]
    m = re.search(r"two converging lenses separated by " + F23 + r" cm\. The eye-piece lens has a focal length of " +
                  F23 + r" cm", s)
    if m and "angular magnification" in s:
        L, f = F(m.group(1)), F(m.group(2))
        return (L - f) / f, [L, f]
    return None


def code23(q: str):
    m = re.search(r"let (\w+) = lambda (\w+): ([^.]+?)\. What is \1\(([-\d.]+)\)\?", q)
    if m:
        v = expr19(m.group(3), {m.group(2): F(m.group(4))})
        if isinstance(v, Fraction):
            version = [F(x) for x in re.findall(r"\bPython (\d)\b", q)]
            return v, [F(x) for x in NUMS23.findall(m.group(3))] + [F(m.group(4))] + version
    return None


TOOLS23 = (("GROUP", group23), ("CHANCE", chance23), ("SHARE", share23), ("COUNT", count23), ("CIRCUIT", circuit23),
           ("CODE", code23))


def places23(c: str):
    s = clean19(str(c)).strip()
    m = re.match(r"[-+]?\d+(?:,\d{3})*(?:\.(\d+))?", s.lstrip("$"))
    return len(m.group(1) or "") if m else None


def tool23(q: str, ch: list):
    """(tool name, index of the one agreeing choice) or None"""
    vals = [number19(c) for c in ch]
    if any(v is None for v in vals):
        return None
    for name, tool in TOOLS23:
        try:
            got = tool(q)
        except (ValueError, ZeroDivisionError, AttributeError, OverflowError):
            got = None
        if not got:
            continue
        x, used = got
        if Counter(numbers23(q)) - Counter(Fraction(u) for u in used):
            continue
        pct = name in ("SHARE", "CHANCE") and bool(re.search(r"%|percent", q))
        ok = [j for j, c in enumerate(ch) if bool(re.search(r"%|percent", str(c))) == pct] or list(range(len(ch)))
        hit = [j for j in ok if vals[j] == x]
        if not hit:
            hit = [j for j in ok if places23(ch[j]) is not None and Fraction(round(x, places23(ch[j]))) == vals[j]]
        if len(hit) == 1:
            return name, hit[0]
    return None


# ---------------- scored on the practice sets ----------------
RES23, WRONG23, KIND23 = {}, [], {}
print(f"\n  {'':<26}{'answered':>10}{'agree':>7}{'with tools':>12}{'was':>8}")
for n, qs in SETS12.items():
    rows, ans = [], 0
    for (q, ch, g), (x, _, _) in zip(qs, READ12[n]):
        c12 = credit12(scores12(x), g)
        s19 = solve19(q, list(ch))
        base = float(s19[1] == g) if s19 else c12
        got = None if s19 else tool23(q, list(ch))
        if got:
            ans += 1
            KIND23.setdefault(got[0], [0, 0])
            KIND23[got[0]][0] += 1
            KIND23[got[0]][1] += got[1] == g
            if got[1] != g and n != "ARC train":
                WRONG23.append((n, q[:110], ch[got[1]], ch[g]))
        rows.append(float(got[1] == g) if got else base)
    RES23[n] = dict(answered=ans, whole=100 * np.mean(rows),
                    was=100 * np.mean([credit12(scores12(x), g) for (_, _, g), (x, _, _) in zip(qs, READ12[n])]))
    agree = sum(1 for (q, ch, g) in qs if (t := (None if solve19(q, list(ch)) else tool23(q, list(ch)))) and t[1] == g)
    print(f"  {n:<26}{ans:>10}{agree:>7}{RES23[n]['whole']:>12.2f}{RES23[n]['was']:>8.2f}")
print("  (with tools = Cells 19 and 23 where they answer, else Cell 12; was = Cell 12 alone)")
print("\n  by tool (all sets: answered, agree with the key)")
for name, _ in TOOLS23:
    if name in KIND23:
        print(f"    {name:<10}{KIND23[name][0]:>5}{KIND23[name][1]:>6}")
print("\n  DISAGREEMENTS with the answer key (practice sets)")
for n, q, said, key in WRONG23:
    print(f"    [{n}] {q}\n      tool: {said}; key: {key}")

print("\n  PREDICTIONS vs RESULTS")
LEDGER.result("B5-TB1", RES23["MMLU validation"]["answered"])
LEDGER.result("B5-TB2", len(WRONG23))
LEDGER.result("B5-TB3", round(RES23["MMLU validation"]["whole"] - RES23["MMLU validation"]["was"], 2))
LEDGER.result("B5-TB4", RES23["MMLU dev"]["answered"])
LEDGER.result("B5-T35", round(time.time() - T23))
if "sync" in globals():
    sync("Cell 23")
print(f"\n  ({time.time() - T23:.0f}s)")
print("  LAST LINE OF MC AI BRAIN 5 CELL 23")


MC AI BRAIN 5 - CELL 23: TOOLBOX I - EXACT TOOLS FOR NAMED PROBLEM KINDS

                              answered  agree  with tools     was
  ARC train                          0      0       43.05   43.02
  ARC-Easy validation                0      0       51.05   51.05
  ARC-Challenge validation           0      0       30.57   30.57
  OpenBookQA validation              0      0       24.82   24.82
  MMLU validation                   19     18       33.16   31.23
  MMLU dev                           6      6       32.43   30.06
  MMLU-Pro validation                3      3       18.53   14.67
  (with tools = Cells 19 and 23 where they answer, else Cell 12; was = Cell 12 alone)

  by tool (all sets: answered, agree with the key)
    GROUP         6     6
    CHANCE        5     5
    SHARE         3     3
    COUNT         4     4
    CIRCUIT       9     8
    CODE          1     1

  DISAGREEMENTS with the answer key (practice sets)
    [MMLU validation] An external circuit has a res

In [33]:
# =====================================================================
# MC AI BRAIN 5 - CELL 24: TOOLBOX II - ALGEBRA, RELATIONS, RECURSION
# Needs Cells 12A, 12B, 19 (clean19, number19, expr19, solve19) and 23
# (tool23, numbers23, places23). Same rules as Toolbox I: a fixed pattern,
# an exact computation, every number of the question used, exactly one
# choice agreeing (rounded to the choice's decimals). Built on the practice
# questions printed by lab/look_cell23.py; the test split is not read.
# TOOLS
#   EXTREMUM  least or greatest value of a quadratic in one letter: three
#             exact evaluations give a, b, c (a fourth checks the degree);
#             the value c - b^2 / 4a when a has the right sign
#   PRIMES    the greatest even prime (2), the least odd prime (3), their
#             sum or product
#   MULTIPLE  "less than N" and "a multiple of k": the one choice that is
#   RELATION  "A scored n more X than B", "C scored n X, which was k times
#             as many X as A": exact linear equations, solved by Gauss-Jordan
#   SETS      three-set inclusion-exclusion: total, three singles, three
#             pairs, none -> all three
#   RECURSE   "f(X) = if x = 1 then v else [...]": the rule run exactly
#             (depth <= 50)
#   RECIPROCAL  divide A by the reciprocal of B: A * B
#   LOGFIT    ln(y) = a + bx at x0: y = e^(a + b x0), matched at the
#             choice's decimals (the one floating-point tool)
#   MOTION    from rest with a = (k m/s^3) t: distance k t^3 / 6
# PREDICTIONS (written before running; built on these practice questions,
#   so B5-TC2 and B5-TC4 are CHECKS of the building)
#   B5-TC1 MMLU validation questions Toolbox II answers: 3-15
#   B5-TC2 practice answers that disagree with the answer key: 0
#   B5-TC3 MMLU validation, Cells 19 + 23 + 24 in place of Cell 12, minus
#          Cell 12, points: 2-5
#   B5-TC4 MMLU dev questions Toolbox II answers: 0-6
#   B5-T36 Cell 24 seconds: <= 300
# =====================================================================
import math
import re
import time
from collections import Counter
from fractions import Fraction

import numpy as np
from mcai.core import at_most, between

print("=" * 64); print("MC AI BRAIN 5 - CELL 24: TOOLBOX II - ALGEBRA, RELATIONS, RECURSION"); print("=" * 64)
T24 = time.time()
for need in ("SETS12", "READ12", "scores12", "credit12", "clean19", "number19", "expr19", "solve19", "tool23",
             "numbers23", "places23"):
    if need not in globals():
        raise RuntimeError(f"run Cells 12A, 12B, 19 and 23 first ({need} is missing)")
for key, claim, rule in (("B5-TC1", "MMLU validation questions Toolbox II answers", between(3, 15)),
                         ("B5-TC2", "practice answers that disagree with the answer key (check)", between(0, 0)),
                         ("B5-TC3", "MMLU validation, Cells 19 + 23 + 24 in place of Cell 12, minus Cell 12, points",
                          between(2, 5)),
                         ("B5-TC4", "MMLU dev questions Toolbox II answers (check)", between(0, 6)),
                         ("B5-T36", "Cell 24 seconds", at_most(300))):
    LEDGER.predict(key, claim, rule)


def tex24(q: str) -> str:
    """LaTeX math as plain maths: $ dropped, \\div, \\cdot, \\times as signs"""
    s = q.replace("$", "").replace("\\div", "/").replace("\\cdot", "*").replace("\\times", "*")
    return clean19(s)


def extremum24(q: str):
    s = tex24(q)
    m = re.search(r"What is the (minimum|maximum) value of (.+?)(?:,| over| for|\?)", s)
    if not m:
        return None
    letters = set(re.findall(r"\b([a-z])\b", m.group(2))) | set(re.findall(r"(?<=\d)([a-z])\b", m.group(2)))
    if len(letters) != 1:
        return None
    v = letters.pop()
    f = [expr19(m.group(2), {v: Fraction(k)}) for k in (0, 1, 2, 3)]
    if not all(isinstance(x, Fraction) for x in f):
        return None
    c = f[0]
    a = (f[2] - 2 * f[1] + f[0]) / 2
    b = f[1] - a - c
    if a == 0 or a * 9 + b * 3 + c != f[3] or (a > 0) != (m.group(1) == "minimum"):
        return None
    return c - b * b / (4 * a), numbers23(m.group(2))


def primes24(q: str):
    s = clean19(q).lower()
    m = re.search(r"(product|sum) of the greatest even prime(?: number)? and the (?:least|smallest) odd prime", s)
    if m:
        return (Fraction(6) if m.group(1) == "product" else Fraction(5)), []
    return None


def multiple24(q: str, ch: list):
    s = clean19(q)
    m = re.search(r"less than (\d+),? and is also a multiple of (\d+)", s)
    if not m or not re.search(r"How many \w+ could", s):
        return None
    n, k = int(m.group(1)), int(m.group(2))
    vals = [number19(c) for c in ch]
    hit = [j for j, v in enumerate(vals) if v is not None and v.denominator == 1 and 0 < v < n and v % k == 0]
    return (hit[0], [n, k]) if len(hit) == 1 else None


NAME24 = r"([A-Z][a-z]+)"


def relation24(q: str):
    s = clean19(q)
    eqs, used, unit = [], [], None
    for sent in re.split(r"(?<=[.?])\s+", s.strip()):
        m = re.fullmatch(NAME24 + r" scored (\d+) more (\w+)(?: [^.]*?)? than " + NAME24 + r"\.", sent)
        if m:
            a, n, u, b = m.groups()
            eqs.append(({a: 1, b: -1}, -Fraction(n)))
            used.append(Fraction(n))
            unit = unit or u
            continue
        m = re.fullmatch(NAME24 + r" scored (\d+) (\w+), which was (twice|three times|four times|half) as many "
                         r"\w+ as " + NAME24 + r"\.", sent)
        if m:
            c, n, u, k, a = m.groups()
            k = {"twice": 2, "three times": 3, "four times": 4, "half": Fraction(1, 2)}[k]
            eqs += [({c: 1}, -Fraction(n)), ({c: 1, a: -k}, Fraction(0))]
            used.append(Fraction(n))
            continue
        m = re.fullmatch(r"How many (\w+) did " + NAME24 + r" score\?", sent)
        if m and eqs:
            names = sorted({k for d, _ in eqs for k in d})
            sol = solve24(eqs, names)
            if sol and m.group(2) in sol:
                return sol[m.group(2)], used
            return None
        return None
    return None


def solve24(eqs: list, names: list):
    """exact Gauss-Jordan for sum(coef * name) + const = 0; None unless every name has one value"""
    n = len(names)
    rows = [[Fraction(d.get(x, 0)) for x in names] + [-Fraction(c)] for d, c in eqs]
    r = 0
    where = {}
    for col in range(n):
        p = next((i for i in range(r, len(rows)) if rows[i][col] != 0), None)
        if p is None:
            return None
        rows[r], rows[p] = rows[p], rows[r]
        rows[r] = [x / rows[r][col] for x in rows[r]]
        for i in range(len(rows)):
            if i != r and rows[i][col] != 0:
                f = rows[i][col]
                rows[i] = [x - f * y for x, y in zip(rows[i], rows[r])]
        where[names[col]] = r
        r += 1
    if any(row[n] != 0 for row in rows[r:]):
        return None
    return {x: rows[i][n] for x, i in where.items()}


def sets24(q: str):
    s = clean19(q).replace("\n", " ")
    m = re.search(r"survey of (\d+)", s)
    single = re.findall(r"(\d+) are enrolled in (?!both)([a-z ]+?)(?:,|\.| and)", s)
    pair = re.findall(r"(\d+) are enrolled in both", s)
    none = re.search(r"(\d+) are not enrolled in any", s)
    if m and len(single) == 3 and len(pair) == 3 and none and "all three" in s:
        t, n0 = Fraction(m.group(1)), Fraction(none.group(1))
        a = [Fraction(x) for x, _ in single]
        b = [Fraction(x) for x in pair]
        return t - n0 - sum(a) + sum(b), [t, n0] + a + b
    return None


def recurse24(q: str):
    m = re.search(r"Let f\(X\) = if x = (\d+) then (\d+) else \[(.+?)\]\. The value of f\((\d+)\) is", q)
    if not m:
        return None
    base, v0, rule, at = int(m.group(1)), Fraction(m.group(2)), m.group(3), int(m.group(4))
    if not re.fullmatch(r"[\dx+\-*/() f]+", rule):
        return None
    memo = {}

    def f(x):
        if x == base:
            return v0
        if abs(x - base) > 50:
            raise ValueError("too deep")
        if x not in memo:
            inner = re.sub(r"f\(x - (\d+)\)", lambda k: f"({f(x - int(k.group(1)))})", rule)
            memo[x] = expr19(inner, {"x": Fraction(x)})
            if not isinstance(memo[x], Fraction):
                raise ValueError("not exact")
        return memo[x]
    return f(at), [Fraction(base), v0, Fraction(at)] + [Fraction(k) for k in re.findall(r"\d+", rule)]


def reciprocal24(q: str):
    s = tex24(q)
    m = re.search(r"Divide (\d+(?:\.\d+)?) by the reciprocal of ([\d\s./*+-]+?)\.?$", s.strip())
    if m:
        b = expr19(m.group(2))
        if isinstance(b, Fraction) and b != 0:
            return Fraction(m.group(1)) * b, numbers23(s)
    return None


def logfit24(q: str):
    s = clean19(q)
    m = re.search(r"ln\(y\) = (-?\d+(?:\.\d+)?) ([+-]) (\d+(?:\.\d+)?)x\. What is the predicted value of y ?when x ?= ?"
                  r"(-?\d+(?:\.\d+)?)\?", s)
    if m:
        a, b, x = float(m.group(1)), float(m.group(3)) * (1 if m.group(2) == "+" else -1), float(m.group(4))
        return math.exp(a + b * x), [Fraction(m.group(1)), Fraction(m.group(3)), Fraction(m.group(4))]
    return None


def motion24(q: str):
    s = clean19(q)
    m = re.search(r"initially at rest experiences a time-varying acceleration given by a = "
                  r"\((\d+(?:\.\d+)?) m/s(?:\^|\*\*)3\)t.*?How far does the object travel in the first (\d+(?:\.\d+)?) seconds", s)
    if m:
        k, t = Fraction(m.group(1)), Fraction(m.group(2))
        return k * t ** 3 / 6, [k, t, Fraction(3), Fraction(0)]
    return None


TOOLS24 = (("EXTREMUM", extremum24), ("PRIMES", primes24), ("RELATION", relation24), ("SETS", sets24),
           ("RECURSE", recurse24), ("RECIPROCAL", reciprocal24), ("LOGFIT", logfit24), ("MOTION", motion24))


def tool24(q: str, ch: list):
    """(tool name, index of the one agreeing choice) or None"""
    got = multiple24(q, ch)
    if got and not (Counter(numbers23(q)) - Counter(Fraction(u) for u in got[1])):
        return "MULTIPLE", got[0]
    vals = [number19(c) for c in ch]
    if any(v is None for v in vals):
        return None
    for name, tool in TOOLS24:
        try:
            got = tool(q)
        except (ValueError, ZeroDivisionError, AttributeError, OverflowError, RecursionError):
            got = None
        if not got:
            continue
        x, used = got
        if Counter(numbers23(q)) - Counter(Fraction(u) for u in used):
            continue
        hit = [j for j, v in enumerate(vals) if isinstance(x, Fraction) and v == x]
        if not hit:
            hit = [j for j, (v, c) in enumerate(zip(vals, ch)) if places23(c) is not None
                   and Fraction(str(round(float(x) if not isinstance(x, Fraction) else x, places23(c)))) == v]
        if len(hit) == 1:
            return name, hit[0]
    return None


# ---------------- scored on the practice sets ----------------
RES24, WRONG24, KIND24 = {}, [], {}
print(f"\n  {'':<26}{'answered':>10}{'agree':>7}{'with tools':>12}{'was':>8}")
for n, qs in SETS12.items():
    rows, ans, agree = [], 0, 0
    for (q, ch, g), (x, _, _) in zip(qs, READ12[n]):
        c12 = credit12(scores12(x), g)
        earlier = solve19(q, list(ch)) or tool23(q, list(ch))
        base = float(earlier[1] == g) if earlier else c12
        got = None if earlier else tool24(q, list(ch))
        if got:
            ans += 1
            agree += got[1] == g
            KIND24.setdefault(got[0], [0, 0])
            KIND24[got[0]][0] += 1
            KIND24[got[0]][1] += got[1] == g
            if got[1] != g and n != "ARC train":
                WRONG24.append((n, q[:110], ch[got[1]], ch[g]))
        rows.append(float(got[1] == g) if got else base)
    RES24[n] = dict(answered=ans, whole=100 * np.mean(rows),
                    was=100 * np.mean([credit12(scores12(x), g) for (_, _, g), (x, _, _) in zip(qs, READ12[n])]))
    print(f"  {n:<26}{ans:>10}{agree:>7}{RES24[n]['whole']:>12.2f}{RES24[n]['was']:>8.2f}")
print("  (with tools = Cells 19, 23 and 24 where they answer, else Cell 12; was = Cell 12 alone)")
print("\n  by tool (all sets: answered, agree with the key)")
for name in ["MULTIPLE"] + [t for t, _ in TOOLS24]:
    if name in KIND24:
        print(f"    {name:<11}{KIND24[name][0]:>5}{KIND24[name][1]:>6}")
print("\n  DISAGREEMENTS with the answer key (practice sets)")
for n, q, said, key in WRONG24:
    print(f"    [{n}] {q}\n      tool: {said}; key: {key}")

print("\n  PREDICTIONS vs RESULTS")
LEDGER.result("B5-TC1", RES24["MMLU validation"]["answered"])
LEDGER.result("B5-TC2", len(WRONG24))
LEDGER.result("B5-TC3", round(RES24["MMLU validation"]["whole"] - RES24["MMLU validation"]["was"], 2))
LEDGER.result("B5-TC4", RES24["MMLU dev"]["answered"])
LEDGER.result("B5-T36", round(time.time() - T24))
if "sync" in globals():
    sync("Cell 24")
print(f"\n  ({time.time() - T24:.0f}s)")
print("  LAST LINE OF MC AI BRAIN 5 CELL 24")


MC AI BRAIN 5 - CELL 24: TOOLBOX II - ALGEBRA, RELATIONS, RECURSION

                              answered  agree  with tools     was
  ARC train                          0      0       43.05   43.02
  ARC-Easy validation                0      0       51.05   51.05
  ARC-Challenge validation           0      0       30.57   30.57
  OpenBookQA validation              0      0       24.82   24.82
  MMLU validation                   10     10       33.65   31.23
  MMLU dev                           0      0       32.43   30.06
  MMLU-Pro validation                0      0       18.53   14.67
  (with tools = Cells 19, 23 and 24 where they answer, else Cell 12; was = Cell 12 alone)

  by tool (all sets: answered, agree with the key)
    MULTIPLE       1     1
    EXTREMUM       2     2
    PRIMES         1     1
    RELATION       1     1
    SETS           1     1
    RECURSE        1     1
    RECIPROCAL     1     1
    LOGFIT         1     1
    MOTION         1     1

  DISAGREEMENTS w

In [34]:
# =====================================================================
# MC AI BRAIN 5 - CELL 25A: TOOLBOX III (PART 1) - EXACT INTEGERS, REMAINDERS
# Needs Cells 0 (fetch, DATA, GITHUB), 12A, 12B, 19 (clean19, number19) and
# 23 (numbers23). Same rules as Toolboxes I and II: a fixed question form,
# an exact computation, every number of the question inside the part the
# tool read. These tools were designed on MATH's training split (Hendrycks
# et al. 2021, as kept in openai/prm800k math_splits/train.jsonl); only the
# rows whose id begins "train/" are read. MMLU was not used to build them.
# EVALUATOR  int25: +, -, *, /, powers, n!, binomials, base-b numerals and
#   a + b + ... + z series (or products) written out when one integer moves
#   by a fixed step; with a modulus m every step is reduced mod m, powers by
#   pow(b, e, m), and n! is 0 mod m once n >= m.
# TOOLS (part 1)
#   REMAINDER  remainder of an expression divided by m; units digit (mod 10);
#              residue mod m; n with 0 <= n < m and n = E (mod m)
#   GCD-LCM    gcd or lcm of two or more integers; gcd times lcm
#   INVERSE    a^-1 mod m
#   DIGITSUM   sum of the digits of an exact value
# PREDICTIONS (written before running; MATH train is where these tools were
#   built, so both are CHECKS of the building)
#   B5-TD1 MATH train questions the part-1 tools answer: 75-95
#   B5-TD2 of those, answers that disagree with MATH's answer key: 0
#   B5-T37 Cell 25A seconds: <= 180
# =====================================================================
import ast
import json
import math
import os
import re
import time
from collections import Counter
from fractions import Fraction

from mcai.core import at_most, between

print("=" * 64); print("MC AI BRAIN 5 - CELL 25A: TOOLBOX III (PART 1) - EXACT INTEGERS, REMAINDERS"); print("=" * 64)
T25 = time.time()
for need in ("fetch", "DATA", "GITHUB", "SETS12", "clean19", "number19", "numbers23"):
    if need not in globals():
        raise RuntimeError(f"run Cells 0, 12A, 12B, 19 and 23 first ({need} is missing)")
for key, claim, rule in (("B5-TD1", "MATH train questions the part-1 tools answer (check)", between(75, 95)),
                         ("B5-TD2", "of those, answers that disagree with MATH's key (check)", between(0, 0)),
                         ("B5-T37", "Cell 25A seconds", at_most(180))):
    LEDGER.predict(key, claim, rule)

# ---------------- data: MATH train, pinned by commit and SHA-256 ----------------
MATH25 = ("openai/prm800k", "7ecc794703b2877f63226f2477a49b34f9b25163", "prm800k/math_splits/train.jsonl",
          "90d96daeac3fe343ebb1e22ce93dd99690f75983e957f88de42f87cffe1e8076")
LFS25 = "https://media.githubusercontent.com/media/{repo}/{commit}/{file}"  # the file is stored with Git LFS
BASE = os.environ.get("MC_AI_FETCH_BASE", "")  # Cell 0's fetch base: Cell 6 reuses the name BASE for its setting
path25, how25, _ = fetch("math", MATH25[0], MATH25[1], MATH25[2], MATH25[3], None, [LFS25])
if how25.startswith("REFUSED"):
    raise RuntimeError(f"MATH train: {how25}")
with open(path25) as f:
    ALL25 = [json.loads(line) for line in f]
MATHROWS25 = [r for r in ALL25 if r["unique_id"].startswith("train/")]
print(f"  MATH train: {len(MATHROWS25):,} problems ({how25}); the file's other {len(ALL25) - len(MATHROWS25):,} rows "
      "(MATH test problems) are skipped")
del ALL25

# ---------------- toolbox III, part 1 ----------------
LIMIT25 = 200000  # bits allowed in an exact intermediate value
FORMAT25 = re.compile(r"(?:Express|Write|Give|Enter|Report) (?:your|the) answer[^.?]*[.?]?|\(Express[^)]*\)", re.I)


def tex25(s: str) -> str:
    """LaTeX and plain maths to Python syntax: C(n,k), F(n) for n!, B('digits',b) for a base-b numeral"""
    s = s.replace("$", " ").replace("\\left", "").replace("\\right", "").replace("\\,", " ").replace("\\!", "")
    s = s.replace("\\cdots", "...").replace("\\ldots", "...").replace("\\dots", "...").replace("\u2026", "...")
    s = s.replace("\\cdot", "*").replace("\\times", "*").replace("\\div", "/").replace("\u00d7", "*")
    s = s.replace("\u2212", "-")
    s = re.sub(r"(?<=\d),(?=\d{3}(?!\d))", "", s).replace("\\[", " ").replace("\\]", " ")
    s = re.sub(r"\\[dt]?binom\s*\{([^{}]+)\}\s*\{([^{}]+)\}", r"C((\1),(\2))", s)
    s = re.sub(r"\\[dt]?frac\s*\{([^{}]+)\}\s*\{([^{}]+)\}", r"((\1)/(\2))", s)
    s = re.sub(r"\\[dt]?frac\s*(\d)(\d)", r"((\1)/(\2))", s)
    s = re.sub(r"\b([0-9A-Za-z]+)_\{?(\d+)\}?", lambda m: f"B('{m.group(1)}',{m.group(2)})", s)
    s = re.sub(r"\(([^()]*)\)!", r"F(\1)", s)
    s = re.sub(r"(\d+)!", r"F(\1)", s)
    s = s.replace("{", "(").replace("}", ")").replace("^", "**").replace("\\", " ")
    s = re.sub(r"\)\s*\(", ")*(", s)
    return re.sub(r"\s+", " ", s).strip(" .?,")


def series25(s: str) -> str:
    """a + b + ... + z (or with *) written out when the terms differ in one integer by a fixed step"""
    for op in ("+", "*"):
        parts = [p.strip() for p in re.split(r"(?<![*])\s*" + re.escape(op) + r"\s*(?![*])", s)]
        if "..." not in parts or len(parts) < 4:
            continue
        k = parts.index("...")
        if parts.count("...") != 1 or k < 2 or k == len(parts) - 1:
            return None
        terms = parts[:k] + parts[k + 1:]
        shapes = [re.sub(r"\d+", "#", t) for t in terms]
        if len(set(shapes)) != 1:
            return None
        nums = [[int(x) for x in re.findall(r"\d+", t)] for t in terms]
        moving = [i for i in range(len(nums[0])) if len({n[i] for n in nums}) > 1]
        if len(moving) != 1:
            return None
        i = moving[0]
        seq = [n[i] for n in nums[:k]]
        tail = [n[i] for n in nums[k:]]
        step = seq[1] - seq[0]
        if step == 0 or any(b - a != step for a, b in zip(seq + tail[:0], seq[1:])) or (tail[-1] - seq[0]) % step:
            return None
        if any(b - a != step for a, b in zip(tail, tail[1:])) or (tail[0] - seq[-1]) % step or \
                (tail[0] - seq[-1]) // step < 1:
            return None
        count = (nums[-1][i] - seq[0]) // step + 1
        if not 2 < count <= 10000:
            return None
        out = []
        for v in range(seq[0], nums[-1][i] + step, step):
            it = iter(range(len(nums[0])))
            out.append(re.sub(r"\d+", lambda m: str(v) if next(it) == i else m.group(0), terms[0]))
        return "(" + f" {op} ".join(out) + ")"
    return s


def int25(s: str, mod: int = None):
    """exact value (int or Fraction) of a maths string, reduced mod `mod` if given; None if out of reach"""
    py = series25(tex25(s))
    if py is None or not re.fullmatch(r"[\d\s+\-*/%()A-Za-z',.]*", py) or not re.search(r"\d", py):
        return None
    if re.search(r"[A-Za-z]", re.sub(r"\b[CFB]\(|'[0-9A-Za-z]+'", "", py)):
        return None
    try:
        tree = ast.parse(py, mode="eval")
    except (SyntaxError, RecursionError, MemoryError, ValueError):
        return None

    def big(x):
        if isinstance(x, int) and x.bit_length() > LIMIT25:
            raise OverflowError("too big")
        return x

    def ev(n, m):
        if isinstance(n, ast.Expression):
            return ev(n.body, m)
        if isinstance(n, ast.Constant) and isinstance(n.value, int) and not isinstance(n.value, bool):
            return n.value % m if m else n.value
        if isinstance(n, ast.UnaryOp) and isinstance(n.op, (ast.USub, ast.UAdd)):
            v = ev(n.operand, m)
            v = -v if isinstance(n.op, ast.USub) else v
            return v % m if m else v
        if isinstance(n, ast.BinOp):
            if isinstance(n.op, ast.Pow):
                e = ev(n.right, None)
                if not isinstance(e, int) or e < 0:
                    if m or e is None or not isinstance(e, int):
                        raise ValueError("exponent")
                b = ev(n.left, m)
                if m:
                    return pow(b, e, m)
                if isinstance(b, int) and abs(b) > 1 and e * b.bit_length() > LIMIT25:
                    raise OverflowError("too big")
                return big(Fraction(b) ** e if e < 0 else b ** e)
            if isinstance(n.op, (ast.Add, ast.Sub, ast.Mult)):
                fam = (ast.Mult,) if isinstance(n.op, ast.Mult) else (ast.Add, ast.Sub)
                chain, node = [], n
                while isinstance(node, ast.BinOp) and isinstance(node.op, fam):
                    chain.append((node.op, node.right))
                    node = node.left
                v = ev(node, m)
                for op, right in reversed(chain):
                    b = ev(right, m)
                    v = v + b if isinstance(op, ast.Add) else v - b if isinstance(op, ast.Sub) else v * b
                    v = big(v % m if m else v)
                return v
            a, b = ev(n.left, m), ev(n.right, m)
            if isinstance(n.op, ast.Div) and not m:
                v = Fraction(a) / Fraction(b)
                v = int(v) if v.denominator == 1 else v
            else:
                raise ValueError("operator")
            return big(v % m if m else v)
        if isinstance(n, ast.Call) and isinstance(n.func, ast.Name) and not n.keywords:
            f = n.func.id
            if f == "B" and len(n.args) == 2 and isinstance(n.args[0], ast.Constant):
                base = ev(n.args[1], None)
                digits = str(n.args[0].value)
                if not 2 <= base <= 36 or any(int(c, 36) >= base for c in digits):
                    raise ValueError("digit")
                v = int(digits, base)
                return v % m if m else v
            args = [ev(a, None) for a in n.args]
            if not all(isinstance(a, int) and a >= 0 for a in args):
                raise ValueError("argument")
            if f == "F" and len(args) == 1:
                k = args[0]
                if m:
                    return 0 if k >= m else math.prod(range(1, k + 1)) % m if k <= 10 ** 6 else None
                if k > 3000:
                    raise OverflowError("too big")
                return math.factorial(k)
            if f == "C" and len(args) == 2 and args[0] <= 20000:
                v = math.comb(args[0], args[1])
                return v % m if m else v
        raise ValueError("form")
    try:
        v = ev(tree, mod)
    except (ValueError, OverflowError, ZeroDivisionError, TypeError, RecursionError):
        return None
    return v if isinstance(v, (int, Fraction)) else None


def math25(s: str) -> str:
    """the maths of a phrase: its $...$ parts, or the phrase itself when it has none"""
    s = re.sub(r"\\\[(.+?)\\\]", r"$\1$", s).replace("$$", "$")
    parts = re.findall(r"\$([^$]+)\$", s)
    if parts:
        rest = re.sub(r"\$[^$]+\$", " ", s).strip(" .?,")
        return parts[0] if len(parts) == 1 and not re.search(r"[A-Za-z0-9]", rest) else None
    return s.strip(" .?,")


def lead25(s: str) -> str:
    return re.sub(r"^(?:the )?(?:(?:sum|product|number|integer|expression|value)\s+)?", "", s.strip())


def factor25(n: int) -> dict:
    """prime factorisation by trial division, n <= 10^12"""
    if not 1 <= n <= 10 ** 12:
        raise ValueError("size")
    out, p = {}, 2
    while p * p <= n:
        while n % p == 0:
            out[p] = out.get(p, 0) + 1
            n //= p
        p += 1 if p == 2 else 2
    if n > 1:
        out[n] = out.get(n, 0) + 1
    return out


def digits25(v: int, b: int) -> str:
    if v < 0:
        raise ValueError("negative")
    out = ""
    while True:
        v, r = divmod(v, b)
        out = "0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ"[r] + out
        if v == 0:
            return out


HEAD25 = r"(?:^|(?<=[.?!:]\s))(?:What is|What's|Find|Compute|Determine|Calculate|Give) "


def remainder25(q: str):
    """(value, consumed pieces of the question) for a remainder, residue, units digit or mod question"""
    m = re.search(HEAD25 + r"the remainder when (.+?)\s*is divided by \$?(\d+)\$?\s*[?.,]", q, re.I)
    if m:
        e = math25(lead25(m.group(1)))
        if e is not None:
            return int25(e, int(m.group(2))), [m.group(0)]
    m = re.search(r"When (?:the )?(?:base-\d+ )?(?:integer |number )?(.+?) is divided by \$?(\d+)\$?,? "
                  r"what is the remainder\?", q, re.I)
    if m:
        e = math25(m.group(1))
        if e is not None:
            return int25(e, int(m.group(2))), [m.group(0)]
    m = re.search(HEAD25 + r"the (?:units|ones) digit of (.+?)\s*[?.]?$", q.strip())
    if m:
        e = math25(lead25(m.group(1)))
        if e is not None and "-" not in e and "/" not in e and "frac" not in e:
            return int25(e, 10), [m.group(0)]
    m = re.search(r"(?:residue of|Compute|Find|Evaluate|What is) \$?(.+?)\s*(?:\\pmod\{?(\d+)\}?|\\bmod\s*\{?(\d+)\}?"
                  r"|\bmod (\d+)|\bmodulo (\d+))\$?\s*[.?]", q)
    if m and "^{-1}" not in m.group(1) and "\\equiv" not in m.group(1):
        mod = int(next(g for g in m.groups()[1:] if g))
        e = math25(m.group(1).strip("$ ")) if "$" not in m.group(1).strip("$ ") else None
        if e is not None:
            return int25(e, mod), [m.group(0)]
    m = re.search(r"(?:integer )?\$?n\$?,? \$?0\s*\\le\s*n\s*(<|\\le)\s*(\d+)\$?,? (?:that )?satisf(?:y|ies)\s*"
                  r"(?:\$\$|\\\[|\$)\s*(?:n\s*\\equiv\s*(.+?)|(.+?)\s*\\equiv\s*n)\s*\\pmod\{?(\d+)\}?", q)
    if m:
        top, mod = int(m.group(2)) + (1 if m.group(1) != "<" else 0), int(m.group(5))
        if top == mod:
            return int25((m.group(3) or m.group(4)).strip(" ~$"), mod), [m.group(0)]
    return None


def gcd25(q: str):
    lst = r"((?:\$?[^$,]+?\$?,\s*)*\$?[^$,]+?\$?,?\s*and\s*\$?[^$?.]+?\$?|\$\\?[^$]+\$)"
    m = re.search(HEAD25 + r"the product of the (?:greatest common (?:divisor|factor) and the least common multiple|"
                  r"least common multiple and the greatest common (?:divisor|factor)) of "
                  r"\$?(\d+)\$? and \$?(\d+)\$?\s*[.?]?\s*$", q.strip())
    if m:
        a, b = int(m.group(1)), int(m.group(2))
        return a * b, [m.group(0)]
    m = re.search(HEAD25 + r"the (greatest common (?:divisor|factor)|least common multiple|GCD|GCF|LCM) of "
                  r"(?:the (?:numbers|integers) )?" + lst + r"\s*[?.]?\s*$", q.strip())
    fn = None
    if m:
        fn, body = m.group(1).lower(), m.group(2)
    else:
        m = re.search(r"\$?\\(gcd|operatorname\{gcd\}|mathop\{\\text\{lcm\}\}|text\{lcm\}|operatorname\{lcm\})"
                      r"\s*[(\[]([^$]+?)[)\]]\$?\s*[?.]?\s*$", q.strip())
        if m:
            fn, body = ("greatest common divisor" if "gcd" in m.group(1) else "least common multiple"), m.group(2)
    if fn is None:
        return None
    items = [x for x in re.split(r",\s*(?:and\s*)?|\s+and\s+", body.replace("$", " ").strip(" ,.?")) if x.strip()]
    vals = [int25(x) for x in items]
    if len(vals) < 2 or not all(isinstance(v, int) and v > 0 for v in vals):
        return None
    v = math.gcd(*vals) if fn.startswith(("greatest", "gcd", "gcf")) else math.lcm(*vals)
    return v, [m.group(0)]


def inverse25(q: str):
    m = re.search(r"(?:modular |multiplicative )?inverse of \$?(\d+)\$?,? (?:modulo|mod) \$?(\d+)\$?", q)
    if not m:
        m = re.search(r"\$?(\d+)\^\{-1\}\s*\\pmod\{?(\d+)\}?\$?", q)
    if not m:
        return None
    a, n = int(m.group(1)), int(m.group(2))
    if n < 2 or math.gcd(a, n) != 1:
        return None
    return pow(a, -1, n), [m.group(0)]


def digitsum25(q: str):
    m = re.search(HEAD25 + r"the sum of (?:all )?(?:of )?the digits (?:of|in) "
                  r"(?:the (?:number|integer|product|sum) )?(.+?)\s*[?.]?$", q.strip())
    if not m or "base" in q:
        return None
    e = math25(m.group(1))
    v = int25(e) if e else None
    if not isinstance(v, int) or v < 0:
        return None
    return sum(map(int, str(v))), [m.group(0)]


TOOLS25 = (("INVERSE", inverse25), ("REMAINDER", remainder25), ("GCD-LCM", gcd25), ("DIGITSUM", digitsum25))


def ask25(q: str):
    """(tool, value) when one tool answers and every number of the question is in the part it read"""
    plain = FORMAT25.sub(" ", q)
    for name, tool in TOOLS25:
        try:
            got = tool(q)
        except (ValueError, ZeroDivisionError, AttributeError, OverflowError, RecursionError, IndexError):
            got = None
        if not got or got[0] is None:
            continue
        v, pieces = got
        used = Counter()
        for p in pieces:
            used += Counter(numbers23(p))
        if Counter(numbers23(plain)) - used:
            continue
        return name, v
    return None


def key25(a: str):
    """MATH's answer as a value: an integer, a fraction, or a numeral in base b"""
    a = re.sub(r"\\text\{\s*([^}]*)\}", r"\1", a).replace("$", "").replace("\\!", "").replace(",", "").strip()
    a = re.sub(r"\^\{?\\circ\}?|\\%|\s*(?:hours|ways|days|cents|dollars|units)\.?$", "", a).strip()
    m = re.fullmatch(r"([0-9A-Z]+)_\{?(\d+)\}?", a)
    return ("BASE", m.group(1), int(m.group(2))) if m else int25(a)


def agree25(v, k) -> bool:
    if isinstance(v, tuple):
        return v[1] == k[1] if isinstance(k, tuple) else isinstance(k, int) and str(k) == v[1]
    return k is not None and not isinstance(k, tuple) and v == k


def math_check25():
    """answered and agreeing, per tool, on MATH train; and the disagreements"""
    by, bad = {}, []
    for r in MATHROWS25:
        got = ask25(r["problem"])
        if not got:
            continue
        ok = agree25(got[1], key25(r["answer"]))
        by.setdefault(got[0], [0, 0])
        by[got[0]][0] += 1
        by[got[0]][1] += ok
        if not ok:
            bad.append((got[0], r["problem"][:110].replace("\n", " "), got[1], r["answer"]))
    print(f"\n  MATH train (where the tools were built)\n    {'tool':<11}{'answered':>9}{'agree':>7}")
    for name, (n, a) in by.items():
        print(f"    {name:<11}{n:>9}{a:>7}")
    for name, q, said, key in bad:
        print(f"    DISAGREES [{name}] {q}\n      tool: {said}; key: {key}")
    return sum(n for n, _ in by.values()), len(bad)


for q, want in (("Find the remainder when $3^{3^{3^3}}$ is divided by 1000.", 387),
                ("What is the ones digit of $1^{2009} + 2^{2009} + 3^{2009} + \\cdots + 2009^{2009}?$", 5),
                ("What is the greatest common divisor of $654321$ and $543210$?", 3),
                ("What is the remainder when the sum $1 + 7 + 13 + 19 + \\cdots + 253 + 259$ is divided by $6$?", 2),
                ("What is the modular inverse of $11$, modulo $1000$?", 91)):
    if ask25(q) is None or ask25(q)[1] != want:
        raise RuntimeError(f"self-check failed: {q} -> {ask25(q)}, want {want}")
print("  self-checks: 5 of 5 (power tower mod 1000, a 2009-term series mod 10, gcd, a 44-term series mod 6, inverse)")
N25A, BAD25A = math_check25()
print("\n  PREDICTIONS vs RESULTS")
LEDGER.result("B5-TD1", N25A)
LEDGER.result("B5-TD2", BAD25A)
LEDGER.result("B5-T37", round(time.time() - T25))
if "sync" in globals():
    sync("Cell 25A")
print(f"\n  ({time.time() - T25:.0f}s)")
print("  LAST LINE OF MC AI BRAIN 5 CELL 25A")


MC AI BRAIN 5 - CELL 25A: TOOLBOX III (PART 1) - EXACT INTEGERS, REMAINDERS
  MATH train: 7,499 problems (downloaded); the file's other 4,501 rows (MATH test problems) are skipped
  self-checks: 5 of 5 (power tower mod 1000, a 2009-term series mod 10, gcd, a 44-term series mod 6, inverse)

  MATH train (where the tools were built)
    tool        answered  agree
    REMAINDER         46     46
    GCD-LCM           36     36
    INVERSE            5      5

  PREDICTIONS vs RESULTS
    B5-TD1 MATH train questions the part-1 tools answer (check): 87 (predicted between 75 and 95) -> HELD
    B5-TD2 of those, answers that disagree with MATH's key (check): 0 (predicted between 0 and 0) -> HELD
    B5-T37 Cell 25A seconds: 2 (predicted <= 180) -> HELD
  sync: pushed 5 files to kaggle/ on main as commit 58b1c129ebe2; GitHub confirms it (690 entries, head af7fe66a367674a0) (3s)

  (5s)
  LAST LINE OF MC AI BRAIN 5 CELL 25A


In [35]:
# =====================================================================
# MC AI BRAIN 5 - CELL 25B: TOOLBOX III (PART 2) - DIVISORS, BASES, COUNTING
# Needs Cell 25A and Cells 12A, 12B, 19, 23, 24 (solve19, tool23, tool24).
# Same rules; built on MATH train like part 1. MMLU was not used to build
# any Toolbox III tool, so the MMLU numbers below are a first look.
# TOOLS (part 2)
#   DIVISORS number or sum of the divisors (all, odd, even or proper);
#            divisors that are perfect squares or cubes; largest or smallest
#            prime factor; sum or number of distinct prime factors
#   BASE     base-b numerals evaluated, the result written in a named base
#   TOBASE   a base-10 number written in base b
#   ARRANGE  distinct orderings of the letters of a word: n! / (a! b! ...)
#   CHOOSE   "Compute <expression>" when it holds a binomial or a factorial;
#            "n choose k"
#   BALLS    n balls into k boxes, each side distinguishable or not, boxes
#            may be empty: k^n, C(n+k-1, k-1), Stirling numbers of the second
#            kind summed to k, partitions of n into at most k parts
#   PICK     k chosen from n when order does not matter: C(n, k)
#   SEAT     n people in a row n!; round a table up to rotation (n-1)!;
#            beads on a bracelet up to rotation and reflection (n-1)!/2
# PREDICTIONS (written before running; B5-TD3 and B5-TD4 are CHECKS of the
#   building on MATH train; the MMLU ones are not)
#   B5-TD3 MATH train questions all of Toolbox III answers: 190-210
#   B5-TD4 of those, answers that disagree with MATH's key: 0
#   B5-TD5 MMLU validation questions Toolbox III answers (where Cells 19, 23
#          and 24 do not): 2-12
#   B5-TD6 of those, answers that disagree with the answer key: <= 1
#   B5-TD7 MMLU validation with Toolbox III, minus without it, points:
#          0.1-0.8
#   B5-TD8 MMLU dev questions Toolbox III answers: 0-3
#   B5-T38 Cell 25B seconds: <= 300
# =====================================================================
import math
import re
import time
from collections import Counter

import numpy as np
from mcai.core import at_most, between

print("=" * 64); print("MC AI BRAIN 5 - CELL 25B: TOOLBOX III (PART 2) - DIVISORS, BASES, COUNTING"); print("=" * 64)
T25B = time.time()
for need in ("ask25", "math_check25", "TOOLS25", "SETS12", "READ12", "scores12", "credit12", "solve19", "tool23",
             "tool24", "number19", "numbers23"):
    if need not in globals():
        raise RuntimeError(f"run Cells 12A, 12B, 19, 23, 24 and 25A first ({need} is missing)")
for key, claim, rule in (("B5-TD3", "MATH train questions all of Toolbox III answers (check)", between(190, 210)),
                         ("B5-TD4", "of those, answers that disagree with MATH's key (check)", between(0, 0)),
                         ("B5-TD5", "MMLU validation questions Toolbox III answers", between(2, 12)),
                         ("B5-TD6", "of those, answers that disagree with the answer key", at_most(1)),
                         ("B5-TD7", "MMLU validation with Toolbox III minus without it, points", between(0.1, 0.8)),
                         ("B5-TD8", "MMLU dev questions Toolbox III answers", between(0, 3)),
                         ("B5-T38", "Cell 25B seconds", at_most(300))):
    LEDGER.predict(key, claim, rule)

# ---------------- toolbox III, part 2 ----------------
ADJ25 = r"(?:(?:positive|distinct|whole[- ]number|natural[- ]number|integer|integral|different)\s*,?\s+)*"
KIND25 = r"((?:(?:odd|even|proper)\s*,?\s+)?)" + ADJ25


def divisors25(q: str):
    s = q.strip()
    of = r"(?:the (?:number|integer) )?(.+?)"
    pats = [(r"[Hh]ow many " + ADJ25 + KIND25 + r"(?:divisors|factors) (?:does|do) " + of + r" have\?", "count"),
            (HEAD25 + r"the (?:number|count) of " + ADJ25 + KIND25 + r"(?:divisors|factors) of " + of + r"\s*[?.]?$",
             "count"),
            (HEAD25 + r"the sum of (?:all )?(?:of )?(?:the )?" + ADJ25 + KIND25 + r"(?:divisors|factors) of " + of +
             r"\s*[?.]?$", "sum"),
            (r"[Hh]ow many " + ADJ25 + r"(?:divisors|factors) of " + of + r" are perfect (squares|cubes)\?", "power"),
            (HEAD25 + r"the (?:largest|greatest|biggest) prime (?:factor|divisor) of " + of + r"\s*[?.]?$", "max"),
            (HEAD25 + r"the (?:smallest|least) prime (?:factor|divisor) of " + of + r"\s*[?.]?$", "min"),
            (HEAD25 + r"the sum of the (distinct |different )?prime (?:factors|divisors) of " + of + r"\s*[?.]?$",
             "psum"),
            (r"[Hh]ow many (?:distinct |different )prime (?:factors|divisors) (?:does|do) " + of + r" have\?",
             "pcount")]
    for p, kind in pats:
        m = re.search(p, s)
        if not m:
            continue
        if kind == "psum" and not m.group(1):
            return None
        e = math25(m.groups()[-1] if kind != "power" else m.group(1))
        n = int25(e) if e else None
        if not isinstance(n, int) or n < 1:
            return None
        f = factor25(n)
        if kind in ("max", "min", "psum", "pcount"):
            if not f:
                return None
            v = {"max": max(f), "min": min(f), "psum": sum(f), "pcount": len(f)}[kind]
            return v, [m.group(0)]
        ds = [1]
        for p_, k_ in f.items():
            ds = [d * p_ ** j for d in ds for j in range(k_ + 1)]
        if kind == "power":
            r = 2 if m.group(2) == "squares" else 3
            ds = [d for d in ds if round(d ** (1 / r)) ** r == d]
        else:
            adj = m.group(1).replace(",", " ").split()
            if "odd" in adj:
                ds = [d for d in ds if d % 2]
            if "even" in adj:
                ds = [d for d in ds if d % 2 == 0]
            if "proper" in adj:
                ds = [d for d in ds if d != n]
        return (len(ds) if kind in ("count", "power") else sum(ds)), [m.group(0)]
    return None


def base25(q: str):
    """numerals in base b: their value, or a sum/difference/product written in a named base"""
    s = q.strip()
    to = re.search(r"(?:in|to) (?:a )?base[- ]?\$?\s*(\d+)\$?", s)
    m = re.search(r"(?:Express|Convert|Write|What is|Find|Compute|Calculate|Evaluate) (?:the value of )?"
                  r"(?:the (?:base-\d+ )?"
                  r"(?:number|integer|numeral) )?\$([^$]+)\$", s)
    if not to or not m:
        return None
    b = int(to.group(1))
    rest = FORMAT25.sub(" ", s.replace(m.group(0), " ").replace(to.group(0), " "))
    if re.search(r"\d", rest) or not re.search(r"_\{?\d+\}?", m.group(1)) and b == 10:
        return None
    if re.search(r"\$", rest) or not 2 <= b <= 36:
        return None
    v = int25(m.group(1))
    if not isinstance(v, int) or v < 0:
        return None
    return ("BASE", digits25(v, b), b), [m.group(0), to.group(0)]


def tobase25(q: str):
    """a base-10 number written in another base"""
    m = re.search(r"(?:Express|Convert|Write|What is) "
                  r"(?:the (?:decimal |base[- ]ten |base[- ]10 )?(?:number|integer) )?"
                  r"\$?(\d+)(?:_\{?10\}?)?\$? (?:in|to) base[- ]?\$?(\d+)\$?", q)
    if not m:
        return None
    b = int(m.group(2))
    if not 2 <= b <= 36:
        return None
    return ("BASE", digits25(int(m.group(1)), b), b), [m.group(0)]


def arrange25(q: str):
    m = re.search(r"(?:ways|arrangements|permutations|rearrangements)[^.?]*?(?:of|in) the (?:letters|word)\s*(?:of|in)?"
                  r"\s*(?:the word )?\W*([A-Za-z]{2,})\W", q + " ")
    if not m or not re.search(r"\b(?:arrange|arrangements|permutations|rearrangements|distinct|different)\b", q):
        return None
    if re.search(r"\b(?:if|such that|vowels?|consonants?|together|begin|begins|start|starts|end|ends|not|adjacent|"
                 r"next|must|circle|round|palindrome|without|except|repeat|two|three|four|five|first|last)\b",
                 q, re.I):
        return None
    w = m.group(1).upper()
    return math.factorial(len(w)) // math.prod(math.factorial(k) for k in Counter(w).values()), [m.group(0)]


def choose25(q: str):
    """Compute/Evaluate/What is <expression with a binomial or factorial>"""
    s = q.strip()
    m = re.fullmatch(r"(?:Compute|Evaluate|Calculate|Simplify|Find|What is(?: the value of)?) \$([^$]+)\$\s*[.?]?"
                     r"(?:\s*" + FORMAT25.pattern + r")?", s)
    if not m:
        m = re.fullmatch(r"(?:Compute|Evaluate|Calculate|What is(?: the value of)?) "
                         r"\$?(\d+)\$? choose \$?(\d+)\$?\s*[.?]?", s)
        if m:
            return math.comb(int(m.group(1)), int(m.group(2))), [m.group(0)]
        return None
    if not re.search(r"binom|\d\s*!|\)\s*!", m.group(1)):
        return None
    v = int25(m.group(1))
    return (v, [m.group(0)]) if v is not None else None


WORDS25 = {w: i for i, w in enumerate("zero one two three four five six seven eight nine ten eleven twelve thirteen "
                                       "fourteen fifteen sixteen seventeen eighteen nineteen twenty".split())}
NUM25 = r"(\d+|" + "|".join(WORDS25) + r")"
STOP25 = (r"\b(?:if|such that|must|cannot|can't|refuse|probability|at least|at most|exactly|president|vice|"
          r"secretary|treasurer|captain and|different gender|consecutive|next to|adjacent|order matters|"
          r"same|both|each|identical|indistinguishable|at random|randomly|chairs? numbered|either|or more)\b")


def n25(w: str) -> int:
    return int(w) if w.isdigit() else WORDS25[w.lower()]


def balls25(q: str):
    """n balls into k boxes, each side distinguishable or not; boxes may be empty"""
    m = re.search(r"How many ways are there to put " + NUM25 + r" ((?:in)?distinguishable )?balls in(?:to)? " + NUM25 +
                  r" ((?:in)?distinguishable )?boxes(?: if the balls are (not )?distinguishable,? (?:(?:but|and) the "
                  r"boxes are( not)?(?: distinguishable)?|and (neither|so) are the boxes))?\?\s*$", q.strip())
    if not m or (m.group(2) is None) != (m.group(4) is None) or (m.group(2) is None) != (" if " in m.group(0)):
        return None
    n, k = n25(m.group(1)), n25(m.group(3))
    if m.group(2) is not None:
        bd, xd = m.group(2) == "distinguishable ", m.group(4) == "distinguishable "
    else:
        bd = m.group(5) is None
        xd = m.group(7) == "so" if m.group(7) else m.group(6) is None
    if n > 60 or k > 60:
        return None
    if bd and xd:
        v = k ** n
    elif xd:
        v = math.comb(n + k - 1, k - 1)
    elif bd:
        st = [[0] * (k + 1) for _ in range(n + 1)]
        st[0][0] = 1
        for i in range(1, n + 1):
            for j in range(1, k + 1):
                st[i][j] = j * st[i - 1][j] + st[i - 1][j - 1]
        v = sum(st[n][:k + 1])
    else:
        p = [[1] * (k + 1)] + [[0] * (k + 1) for _ in range(n)]
        for i in range(1, n + 1):
            for j in range(1, k + 1):
                p[i][j] = p[i][j - 1] + (p[i - j][j] if i >= j else 0)
        v = p[n][k]
    return v, [m.group(0)]


def pick25(q: str):
    """k chosen from n where order does not matter"""
    if re.search(STOP25, re.sub(r"if the order[^.?]*does(?: not|n't) matter", "", q), re.I):
        return None
    pats = [r"(?:choose|select|pick) " + NUM25 + r" (?:[a-z-]+ ){0,2}(?:out of|from)(?: (?:a|the|his|her|my|our) "
            r"(?:group|team|club|set|class|box|bag|list|pool)?(?: of)?)? " + NUM25 + r"\b",
            r"(?:choose|select|pick|form) (?:a|an) " + NUM25 +
            r"-(?:person|student|member|people) (?:\w+ )?committee "
            r"(?:from|out of) (?:a|the|my|our) (?:group|club|team|class) of " + NUM25 + r"\b",
            r"[Ff]rom a (?:group|club|team|class) of " + NUM25 + r" (?:\w+), how many (?:different )?" + NUM25 +
            r"-(?:person|student|member) committees can be (?:chosen|selected|formed)\?"]
    for i, p in enumerate(pats):
        m = re.search(p, q)
        if m:
            k, n = (n25(m.group(2)), n25(m.group(1))) if i == 2 else (n25(m.group(1)), n25(m.group(2)))
            if not 0 < k <= n <= 1000 or len(re.findall(r"\d+", q)) > 2:
                return None
            return math.comb(n, k), [m.group(0)]
    return None


def seat25(q: str):
    """n people in a row (n!), around a round table up to rotation ((n-1)!), beads on a bracelet ((n-1)!/2)"""
    s = q.strip()
    if re.search(STOP25, re.sub(r"\((?:Two|Note)[^)]*\)", "", s), re.I):
        return None
    m = re.search(r"In how many (?:different )?ways can " + NUM25 + r" (?:people|students|children|friends) "
                  r"(?:be seated|sit|stand|line up)(?: in a (?:row|line|straight line)(?: of chairs)?)?\?\s*$", s)
    if m:
        return math.factorial(n25(m.group(1))), [m.group(0)]
    m = re.search(r"In how many ways can " + NUM25 + r" (?:people|students|children|friends) (?:be seated|sit) "
                  r"around a round table\?\s*\((?:Two|Note that two) (?:seatings|arrangements) are "
                  r"(?:considered )?the same if "
                  r"one is a rotation of the other\.?\)\s*$", s)
    if m:
        return math.factorial(n25(m.group(1)) - 1), [m.group(0)]
    m = re.search(r"In how many ways can " + NUM25 + r" distinct beads be placed on a bracelet\?\s*\(Note that two "
                  r"arrangements are the same if one can be rotated or reflected to produce the other\.?\)\s*$", s)
    if m:
        k = n25(m.group(1))
        return (math.factorial(k - 1) // 2 if k > 2 else 1), [m.group(0)]
    return None


PART2 = (("DIVISORS", divisors25), ("BASE", base25), ("TOBASE", tobase25), ("ARRANGE", arrange25),
         ("CHOOSE", choose25), ("BALLS", balls25), ("PICK", pick25), ("SEAT", seat25))
TOOLS25 = tuple(t for t in TOOLS25 if t[0] not in dict(PART2)) + PART2  # ask25 reads TOOLS25 when it runs


def same25(v, c: str) -> bool:
    """does choice text c state the value v (a number, or a numeral in base b)"""
    if isinstance(v, tuple):
        t = re.sub(r"\s|\$|\\text\{[^}]*\}|_\{?\d+\}?|\(base \d+\)|base \d+", "", str(c))
        return t.upper() == v[1]
    x = number19(c)
    return x is not None and x == v and re.fullmatch(r"\s*\$?[-+]?[\d,]+(?:\.\d+)?(?:\s*/\s*\d+)?\$?\s*",
                                                       str(c)) is not None


def tool25(q: str, ch: list):
    """(tool name, index of the one agreeing choice) or None"""
    got = ask25(q)
    if not got:
        return None
    hit = [j for j, c in enumerate(ch) if same25(got[1], c)]
    return (got[0], hit[0]) if len(hit) == 1 else None


for q, want in (("What is the sum of the positive odd divisors of $60$?", 24),
                ("What is the value of $25_{10}+36_{10}$ in base 3?", ("BASE", "2021", 3)),
                ("How many ways are there to put 5 balls in 3 boxes if the balls are distinguishable but the boxes "
                 "are not?", 41),
                ("How many ways are there to arrange the letters of the word BANANA?", 60)):
    if ask25(q) is None or ask25(q)[1] != want:
        raise RuntimeError(f"self-check failed: {q} -> {ask25(q)}, want {want}")
print("  self-checks: 4 of 4 (odd divisors, base 3, Stirling numbers, letters of BANANA)")
N25B, BAD25B = math_check25()

# ---------------- scored on the practice sets ----------------
RES25, WRONG25, TALLY25, EXACT25 = {}, [], {}, {}
print(f"\n  {'':<26}{'answered':>10}{'agree':>7}{'with III':>10}{'without':>9}")
for n, qs in SETS12.items():
    rows, prev, ans, agree, ex = [], [], 0, 0, [0, 0]
    for (q, ch, g), (x, _, _) in zip(qs, READ12[n]):
        c12 = credit12(scores12(x), g)
        earlier = solve19(q, list(ch)) or tool23(q, list(ch)) or tool24(q, list(ch))
        base = float(earlier[1] == g) if earlier else c12
        got = None if earlier else tool25(q, list(ch))
        if got:
            ans += 1
            agree += got[1] == g
            TALLY25.setdefault(got[0], [0, 0])
            TALLY25[got[0]][0] += 1
            TALLY25[got[0]][1] += got[1] == g
            if got[1] != g and n != "ARC train":
                WRONG25.append((n, q[:110], ch[got[1]], ch[g]))
        final = earlier or got
        if final:
            ex[0] += 1
            ex[1] += final[1] == g
        rows.append(float(got[1] == g) if got else base)
        prev.append(base)
    RES25[n] = dict(answered=ans, agree=agree, whole=100 * np.mean(rows), was=100 * np.mean(prev))
    EXACT25[n] = (ex[0], ex[1], len(qs))
    print(f"  {n:<26}{ans:>10}{agree:>7}{RES25[n]['whole']:>10.2f}{RES25[n]['was']:>9.2f}")
print("  (with III = Cells 19, 23, 24, 25 where they answer, else Cell 12; without = the same minus Cell 25)")
print("\n  by tool (all sets: answered, agree with the key)")
for name, _ in TOOLS25:
    if name in TALLY25:
        print(f"    {name:<11}{TALLY25[name][0]:>5}{TALLY25[name][1]:>6}")
print("\n  EXACT BASE (Cells 19, 23, 24, 25 together: answered, agree, share of the set)")
for n, (a, r, t) in EXACT25.items():
    print(f"    {n:<26}{a:>5}{r:>6}{100 * a / t:>8.1f}%")
print("\n  DISAGREEMENTS with the answer key (practice sets)")
for n, q, said, key in WRONG25:
    print(f"    [{n}] {q}\n      tool: {said}; key: {key}")

print("\n  PREDICTIONS vs RESULTS")
LEDGER.result("B5-TD3", N25B)
LEDGER.result("B5-TD4", BAD25B)
LEDGER.result("B5-TD5", RES25["MMLU validation"]["answered"])
LEDGER.result("B5-TD6", RES25["MMLU validation"]["answered"] - RES25["MMLU validation"]["agree"])
LEDGER.result("B5-TD7", round(RES25["MMLU validation"]["whole"] - RES25["MMLU validation"]["was"], 2))
LEDGER.result("B5-TD8", RES25["MMLU dev"]["answered"])
LEDGER.result("B5-T38", round(time.time() - T25B))
if "sync" in globals():
    sync("Cell 25B")
print(f"\n  ({time.time() - T25B:.0f}s)")
print("  LAST LINE OF MC AI BRAIN 5 CELL 25B")


MC AI BRAIN 5 - CELL 25B: TOOLBOX III (PART 2) - DIVISORS, BASES, COUNTING
  self-checks: 4 of 4 (odd divisors, base 3, Stirling numbers, letters of BANANA)

  MATH train (where the tools were built)
    tool        answered  agree
    REMAINDER         46     46
    GCD-LCM           36     36
    CHOOSE            26     26
    BALLS             12     12
    BASE              18     18
    ARRANGE           10     10
    PICK               5      5
    INVERSE            5      5
    DIVISORS          42     42
    SEAT               3      3

                              answered  agree  with III  without
  ARC train                          0      0     43.05    43.05
  ARC-Easy validation                0      0     51.05    51.05
  ARC-Challenge validation           0      0     30.57    30.57
  OpenBookQA validation              0      0     24.82    24.82
  MMLU validation                    0      0     33.65    33.65
  MMLU dev                           0      0     32.43  

In [36]:
# =====================================================================
# MC AI BRAIN 5 - CELL 25C: WHY TOOLBOX III ANSWERED NO MMLU QUESTION (A LOOK)
# Needs Cells 12A, 12B, 19, 23, 24, 25A and 25B. Prints the MMLU dev and
# validation (practice) questions about remainders, divisors, bases or
# counting that Cells 19, 23 and 24 leave open, and for each one the first
# place Toolbox III stopped:
#   NO FORM     no tool's question form matched the wording
#   NO VALUE    a form matched but the exact evaluator could not read the maths
#   UNUSED      a value was found but numbers of the question were not read
#   NO CHOICE   a value was found but no choice states it
#   TWO CHOICES a value was found and more than one choice states it
# (a tool whose one agreeing choice was found would have answered: such
#  questions are not open and are skipped)
# Nothing here is scored and no prediction is made; the test split is not read.
# =====================================================================
import re
import time
from collections import Counter

print("=" * 64); print("MC AI BRAIN 5 - CELL 25C: WHY TOOLBOX III ANSWERED NO MMLU QUESTION"); print("=" * 64)
T25C = time.time()
for need in ("TOOLS25", "FORMAT25", "same25", "tool25", "SETS12", "solve19", "tool23", "tool24", "numbers23"):
    if need not in globals():
        raise RuntimeError(f"run Cells 12A, 12B, 19, 23, 24, 25A and 25B first ({need} is missing)")
TOPIC25 = re.compile(r"remainder|divided by|\bmod\b|modulo|congruen|\\equiv|greatest common|least common|\bgcd\b|"
                     r"\bgcf\b|\blcm\b|divisor|factors?\b|prime|\bbase\b|digit|inverse|arrange|\bways\b|choose|"
                     r"combination|permutation|committee|factorial|\d!", re.I)


def why25(q: str, ch: list):
    """(tool or '-', stop reason, detail) for the first place Toolbox III stopped"""
    plain = FORMAT25.sub(" ", q)
    best = ("-", "NO FORM", "")
    rank = {"NO FORM": 0, "NO VALUE": 1, "UNUSED": 2, "NO CHOICE": 3, "TWO CHOICES": 4}
    for name, tool in TOOLS25:
        try:
            got = tool(q)
        except (ValueError, ZeroDivisionError, AttributeError, OverflowError, RecursionError, IndexError) as err:
            got = (None, [f"error: {type(err).__name__}"])
        if not got:
            continue
        v, pieces = got
        if v is None:
            here = (name, "NO VALUE", pieces[0][:60] if pieces else "")
        else:
            used = Counter()
            for p in pieces:
                used += Counter(numbers23(p))
            left = Counter(numbers23(plain)) - used
            hits = [j for j, c in enumerate(ch) if same25(v, c)]
            if left:
                here = (name, "UNUSED", f"value {v}; numbers not read: {sorted(map(str, left.elements()))}")
            elif not hits:
                here = (name, "NO CHOICE", f"value {v}")
            elif len(hits) > 1:
                here = (name, "TWO CHOICES", f"value {v}")
            else:
                continue
        if rank[here[1]] > rank[best[1]]:
            best = here
    return best


OPEN25, SHOWN25 = Counter(), 0
for n in ("MMLU dev", "MMLU validation"):
    for i, (q, ch, g) in enumerate(SETS12[n]):
        if not TOPIC25.search(q) or solve19(q, list(ch)) or tool23(q, list(ch)) or tool24(q, list(ch)) or \
                tool25(q, list(ch)):
            continue
        name, why, detail = why25(q, list(ch))
        OPEN25[why] += 1
        if SHOWN25 < 90:
            SHOWN25 += 1
            subj = SUBJ12[i] if n == "MMLU validation" and "SUBJ12" in globals() else n
            text = re.sub(r"\s+", " ", q)[:170]
            print(f"\n[{subj}] {why} {name} {detail}\n  {text}\n  " +
                  " || ".join(("*" if k == g else "") + re.sub(r"\s+", " ", str(c))[:30] for k, c in enumerate(ch)))
print(f"\n  open questions on these topics, by where Toolbox III stopped (dev and validation; {SHOWN25} shown)")
for why, k in OPEN25.most_common():
    print(f"    {why:<12}{k:>5}")
print("  (* marks the key's answer)")
if "sync" in globals():
    sync("Cell 25C")
print(f"\n  ({time.time() - T25C:.0f}s)")
print("  LAST LINE OF MC AI BRAIN 5 CELL 25C")


MC AI BRAIN 5 - CELL 25C: WHY TOOLBOX III ANSWERED NO MMLU QUESTION

[MMLU dev] NO FORM - 
  Statement 1 | If aH is an element of a factor group, then |aH| divides |a|. Statement 2 | If H and K are subgroups of G then HK is a subgroup of G.
  True, True || *False, False || True, False || False, True

[MMLU dev] NO FORM - 
  A solution contains 2.00 mole of acetic acid, CH3COOH, and 1.00 mole of calcium acetate, Ca(CH3COO)2. The solution is able to resist the addition of a small amount of str
  0.500 mole || 1.00 mole || *2.00 mole || 3.00 mole

[MMLU dev] NO FORM - 
  This question refers to the following information. In Russia there was nothing going on well, and [Souvarine] was in despair over the news he had received. His old compan
  Capitalist || Scientific || *Communist || Existentialist

[MMLU dev] NO FORM - 
  This question refers to the following information. "Society in every state is a blessing, but government even in its best state is but a necessary evil; in its worst stat

In [37]:
# =====================================================================
# MC AI BRAIN 5 - CELL 26A: A STORY READER BY UNITS
# Needs Cell 22 (its GSM8K, ASDiv and SVAMP files). Same rule as the
# toolboxes: exact, and silent unless sure.
# STORY  a word problem is answered only when every sentence is of a form
#   the reader takes (no word that hides an operation: gave, rest, each, a
#   day, half, budget ...) and the question asks for a total or a cost.
#   Each number is read with its unit (the head noun after it: "3 feet of
#   wood" -> foot). A rate ($7 per foot, $23 each, $3.00 for students) is
#   multiplied by the one count in its unit; "N more/fewer X than ..." adds
#   to or takes from the one earlier amount it names (by name, by "the first
#   box", or by its describing words); the answer is the sum of the amounts
#   in the asked unit, and every number of the story must be in it (number
#   words such as "two boxes" excepted). The idea of units deciding which
#   numbers combine is Roy & Roth's unit dependency graph (2017).
# BUILT ON GSM8K train, ASDiv (only Brain 2's DEV part: in file order, every
#   5th problem is Brain 2's TEST and is skipped) and SVAMP; GSM8K test is
#   not read. Four MMLU validation stories were seen while building, so the
#   MMLU numbers (Cell 26B) are CHECKS.
# PREDICTIONS (written before running)
#   B5-TE1 GSM8K train, ASDiv dev and SVAMP stories the reader answers
#          (check): 20-35
#   B5-TE2 of those, answers that disagree with the dataset (check): 0
#   B5-T39 Cell 26A seconds: <= 120
# =====================================================================
import json
import re
import time
import xml.etree.ElementTree as ET
from fractions import Fraction

from mcai.core import at_most, between

print("=" * 64); print("MC AI BRAIN 5 - CELL 26A: A STORY READER BY UNITS"); print("=" * 64)
T26 = time.time()
if "GOT22" not in globals():
    raise RuntimeError("run Cell 22 first (GOT22, its GSM8K, ASDiv and SVAMP files, is missing)")
for key, claim, rule in (("B5-TE1", "GSM8K train, ASDiv dev and SVAMP stories answered (check)", between(20, 35)),
                         ("B5-TE2", "of those, answers that disagree with the dataset (check)", between(0, 0)),
                         ("B5-T39", "Cell 26A seconds", at_most(120))):
    LEDGER.predict(key, claim, rule)

# ---------------- the story reader ----------------
UNITS26 = ("zero one two three four five six seven eight nine ten eleven twelve thirteen fourteen fifteen sixteen "
           "seventeen eighteen nineteen").split()
TENS26 = dict(twenty=20, thirty=30, forty=40, fifty=50, sixty=60, seventy=70, eighty=80, ninety=90)
WORDS26 = {w: i for i, w in enumerate(UNITS26)}
WORDS26.update(TENS26)
for t, tv in TENS26.items():
    WORDS26.update({f"{t}-{u}": tv + i for i, u in enumerate(UNITS26[1:10], 1)})
WORDS26["hundred"] = 100
IRREG26 = dict(feet="foot", people="person", children="child", teeth="tooth", men="man", women="woman", mice="mouse",
               geese="goose", dollars="$", dollar="$", knives="knife", leaves="leaf", loaves="loaf", shelves="shelf",
               wolves="wolf", inches="inch", glasses="glass", has="have", does="do")
SKIP26 = set("a an the of to in on at for and or with his her their its my our your this that these those is are was "
             "were be been have had do did will would can could it they he she we you i which who what how total all "
             "than as more fewer less".split())
NUM26 = r"(?<![-\w])(\d+(?:,\d{3})*(?:\.\d+)?|" + "|".join(sorted(WORDS26, key=len, reverse=True)) + r")(?![-\w])"
NUMWORD26 = re.compile(r"\b(?:" + "|".join(list(UNITS26) + list(TENS26)) + r"|hundred|thousand|million|billion)\b",
                       re.I)
WHOLE26 = r"(?<![-\w])(?:" + "|".join(sorted(WORDS26, key=len, reverse=True)) + r")(?![-\w])"
CAPS26 = set("The There A An In On At He She They It His Her Their We I You My Our Your This That These Those How "
             "What Each Every If When".split())
HIDDEN26 = re.compile(
    r"\b(?:gave|give|gives|giving|away|rest|left|remains?|remaining|budget|change|kept|keeps?|sold|sells?|lost|"
    r"loses?|ate|eats?|took|takes?|shared?|split|between|owes?|twice|thrice|double|triple|half|halves|thirds?|"
    r"quarters?|fourths?|fifths?|tenths?|dozens?|percent|days?|daily|weeks?|weekly|months?|years?|yearly|hours?|"
    r"minutes?|seconds|morning|afternoon|evening|night|monday|tuesday|wednesday|thursday|friday|saturday|sunday|"
    r"every|per|each|times|instead|except|but|only|if|when|after|before|then|later|until|returned?|back|out|off|"
    r"used|uses?|free|discount|tip|saved?|savings|earns?|earned|paid|pays?|spends?|spent|cheaper|expensive|older|"
    r"younger|taller|shorter|longer|wider|heavier|lighter|faster|slower|ago|rows?|columns?|groups?|packs?|"
    r"packages?|bags?|cases?|piles?|stacks?|sets?|teams?|batch\w*|same|equal|equally|evenly|another|other|others|"
    r"again|extra|remainder|some|several|most|least|both|neither|either|none|not|no|without|whole|full|empty|by|"
    r"square|area|perimeter|long|wide|tall|high|deep|from|into|onto|among|divid\w*|multipl\w*|subtract\w*|ratio|"
    r"rate|speed|average|mean|age|aged|old|cents?|dimes?|nickels?|pennies|penny|worth|win|won|lose)\b|"
    r"%|\d/\d|\b(?:that|which|who) (?:hold|holds|have|has|contain|contains)\b|\b(?:bills?|containing|holding)\b|"
    r"\d+ [a-z]+ of (?:\d|one|two|three|four|five|six|seven|eight|nine|ten)\b", re.I)
TOTAL26 = re.compile(r"\b(?:total|in all|altogether|combined|together)\b|"
                     r"\bhow much (?:will|did|does|do|would) .{0,60}\bcost\b", re.I)
RATEFORM26 = re.compile(r"\$[\d.,]+ (?:per|a|an|for each|for every|every) [a-z]+(?: of [a-z]+)?|"
                        r"\b[Ee]ach [a-z]+(?: of [a-z]+)? (?:costs?|is|sells for|was) \$[\d.,]+|"
                        r"\b(?:that|which) (?:costs?|sells? for) \$[\d.,]+ each\b|"
                        r",? not including (?:the )?tax,?|\bfor a group of\b")


def lemma26(w: str) -> str:
    w = w.lower().strip(".,;:!?'\"()")
    if w in IRREG26:
        return IRREG26[w]
    if len(w) > 4 and w.endswith("ies"):
        return w[:-3] + "y"
    if len(w) > 3 and re.search(r"(?:ches|shes|sses|xes|zes)$", w):
        return w[:-2]
    if len(w) > 3 and w.endswith("s") and not w.endswith(("ss", "us", "is")):
        return w[:-1]
    return w


def val26(t: str) -> Fraction:
    t = t.lower()
    return Fraction(WORDS26[t]) if t in WORDS26 else Fraction(t.replace(",", ""))


def nouns26(text: str) -> set:
    """the content words of the few words after a number: the quantity's unit, loosely"""
    out = set()
    for w in re.findall(r"[A-Za-z][A-Za-z-]*", text)[:4]:
        lw = lemma26(w)
        if lw in ("than", "as", "per", "each", "which", "that", "who", "and", "for", "but", "to", "in", "on", "at"):
            break
        if lw not in SKIP26:
            out.add(lw)
    return out


def head26(text: str):
    """the head noun of the words after a number: 'feet of wood' -> foot, 'sticker book' -> book"""
    words = []
    for w in re.findall(r"[A-Za-z][A-Za-z-]*", text)[:4]:
        lw = lemma26(w)
        if lw in ("of", "than", "as", "per", "each", "which", "that", "who", "and", "for", "but", "to", "in", "on",
                  "at"):
            break
        if lw not in SKIP26:
            words.append(lw)
    return words[-1] if words else None


class Q26:
    """one quantity: value, unit words, for a rate the unit words it is counted per, and whose it is"""
    def __init__(self, v, unit, per=None, word=False, subj=None):
        self.v, self.unit, self.per, self.word, self.subj = v, set(unit), set(per) if per else None, word, subj
        self.head = None


def subject26(s: str, last):
    """who or what a sentence is about: a name, 'first box' style, or the previous subject for a pronoun"""
    first = re.match(r"\s*(?:In [^,]+, )?([A-Z][a-z]+)", s)
    if not first:
        return None
    w = first.group(1)
    if w in ("He", "She", "They", "It", "His", "Her"):
        return last
    if w in CAPS26:
        o = re.search(r"\bThe (first|second|third|fourth|fifth) ([a-z]+)", s)
        return (o.group(1) + " " + lemma26(o.group(2))) if o else None
    return w


def compare26(qs: list, m, subj):
    """the one earlier amount a 'N more/fewer X than ...' phrase refers to, or None"""
    own = nouns26(m.group(3)) if m.group(3).strip() else set()
    ref = m.group(4)
    words = {lemma26(w) for w in re.findall(r"[A-Za-z]+", ref)} - SKIP26 - {"do", "have"}
    names = set(re.findall(r"\b[A-Z][a-z]+", ref)) - CAPS26
    o = re.search(r"\bthe (first|second|third|fourth|fifth) ([a-z]+)", ref)
    if o:
        names = {o.group(1) + " " + lemma26(o.group(2))}
    cands = []
    for x in qs:
        if x.per is not None:
            continue
        if names:
            ok = x.subj in names and subj not in names
        else:
            ok = bool(words) and words <= x.unit and bool(own) and own != x.unit and bool(own & x.unit)
        if ok:
            cands.append(x)
    if len(cands) != 1:
        return None
    unit = own if own and own & cands[0].unit else cands[0].unit
    sign = 1 if m.group(2) == "more" else -1
    got = Q26(cands[0].v + sign * val26(m.group(1)), unit, subj=subj)
    got.head = head26(m.group(3)) if m.group(3).strip() else cands[0].head
    return got


def quantities26(story: str):
    """the quantities of the story, or None for a form this reader does not take"""
    qs, last = [], None
    for sent in re.split(r"(?<=[.!;])\s+", story.strip()):
        s = " " + sent.strip() + " "
        subj = subject26(s, last)
        last = subj
        taken = []

        def free(m):
            return not any(a <= m.start() < b for a, b in taken)
        for m in re.finditer(r"\$" + NUM26 + r" (?:per|a|an|for each|for every|every) ([a-z]+(?: of [a-z]+)?)", s):
            qs.append(Q26(val26(m.group(1)), {"$"}, {head26(m.group(2))}, subj=subj))
            taken.append(m.span())
        for m in re.finditer(r"\$" + NUM26 + r" for (?:the |all )?([a-z]+s)\b", s):
            if free(m):
                qs.append(Q26(val26(m.group(1)), {"$"}, {lemma26(m.group(2))}, subj=subj))
                taken.append(m.span())
        for m in re.finditer(r"\b[Ee]ach ([a-z]+(?: of [a-z]+)?) (?:costs?|is|sells for|was) \$" + NUM26, s):
            qs.append(Q26(val26(m.group(2)), {"$"}, {head26(m.group(1))}, subj=subj))
            taken.append(m.span())
        for m in re.finditer(r"\b" + NUM26 + r" ([a-z]+(?: [a-z]+)?) (?:that|which) (?:costs?|sells? for) \$" + NUM26 +
                             r" each\b", s):
            u = nouns26(m.group(2))
            qs.append(Q26(val26(m.group(1)), u, word=not m.group(1)[0].isdigit(), subj=subj))
            qs[-1].head = head26(m.group(2))
            qs.append(Q26(val26(m.group(3)), {"$"}, {head26(m.group(2))}, subj=subj))
            taken.append(m.span())
        for m in re.finditer(r"\b" + NUM26 + r" (more|fewer|less) ((?:[a-z]+ ){0,3})than ([^,.;?]+)", s):
            if free(m):
                got = compare26(qs, m, subj)
                if got is None:
                    return None
                qs.append(got)
                taken.append(m.span())
        for m in re.finditer(r"\$" + NUM26 + r"\b", s):
            if free(m):
                qs.append(Q26(val26(m.group(1)), {"$"}, subj=subj))
                qs[-1].head = "$"
                taken.append(m.span())
        for m in re.finditer(r"\b(?:one|a|an)\b", s, re.I):
            taken.append(m.span())
        for m in re.finditer(r"(?<![\w$.,])" + NUM26 + r"\b(?![.,]\d)", s, re.I):
            if not free(m):
                continue
            after = s[m.end():]
            u = {"$"} if re.match(r"\s*dollars?\b", after) else nouns26(re.sub(r"\bof\b", " ", after))
            if not u:
                return None
            qs.append(Q26(val26(m.group(1)), u, word=not m.group(1)[0].isdigit(), subj=subj))
            qs[-1].head = "$" if "$" in u else head26(after)
            taken.append(m.span())
    return qs


def ask26(question: str):
    """the unit words the question asks the total of, or None"""
    s = question.strip()
    if re.search(r"\bhow much (?:will|did|does|do|would) .{0,60}\bcost\b|\btotal cost\b|\bhow much money\b", s, re.I):
        return "$"
    m = re.search(r"\b(?:how many|total number of|number of) ([a-z]+(?: [a-z]+)?)", s, re.I)
    if not m:
        return None
    words = [w for w in m.group(1).split() if lemma26(w) not in SKIP26 and lemma26(w) != "total"]
    plural = [w for w in words if w.endswith("s") or w in IRREG26]
    return lemma26((plural or words or [""])[-1]) or None


def plain26(sent: str) -> bool:
    """no word of the sentence (outside the rate forms the reader takes) hides an operation"""
    return not HIDDEN26.search(RATEFORM26.sub(" ", sent))


def story26(text: str):
    """(exact total, unit words) for a story of amounts, rates times counts and comparisons; None otherwise"""
    parts = re.split(r"(?<=[.!?])\s+(?=[A-Z])", re.sub(r"\s+", " ", text.strip()))
    if len(parts) < 2 or not parts[-1].endswith("?") or any(p.endswith("?") for p in parts[:-1]):
        return None
    if not all(plain26(p) for p in parts) or not TOTAL26.search(parts[-1]):
        return None
    asked = ask26(parts[-1])
    if not asked:
        return None
    people = {w for w in re.findall(r"\b[A-Z][a-z]+", " ".join(parts[:-1])) if w not in CAPS26}
    named = {w for w in re.findall(r"\b[A-Z][a-z]+", parts[-1][1:]) if w not in CAPS26}
    if named and named != people:
        return None
    extra = ""
    if re.search(r"\d", parts[-1]):
        m = re.search(r"\bfor (?:a group of )?(" + NUM26 + r" [a-z]+(?:,? (?:and )?" + NUM26 + r" [a-z]+)*)", parts[-1])
        if not m or re.search(r"\d", parts[-1].replace(m.group(1), "")):
            return None
        extra = " There are " + m.group(1) + "."
    story = " ".join(parts[:-1]) + extra
    if len(NUMWORD26.findall(story)) != len(re.findall(WHOLE26, story, re.I)):
        return None
    qs = quantities26(story)
    if not qs:
        return None
    counts = [x for x in qs if not x.per]
    used, out = set(), []
    for r in (x for x in qs if x.per):
        hit = [i for i, c in enumerate(counts) if i not in used and c.head in r.per and c.head != "$"]
        if len(hit) != 1:
            return None
        used.add(hit[0])
        out.append(Q26(r.v * counts[hit[0]].v, {"$"}))
        out[-1].head = "$"
    out += [c for i, c in enumerate(counts) if i not in used]
    keep = [x for x in out if x.head == asked]
    if not keep or any(not x.word for x in out if x.head != asked):
        return None
    marks = {lemma26(w) for w in re.findall(r"[a-z]+", parts[-1].lower())} - {asked} - SKIP26
    if any(any(w in x.unit for x in keep) and not all(w in x.unit for x in keep) for w in marks):
        return None
    return sum((x.v for x in keep), Fraction(0)), asked


# ---------------- built on: GSM8K train, ASDiv dev, SVAMP ----------------
with open(GOT22["train.jsonl"]) as f:
    DESIGN26 = [("GSM8K train", r["question"], r["answer"].split("####")[-1]) for r in map(json.loads, f)]
for i, p in enumerate(ET.parse(GOT22["ASDiv.xml"]).getroot().iter("Problem")):
    if i % 5 != 4:
        DESIGN26.append(("ASDiv dev", (p.findtext("Body") or "").strip() + " " + (p.findtext("Question") or "").strip(),
                         p.findtext("Answer") or ""))
with open(GOT22["SVAMP.json"]) as f:
    DESIGN26 += [("SVAMP", p["Body"].strip() + " " + p["Question"].strip(), str(p["Answer"])) for p in json.load(f)]
BY26, BAD26 = {}, []
for src, q, a in DESIGN26:
    try:
        got = story26(q)
    except (ValueError, ZeroDivisionError, KeyError):
        got = None
    if not got:
        continue
    m = re.match(r"\s*(-?[\d,]*\.?\d+)", a)
    ok = m is not None and got[0] == Fraction(m.group(1).replace(",", ""))
    BY26.setdefault(src, [0, 0])
    BY26[src][0] += 1
    BY26[src][1] += ok
    if not ok:
        BAD26.append((src, q[:110], got[0], a))
print(f"\n  STORY on its design sets\n    {'':<14}{'answered':>9}{'agree':>7}")
for src, (n, a) in BY26.items():
    print(f"    {src:<14}{n:>9}{a:>7}")
for src, q, said, a in BAD26:
    print(f"    DISAGREES [{src}] {q}\n      reader: {said}; dataset: {a}")
print("\n  PREDICTIONS vs RESULTS")
LEDGER.result("B5-TE1", sum(k for k, _ in BY26.values()))
LEDGER.result("B5-TE2", len(BAD26))
LEDGER.result("B5-T39", round(time.time() - T26))
if "sync" in globals():
    sync("Cell 26A")
print(f"\n  ({time.time() - T26:.0f}s)")
print("  LAST LINE OF MC AI BRAIN 5 CELL 26A")


MC AI BRAIN 5 - CELL 26A: A STORY READER BY UNITS

  STORY on its design sets
                   answered  agree
    GSM8K train           9      9
    ASDiv dev            15     15
    SVAMP                 2      2

  PREDICTIONS vs RESULTS
    B5-TE1 GSM8K train, ASDiv dev and SVAMP stories answered (check): 26 (predicted between 20 and 35) -> HELD
    B5-TE2 of those, answers that disagree with the dataset (check): 0 (predicted between 0 and 0) -> HELD
    B5-T39 Cell 26A seconds: 1 (predicted <= 120) -> HELD
  sync: pushed 5 files to kaggle/ on main as commit 38c66cbc9560; GitHub confirms it (710 entries, head b6f26d9bf9997a1d) (3s)

  (4s)
  LAST LINE OF MC AI BRAIN 5 CELL 26A


In [38]:
# =====================================================================
# MC AI BRAIN 5 - CELL 26B: ONE-LETTER EQUATIONS, AND THE SCORE WITH STORIES
# Needs Cell 26A and Cells 12A, 12B, 19, 23, 24, 25A, 25B.
# SOLVE  "Solve (for v): <equation>" with one letter, linear in it: the
#   left minus the right side at v = 0, 1, 2 (exact) gives the root. Built
#   on MATH train's "Solve" problems. Three MMLU validation equations were
#   seen while building, so the MMLU numbers below are CHECKS.
# PREDICTIONS (written before running)
#   B5-TE3 MMLU validation questions Cells 26A + 26B answer (check): 5-9
#   B5-TE4 of those, answers that disagree with the key (check): 1 (the
#          key gives 69.6 for -2k = -34.8, whose root is 17.4)
#   B5-TE5 MMLU validation with Cell 26, minus without it, points: 0.2-0.6
#   B5-TE6 MMLU dev questions Cell 26 answers: 0-3
#   B5-T40 Cell 26B seconds: <= 120
# =====================================================================
import re
import time
from fractions import Fraction

import numpy as np
from mcai.core import at_most, between

print("=" * 64); print("MC AI BRAIN 5 - CELL 26B: ONE-LETTER EQUATIONS, AND THE SCORE WITH STORIES"); print("=" * 64)
T26B = time.time()
for need in ("story26", "SETS12", "READ12", "scores12", "credit12", "solve19", "tool23", "tool24", "tool25",
             "expr19", "number19", "DASH19", "tex25", "int25", "key25", "MATHROWS25"):
    if need not in globals():
        raise RuntimeError(f"run Cells 12A, 12B, 19, 23, 24, 25A, 25B and 26A first ({need} is missing)")
for key, claim, rule in (("B5-TE3", "MMLU validation questions Cell 26 answers (check)", between(5, 9)),
                         ("B5-TE4", "of those, answers that disagree with the key (check)", between(1, 1)),
                         ("B5-TE5", "MMLU validation with Cell 26 minus without it, points", between(0.2, 0.6)),
                         ("B5-TE6", "MMLU dev questions Cell 26 answers", between(0, 3)),
                         ("B5-T40", "Cell 26B seconds", at_most(120))):
    LEDGER.predict(key, claim, rule)


# ---------------- one-letter linear equations ----------------
def solve26(q: str, ch: list):
    """(index of the one choice equal to the root, root) for 'Solve (for v): <linear equation in v>', else None"""
    m = re.fullmatch(r"\s*Solve(?: for \$?([a-z])\$?)?\s*[.:]?\s*\$?([^$]+?)\$?\s*\.?\s*", q)
    if not m or m.group(2).count("=") != 1:
        return None
    eq = "".join(DASH19.get(c, c) for c in tex25(m.group(2)))
    letters = set(re.findall(r"(?<![A-Za-z])[a-z](?![A-Za-z])", eq))
    if len(letters) != 1 or (m.group(1) and letters != {m.group(1)}):
        return None
    v = letters.pop()
    left, right = eq.split("=")
    f = [expr19(f"({left})-({right})", {v: Fraction(t)}) for t in (0, 1, 2)]
    if not all(isinstance(x, Fraction) and not isinstance(x, bool) for x in f):
        return None
    a = f[1] - f[0]
    if a == 0 or f[2] - f[1] != a:
        return None
    root = -f[0] / a
    vals = []
    for c in ch:
        t = "".join(DASH19.get(k, k) for k in str(c)).strip()
        x = number19(t) if re.fullmatch(r"[-+]?\d+(?:\.\d+)?(?:\s*/\s*\d+)?", t) else int25(t)
        vals.append(x)
    hit = [j for j, x in enumerate(vals) if x is not None and x == root]
    return (hit[0], root) if len(hit) == 1 else None


def tool26(q: str, ch: list):
    """(tool name, index of the one agreeing choice) or None"""
    got = solve26(q, ch)
    if got:
        return "SOLVE", got[0]
    try:
        got = story26(q)
    except (ValueError, ZeroDivisionError, KeyError):
        got = None
    if not got:
        return None
    hit = [j for j, c in enumerate(ch) if number19(c) == got[0]]
    return ("STORY", hit[0]) if len(hit) == 1 else None


S26 = [0, 0]
for r in MATHROWS25:
    if r["problem"].lstrip().startswith("Solve"):
        k = key25(r["answer"])
        if isinstance(k, Fraction) or isinstance(k, int):
            got = solve26(r["problem"], [r["answer"].replace("$", ""), "999983", "-777773", "123457"])
            if got:
                S26[0] += 1
                S26[1] += got[0] == 0
print(f"  SOLVE on MATH train: {S26[0]} answered, {S26[1]} agree")

# ---------------- scored on the practice sets ----------------
RES26, WRONG26, TALLY26, EXACT26 = {}, [], {}, {}
print(f"\n  {'':<26}{'answered':>10}{'agree':>7}{'with 26':>9}{'without':>9}")
for n, qs in SETS12.items():
    rows, prev, ans, agree, ex = [], [], 0, 0, [0, 0]
    for (q, ch, g), (x, _, _) in zip(qs, READ12[n]):
        earlier = solve19(q, list(ch)) or tool23(q, list(ch)) or tool24(q, list(ch)) or tool25(q, list(ch))
        base = float(earlier[1] == g) if earlier else credit12(scores12(x), g)
        got = None if earlier else tool26(q, list(ch))
        if got:
            ans += 1
            agree += got[1] == g
            TALLY26.setdefault(got[0], [0, 0])
            TALLY26[got[0]][0] += 1
            TALLY26[got[0]][1] += got[1] == g
            if got[1] != g and n != "ARC train":
                WRONG26.append((n, q[:110], ch[got[1]], ch[g]))
        final = earlier or got
        if final:
            ex[0] += 1
            ex[1] += final[1] == g
        rows.append(float(got[1] == g) if got else base)
        prev.append(base)
    RES26[n] = dict(answered=ans, agree=agree, whole=100 * np.mean(rows), was=100 * np.mean(prev))
    EXACT26[n] = (ex[0], ex[1], len(qs))
    print(f"  {n:<26}{ans:>10}{agree:>7}{RES26[n]['whole']:>9.2f}{RES26[n]['was']:>9.2f}")
print("  (with 26 = Cells 19, 23, 24, 25, 26 where they answer, else Cell 12; without = the same minus Cell 26)")
print("\n  by tool (all sets: answered, agree with the key)")
for name, (k, a) in TALLY26.items():
    print(f"    {name:<8}{k:>5}{a:>6}")
print("\n  EXACT BASE (Cells 19, 23, 24, 25, 26 together: answered, agree, share of the set)")
for n, (a, r, t) in EXACT26.items():
    print(f"    {n:<26}{a:>5}{r:>6}{100 * a / t:>8.1f}%")
print("\n  DISAGREEMENTS with the answer key (practice sets)")
for n, q, said, key in WRONG26:
    print(f"    [{n}] {q}\n      tool: {said}; key: {key}")

print("\n  PREDICTIONS vs RESULTS")
LEDGER.result("B5-TE3", RES26["MMLU validation"]["answered"])
LEDGER.result("B5-TE4", RES26["MMLU validation"]["answered"] - RES26["MMLU validation"]["agree"])
LEDGER.result("B5-TE5", round(RES26["MMLU validation"]["whole"] - RES26["MMLU validation"]["was"], 2))
LEDGER.result("B5-TE6", RES26["MMLU dev"]["answered"])
LEDGER.result("B5-T40", round(time.time() - T26B))
if "sync" in globals():
    sync("Cell 26B")
print(f"\n  ({time.time() - T26B:.0f}s)")
print("  LAST LINE OF MC AI BRAIN 5 CELL 26B")


MC AI BRAIN 5 - CELL 26B: ONE-LETTER EQUATIONS, AND THE SCORE WITH STORIES
  SOLVE on MATH train: 8 answered, 8 agree

                              answered  agree  with 26  without
  ARC train                          0      0    43.05    43.05
  ARC-Easy validation                0      0    51.05    51.05
  ARC-Challenge validation           0      0    30.57    30.57
  OpenBookQA validation              0      0    24.82    24.82
  MMLU validation                    7      6    33.93    33.65
  MMLU dev                           0      0    32.43    32.43
  MMLU-Pro validation                0      0    18.53    18.53
  (with 26 = Cells 19, 23, 24, 25, 26 where they answer, else Cell 12; without = the same minus Cell 26)

  by tool (all sets: answered, agree with the key)
    STORY       4     4
    SOLVE       3     2

  EXACT BASE (Cells 19, 23, 24, 25, 26 together: answered, agree, share of the set)
    ARC train                     1     1     0.0%
    ARC-Easy validation     

In [39]:
# =====================================================================
# MC AI BRAIN 5 - CELL 27: STATISTICS BY FORMULA
# Needs Cells 12A, 12B, 19, 23, 24, 25A, 25B, 26A and 26B. Same rule as the
# toolboxes: a fixed reading, an exact (or exactly rounded) formula, every
# number of the question read, exactly one choice agreeing at the choice's
# own precision.
# TOOLS
#   NORMAL-PCT  the P-th percentile of a normal distribution against the
#               mean: z(P) x standard deviation, below or above
#   NORMAL-IQR  the quartiles of a normal distribution: mean +- z(0.75) sd
#   BAYES       P(condition | positive) from a base rate, a hit rate and a
#               false-alarm rate, each told apart by its own words
#   BINOMIAL    all n of n tries; at least k of n drawn with replacement
#               above a quartile (p = 1/4); k or more sixes in n fair tosses
#               (against interval choices); the least number of n fair
#               red/black calls significant at a level (exact tail sums)
#   R-SQUARED   the share of variation explained by a line: r squared
#   z(p) is Python's statistics.NormalDist().inv_cdf; binomial tails are
#   exact Fraction sums.
# BUILT ON the open MMLU practice questions of these kinds (read while
#   building, so B5-TF1..TF3 are CHECKS) and on textbook values, checked
#   below. No outside question set with answers was found for these kinds:
#   the OpenStax statistics exercises are free-form and multi-part.
# PREDICTIONS (written before running)
#   B5-TF1 MMLU validation questions Cell 27 answers (check): 8-11
#   B5-TF2 of those, answers that disagree with the key (check): 0
#   B5-TF3 MMLU validation with Cell 27, minus without it, points: 0.3-0.6
#   B5-TF4 MMLU dev questions Cell 27 answers: 0-2
#   B5-T41 Cell 27 seconds: <= 120
# =====================================================================
import math
import re
import time
from collections import Counter
from fractions import Fraction
from statistics import NormalDist

import numpy as np
from mcai.core import at_most, between

print("=" * 64); print("MC AI BRAIN 5 - CELL 27: STATISTICS BY FORMULA"); print("=" * 64)
T27 = time.time()
for need in ("tool26", "SETS12", "READ12", "scores12", "credit12", "solve19", "tool23", "tool24", "tool25",
             "numbers23"):
    if need not in globals():
        raise RuntimeError(f"run Cells 12A, 12B, 19, 23, 24, 25A, 25B, 26A and 26B first ({need} is missing)")
for key, claim, rule in (("B5-TF1", "MMLU validation questions Cell 27 answers (check)", between(8, 11)),
                         ("B5-TF2", "of those, answers that disagree with the key (check)", between(0, 0)),
                         ("B5-TF3", "MMLU validation with Cell 27 minus without it, points", between(0.3, 0.6)),
                         ("B5-TF4", "MMLU dev questions Cell 27 answers", between(0, 2)),
                         ("B5-T41", "Cell 27 seconds", at_most(120))):
    LEDGER.predict(key, claim, rule)

# ---------------- the tools ----------------
STD27 = NormalDist()
PCT27 = r"(\d+(?:\.\d+)?)\s*(?:%|percent)"
PROB27 = r"(\d+(?:\.\d+)?\s*(?:%|percent)|0?\.\d+)"
WORD27 = dict(one=1, two=2, three=3, four=4, five=5, six=6, seven=7, eight=8, nine=9, ten=10)


def p27(t: str) -> float:
    t = t.strip()
    return float(re.sub(r"\s*(?:%|percent)", "", t)) / 100 if re.search(r"%|percent", t) else float(t)


def n27(t: str) -> int:
    return WORD27[t.lower()] if t.lower() in WORD27 else int(t.replace(",", ""))


def num27(s: str):
    """the numbers of a choice: 652,000 - 628,000 -> [652000.0, 628000.0]"""
    return [float(x.replace(",", "")) for x in re.findall(r"-?\d[\d,]*(?:\.\d+)?|-?\.\d+", s)]


def near27(x: float, c: str, i: int = 0) -> bool:
    """x written to the precision of the i-th number of choice c"""
    m = re.findall(r"-?\d[\d,]*(?:\.\d+)?|-?\.\d+", c)
    if len(m) <= i:
        return False
    t = m[i].replace(",", "")
    if "." in t:
        k = len(t.split(".")[1])
        return round(x, k) == round(float(t), k)
    z = len(t) - len(t.rstrip("0")) if t.strip("-0") else 0
    step = 10 ** min(z, 6)
    return round(x / step) * step == float(t)


def normpct27(q: str, ch: list):
    """the value at the P-th percentile of a normal distribution, compared with the mean"""
    m = re.search(r"normal(?:ly distributed)?\b.*?standard deviation (?:of )?(\d+(?:\.\d+)?)\s*([a-z]+)?\.?", q)
    p = re.search(r"\b(?:in|at) the (\d+)(?:st|nd|rd|th) percentile\b.*?compare (?:to|with) the mean", q)
    if not m or not p:
        return None
    d = STD27.inv_cdf(int(p.group(1)) / 100) * float(m.group(1))
    side = "below" if d < 0 else "above"
    hit = [j for j, c in enumerate(ch) if side in c and near27(abs(d), c)]
    mean = re.search(r"\bmean (?:of )?(\d+(?:\.\d+)?)", q)
    used = [m.group(1), p.group(1)] + ([mean.group(1)] if mean else [])
    return (hit, used) if len(hit) == 1 else None


def iqr27(q: str, ch: list):
    """the quartiles of a normal distribution, mean +- 0.6745 standard deviations"""
    m = re.search(r"normal distribution with mean ([\d,.]+)\b.*?standard deviation ([\d,.]+)\b.*?"
                  r"What is the interquartile range", q)
    if not m:
        return None
    mu, sd = float(m.group(1).replace(",", "")), float(m.group(2).replace(",", ""))
    z = STD27.inv_cdf(0.75) * sd
    hit = [j for j, c in enumerate(ch) if len(num27(c)) == 2 and near27(mu + z, c, 0) and near27(mu - z, c, 1)]
    return (hit, [m.group(1), m.group(2)]) if len(hit) == 1 else None


def bayes27(q: str, ch: list):
    """P(condition | positive result) from a base rate, a hit rate and a false-alarm rate"""
    s = re.sub(r"\([^)]*\)", " ", q)
    if not re.search(r"\bprobability\b", s) or not re.search(r"\b(?:if|given)\b[^.?]*\b(?:positive|alarm)", s, re.I):
        return None
    roles, used = {}, []
    for sent in re.split(r"(?<=[.;?])\s+", s):
        test = re.search(r"\b(?:positive|detect\w*|alarm)\b", sent, re.I)
        found = list(re.finditer(PROB27, sent))
        for i, m in enumerate(found):
            window = sent[m.start():found[i + 1].start() if i + 1 < len(found) else len(sent)].lower()
            if test and re.search(r"\b(?:do not|does not|don't|doesn't|without|not have)\b", window):
                role = "false"
            elif test:
                role = "hit"
            elif re.search(r"\b(?:have|has|actually|are|is)\b", sent):
                role = "base"
            else:
                return None
            if role in roles:
                return None
            roles[role] = p27(m.group(1))
            used.append(m.group(1))
    if len(roles) != 3:
        return None
    post = roles["base"] * roles["hit"] / (roles["base"] * roles["hit"] + (1 - roles["base"]) * roles["false"])
    got = [j for j, c in enumerate(ch) if len(num27(c)) == 1 and near27(post, c)]
    return (got, used) if len(got) == 1 else None


def tail27(n: int, p: Fraction, k: int) -> Fraction:
    """P(X >= k) for X ~ Binomial(n, p), exact"""
    return sum((Fraction(math.comb(n, j)) * p ** j * (1 - p) ** (n - j) for j in range(k, n + 1)), Fraction(0))


def binom27(q: str, ch: list):
    """binomial chances: all n in n tries, at least k of n, k or more, and the least k significant at a level"""
    s = re.sub(r"\s+", " ", q)
    m = re.search(r"(?:average|probability|chance) is (0?\.\d+)\b.*?probability that (?:he|she|it|they) will get "
                  r"(\w+) \w+ in (\w+) [\w-]+\?", s)
    if m and m.group(2) == m.group(3):
        v = float(m.group(1)) ** n27(m.group(2))
        got = [j for j, c in enumerate(ch) if len(num27(c)) == 1 and near27(v, c)]
        return (got, [m.group(1)]) if len(got) == 1 else None
    m = re.search(r"the (third|first) quartile for .*? was (\d+)\. If (\w+) \w+ are selected at random \(with "
                  r"replacement\).*?probability that at least (\w+) of them had (more|fewer) than (\d+)\b", s)
    if m and m.group(2) == m.group(6) and (m.group(1) == "third") == (m.group(5) == "more"):
        v = float(tail27(n27(m.group(3)), Fraction(1, 4), n27(m.group(4))))
        got = [j for j, c in enumerate(ch) if len(num27(c)) == 1 and near27(v, c)]
        roster = re.search(r"\bFor the (\d+) \w+ on the roster\b", s)  # drawn with replacement: the size is not used
        return (got, [m.group(2), m.group(6)] + ([roster.group(1)] if roster else [])) if len(got) == 1 else None
    m = re.search(r"A fair die is tossed (\d+) times\. The probability that a (one|two|three|four|five|six) comes up "
                  r"on (\d+) or more of the tosses is", s)
    if m:
        v = float(tail27(int(m.group(1)), Fraction(1, 6), int(m.group(3))))
        got = []
        for j, c in enumerate(ch):
            b = re.fullmatch(r"\s*between (0?\.\d+) and (0?\.\d+)\s*", c)
            g = re.fullmatch(r"\s*greater than (0?\.\d+)\s*", c)
            if b and float(b.group(1)) < v < float(b.group(2)) or g and v > float(g.group(1)):
                got.append(j)
        return (got, [m.group(1), m.group(3)]) if len(got) == 1 else None
    m = re.search(r"\b(One hundred|one hundred|\d+) cards are dealt\b", s)
    lev = re.search(r"\b(\d+) percent level of significance\b", s)
    if m and lev and re.search(r"\bhalf the cards are red\b", s) and re.search(r"\breplaced after\b", s) and \
            re.search(r"\blowest number of cards\b.*\bcorrectly\b", s):
        n, a = (100 if not m.group(1).isdigit() else int(m.group(1))), Fraction(int(lev.group(1)), 100)
        k = next(k for k in range(n + 1) if tail27(n, Fraction(1, 2), k) <= a)
        got = [j for j, c in enumerate(ch) if num27(c) == [float(k)]]
        return (got, [lev.group(1)] + ([m.group(1)] if m.group(1).isdigit() else [])) if len(got) == 1 else None
    return None


def r2_27(q: str, ch: list):
    """the share of variation explained by a linear fit: r squared"""
    m = re.search(r"\br\s*=\s*(-?0?\.\d+)\b.*?percentage of (?:the )?variation in \w+ (?:is )?explained", q)
    if not m:
        return None
    v = 100 * float(m.group(1)) ** 2
    got = [j for j, c in enumerate(ch) if "%" in c and near27(v, c)]
    return (got, [m.group(1)]) if len(got) == 1 else None


TOOLS27 = (("NORMAL-PCT", normpct27), ("NORMAL-IQR", iqr27), ("BAYES", bayes27), ("BINOMIAL", binom27),
           ("R-SQUARED", r2_27))
LABEL27 = re.compile(r"\b\d+-(?:ounce|pound|gram|inch|foot|year|day|hour|minute)\b")


def tool27(q: str, ch: list):
    """(tool name, index of the one agreeing choice) when every number of the question was read"""
    for name, tool in TOOLS27:
        try:
            got = tool(q, list(map(str, ch)))
        except (ValueError, ZeroDivisionError, OverflowError, StopIteration):
            got = None
        if not got:
            continue
        hits, used = got
        left = Counter(numbers23(LABEL27.sub(" ", q))) - Counter(Fraction(u.replace(",", "").replace("%", "").strip())
                                                                 for u in used)
        if left:
            continue
        return name, hits[0]
    return None


# ---------------- textbook values ----------------
for what, got, want in (("z(0.28)", round(STD27.inv_cdf(0.28), 4), -0.5828),
                        ("z(0.75)", round(STD27.inv_cdf(0.75), 4), 0.6745),
                        ("P(X >= 59), n = 100, p = 1/2", round(float(tail27(100, Fraction(1, 2), 59)), 4), 0.0443),
                        ("P(X >= 58), n = 100, p = 1/2", round(float(tail27(100, Fraction(1, 2), 58)), 4), 0.0666),
                        ("Bayes: 1%, 90%, 9% -> 0.0917", (tool27(
                            "Suppose 1% of women have breast cancer. A mammogram gives a positive result for 90% of "
                            "women who have cancer and for 9% of women who do not have cancer. If a woman tests positive, "
                            "what is the probability she has cancer?", ["0.5", "0.0917", "0.9", "0.25"]) or [0, -1])[1],
                         1)):
    if got != want:
        raise RuntimeError(f"self-check failed: {what} = {got}, want {want}")
print("  self-checks: 5 of 5 (two normal quantiles, two binomial tails, a textbook Bayes example)")

# ---------------- scored on the practice sets ----------------
RES27, WRONG27, TALLY27, EXACT27 = {}, [], {}, {}
print(f"\n  {'':<26}{'answered':>10}{'agree':>7}{'with 27':>9}{'without':>9}")
for n, qs in SETS12.items():
    rows, prev, ans, agree, ex = [], [], 0, 0, [0, 0]
    for (q, ch, g), (x, _, _) in zip(qs, READ12[n]):
        earlier = solve19(q, list(ch)) or tool23(q, list(ch)) or tool24(q, list(ch)) or tool25(q, list(ch)) or \
            tool26(q, list(ch))
        base = float(earlier[1] == g) if earlier else credit12(scores12(x), g)
        got = None if earlier else tool27(q, list(ch))
        if got:
            ans += 1
            agree += got[1] == g
            TALLY27.setdefault(got[0], [0, 0])
            TALLY27[got[0]][0] += 1
            TALLY27[got[0]][1] += got[1] == g
            if got[1] != g and n != "ARC train":
                WRONG27.append((n, q[:110], ch[got[1]], ch[g]))
        final = earlier or got
        if final:
            ex[0] += 1
            ex[1] += final[1] == g
        rows.append(float(got[1] == g) if got else base)
        prev.append(base)
    RES27[n] = dict(answered=ans, agree=agree, whole=100 * np.mean(rows), was=100 * np.mean(prev))
    EXACT27[n] = (ex[0], ex[1], len(qs))
    print(f"  {n:<26}{ans:>10}{agree:>7}{RES27[n]['whole']:>9.2f}{RES27[n]['was']:>9.2f}")
print("  (with 27 = Cells 19 and 23-27 where they answer, else Cell 12; without = the same minus Cell 27)")
print("\n  by tool (all sets: answered, agree with the key)")
for name, (k, a) in TALLY27.items():
    print(f"    {name:<11}{k:>5}{a:>6}")
print("\n  EXACT BASE (Cells 19 and 23-27 together: answered, agree, share of the set)")
for n, (a, r, t) in EXACT27.items():
    print(f"    {n:<26}{a:>5}{r:>6}{100 * a / t:>8.1f}%")
print("\n  DISAGREEMENTS with the answer key (practice sets)")
for n, q, said, key in WRONG27:
    print(f"    [{n}] {q}\n      tool: {said}; key: {key}")

print("\n  PREDICTIONS vs RESULTS")
LEDGER.result("B5-TF1", RES27["MMLU validation"]["answered"])
LEDGER.result("B5-TF2", RES27["MMLU validation"]["answered"] - RES27["MMLU validation"]["agree"])
LEDGER.result("B5-TF3", round(RES27["MMLU validation"]["whole"] - RES27["MMLU validation"]["was"], 2))
LEDGER.result("B5-TF4", RES27["MMLU dev"]["answered"])
LEDGER.result("B5-T41", round(time.time() - T27))
if "sync" in globals():
    sync("Cell 27")
print(f"\n  ({time.time() - T27:.0f}s)")
print("  LAST LINE OF MC AI BRAIN 5 CELL 27")


MC AI BRAIN 5 - CELL 27: STATISTICS BY FORMULA
  self-checks: 5 of 5 (two normal quantiles, two binomial tails, a textbook Bayes example)

                              answered  agree  with 27  without
  ARC train                          0      0    43.05    43.05
  ARC-Easy validation                0      0    51.05    51.05
  ARC-Challenge validation           0      0    30.57    30.57
  OpenBookQA validation              0      0    24.82    24.82
  MMLU validation                    8      8    34.27    33.93
  MMLU dev                           0      0    32.43    32.43
  MMLU-Pro validation                0      0    18.53    18.53
  (with 27 = Cells 19 and 23-27 where they answer, else Cell 12; without = the same minus Cell 27)

  by tool (all sets: answered, agree with the key)
    BINOMIAL       4     4
    BAYES          1     1
    NORMAL-PCT     2     2
    NORMAL-IQR     1     1

  EXACT BASE (Cells 19 and 23-27 together: answered, agree, share of the set)
    ARC tra

In [40]:
# =====================================================================
# MC AI BRAIN 5 - CELL 28: SMALL READINGS - FRACTIONS, PATTERNS, RATIOS, FORMULAS
# Needs Cells 12A, 12B, 19, 22, 23, 24, 25A, 25B, 26A, 26B and 27. Same rule
# as the toolboxes: a fixed reading, an exact (or exactly rounded) value,
# every number of the question read, exactly one choice agreeing.
# TOOLS
#   MIXED        a decimal written as a mixed number (5.875 = 5 and 7 over 8)
#   POWER        the square, cube or (whole) square root of a number
#   TIMES        how many times as many X in A than in B, from two counts
#   RATIO        N of A with A : B given -> the amount of B
#   PATTERN      a list with one constant difference or ratio -> its rule
#   FRACTION-OF  a fraction of N things; a fraction of a fraction amount
#   CUBES        how many positive squares or cubes divide N (from N's prime
#                powers: the product of (e // k + 1))
#   PHYSICS      apparent weight in an accelerating lift m(g +- a); a level
#                shot's speed from its drop d / sqrt(2h / g); decibels to
#                the share blocked 1 - 10^(-dL/10); rest mass sqrt(E^2 - p^2)
#                (g = 9.8 m/s^2)
#   SAVING       saving = income - (autonomous spending + MPC x income)
# BUILT ON the open MMLU validation questions of these kinds (read while
#   building), so B5-TG1..TG3 are CHECKS. These readings are shaped to that
#   wording: on 17,000 outside problems (ASDiv dev, MATH train, GSM8K train,
#   SVAMP) none of them fired when this was built, so their precision
#   outside MMLU cannot be measured; B5-TG5 records that count.
# PREDICTIONS (written before running)
#   B5-TG1 MMLU validation questions Cell 28 answers (check): 11-14
#   B5-TG2 of those, answers that disagree with the key (check): 0
#   B5-TG3 MMLU validation with Cell 28, minus without it, points: 0.4-0.9
#   B5-TG4 MMLU dev questions Cell 28 answers: 0-2
#   B5-TG5 outside problems any Cell 28 tool reads (check): 0-5
#   B5-T42 Cell 28 seconds: <= 120
# =====================================================================
import json
import math
import re
import time
import xml.etree.ElementTree as ET
from collections import Counter
from fractions import Fraction

import numpy as np
from mcai.core import at_most, between

print("=" * 64)
print("MC AI BRAIN 5 - CELL 28: SMALL READINGS - FRACTIONS, PATTERNS, RATIOS, FORMULAS")
print("=" * 64)
T28 = time.time()
for need in ("tool27", "num27", "near27", "lemma26", "SKIP26", "tool26", "int25", "factor25", "tool25", "GOT22",
             "MATHROWS25", "SETS12", "READ12", "scores12", "credit12", "solve19", "tool23", "tool24", "numbers23"):
    if need not in globals():
        raise RuntimeError(f"run Cells 12A, 12B, 19, 22, 23-27 first ({need} is missing)")
for key, claim, rule in (("B5-TG1", "MMLU validation questions Cell 28 answers (check)", between(11, 14)),
                         ("B5-TG2", "of those, answers that disagree with the key (check)", between(0, 0)),
                         ("B5-TG3", "MMLU validation with Cell 28 minus without it, points", between(0.4, 0.9)),
                         ("B5-TG4", "MMLU dev questions Cell 28 answers", between(0, 2)),
                         ("B5-TG5", "outside problems any Cell 28 tool reads (check)", between(0, 5)),
                         ("B5-T42", "Cell 28 seconds", at_most(120))):
    LEDGER.predict(key, claim, rule)

# ---------------- the tools ----------------
G28 = 9.8  # m/s^2, the value high-school texts use
FRAC28 = r"(?:\\[dt]?frac\{(\d+)\}\{(\d+)\}|\\frac(\d)(\d)|(\d+)/(\d+))"


def frac28(m, i: int = 1) -> Fraction:
    """the fraction of a FRAC28 match whose groups start at i"""
    g = [m.group(k) for k in range(i, i + 6)]
    a, b = next((g[k], g[k + 1]) for k in (0, 2, 4) if g[k] is not None)
    return Fraction(int(a), int(b))


def choice28(c: str):
    """a choice's exact value: 7, -2.5, 5/24, \\frac{5}{24}, '5 and 7 over 8', '4 times'"""
    t = str(c).strip().replace("$", "").replace("\u2212", "-")
    m = re.fullmatch(r"(\d+) and (\d+) over (\d+)", t)
    if m:
        return int(m.group(1)) + Fraction(int(m.group(2)), int(m.group(3)))
    m = re.fullmatch(r"(\d+) over (\d+)", t)
    if m:
        return Fraction(int(m.group(1)), int(m.group(2)))
    m = re.fullmatch(r"(-?)\\[dt]?frac\{(\d+)\}\{(\d+)\}", t)
    if m:
        return (-1 if m.group(1) else 1) * Fraction(int(m.group(2)), int(m.group(3)))
    t = re.sub(r"\s*(?:times|[A-Za-z]+(?:/[A-Za-z0-9^]+)?)\s*$", "", t)
    try:
        return Fraction(t.replace(",", ""))
    except (ValueError, ZeroDivisionError):
        return None


def pick28(v, ch: list, near: bool = False):
    """the one choice equal to v (or, with near, equal at the choice's own precision)"""
    if near:
        hit = [j for j, c in enumerate(ch) if len(num27(str(c))) == 1 and near27(float(v), str(c))]
    else:
        hit = [j for j, c in enumerate(ch) if choice28(c) is not None and choice28(c) == v]
    return hit if len(hit) == 1 else None


def mixed28(q, ch):
    m = re.fullmatch(r"\s*Write (\d+\.\d+) as a mixed number or (?:a )?fraction in simplest form\.?\s*", q)
    if not m:
        return None
    v = Fraction(m.group(1))
    hit = [j for j, c in enumerate(ch) if choice28(c) == v and (re.search(r" and ", str(c)) or v < 1)]
    return (hit, [m.group(1)]) if len(hit) == 1 else None


def power28(q, ch):
    m = re.fullmatch(r"\s*What is the (square|cube|square root) of (\d+(?:\.\d+)?)\?\s*", q)
    if not m:
        return None
    x = Fraction(m.group(2))
    if m.group(1) == "square root":
        r = math.isqrt(int(x)) if x.denominator == 1 else -1
        if r * r != x:
            return None
        v = Fraction(r)
    else:
        v = x ** (2 if m.group(1) == "square" else 3)
    hit = pick28(v, ch)
    return (hit, [m.group(2)]) if hit else None


def times28(q, ch):
    """'There are 7 days in a week. ... 28 days in February. How many times as many days are there in A than in B?'"""
    parts = re.split(r"(?<=[.?])\s+", q.strip())
    m = re.fullmatch(r"How many times as many (\w+) are there in (.+?) than (?:are )?in (.+?)\?", parts[-1])
    if not m or len(parts) != 3:
        return None
    unit = lemma26(m.group(1))
    found = []
    for p in parts[:2]:
        k = re.findall(r"\b(\d+) (\w+)\b", p)
        if len(k) != 1 or lemma26(k[0][1]) != unit:
            return None
        found.append((Fraction(k[0][0]), {lemma26(w) for w in re.findall(r"[a-z]+", p.lower())} - SKIP26 - {unit}))

    def which(text):
        words = {lemma26(w) for w in re.findall(r"[a-z]+", text.lower())} - SKIP26 - {"one", "month", "this", "year"}
        got = [i for i, (_, ctx) in enumerate(found) if words & ctx]
        return got[0] if len(got) == 1 else None
    a, b = which(m.group(2)), which(m.group(3))
    if a is None or b is None or a == b:
        return None
    hit = pick28(found[a][0] / found[b][0], ch)
    return (hit, [str(found[0][0]), str(found[1][0])]) if hit else None


def ratio28(q, ch):
    """'X has 6 teaspoons. The ratio of teaspoons to tablespoons is 3 : 1. How many tablespoons ...?'"""
    r = re.search(r"\bratio of (\w+) to (\w+) is (\d+)\s*:\s*(\d+)\b", q)
    if not r:
        return None
    a, b = lemma26(r.group(1)), lemma26(r.group(2))
    have = [(Fraction(n), lemma26(w)) for n, w in re.findall(r"\b(\d+(?:\.\d+)?) (\w+)\b", q[:r.start()])]
    ask = re.search(r"How many (\w+)\b[^.?]*\?\s*$", q)
    if len(have) != 1 or not ask or have[0][1] not in (a, b) or lemma26(ask.group(1)) not in (a, b) or \
            lemma26(ask.group(1)) == have[0][1] or re.search(r"\d", q[r.end():]):
        return None
    ra, rb = Fraction(int(r.group(3))), Fraction(int(r.group(4)))
    v = have[0][0] * (rb / ra if have[0][1] == a else ra / rb)
    hit = pick28(v, ch)
    return (hit, [str(have[0][0]), r.group(3), r.group(4)]) if hit else None


def pattern28(q, ch):
    m = re.search(r"rule for the pattern\?\s*((?:-?\d+,\s*){3,}-?\d+),?\s*(?:\.\.\.|\u2026)?\s*$", q)
    if not m:
        return None
    xs = [Fraction(x) for x in re.findall(r"-?\d+", m.group(1))]
    d = {b - a for a, b in zip(xs, xs[1:])}
    r = {b / a for a, b in zip(xs, xs[1:])} if 0 not in xs else set()
    rules = set()
    if len(d) == 1:
        dv = d.pop()
        rules.add(("add", dv) if dv > 0 else ("subtract", -dv))
    if len(r) == 1:
        rv = r.pop()
        rules.add(("multiply by", rv) if rv > 1 else ("divide by", 1 / rv))
    hit = []
    for j, c in enumerate(ch):
        k = re.fullmatch(r"\s*(add|subtract|multiply by|divide by) (\d+)\s*", str(c))
        if k and (k.group(1), Fraction(int(k.group(2)))) in rules:
            hit.append(j)
    return (hit, list(map(str, xs))) if len(hit) == 1 else None


def fracof28(q, ch):
    """'17/40 of the 160 aliens had 3 eyes. How many aliens had 3 eyes?' and a fraction of a fraction amount"""
    s = re.sub(r"\$", "", q)
    m = re.search(FRAC28 + r" of the (\d+) (\w+)\b(.*?)\.\s*How many (\w+)\b(.*?)\?\s*$", s)
    if m and lemma26(m.group(8)) == lemma26(m.group(10)) and m.group(9).strip() == m.group(11).strip():
        v = frac28(m) * int(m.group(7))
        if v.denominator != 1:
            return None
        hit = pick28(v, ch)
        same = re.findall(r"\d+", m.group(9))
        return (hit, [str(frac28(m).numerator), str(frac28(m).denominator), m.group(7)] + same * 2) if hit else None
    m = re.search(r"\bhas " + FRAC28 + r" of a (\w+) of (\w+)\. If (?:he|she|they) "
                  r"(?:eats?|uses?|drinks?|gives? away) " +
                  FRAC28 + r" of the (\w+) (?:he|she|they) "
                  r"(?:has|have), how many (\w+) of (\w+) does", s)
    if m and lemma26(m.group(7)) == lemma26(m.group(16)) and m.group(8) == m.group(15) == m.group(17):
        v = frac28(m, 1) * frac28(m, 9)
        hit = pick28(v, ch)
        used = [str(x) for f in (frac28(m, 1), frac28(m, 9)) for x in (f.numerator, f.denominator)]
        return (hit, used) if hit else None
    return None


def cubes28(q, ch):
    m = re.fullmatch(r"\s*How many positive (?:perfect )?(squares|cubes) divide \$?(.+?)\$?\s*\?\s*", q)
    if not m:
        return None
    n = int25(m.group(2).replace("\\,", ""))
    if not isinstance(n, int) or n < 1:
        return None
    k = 2 if m.group(1) == "squares" else 3
    v = Fraction(math.prod(e // k + 1 for e in factor25(n).values()))
    hit = pick28(v, ch)
    return (hit, numbers23(m.group(2))) if hit else None


def physics28(q, ch):
    s = re.sub(r"\s+", " ", q)
    m = re.search(r"\bA (\d+(?:\.\d+)?) kg mass is placed in an elevator that is accelerating (up|down)wards at "
                  r"(\d+(?:\.\d+)?) m/s(?:2|\^2|\*\*2)\. The apparent weight of this mass is", s)
    if m:
        v = float(m.group(1)) * (G28 + (1 if m.group(2) == "up" else -1) * float(m.group(3)))
        hit = pick28(v, ch, near=True)
        return (hit, [m.group(1), m.group(3), "2"]) if hit else None
    m = re.search(r"\baimed horizontally, directly at the center of a target (\d+(?:\.\d+)?) m away\. The arrow hits "
                  r"(\d*\.\d+|\d+) m below the center of the target\..*?initial speed", s)
    if m:
        v = float(m.group(1)) / math.sqrt(2 * float(m.group(2)) / G28)
        hit = pick28(v, ch, near=True)
        return (hit, [m.group(1), m.group(2)]) if hit else None
    m = re.search(r"\breduces the sound of .*? from (\d+) dB to (\d+) dB\. What percent of the noise is being blocked",
                  s)
    if m:
        v = 100 * (1 - 10 ** (-(int(m.group(1)) - int(m.group(2))) / 10))
        hit = pick28(v, [str(c) for c in ch], near=True)
        return (hit, [m.group(1), m.group(2)]) if hit else None
    m = re.search(r"\bThe rest mass of a particle with total energy (\d+(?:\.\d+)?) GeV and momentum (\d+(?:\.\d+)?) "
                  r"GeV/c is approximately", s)
    if m and float(m.group(1)) > float(m.group(2)):
        v = math.sqrt(float(m.group(1)) ** 2 - float(m.group(2)) ** 2)
        hit = pick28(v, [re.sub(r"\s*GeV/c\^?2$", "", str(c)) for c in ch], near=True)
        return (hit, [m.group(1), m.group(2), "2"]) if hit else None
    return None


def saving28(q, ch):
    """saving = income - consumption, with consumption = autonomous + MPC x income"""
    m = re.search(r"would spend \$(\d+(?:\.\d+)?) per (week|month|year) even if (?:she|he|they) made no income\. "
                  r"(?:Her|His|Their) \2ly income is \$([\d,]+(?:\.\d+)?) and (?:her|his|their) marginal propensity to "
                  r"consume is (0?\.\d+)\. How much does \w+ save per \2\?", re.sub(r"\s+", " ", q))
    if not m:
        return None
    y = Fraction(m.group(3).replace(",", ""))
    v = y - (Fraction(m.group(1)) + Fraction(m.group(4)) * y)
    hit = pick28(v, [str(c).replace(" ", "") for c in ch])
    return (hit, [m.group(1), m.group(3), m.group(4)]) if hit else None


TOOLS28 = (("MIXED", mixed28), ("POWER", power28), ("TIMES", times28), ("RATIO", ratio28), ("PATTERN", pattern28),
           ("FRACTION-OF", fracof28), ("CUBES", cubes28), ("PHYSICS", physics28), ("SAVING", saving28))


def tool28(q: str, ch: list):
    """(tool name, index of the one agreeing choice) when every number of the question was read"""
    for name, tool in TOOLS28:
        try:
            got = tool(q, list(ch))
        except (ValueError, ZeroDivisionError, OverflowError, AttributeError, KeyError):
            got = None
        if not got:
            continue
        hits, used = got
        if Counter(numbers23(q)) - Counter(Fraction(str(u).replace(",", "")) for u in used):
            continue
        return name, hits[0]
    return None


# ---------------- outside problems: does any reading fire? ----------------
OUT28 = [r["problem"] for r in MATHROWS25]
with open(GOT22["train.jsonl"]) as f:
    OUT28 += [r["question"] for r in map(json.loads, f)]
for i, p in enumerate(ET.parse(GOT22["ASDiv.xml"]).getroot().iter("Problem")):
    if i % 5 != 4:
        OUT28.append((p.findtext("Body") or "").strip() + " " + (p.findtext("Question") or "").strip())
with open(GOT22["SVAMP.json"]) as f:
    OUT28 += [p["Body"].strip() + " " + p["Question"].strip() for p in json.load(f)]
FIRED28 = 0
for q in OUT28:
    for name, tool in TOOLS28:
        try:
            hits = tool(q, ["0", "1", "2", "3", "4", "5", "6", "7", "8", "9", "10"])
        except (ValueError, ZeroDivisionError, OverflowError, AttributeError, KeyError):
            hits = None
        if hits is not None:
            FIRED28 += 1
            print(f"  outside [{name}] {q[:100]}")
            break
print(f"  outside problems read by a Cell 28 tool: {FIRED28} of {len(OUT28):,}")

# ---------------- scored on the practice sets ----------------
RES28, WRONG28, TALLY28, EXACT28 = {}, [], {}, {}
print(f"\n  {'':<26}{'answered':>10}{'agree':>7}{'with 28':>9}{'without':>9}")
for n, qs in SETS12.items():
    rows, prev, ans, agree, ex = [], [], 0, 0, [0, 0]
    for (q, ch, g), (x, _, _) in zip(qs, READ12[n]):
        earlier = solve19(q, list(ch)) or tool23(q, list(ch)) or tool24(q, list(ch)) or tool25(q, list(ch)) or \
            tool26(q, list(ch)) or tool27(q, list(ch))
        base = float(earlier[1] == g) if earlier else credit12(scores12(x), g)
        got = None if earlier else tool28(q, list(ch))
        if got:
            ans += 1
            agree += got[1] == g
            TALLY28.setdefault(got[0], [0, 0])
            TALLY28[got[0]][0] += 1
            TALLY28[got[0]][1] += got[1] == g
            if got[1] != g and n != "ARC train":
                WRONG28.append((n, q[:110], ch[got[1]], ch[g]))
        final = earlier or got
        if final:
            ex[0] += 1
            ex[1] += final[1] == g
        rows.append(float(got[1] == g) if got else base)
        prev.append(base)
    RES28[n] = dict(answered=ans, agree=agree, whole=100 * np.mean(rows), was=100 * np.mean(prev))
    EXACT28[n] = (ex[0], ex[1], len(qs))
    print(f"  {n:<26}{ans:>10}{agree:>7}{RES28[n]['whole']:>9.2f}{RES28[n]['was']:>9.2f}")
print("  (with 28 = Cells 19 and 23-28 where they answer, else Cell 12; without = the same minus Cell 28)")
print("\n  by tool (all sets: answered, agree with the key)")
for name, (k, a) in TALLY28.items():
    print(f"    {name:<12}{k:>5}{a:>6}")
print("\n  EXACT BASE (Cells 19 and 23-28 together: answered, agree, share of the set)")
for n, (a, r, t) in EXACT28.items():
    print(f"    {n:<26}{a:>5}{r:>6}{100 * a / t:>8.1f}%")
print("\n  DISAGREEMENTS with the answer key (practice sets)")
for n, q, said, key in WRONG28:
    print(f"    [{n}] {q}\n      tool: {said}; key: {key}")

print("\n  PREDICTIONS vs RESULTS")
LEDGER.result("B5-TG1", RES28["MMLU validation"]["answered"])
LEDGER.result("B5-TG2", RES28["MMLU validation"]["answered"] - RES28["MMLU validation"]["agree"])
LEDGER.result("B5-TG3", round(RES28["MMLU validation"]["whole"] - RES28["MMLU validation"]["was"], 2))
LEDGER.result("B5-TG4", RES28["MMLU dev"]["answered"])
LEDGER.result("B5-TG5", FIRED28)
LEDGER.result("B5-T42", round(time.time() - T28))
if "sync" in globals():
    sync("Cell 28")
print(f"\n  ({time.time() - T28:.0f}s)")
print("  LAST LINE OF MC AI BRAIN 5 CELL 28")


MC AI BRAIN 5 - CELL 28: SMALL READINGS - FRACTIONS, PATTERNS, RATIOS, FORMULAS
  outside problems read by a Cell 28 tool: 0 of 17,816

                              answered  agree  with 28  without
  ARC train                          0      0    43.05    43.05
  ARC-Easy validation                0      0    51.05    51.05
  ARC-Challenge validation           0      0    30.57    30.57
  OpenBookQA validation              0      0    24.82    24.82
  MMLU validation                   13     13    34.97    34.27
  MMLU dev                           0      0    32.43    32.43
  MMLU-Pro validation                0      0    18.53    18.53
  (with 28 = Cells 19 and 23-28 where they answer, else Cell 12; without = the same minus Cell 28)

  by tool (all sets: answered, agree with the key)
    PHYSICS         4     4
    MIXED           1     1
    RATIO           1     1
    TIMES           1     1
    POWER           1     1
    PATTERN         1     1
    SAVING          1     1
    F

In [41]:
# =====================================================================
# MC AI BRAIN 5 - CELL 29A: A STORY READER BUILT ON CELL 16'S PARSER
# Needs Cell 16 (read16, base16: the tagger and dependency parser learned
# by counting), Cell 22 (GSM8K, ASDiv, SVAMP files) and Cell 26A (WORDS26,
# val26, NUM26).
# READING  each sentence through Cell 16's tagger and parser: the main verb
#   (two main verbs: refused), its subject, each number with the noun it
#   counts ("pieces of candy" kept whole), "$ N" as money, the receiver of
#   "give" ("gave him", "gave 3 to Mary").
# VERB KINDS (ARIS, Hosseini et al. 2014): have/be state an amount; buy,
#   find, get ... add; eat, lose, sell ... take away; give moves an amount
#   from giver to receiver; buying with money takes money away. Any other
#   verb leaves the story unread.
# WORLD STATE  each owner's amount event by event; one unknown solved from
#   a final amount; "N more/fewer E than R" solved in either order.
# QUESTIONS  how many E does X have (now/left); how many did X <verb>; how
#   many at first; how many more does X have than Y; "If A, how many ...?".
#   Hidden operations (each, per, half, a day, the rest, together) refused.
# BUILT ON ASDiv dev (Brain 2's TEST part, every 5th problem, skipped),
#   SVAMP and GSM8K train; no MMLU question was used. Only stories that
#   pass the cheap word checks are parsed (about 0.15 s a sentence).
# This cell defines the reader and checks it on three stories; Cell 29B
# scores it on its design sets and on MMLU.
# PREDICTIONS (written before running)
#   B5-T43 Cell 29A seconds: <= 60
# =====================================================================
import re
import time
from fractions import Fraction

from mcai.core import at_most

print("=" * 64); print("MC AI BRAIN 5 - CELL 29A: A STORY READER BUILT ON CELL 16'S PARSER"); print("=" * 64)
T29 = time.time()
for need in ("read16", "base16", "GOT22", "WORDS26", "val26", "NUM26"):
    if need not in globals():
        raise RuntimeError(f"run Cells 16, 22 and 26A first ({need} is missing)")
LEDGER.predict("B5-T43", "Cell 29A seconds", at_most(60))

# ---------------- the reader ----------------
KIND29 = {}
for k29, ws29 in (("+", "buy find pick get receive collect gather catch make bake grow plant add join win borrow earn "
                        "bring score place put"),
                  ("-", "eat lose sell spend use break throw drink pop send donate lend remove spill give pay"),
                  ("=", "have be")):
    KIND29.update({w29: k29 for w29 in ws29.split()})
REFUSE29 = re.compile(
    r"\b(?:each|every|per|times|twice|half|halves|thirds?|fourths?|fifths?|sixths?|sevenths?|eighths?|ninths?|"
    r"tenths?|quarters?|percent|dozens?|average|ratio|equal\w*|share\w*|split|between|among|both|same|double|triple|"
    r"thrice|instead|if|would|could|should|might|about|nearest|round\w*|estimate|approximately|worth|price|"
    r"cents?|dimes?|nickels?|pennies|penny|rest|other|others|except|only|least|most|together|combined|but|"
    r"however|absent|present|missing|everyday|daily|weekly|days?|weeks?|months?|years?|hours?|minutes?|total)\b|"
    r"%|\d/\d|"
    r"\d+-\w+", re.I)
PRON29 = {"he", "she", "they", "it", "him", "her", "them"}
AUXV29 = {"be", "have", "do", "will", "can", "may", "must", "shall"}


def words29(text: str):
    """[(form, tag, head, relation)] from Cell 16's parser, with $ amounts kept as one word"""
    return read16(re.sub(r"\$\s*(\d)", r"$ \1", text))


def frame29(sent: str, last):
    """one sentence as (subject, verb base, kind, amounts, receiver); amounts = [(value or 'x', noun, modifiers)];
    None when a part is out of reach"""
    toks = words29(sent)
    forms = [f for f, _, _, _ in toks]
    tags = [t for _, t, _, _ in toks]
    low = [f.lower() for f in forms]
    main = None
    for i, t in enumerate(tags):
        if not t.startswith("VB") or (i and tags[i - 1] == "TO"):
            continue
        nxt = next((j for j in range(i + 1, min(i + 4, len(tags))) if tags[j].startswith("VB")), None)
        b = base16(forms[i], t)
        if b in AUXV29 and nxt is not None and not any(tags[j] in ("CD",) or tags[j].startswith("NN")
                                                       for j in range(i + 1, nxt)):
            continue
        main = i
        break
    mains = []
    for i, t in enumerate(tags):
        if t.startswith("VB") and not (i and (tags[i - 1] == "TO" or low[i - 1] in ("that", "which", "who"))):
            nxt = next((j for j in range(i + 1, min(i + 4, len(tags))) if tags[j].startswith("VB")), None)
            if base16(forms[i], t) in AUXV29 and nxt is not None and not any(
                    tags[j] == "CD" or tags[j].startswith("NN") for j in range(i + 1, nxt)):
                continue
            mains.append(i)
    if len(mains) > 1:
        return None
    if main is None:
        if low[:1] == ["there"] and any(w in ("is", "are", "was", "were") for w in low[1:3]):
            main = next(i for i, w in enumerate(low) if w in ("is", "are", "was", "were"))
        else:
            return None
    verb = base16(forms[main], tags[main])
    if verb not in KIND29:
        return None
    kind = KIND29[verb]
    rest = low[main + 1:main + 3]
    if verb == "give" and not (rest[:1] and (rest[0] in ("away",) or rest[0] in PRON29 or tags[main + 1] == "NNP" or
                                             re.search(r"\bto\b", " ".join(low[main + 1:])))):
        return None
    if verb == "take" or (verb == "put" and not re.search(r"\b(?:in|into|on|onto)\b", " ".join(low[main:]))):
        return None
    subj = None
    for j in range(main - 1, -1, -1):
        if tags[j] in ("NNP", "NNPS") or low[j] in PRON29 or low[j] in ("i", "we", "you"):
            subj = last if low[j] in PRON29 else forms[j]
            k = j - 1
            if k >= 0 and low[k] in ("his", "her", "their", "my") and tags[j].startswith("NN"):
                subj = low[k] + " " + low[j]
            break
        if tags[j].startswith("NN") and j + 1 == main:
            k = j - 1
            subj = (low[k] + " " + low[j]) if k >= 0 and low[k] in ("his", "her", "their", "my") else low[j]
            break
    if low[:1] == ["there"]:
        subj = None
    recv = None
    if verb == "give":
        after = low[main + 1] if main + 1 < len(low) else ""
        if after in PRON29:
            recv = last if after in ("him", "her", "them") else after
        elif main + 1 < len(tags) and tags[main + 1] == "NNP":
            recv = forms[main + 1]
        else:
            m = re.search(r"\bto (?:his |her |their )?([A-Z]\w+|him|her|them|\w+)\b", sent)
            recv = (m.group(1) if m else None) if "away" not in low else None
    amounts = []
    i = 0
    while i < len(toks):
        f, t = forms[i], tags[i]
        money = f == "$" and i + 1 < len(toks) and tags[i + 1] == "CD"
        if money:
            i += 1
            f = forms[i]
        isnum = tags[i] == "CD" or low[i] in WORDS26
        if not isnum and not money and low[i] not in ("some", "several", "more", "another"):
            i += 1
            continue
        if low[i] in ("some", "several", "more", "another") and not money:
            if low[i] == "another" and i + 1 < len(tags) and tags[i + 1] == "CD":
                i += 1
                continue
            if low[i] == "more" and i and (tags[i - 1] == "CD" or low[i - 1] in WORDS26):
                i += 1
                continue
            j = i + 1
            while j < len(tags) and (tags[j].startswith("JJ") or tags[j].startswith("NN")):
                j += 1
            if j - 1 > i and tags[j - 1].startswith("NN"):
                amounts.append(("x", base16(forms[j - 1], "NN"), tuple(low[i + 1:j - 1])))
            i = j
            continue
        try:
            v = Fraction(f.replace(",", "")) if re.fullmatch(r"\d+(?:,\d{3})*(?:\.\d+)?", f) else \
                Fraction(WORDS26[f.lower()])
        except (KeyError, ValueError):
            return None
        if money:
            amounts.append((v, "$", ()))
            i += 1
            continue
        j = i + 1
        if j < len(low) and low[j] in ("more", "additional", "new", "extra"):
            j += 1
        k = j
        while k < len(tags) and (tags[k].startswith("JJ") or tags[k].startswith("NN")) and low[k] not in ("more",):
            k += 1
        if k > j and tags[k - 1].startswith("NN"):
            noun = base16(forms[k - 1], "NN")
            if k + 1 < len(tags) and low[k] == "of" and tags[k + 1].startswith("NN"):
                noun, k = noun + " of " + base16(forms[k + 1], "NN"), k + 2
            amounts.append((v, noun, tuple(low[j:k - 1])))
        elif low[i] == "one":
            pass
        elif j > i + 1 or (j < len(low) and low[j] in (".", ",", "and")):
            amounts.append((v, None, ()))
        else:
            return None
        i = k
    return dict(subj=subj, verb=verb, kind=kind, amounts=amounts, recv=recv,
                now=bool(re.search(r"\b(?:now|then|left|in the end|finally|still|after that)\b", sent, re.I)))


def compare29(sent: str, last, ent):
    """(subject, referent, difference) of 'S has N more/fewer E than R' or 'S has N E more/less than R'"""
    if len(re.findall(r"\bthan\b", sent)) != 1 or re.search(r"\b(?:who|which|and)\b", sent):
        return None
    m = re.fullmatch(r"\s*([A-Z]\w+|He|She|They) (?:has|have|had|got|picked|found|bought|collected|sold|scored|ate|"
                     r"made|caught|baked|read|drew|grew|planted|won) " + NUM26 +
                     r" (?:(more|fewer|less) ((?:[a-z]+ ){0,3})than|((?:[a-z]+ ){0,3})(more|fewer|less) than) "
                     r"(?:his |her )?([A-Z]\w+|him|her)(?: does| did| has)?\.\s*",
                     sent)
    if not m:
        return None
    subj = last if m.group(1) in ("He", "She", "They") else m.group(1)
    word = m.group(3) or m.group(6)
    noun = (m.group(4) or m.group(5) or "").split()
    if noun:
        n = base16(noun[-1], "NNS")
        if n != ent.split(" of ")[0]:
            return None
    try:
        d = val26(m.group(2))
    except (KeyError, ValueError):
        return None
    ref = last if m.group(7) in ("him", "her") else m.group(7)
    if subj == ref:
        return None
    return subj, ref, d if word == "more" else -d


def ask29(q: str):
    """(entity, modifiers, owner, verb, asks_first) of a 'How many ...' / 'How much money ...' question"""
    toks = words29(q)
    low = [f.lower() for f, _, _, _ in toks]
    tags = [t for _, t, _, _ in toks]
    if low[:3] == ["how", "much", "money"]:
        ent, mods, k = "$", (), 3
    elif low[:2] == ["how", "many"]:
        k = 2
        while k < len(tags) and (tags[k].startswith("JJ") or tags[k].startswith("NN")):
            k += 1
        if k == 2 or not tags[k - 1].startswith("NN"):
            return None
        ent, mods = base16(toks[k - 1][0], "NN"), tuple(low[2:k - 1])
        if k + 1 < len(tags) and low[k] == "of" and tags[k + 1].startswith("NN"):
            ent, k = ent + " of " + base16(toks[k + 1][0], "NN"), k + 2
    else:
        return None
    owner, verb = None, None
    for j in range(k, len(toks)):
        if tags[j] in ("NNP", "NNPS") or low[j] in PRON29:
            owner = owner or (low[j] if low[j] in PRON29 else toks[j][0])
        if tags[j].startswith("VB"):
            b = base16(toks[j][0], tags[j])
            if b in KIND29 and KIND29[b] != "=":
                verb = verb or b
    first = bool(re.search(r"\b(?:at first|to begin with|to start with|in the beginning|originally)\b", q, re.I))
    return ent, mods, owner, verb, first


def story29(text: str):
    """(exact answer, entity) for a story of owners' amounts changed by gains, losses and gifts; None otherwise"""
    t = re.sub(r"\b(Mr|Mrs|Ms|Dr|St)\.", r"\1", re.sub(r"\s+", " ", text.strip()))
    parts = re.split(r"(?<=[.!?])\s+(?=[A-Z0-9$])", t)
    if not parts[-1].endswith("?") or any(p.endswith("?") for p in parts[:-1]):
        return None
    cond = re.fullmatch(r"(?:If|Now if|And if) ([^?]+?), ((?:how|what) [^?]+\?)", parts[-1], re.I)
    if cond:
        parts = parts[:-1] + [cond.group(1)[0].upper() + cond.group(1)[1:] + ".", cond.group(2)[0].upper() +
                              cond.group(2)[1:]]
        t = " ".join(parts)
    if len(parts) < 2 or REFUSE29.search(t) or re.search(r"\d", parts[-1]):
        return None
    cmpq = re.fullmatch(r"How many (?:more|fewer) (.+?) (?:does|did|do) ([A-Z]\w+) (have|\w+) than ([A-Z]\w+)"
                        r"(?: does| did)?\?",
                        parts[-1])
    if not cmpq and re.search(r"\b(?:than|more|fewer|less|altogether|in all|total|together|combined|and)\b", parts[-1],
                              re.I):
        return None
    if cmpq:
        parts[-1] = f"How many {cmpq.group(1)} does {cmpq.group(2)} have?"
    asked = ask29(parts[-1])
    if not asked:
        return None
    ent, mods, qowner, qverb, first = asked
    state, start, events, final, last, unknown = {}, {}, [], {}, None, None
    rels = []
    for p in parts[:-1]:
        if re.search(r"\bthan\b", p):
            got = compare29(p, last, ent)
            if got is None:
                return None
            rels.append(got)
            last = got[0]
            continue
        fr = frame29(p, last)
        if fr is None:
            return None
        last = fr["subj"] or last
        mine = [(v, n, m) for v, n, m in fr["amounts"] if n in (ent, None)]
        if len(mine) != len(fr["amounts"]):
            return None
        if not mine:
            if fr["kind"] != "=":
                return None
            continue
        if any(n is None for _, n, _ in mine) and not events and not start:
            return None
        if mods and any(m != mods for _, _, m in mine):
            return None
        if "x" in [v for v, _, _ in mine] and len(mine) > 1:
            return None
        who = fr["subj"]
        amount = "x" if mine[0][0] == "x" else sum((v for v, _, _ in mine), Fraction(0))
        if ent == "$" and fr["verb"] in ("buy", "pay"):
            fr["kind"] = "-"
        if amount == "x":
            if unknown is not None:
                return None
        if fr["kind"] == "=":
            if who not in start and who not in state:
                start[who] = amount
                state[who] = amount
                if amount == "x":
                    unknown = ("start", who)
            elif fr["now"] and amount != "x" and who not in final:
                final[who] = amount
            else:
                return None
            continue
        sign = 1 if fr["kind"] == "+" else -1
        if fr["verb"] == "give" and fr["recv"]:
            targets = [(who, -1), (fr["recv"], 1)]
        else:
            targets = [(who, sign)]
        if amount == "x":
            unknown = ("change", fr["verb"], fr["kind"], targets)
        events.append((fr["verb"], targets, amount, who))
    if rels:
        if events or unknown is not None or final or qverb:
            return None
        known = dict(start)
        for _ in range(len(rels) + 1):
            for subj, ref, d in rels:
                if ref in known and subj not in known:
                    known[subj] = known[ref] + d
                elif subj in known and ref not in known:
                    known[ref] = known[subj] - d
        if any(not isinstance(v, Fraction) for v in known.values()):
            return None
        for subj, ref, d in rels:
            if subj not in known or ref not in known or known[subj] - known[ref] != d:
                return None
        if cmpq:
            a, b = cmpq.group(2), cmpq.group(4)
            if a not in known or b not in known:
                return None
            v = (known[a] - known[b]) if re.match(r"How many more", t.split("?")[0].split(". ")[-1]) else \
                (known[b] - known[a])
            return (v, ent) if v >= 0 else None
        if qowner not in known or known[qowner] < 0:
            return None
        return known[qowner], ent
    if cmpq:
        if events or unknown is not None or final:
            return None
        a, b = cmpq.group(2), cmpq.group(4)
        if a not in start or b not in start or "x" in (start[a], start[b]):
            return None
        more = re.search(r"How many more", t)
        v = (start[a] - start[b]) if more else (start[b] - start[a])
        return (v, ent) if v >= 0 else None
    owners = set(start) | {o for _, ts, _, _ in events for o, _ in ts}
    target = qowner if qowner in owners else (last if qowner in PRON29 else (next(iter(owners)) if len(owners) == 1
                                                                             else None))
    if target is None:
        return None
    if unknown is None:
        if final or first:
            return None
        if qverb:
            mine = [a for verb, _, a, w in events if w == target and verb == qverb]
            if not mine or any(w == target and verb != qverb for verb, _, _, w in events):
                return None
            return sum(mine, Fraction(0)), ent
        if target not in start:
            return None
        total = start[target]
        for _, ts, a, _ in events:
            for o, s in ts:
                if o == target:
                    total += s * a
        return total, ent
    if target not in final or (unknown[0] == "start") != first:
        return None
    if unknown[0] == "change" and (not qverb or KIND29[qverb] != unknown[2]):
        return None
    if unknown[0] == "start" and unknown[1] != target:
        return None
    const, coef = (Fraction(0), Fraction(1)) if unknown[0] == "start" else (start.get(target, None), Fraction(0))
    if const is None:
        return None
    for _, ts, a, _ in events:
        for o, s in ts:
            if o == target:
                if a == "x":
                    coef += s
                else:
                    const += s * a
    if coef == 0:
        return None
    x = (final[target] - const) / coef
    return (x, ent) if x >= 0 else None


def safe29(text: str):
    """story29, with any parser surprise read as silence"""
    try:
        return story29(text)
    except (ValueError, KeyError, IndexError, StopIteration, TypeError, ZeroDivisionError):
        return None


for q, want in (("Tommy had some balloons. His mom gave him 34 more balloons for his birthday. Then, Tommy had 60 "
                 "balloons. How many balloons did Tommy have to start with?", 26),
                ("Laurie has 12 more marbles than Kurt. Kurt has 45 marbles less than Dennis. Dennis has 70 marbles. "
                 "How many marbles does Laurie have?", 37),
                ("Dan has $ 7. He bought a candy bar for $ 2 and a chocolate for $ 3. How much money is left?", 2)):
    got = safe29(q)
    if not got or got[0] != want:
        raise RuntimeError(f"self-check failed: {q[:60]} -> {got}, want {want}")
print("  self-checks: 3 of 3 (an unknown start, a chain of comparisons, money spent)")
print("\n  PREDICTIONS vs RESULTS")
LEDGER.result("B5-T43", round(time.time() - T29))
if "sync" in globals():
    sync("Cell 29A")
print(f"\n  ({time.time() - T29:.0f}s)")
print("  LAST LINE OF MC AI BRAIN 5 CELL 29A")


MC AI BRAIN 5 - CELL 29A: A STORY READER BUILT ON CELL 16'S PARSER
  self-checks: 3 of 3 (an unknown start, a chain of comparisons, money spent)

  PREDICTIONS vs RESULTS
    B5-T43 Cell 29A seconds: 0 (predicted <= 60) -> HELD
  sync: pushed 5 files to kaggle/ on main as commit bb0cd262474a; GitHub confirms it (744 entries, head 18dc6259507c05bc) (5s)

  (5s)
  LAST LINE OF MC AI BRAIN 5 CELL 29A


In [42]:
# =====================================================================
# MC AI BRAIN 5 - CELL 29B: THE PARSER STORY READER, SCORED
# Needs Cell 29A and Cells 12A, 12B, 19, 22, 23-28.
# PREDICTIONS (written before running; B5-TH1, B5-TH2 are CHECKS: ASDiv dev,
#   SVAMP and GSM8K train were used to build the reader. MMLU was not.)
#   B5-TH1 design-set stories the reader answers (check): 36-50
#   B5-TH2 of those, answers that disagree with the dataset (check): 0
#   B5-TH3 MMLU validation questions the reader answers (where Cells 19 and
#          23-28 do not): 0-3
#   B5-TH4 of those, answers that disagree with the key: 0
#   B5-TH5 MMLU dev questions it answers: 0-2
#   B5-T44 Cell 29B seconds: <= 900
# =====================================================================
import json
import re
import time
import xml.etree.ElementTree as ET
from fractions import Fraction

import numpy as np
from mcai.core import at_most, between

print("=" * 64); print("MC AI BRAIN 5 - CELL 29B: THE PARSER STORY READER, SCORED"); print("=" * 64)
T29B = time.time()
for need in ("safe29", "tool28", "tool27", "tool26", "tool25", "SETS12", "READ12", "scores12", "credit12", "solve19",
             "tool23", "tool24", "number19", "GOT22"):
    if need not in globals():
        raise RuntimeError(f"run Cells 12A, 12B, 19, 22, 23-28 and 29A first ({need} is missing)")
for key, claim, rule in (("B5-TH1", "design-set stories the parser reader answers (check)", between(36, 50)),
                         ("B5-TH2", "of those, answers that disagree with the dataset (check)", between(0, 0)),
                         ("B5-TH3", "MMLU validation questions the parser reader answers", between(0, 3)),
                         ("B5-TH4", "of those, answers that disagree with the key", between(0, 0)),
                         ("B5-TH5", "MMLU dev questions the parser reader answers", between(0, 2)),
                         ("B5-T44", "Cell 29B seconds", at_most(900))):
    LEDGER.predict(key, claim, rule)

# ---------------- built on: ASDiv dev, SVAMP, GSM8K train ----------------
with open(GOT22["train.jsonl"]) as f:
    DESIGN29 = [("GSM8K train", r["question"], r["answer"].split("####")[-1]) for r in map(json.loads, f)]
for i, p in enumerate(ET.parse(GOT22["ASDiv.xml"]).getroot().iter("Problem")):
    if i % 5 != 4:
        DESIGN29.append(("ASDiv dev", (p.findtext("Body") or "").strip() + " " + (p.findtext("Question") or "").strip(),
                         p.findtext("Answer") or ""))
with open(GOT22["SVAMP.json"]) as f:
    DESIGN29 += [("SVAMP", p["Body"].strip() + " " + p["Question"].strip(), str(p["Answer"])) for p in json.load(f)]
BY29, BAD29 = {}, []
for src, q, a in DESIGN29:
    got = safe29(q)
    if not got:
        continue
    m = re.match(r"\s*(-?[\d,]*\.?\d+)", a)
    ok = m is not None and got[0] == Fraction(m.group(1).replace(",", ""))
    BY29.setdefault(src, [0, 0])
    BY29[src][0] += 1
    BY29[src][1] += ok
    if not ok:
        BAD29.append((src, q[:110], got[0], a))
print(f"\n  the parser reader on its design sets ({time.time() - T29B:.0f}s)\n    {'':<14}{'answered':>9}{'agree':>7}")
for src, (n, a) in BY29.items():
    print(f"    {src:<14}{n:>9}{a:>7}")
for src, q, said, a in BAD29:
    print(f"    DISAGREES [{src}] {q}\n      reader: {said}; dataset: {a}")



def tool29(q: str, ch: list):
    """('STORY-III', index of the one agreeing choice) or None"""
    got = safe29(q)
    if not got:
        return None
    hit = [j for j, c in enumerate(ch) if number19(c) == got[0]]
    return ("STORY-III", hit[0]) if len(hit) == 1 else None


# ---------------- scored on the practice sets ----------------
RES29, WRONG29, EXACT29 = {}, [], {}
print(f"\n  {'':<26}{'answered':>10}{'agree':>7}{'with 29':>9}{'without':>9}")
for n, qs in SETS12.items():
    rows, prev, ans, agree, ex = [], [], 0, 0, [0, 0]
    for (q, ch, g), (x, _, _) in zip(qs, READ12[n]):
        earlier = solve19(q, list(ch)) or tool23(q, list(ch)) or tool24(q, list(ch)) or tool25(q, list(ch)) or \
            tool26(q, list(ch)) or tool27(q, list(ch)) or tool28(q, list(ch))
        base = float(earlier[1] == g) if earlier else credit12(scores12(x), g)
        got = None if earlier else tool29(q, list(ch))
        if got:
            ans += 1
            agree += got[1] == g
            if got[1] != g and n != "ARC train":
                WRONG29.append((n, q[:110], ch[got[1]], ch[g]))
        final = earlier or got
        if final:
            ex[0] += 1
            ex[1] += final[1] == g
        rows.append(float(got[1] == g) if got else base)
        prev.append(base)
    RES29[n] = dict(answered=ans, agree=agree, whole=100 * np.mean(rows), was=100 * np.mean(prev))
    EXACT29[n] = (ex[0], ex[1], len(qs))
    print(f"  {n:<26}{ans:>10}{agree:>7}{RES29[n]['whole']:>9.2f}{RES29[n]['was']:>9.2f}")
print("\n  EXACT BASE (Cells 19 and 23-29 together: answered, agree, share of the set)")
for n, (a, r, t) in EXACT29.items():
    print(f"    {n:<26}{a:>5}{r:>6}{100 * a / t:>8.1f}%")
print("\n  DISAGREEMENTS with the answer key (practice sets)")
for n, q, said, key in WRONG29:
    print(f"    [{n}] {q}\n      tool: {said}; key: {key}")

print("\n  PREDICTIONS vs RESULTS")
LEDGER.result("B5-TH1", sum(k for k, _ in BY29.values()))
LEDGER.result("B5-TH2", len(BAD29))
LEDGER.result("B5-TH3", RES29["MMLU validation"]["answered"])
LEDGER.result("B5-TH4", RES29["MMLU validation"]["answered"] - RES29["MMLU validation"]["agree"])
LEDGER.result("B5-TH5", RES29["MMLU dev"]["answered"])
LEDGER.result("B5-T44", round(time.time() - T29B))
if "sync" in globals():
    sync("Cell 29B")
print(f"\n  ({time.time() - T29B:.0f}s)")
print("  LAST LINE OF MC AI BRAIN 5 CELL 29B")


MC AI BRAIN 5 - CELL 29B: THE PARSER STORY READER, SCORED

  the parser reader on its design sets (9s)
                   answered  agree
    GSM8K train           2      2
    ASDiv dev            28     28
    SVAMP                13     13

                              answered  agree  with 29  without
  ARC train                          0      0    43.05    43.05
  ARC-Easy validation                0      0    51.05    51.05
  ARC-Challenge validation           0      0    30.57    30.57
  OpenBookQA validation              0      0    24.82    24.82
  MMLU validation                    0      0    34.97    34.97
  MMLU dev                           0      0    32.43    32.43
  MMLU-Pro validation                0      0    18.53    18.53

  EXACT BASE (Cells 19 and 23-29 together: answered, agree, share of the set)
    ARC train                     1     1     0.0%
    ARC-Easy validation           0     0     0.0%
    ARC-Challenge validation      0     0     0.0%
    OpenBook

In [43]:
# =====================================================================
# MC AI BRAIN 5 - CELL 30: TABLES - THE PERIODIC TABLE AND UNITS
# Needs Cells 0 (fetch, GITHUB), 12A, 12B, 19, 23-29. Answers from two pinned
# public tables, never from words the cell writes itself:
#   PeriodicTableJSON (Bowserinator/Periodic-Table-JSON, CC BY-SA 3.0, data
#     as cited by Wikipedia): name, symbol, number, group, category, phase,
#     shells, electron configuration of 119 elements
#   Pint's unit definitions (hgrecco/pint default_en.txt, BSD licence),
#     read by this cell into exact factors of the SI base units: inch =
#     yard / 36, yard = 0.9144 m, ounce = pound / 16 ...
# TOOLS  SYMBOL (the symbol of a named element); CONFIG (an electron
#   configuration -> the element whose number is the sum of electrons);
#   SHELL (electrons in a numbered or the outer shell); CATEGORY (which
#   group or family; the one non-metal among metals); GROUP (most similar
#   chemical properties: the same column); ION (the usual charge of groups
#   1, 2, 13, 15, 16, 17); PHASE (solid, liquid or gas at room temperature);
#   NUCLEONS (mass = protons + neutrons; neutrons = mass - atomic number);
#   UNITS ("how many A are in N B?", the same dimension only).
# Every tool answers only when exactly one choice agrees.
# BUILT ON the element questions of ARC train and of the MMLU looks (lab/
#   look_tables.py, lab/look_facts.py), so B5-TI1..TI4 are CHECKS; the ARC
#   validation sets were not looked at (B5-TI5).
# PREDICTIONS (written before running)
#   B5-TI1 ARC train questions the table tools answer (check): 5-15
#   B5-TI2 of those, answers that disagree with the key (check): <= 1
#   B5-TI3 MMLU validation questions they answer (check): 2-6
#   B5-TI4 of those, answers that disagree with the key (check): 0
#   B5-TI5 ARC-Easy + ARC-Challenge validation questions they answer: 1-10
#   B5-T45 Cell 30 seconds: <= 120
# =====================================================================
import json
import os
import re
import time
from fractions import Fraction

import numpy as np
from mcai.core import at_most, between

print("=" * 64); print("MC AI BRAIN 5 - CELL 30: TABLES - THE PERIODIC TABLE AND UNITS"); print("=" * 64)
T30 = time.time()
for need in ("fetch", "GITHUB", "tool29", "tool28", "tool27", "tool26", "tool25", "SETS12", "READ12", "scores12",
             "credit12", "solve19", "tool23", "tool24"):
    if need not in globals():
        raise RuntimeError(f"run Cells 0, 12A, 12B, 19 and 23-29 first ({need} is missing)")
for key, claim, rule in (("B5-TI1", "ARC train questions the table tools answer (check)", between(5, 15)),
                         ("B5-TI2", "of those, answers that disagree with the key (check)", at_most(1)),
                         ("B5-TI3", "MMLU validation questions the table tools answer (check)", between(2, 6)),
                         ("B5-TI4", "of those, answers that disagree with the key (check)", between(0, 0)),
                         ("B5-TI5", "ARC-Easy + ARC-Challenge validation questions they answer", between(1, 10)),
                         ("B5-T45", "Cell 30 seconds", at_most(120))):
    LEDGER.predict(key, claim, rule)

BASE = os.environ.get("MC_AI_FETCH_BASE", "")  # Cell 0's fetch base: Cell 6 reuses the name BASE for its setting
TABLES30 = {"PeriodicTableJSON.json": ("Bowserinator/Periodic-Table-JSON", "a78c7a48fe7235cebd6153161a9e7fa70f0cfb30",
                                       "PeriodicTableJSON.json",
                                       "05e478cb1192c5bbfad8d9600fd3b4c7902c0966ad5d0408bfcc4d5092424c0d"),
            "default_en.txt": ("hgrecco/pint", "072281c5f1b3d5fe30d097cbb0791be03ee24d45", "pint/default_en.txt",
                               "5eb5d618172a8af47d01398f9642490216452bd410ed08ca5bdd311b856ecf7b")}
GOT30 = {}
for name, (repo, commit, rel, want) in TABLES30.items():
    path, how, _ = fetch("tables", repo, commit, rel, want, None, [GITHUB])
    if how.startswith("REFUSED"):
        raise RuntimeError(f"{name}: {how}")
    GOT30[name] = path

# ---------------- the tools ----------------
WORDN30 = {w: i for i, w in enumerate("zero one two three four five six seven eight nine ten".split())}
WORDN30["none"] = 0
ORD30 = dict(first=1, second=2, third=3, fourth=4, fifth=5, sixth=6, seventh=7)
CHARGE30 = {1: 1, 2: 2, 13: 3, 15: -3, 16: -2, 17: -1}


def elements30(path) -> list:
    with open(path) as f:
        rows = json.load(f)["elements"]
    for e in rows:
        e["low"] = e["name"].lower()
        e["metal"] = "metal" in e["category"] and "nonmetal" not in e["category"] and "metalloid" not in e["category"]
        e["kinds"] = {e["category"]} | ({"halogen"} if e["group"] == 17 else set()) | \
            ({"metal"} if e["metal"] else {"nonmetal"} if "nonmetal" in e["category"] or e["group"] == 18 else set())
    return rows


def find30(text: str, table: list) -> list:
    """the elements a text names: by name, or by symbol in parentheses or after a comma ('beryllium, Be')"""
    low = text.lower()
    out = [e for e in table if re.search(r"\b" + e["low"] + r"\b", low)]
    for e in table:
        if e not in out and re.search(r"(?:\(|, )" + e["symbol"] + r"\b(?![a-z])", text):
            out.append(e)
    return out


def choice_el30(c: str, table: list):
    """the one element a choice names (a name, a symbol alone, or both), else None"""
    c = str(c).strip().rstrip(".")
    by_sym = [e for e in table if c == e["symbol"]]
    if by_sym:
        return by_sym[0]
    got = find30(c, table)
    return got[0] if len(got) == 1 else None


def tool30_elements(q: str, ch: list, table: list):
    s = re.sub(r"\s+", " ", q)
    low = s.lower()
    els = [choice_el30(c, table) for c in ch]
    named = find30(s, table)
    one = lambda xs: xs if len(xs) == 1 else None
    m = re.search(r"\bchemical symbol (?:for|of|represents)(?: the element)? (\w+)|\bsymbol (?:for|of) (\w+)", low)
    if m:
        e = [x for x in table if x["low"] == (m.group(1) or m.group(2))]
        if e:
            return "SYMBOL", one([j for j, c in enumerate(ch) if str(c).strip() == e[0]["symbol"]])
    m = re.search(r"\b((?:[1-7][spdf]\s?\^?\{?\d{1,2}\}?\s*){3,})", s)
    if m and "element" in low and not re.search(r"\bions?\b|\bcharge\b", low):
        z = sum(int(x) for x in re.findall(r"[spdf]\s?\^?\{?(\d{1,2})\}?", m.group(1)))
        return "CONFIG", one([j for j, e in enumerate(els) if e and e["number"] == z])
    m = re.search(r"\bhow many electrons (?:are )?(?:there )?in the (first|second|third|fourth|fifth|outer|outermost) "
                  r"(?:electron )?shell of (\w+)", low)
    if m and len(named) == 1:
        sh = named[0]["shells"]
        k = len(sh) if m.group(1) in ("outer", "outermost") else ORD30[m.group(1)]
        v = sh[k - 1] if k <= len(sh) else 0
        vals = [WORDN30.get(str(c).strip().lower().rstrip("."), None) if not str(c).strip().isdigit() else int(c)
                for c in ch]
        return "SHELL", one([j for j, x in enumerate(vals) if x == v])
    m = re.search(r"\b(\w+) belongs to which (?:metal )?(?:group|family|category)\b", low)
    if m and len(named) == 1:
        cat = set(named[0]["category"].split()) - {"metal"}
        return "CATEGORY", one([j for j, c in enumerate(ch)
                                if set(str(c).lower().split()) - {"metal", "metals"} == cat])
    if re.search(r"\ball of the following are metals except\b", low) and all(els):
        return "CATEGORY", one([j for j, e in enumerate(els) if not e["metal"]])
    m = re.search(r"\bmost similar to (?:those of )?(\w+)", low)
    if m and len(named) == 1 and all(els) and named[0]["group"] in range(1, 19):
        return "GROUP", one([j for j, e in enumerate(els) if e["group"] == named[0]["group"] and e is not named[0]])
    m = re.search(r"\bmost likely to form an ion with a (negative|positive) (\w+) charge\b", low)
    if m and all(els):
        k = WORDN30.get(m.group(2), int(m.group(2)) if m.group(2).isdigit() else None)
        if k:
            want = k if m.group(1) == "positive" else -k
            return "ION", one([j for j, e in enumerate(els) if CHARGE30.get(e["group"]) == want])
    m = re.search(r"\b(\w+)(?: \(\w+\))? is different from other metals because (?:\w+ )?is\b", low)
    if m and len(named) == 1:
        ph = named[0]["phase"].lower()
        return "PHASE", one([j for j, c in enumerate(ch)
                             if re.search(r"\b" + ph + r" at room temperature", str(c).lower())
                             and not re.search(r"\bnot\b", str(c).lower())])
    return None


def nucleons30(q: str, ch: list):
    low = re.sub(r"\s+", " ", q.lower())
    p = re.search(r"\b(\d+) protons\b", low)
    n = re.search(r"\b(\d+) neutrons\b", low)
    z = re.search(r"\batomic number (?:of )?(\d+)\b", low)
    a = re.search(r"\batomic mass (?:of )?(\d+)\b", low)
    vals = [int(c) if re.fullmatch(r"\s*\d+\s*", str(c)) else None for c in ch]
    if p and n and re.search(r"\b(?:atomic mass|mass number|mass of (?:the|this) atom)\b", low) and \
            not re.search(r"\bhow many (?:protons|neutrons|electrons)\b", low):
        v = int(p.group(1)) + int(n.group(1))
    elif z and a and re.search(r"\bparticles (?:are )?(?:found )?in (?:its|the) nucleus\b", low):
        v = int(a.group(1))
    elif z and a and re.search(r"\bhow many neutrons\b", low):
        v = int(a.group(1)) - int(z.group(1))
    else:
        return None
    hit = [j for j, x in enumerate(vals) if x == v]
    return ("NUCLEONS", hit) if len(hit) == 1 else None


# ---------------- units: Pint's definition file read into exact factors ----------------
def units30(path) -> dict:
    """{unit name or alias: (Fraction factor in base units, dimensions)} for the definitions that resolve"""
    raw, prefix = {}, {}
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.split("#")[0].strip()
            if not line or line.startswith("@") or "=" not in line:
                continue
            parts = [x.strip() for x in line.split("=")]
            name = parts[0]
            if name.endswith("-"):
                try:
                    prefix[name[:-1]] = Fraction(parts[1])
                except ValueError:
                    pass
                continue
            if not re.fullmatch(r"[A-Za-z_]+", name):
                continue
            raw[name] = parts[1]
            for alias in parts[2:]:
                if re.fullmatch(r"[A-Za-z_]+", alias) and alias not in raw:
                    raw.setdefault("@" + alias, name)
    done = {}

    def get(u, depth=0):
        if u in done:
            return done[u]
        if depth > 30:
            return None
        if "@" + u in raw and u not in raw:
            return get(raw["@" + u], depth + 1)
        if u not in raw:
            for p, v in prefix.items():
                if u.startswith(p) and u[len(p):] in raw:
                    b = get(u[len(p):], depth + 1)
                    return (v * b[0], b[1]) if b else None
            return None
        expr = raw[u]
        if re.fullmatch(r"\[\w+\]", expr):
            done[u] = (Fraction(1), ((expr, 1),))
            return done[u]
        expr = re.sub(r"(?<=[\d.])\s+(?=[A-Za-z_])", " * ", expr)
        toks = re.findall(r"\d+(?:\.\d+)?(?:e-?\d+)?|[A-Za-z_]+|\*\*|[*/()]", expr)
        if "".join(toks) != expr.replace(" ", ""):
            return None
        py = []
        for t in toks:
            if re.match(r"[A-Za-z_]", t):
                b = get(t, depth + 1)
                if b is None:
                    return None
                py.append(f"U({len(py)})")
                py[-1] = ("U", b)
            else:
                py.append(t)
        try:
            val = evalu30(py)
        except (ValueError, ZeroDivisionError, TypeError, IndexError):
            return None
        done[u] = val
        return val
    out = {}
    for u in list(raw):
        name = u[1:] if u.startswith("@") else u
        got = get(name)
        if got:
            out[name] = got
    for p in prefix:
        for base in ("meter", "gram", "liter", "second"):
            got = get(p + base)
            if got:
                out[p + base] = got
    return out


def evalu30(toks: list):
    """(factor, dims) of a token list of numbers, units, * / ** and brackets, left to right with ** first"""
    def atom(i):
        t = toks[i]
        if isinstance(t, tuple):
            v, i = t[1], i + 1
        elif t == "(":
            v, i = expr(i + 1)
            i += 1
        else:
            v, i = (Fraction(t), ()), i + 1
        if i < len(toks) and toks[i] == "**":
            k = int(toks[i + 1])
            v, i = (v[0] ** k, tuple((d, e * k) for d, e in v[1])), i + 2
        return v, i

    def expr(i):
        v, i = atom(i)
        while i < len(toks) and toks[i] in ("*", "/"):
            op = toks[i]
            w, i = atom(i + 1)
            dims = dict(v[1])
            for d, e in w[1]:
                dims[d] = dims.get(d, 0) + (e if op == "*" else -e)
            v = (v[0] * w[0] if op == "*" else v[0] / w[0], tuple(sorted((d, e) for d, e in dims.items() if e)))
        return v, i
    v, i = expr(0)
    if i != len(toks):
        raise ValueError("trailing")
    return v


def unitname30(w: str, units: dict):
    w = w.lower().strip(".,?")
    for cand in (w, w[:-1] if w.endswith("s") else w, w[:-2] if w.endswith("es") else w,
                 {"feet": "foot", "inches": "inch"}.get(w, w), w.replace(" ", "_")):
        if cand in units:
            return cand
    return None


def convert30(q: str, ch: list, units: dict):
    low = re.sub(r"\s+", " ", q.lower())
    m = re.fullmatch(r"\s*how many ((?:fluid |square |cubic )?[a-z]+) (?:are )?(?:there )?in (a|an|one|\d+(?:\.\d+)?) "
                     r"((?:fluid |square |cubic )?[a-z]+)\s*\?\s*", low)
    if not m:
        return None
    a, b = unitname30(m.group(1), units), unitname30(m.group(3), units)
    if not a or not b or a == b or units[a][1] != units[b][1]:
        return None
    n = Fraction(1) if m.group(2) in ("a", "an", "one") else Fraction(m.group(2))
    v = n * units[b][0] / units[a][0]
    hit = [j for j, c in enumerate(ch) if re.fullmatch(r"\s*[\d,]+(?:\.\d+)?\s*", str(c)) and
           Fraction(str(c).replace(",", "").strip()) == v]
    return ("UNITS", hit) if len(hit) == 1 else None


def tool30(q: str, ch: list):
    """(tool name, index of the one agreeing choice) or None"""
    for got in (tool30_elements(q, list(map(str, ch)), TABLE30), nucleons30(q, ch),
                convert30(q, ch, UNITS30)):
        if got and got[1]:
            return got[0], got[1][0]
    return None



TABLE30 = elements30(GOT30["PeriodicTableJSON.json"])
UNITS30 = units30(GOT30["default_en.txt"])
print(f"  tables: {len(TABLE30)} elements; {len(UNITS30)} units read into exact factors "
      f"(a pound is {float(UNITS30['pound'][0])} g)")
for q, ch, want in (("How many ounces are in a pound?", ["4", "12", "16", "32"], 2),
                    ("How many feet are in a mile?", ["1760", "3280", "5280", "5000"], 2),
                    ("The electron configuration 1s2 2s2 2p6 3s2 3p6 4s2 signifies the ground state of the element",
                     ["V", "Ti", "Co", "Ca"], 3)):
    got = tool30(q, ch)
    if not got or got[1] != want:
        raise RuntimeError(f"self-check failed: {q[:50]} -> {got}")
print("  self-checks: 3 of 3 (ounces in a pound, feet in a mile, an electron configuration)")



# ---------------- scored on the practice sets ----------------
RES30, WRONG30, TALLY30, EXACT30 = {}, [], {}, {}
print(f"\n  {'':<26}{'answered':>10}{'agree':>7}{'with 30':>9}{'without':>9}")
for n, qs in SETS12.items():
    rows, prev, ans, agree, ex = [], [], 0, 0, [0, 0]
    for (q, ch, g), (x, _, _) in zip(qs, READ12[n]):
        earlier = solve19(q, list(ch)) or tool23(q, list(ch)) or tool24(q, list(ch)) or tool25(q, list(ch)) or \
            tool26(q, list(ch)) or tool27(q, list(ch)) or tool28(q, list(ch)) or tool29(q, list(ch))
        base = float(earlier[1] == g) if earlier else credit12(scores12(x), g)
        got = None if earlier else tool30(q, list(ch))
        if got:
            ans += 1
            agree += got[1] == g
            TALLY30.setdefault(got[0], [0, 0])
            TALLY30[got[0]][0] += 1
            TALLY30[got[0]][1] += got[1] == g
            if got[1] != g:
                WRONG30.append((n, q[:110], ch[got[1]], ch[g]))
        final = earlier or got
        if final:
            ex[0] += 1
            ex[1] += final[1] == g
        rows.append(float(got[1] == g) if got else base)
        prev.append(base)
    RES30[n] = dict(answered=ans, agree=agree, whole=100 * np.mean(rows), was=100 * np.mean(prev))
    EXACT30[n] = (ex[0], ex[1], len(qs))
    print(f"  {n:<26}{ans:>10}{agree:>7}{RES30[n]['whole']:>9.2f}{RES30[n]['was']:>9.2f}")
print("\n  by tool (all sets: answered, agree with the key)")
for name, (k, a) in TALLY30.items():
    print(f"    {name:<10}{k:>5}{a:>6}")
print("\n  EXACT BASE (Cells 19 and 23-30 together: answered, agree, share of the set)")
for n, (a, r, t) in EXACT30.items():
    print(f"    {n:<26}{a:>5}{r:>6}{100 * a / t:>8.1f}%")
print("\n  DISAGREEMENTS with the answer key (all sets)")
for n, q, said, key in WRONG30:
    print(f"    [{n}] {q}\n      tool: {said}; key: {key}")

print("\n  PREDICTIONS vs RESULTS")
LEDGER.result("B5-TI1", RES30["ARC train"]["answered"])
LEDGER.result("B5-TI2", RES30["ARC train"]["answered"] - RES30["ARC train"]["agree"])
LEDGER.result("B5-TI3", RES30["MMLU validation"]["answered"])
LEDGER.result("B5-TI4", RES30["MMLU validation"]["answered"] - RES30["MMLU validation"]["agree"])
LEDGER.result("B5-TI5", RES30["ARC-Easy validation"]["answered"] + RES30["ARC-Challenge validation"]["answered"])
LEDGER.result("B5-T45", round(time.time() - T30))
if "sync" in globals():
    sync("Cell 30")
print(f"\n  ({time.time() - T30:.0f}s)")
print("  LAST LINE OF MC AI BRAIN 5 CELL 30")


MC AI BRAIN 5 - CELL 30: TABLES - THE PERIODIC TABLE AND UNITS
  tables: 119 elements; 614 units read into exact factors (a pound is 453.59237 g)
  self-checks: 3 of 3 (ounces in a pound, feet in a mile, an electron configuration)

                              answered  agree  with 30  without
  ARC train                         10     10    43.17    43.05
  ARC-Easy validation                0      0    51.05    51.05
  ARC-Challenge validation           0      0    30.57    30.57
  OpenBookQA validation              0      0    24.82    24.82
  MMLU validation                    4      4    35.09    34.97
  MMLU dev                           0      0    32.43    32.43
  MMLU-Pro validation                0      0    18.53    18.53

  by tool (all sets: answered, agree with the key)
    SYMBOL        2     2
    NUCLEONS      4     4
    CATEGORY      2     2
    ION           1     1
    PHASE         1     1
    GROUP         1     1
    SHELL         1     1
    CONFIG        1   